# $J/\psi+D^*$ — nominal Run-2 2D template fit + data-driven cross-check (HIST)

This is the **single nominal analysis notebook** for the convener-facing Run-2 workflow. The central SPS/DPS templates are **binned histograms (HIST)**; there is no alternative template-estimator branch in this notebook.

The notebook runs the full chain from the aligned Run-2 data/sWeights and the SPS/DPS MC samples through the efficiency response, the 2D profile likelihood, closure and goodness-of-fit diagnostics, the mixed-event data-driven cross-check, and the final method comparison.

Presentation diagnostics are intentionally drawn **inline** and also saved as PNG/PDF. The recommended narrative is:

1. data before/after signal sPlot weighting;
2. MC before/after efficiency correction;
3. pre-fit data vs pure SPS/DPS template shapes;
4. nominal post-fit template result and fit-quality diagnostics;
5. construction of the mixed-event data-driven method;
6. data-driven result;
7. final template-fit vs data-driven comparison.

All physics/configuration switches remain explicit in Section 1. The nominal defaults use one free parameter, the full folded $|\Delta\phi|\leq\pi$ range, the legacy `6x6` key (whose explicit edge arrays define the actual grid), automatic DPS-enriched-region optimization, and the post-fit purity scale in the data-driven cross-check.

In [ ]:
from pathlib import Path
import os
import re
import math
import json
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.legend import Legend
from matplotlib.text import Text

try:
    import uproot
except Exception as exc:
    raise RuntimeError("Install uproot before running this notebook.") from exc

try:
    from scipy.stats import beta as beta_distribution, chi2 as chi2_distribution, spearmanr
    from scipy.optimize import minimize_scalar
except Exception as exc:
    raise RuntimeError("Install scipy before running this notebook.") from exc

try:
    from iminuit import Minuit
except Exception as exc:
    raise RuntimeError(
        "This notebook uses iminuit for the profiled likelihood. Install iminuit first."
    ) from exc

try:
    import mplhep as hep
    HAS_MPLHEP = True
except Exception:
    HAS_MPLHEP = False

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def configure_plot_style():
    if HAS_MPLHEP:
        plt.style.use(hep.style.CMS)
    plt.rcParams.update({
        "figure.dpi": 105,
        "savefig.dpi": 220,
        "axes.linewidth": 1.2,
        "axes.labelsize": AXIS_LABEL_FONT_SIZE,
        "axes.titlesize": PANEL_TITLE_FONT_SIZE,
        "xtick.labelsize": TICK_FONT_SIZE,
        "ytick.labelsize": TICK_FONT_SIZE,
        "legend.fontsize": LEGEND_FONT_SIZE,
        "figure.titlesize": FIGURE_TITLE_FONT_SIZE,
        "legend.frameon": True,
        "legend.framealpha": 0.92,
        "legend.facecolor": "white",
        "legend.edgecolor": "0.75",
        "mathtext.fontset": "stixsans",
        "axes.formatter.use_mathtext": True,
        "axes.grid": False,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
    })


# ================================================================
# Centralized presentation labels and layout QA
# ================================================================
XLABEL_DPHI = r"$|\Delta\phi(J/\psi,D^*)|$"
XLABEL_DY = r"$|\Delta y(J/\psi,D^*)|$"
YLABEL_NORMALIZED = "Normalized yield / bin"
YLABEL_DPHI_XSEC = (
    r"$\mathrm{d}\sigma_{\mathrm{fid}}/"
    r"\mathrm{d}|\Delta\phi|\;[\mathrm{pb/rad}]$"
)
YLABEL_DY_XSEC = (
    r"$\mathrm{d}\sigma_{\mathrm{fid}}/"
    r"\mathrm{d}|\Delta y|\;[\mathrm{pb}]$"
)
LABEL_FDPS = r"$f_{\mathrm{DPS}}$"
LABEL_FDPS_DD = r"$f_{\mathrm{DPS}}^{\mathrm{DD}}$"
LABEL_PDPS = r"$P_{\mathrm{DPS}}$"
YLABEL_DATA_PREDICTION = r"$\mathrm{Data}/\mathrm{MC}$"
CBAR_NORMALIZED_2D = "Normalized probability / 2D bin"

CMS_FONT_SIZE = 20.0
CMS_EXTRA_FONT_SIZE = 15.0
LUMI_FONT_SIZE = 14.0
AXIS_LABEL_FONT_SIZE = 20.0
TICK_FONT_SIZE = 11.0
LEGEND_FONT_SIZE = 11.0
PANEL_TITLE_FONT_SIZE = 14.0
FIGURE_TITLE_FONT_SIZE = 16.0
CMS_INTERNAL_PAD_PT = 6.0
CMS_GAP_MIN_PT = 4.0
CMS_GAP_MAX_PT = 10.0
LAYOUT_CANVAS_SAFETY_PX = 2.0
RASTER_NEAR_WHITE_THRESHOLD = 248
RASTER_MARGIN_MAX_FRACTION = 0.18
REGULAR_GRID_MAX_GAP_RATIO = 0.50
COLORBAR_GRID_MAX_GAP_RATIO = 0.68
REGULAR_GRID_MIN_OCCUPANCY = 0.50

LAYOUT_AUDIT_STRICT = bool(int(os.environ.get(
    "JPSIDSTAR_LAYOUT_AUDIT_STRICT", "1"
)))

PRESENTATION_LAYOUT_AUDIT_PREFIXES = (
    "data_splot_before_after_",
    "prefit_component_",
    "mc_efficiency_before_after_",
    "prefit_data_",
    "prefit_SPS_DPS_",
    "profile_",
    "gof_toys_",
    "templatefit_closure_",
    "prediction_uncertainty_breakdown_",
    "templatefit_postfit_",
    "dps_mc_vs_mixed_",
    "data_driven_statistical_",
    "data_driven_region_systematic_",
    "data_driven_enriched_region_",
    "data_driven_intermediate_",
    "final_fdps_profile_",
)
FIGURE_LAYOUT_AUDIT = []


def estimator_stem(stem):
    """Nominal HIST notebook: preserve the requested figure stem unchanged."""
    return str(stem)


def _is_presentation_stem(stem):
    return str(stem).startswith(PRESENTATION_LAYOUT_AUDIT_PREFIXES)


def _points_to_pixels(fig, points):
    return float(points) * float(fig.dpi) / 72.0


def _main_plot_axes(fig):
    axes = []
    for ax in fig.axes:
        gid = str(ax.get_gid() or "")
        is_colorbar = (
            gid == "layout:colorbar"
            or ax.get_label() == "<colorbar>"
            or getattr(ax, "_colorbar", None) is not None
        )
        if ax.get_visible() and not is_colorbar and gid != "layout:ignore":
            axes.append(ax)
    return axes


def _bbox_union(boxes):
    boxes = list(boxes)
    if not boxes:
        return None
    return boxes[0].union(boxes)


def _axes_share_x(first, second):
    try:
        return bool(first.get_shared_x_axes().joined(first, second))
    except Exception:
        return False


def _header_axes(fig, candidates=None):
    """Return independent main panels; compact shared-x ratio axes are excluded."""
    main_axes = _main_plot_axes(fig)
    candidate_ids = None if candidates is None else {id(ax) for ax in candidates}
    ratio_ids = {
        id(ax) for ax in main_axes
        if str(ax.get_gid() or "") == "layout:ratio_axis"
    }
    positions = {id(ax): ax.get_position() for ax in main_axes}
    for first_index, first in enumerate(main_axes):
        first_box = positions[id(first)]
        for second in main_axes[first_index + 1:]:
            second_box = positions[id(second)]
            same_x_span = (
                abs(first_box.x0 - second_box.x0) < 1.0e-4
                and abs(first_box.x1 - second_box.x1) < 1.0e-4
            )
            height_ratio = max(first_box.height, second_box.height) / max(
                min(first_box.height, second_box.height), 1.0e-12,
            )
            if same_x_span and height_ratio > 1.8 and _axes_share_x(first, second):
                shorter = first if first_box.height < second_box.height else second
                ratio_ids.add(id(shorter))
    return [
        ax for ax in main_axes
        if id(ax) not in ratio_ids
        and (candidate_ids is None or id(ax) in candidate_ids)
    ]


def _axis_owner(fig, ax):
    for index, candidate in enumerate(_main_plot_axes(fig)):
        if candidate is ax:
            return f"axis{index}"
    raise ValueError("CMS header owner is not a main plotting axis")


def _remove_central_header(fig, owners=None):
    owners = None if owners is None else {str(owner) for owner in owners}
    for artist in list(fig.texts):
        gid = str(artist.get_gid() or "")
        match = re.fullmatch(
            r"layout:cms_header:(axis[0-9]+):(cms|internal|lumi)", gid,
        )
        if gid.startswith("layout:cms_header:") and (
            owners is None or (match is not None and match.group(1) in owners)
        ):
            artist.remove()


def _draw_cms_header_for_axis(fig, ax, owner, lumi_fb, label="Internal"):
    """Draw one complete, renderer-positioned header for one owning main axis."""
    if ax.figure is not fig:
        raise ValueError("Axis does not belong to the supplied figure")
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axis_box = ax.get_window_extent(renderer)
    anchor_top_px = axis_box.y1
    title = ax.title
    if title.get_visible() and title.get_text().strip():
        anchor_top_px = max(anchor_top_px, title.get_window_extent(renderer).y1)
    baseline_y_px = anchor_top_px + _points_to_pixels(fig, 2.5)
    inv = fig.transFigure.inverted()
    left_fig, baseline_fig = inv.transform((axis_box.x0, baseline_y_px))
    right_fig = inv.transform((axis_box.x1, baseline_y_px))[0]

    cms = fig.text(
        left_fig, baseline_fig, "CMS", transform=fig.transFigure,
        ha="left", va="bottom", fontsize=CMS_FONT_SIZE,
        fontweight="bold", clip_on=False,
    )
    cms.set_gid(f"layout:cms_header:{owner}:cms")
    fig.canvas.draw()
    cms_box = cms.get_window_extent(fig.canvas.get_renderer())
    internal_x_px = cms_box.x1 + _points_to_pixels(fig, CMS_INTERNAL_PAD_PT)
    internal_x_fig = inv.transform((internal_x_px, baseline_y_px))[0]
    internal = fig.text(
        internal_x_fig, baseline_fig, str(label), transform=fig.transFigure,
        ha="left", va="bottom", fontsize=CMS_EXTRA_FONT_SIZE,
        style="italic", clip_on=False,
    )
    internal.set_gid(f"layout:cms_header:{owner}:internal")

    value = np.asarray(lumi_fb, dtype=float)
    if value.size != 1:
        raise ValueError(f"lumi_fb must be scalar, got shape {value.shape}")
    lumi = fig.text(
        right_fig, baseline_fig,
        rf"{float(value.reshape(-1)[0]):.1f} fb$^{{-1}}$ (13 TeV)",
        transform=fig.transFigure, ha="right", va="bottom",
        fontsize=LUMI_FONT_SIZE, clip_on=False,
    )
    lumi.set_gid(f"layout:cms_header:{owner}:lumi")
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    internal_box = internal.get_window_extent(renderer)
    lumi_box = lumi.get_window_extent(renderer)

    header_lines = 1
    if internal_box.x1 + _points_to_pixels(fig, 4.0) > lumi_box.x0:
        header_lines = 2
        lumi_y_px = max(cms_box.y1, internal_box.y1) + _points_to_pixels(fig, 2.5)
        lumi.set_y(inv.transform((axis_box.x1, lumi_y_px))[1])
        fig.canvas.draw()
    return {
        "artists": {"cms": cms, "internal": internal, "lumi": lumi},
        "lines": int(header_lines),
    }


def _draw_cms_headers(fig, lumi_fb, label="Internal", axes=None):
    """Draw one complete CMS/Internal/luminosity header per independent panel."""
    candidates = None if axes is None else list(np.ravel(axes))
    if candidates is not None and any(ax.figure is not fig for ax in candidates):
        raise ValueError("All axes must belong to the supplied figure")
    selected = _header_axes(fig, candidates=candidates)
    if not selected:
        raise RuntimeError("CMS header requires at least one eligible main plotting axis")
    _remove_central_header(fig)
    fig._layout_header_axis_map = {}
    fig._layout_header_lines = {}
    rendered = {}
    for ax in selected:
        owner = _axis_owner(fig, ax)
        result = _draw_cms_header_for_axis(
            fig, ax, owner, lumi_fb=lumi_fb, label=label,
        )
        fig._layout_header_axis_map[owner] = ax
        fig._layout_header_lines[owner] = result["lines"]
        rendered[owner] = result["artists"]
    fig._layout_header_mode = "per_panel"
    return rendered


def cms_label_axis(fig, ax, lumi_fb, label="Internal"):
    """Label only the supplied independent main axis."""
    if ax.figure is not fig:
        raise ValueError("Axis does not belong to the supplied figure")
    if ax not in _header_axes(fig, candidates=[ax]):
        return {}
    owner = _axis_owner(fig, ax)
    _remove_central_header(fig, owners=[owner])
    result = _draw_cms_header_for_axis(
        fig, ax, owner, lumi_fb=lumi_fb, label=label,
    )
    if not hasattr(fig, "_layout_header_axis_map"):
        fig._layout_header_axis_map = {}
    if not hasattr(fig, "_layout_header_lines"):
        fig._layout_header_lines = {}
    fig._layout_header_axis_map[owner] = ax
    fig._layout_header_lines[owner] = result["lines"]
    fig._layout_header_mode = "per_panel"
    return result["artists"]


def cms_label_figure(fig, axes, lumi_fb, label="Internal"):
    """Label each eligible independent main axis in the supplied collection."""
    return _draw_cms_headers(
        fig, lumi_fb=lumi_fb, label=label, axes=axes,
    )


def _outside_view_tick(fig, artist):
    if not isinstance(artist, Text):
        return False
    for ax in fig.axes:
        if any(artist is label for label in ax.get_xticklabels() + ax.get_xticklabels(minor=True)):
            x = float(artist.get_position()[0])
            xlo, xhi = sorted(map(float, ax.get_xlim()))
            return x < xlo or x > xhi
        if any(artist is label for label in ax.get_yticklabels() + ax.get_yticklabels(minor=True)):
            y = float(artist.get_position()[1])
            ylo, yhi = sorted(map(float, ax.get_ylim()))
            return y < ylo or y > yhi
    return False


def _intersection_area(first, second):
    width = max(0.0, min(first.x1, second.x1) - max(first.x0, second.x0))
    height = max(0.0, min(first.y1, second.y1) - max(first.y0, second.y0))
    return width * height


def _semantic_layout_audit(fig, stem):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    canvas = fig.bbox
    clipping = []
    semantic = []
    artists = fig.findobj(match=lambda artist: isinstance(artist, (Text, Legend)))
    for artist in artists:
        if not artist.get_visible():
            continue
        if isinstance(artist, Text) and not artist.get_text().strip():
            continue
        if _outside_view_tick(fig, artist):
            continue
        bbox = artist.get_window_extent(renderer=renderer)
        bounds = np.asarray([bbox.x0, bbox.y0, bbox.x1, bbox.y1], dtype=float)
        if not np.all(np.isfinite(bounds)):
            clipping.append(f"{type(artist).__name__}: non-finite bounds")
            continue
        safety = LAYOUT_CANVAS_SAFETY_PX
        if (
            bbox.x0 < canvas.x0 + safety
            or bbox.y0 < canvas.y0 + safety
            or bbox.x1 > canvas.x1 - safety
            or bbox.y1 > canvas.y1 - safety
        ):
            label = artist.get_text() if isinstance(artist, Text) else "legend"
            clipping.append(
                f"{type(artist).__name__} {label!r} bbox="
                f"{tuple(np.round(bounds, 1))} canvas="
                f"{tuple(np.round([canvas.x0, canvas.y0, canvas.x1, canvas.y1], 1))}"
            )

    for ax in fig.axes:
        if not ax.get_visible():
            continue
        x_label = ax.xaxis.label
        y_label = ax.yaxis.label
        visible_x_ticks = [tick for tick in ax.get_xticklabels() if tick.get_visible() and tick.get_text()]
        visible_y_ticks = [tick for tick in ax.get_yticklabels() if tick.get_visible() and tick.get_text()]
        if x_label.get_visible() and x_label.get_text().strip():
            label_box = x_label.get_window_extent(renderer)
            if any(_intersection_area(label_box, tick.get_window_extent(renderer)) > 1.0 for tick in visible_x_ticks):
                semantic.append(f"axis x-label/tick-label overlap on axes {ax.get_label()!r}")
        if y_label.get_visible() and y_label.get_text().strip():
            label_box = y_label.get_window_extent(renderer)
            if any(_intersection_area(label_box, tick.get_window_extent(renderer)) > 1.0 for tick in visible_y_ticks):
                semantic.append(f"axis y-label/tick-label overlap on axes {ax.get_label()!r}")

        legends = [child for child in ax.get_children() if isinstance(child, Legend) and child.get_visible()]
        annotations = [
            text for text in ax.texts
            if text.get_visible() and text.get_text().strip()
            and not str(text.get_gid() or "").startswith("layout:cms_header:")
        ]
        for legend in legends:
            legend_box = legend.get_window_extent(renderer)
            for annotation in annotations:
                annotation_box = annotation.get_window_extent(renderer)
                overlap = _intersection_area(legend_box, annotation_box)
                smaller = max(1.0, min(
                    legend_box.width * legend_box.height,
                    annotation_box.width * annotation_box.height,
                ))
                if overlap / smaller > 0.10:
                    semantic.append(
                        "legend/annotation overlap: "
                        + repr(annotation.get_text().replace("\n", " / ")[:80])
                    )

    header_artists = [
        (str(text.get_gid()), text)
        for text in fig.texts
        if str(text.get_gid() or "").startswith("layout:cms_header:")
    ]
    suptitle = getattr(fig, "_suptitle", None)
    if suptitle is not None and suptitle.get_visible() and suptitle.get_text().strip():
        suptitle_box = suptitle.get_window_extent(renderer)
        for gid, text in header_artists:
            if _intersection_area(suptitle_box, text.get_window_extent(renderer)) > 1.0:
                semantic.append(f"suptitle/CMS-header overlap with {gid}")

    colorbar_issues = []
    for ax in fig.axes:
        is_colorbar = ax.get_label() == "<colorbar>" or getattr(ax, "_colorbar", None) is not None
        if is_colorbar:
            label = ax.yaxis.label if ax.yaxis.label.get_text().strip() else ax.xaxis.label
            if label.get_text().strip():
                box = label.get_window_extent(renderer)
                if box.x0 < canvas.x0 + 2 or box.y0 < canvas.y0 + 2 or box.x1 > canvas.x1 - 2 or box.y1 > canvas.y1 - 2:
                    colorbar_issues.append("colorbar label clipping: " + repr(label.get_text()))

    return {
        "bbox_status": "PASS" if not clipping else "FAIL",
        "semantic_status": "PASS" if not semantic and not colorbar_issues else "FAIL",
        "bbox_issues": clipping,
        "semantic_issues": semantic + colorbar_issues,
    }


def _cms_spacing_audit(fig):
    """Audit exact per-panel CMS ownership and rendered geometry."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    canvas = fig.bbox
    tagged = {}
    issues = []
    gid_pattern = re.compile(
        r"^layout:cms_header:(axis[0-9]+):(cms|internal|lumi)$"
    )
    for text in fig.texts:
        gid = str(text.get_gid() or "")
        if not gid.startswith("layout:cms_header:"):
            continue
        match = gid_pattern.fullmatch(gid)
        if match is None:
            issues.append(f"malformed CMS header GID {gid!r}")
            continue
        owner, part = match.groups()
        tagged.setdefault(owner, {}).setdefault(part, []).append(text)

    expected_axes = _header_axes(fig)
    expected = {_axis_owner(fig, ax): ax for ax in expected_axes}
    unexpected = sorted(set(tagged).difference(expected))
    missing_owners = sorted(set(expected).difference(tagged))
    if unexpected:
        issues.append("unexpected CMS header owners: " + ", ".join(unexpected))
    if missing_owners:
        issues.append("missing CMS header owners: " + ", ".join(missing_owners))

    required_parts = ("cms", "internal", "lumi")
    panel_status = []
    gaps = []
    main_axes = _main_plot_axes(fig)
    suptitle = getattr(fig, "_suptitle", None)
    suptitle_box = None
    if suptitle is not None and suptitle.get_visible() and suptitle.get_text().strip():
        suptitle_box = suptitle.get_window_extent(renderer)

    for owner, ax in expected.items():
        panel_issues = []
        owner_artists = tagged.get(owner, {})
        for part in required_parts:
            count = len(owner_artists.get(part, []))
            if count != 1:
                panel_issues.append(
                    f"expected exactly one {part} artist, found {count}"
                )
        gap_pt = np.nan
        cms_left_delta_px = np.nan
        lumi_right_delta_px = np.nan
        upper_panel_intersection_max_px2 = 0.0
        if not panel_issues:
            artists = {
                part: owner_artists[part][0] for part in required_parts
            }
            boxes = {
                part: artist.get_window_extent(renderer)
                for part, artist in artists.items()
            }
            gap_pt = (
                boxes["internal"].x0 - boxes["cms"].x1
            ) * 72.0 / float(fig.dpi)
            gaps.append(float(gap_pt))
            if boxes["internal"].x0 <= boxes["cms"].x1:
                panel_issues.append("CMS/Internal overlap")
            if not (CMS_GAP_MIN_PT <= gap_pt <= CMS_GAP_MAX_PT):
                panel_issues.append(
                    f"CMS/Internal rendered gap {gap_pt:.2f} pt outside "
                    f"[{CMS_GAP_MIN_PT:.1f}, {CMS_GAP_MAX_PT:.1f}] pt"
                )
            for left_name in ("cms", "internal"):
                if _intersection_area(boxes[left_name], boxes["lumi"]) > 1.0:
                    panel_issues.append(f"{left_name}/luminosity overlap")

            axis_box = ax.get_window_extent(renderer)
            cms_left_delta_px = float(boxes["cms"].x0 - axis_box.x0)
            lumi_right_delta_px = float(boxes["lumi"].x1 - axis_box.x1)
            if abs(cms_left_delta_px) > 1.0:
                panel_issues.append(
                    f"CMS left edge misaligned by {cms_left_delta_px:.2f} px"
                )
            if abs(lumi_right_delta_px) > 1.0:
                panel_issues.append(
                    f"luminosity right edge misaligned by {lumi_right_delta_px:.2f} px"
                )
            for part, box in boxes.items():
                if box.x0 < axis_box.x0 - 1.0 or box.x1 > axis_box.x1 + 1.0:
                    panel_issues.append(f"{part} leaves owning panel horizontal span")
                if (
                    box.x0 < canvas.x0 + LAYOUT_CANVAS_SAFETY_PX
                    or box.y0 < canvas.y0 + LAYOUT_CANVAS_SAFETY_PX
                    or box.x1 > canvas.x1 - LAYOUT_CANVAS_SAFETY_PX
                    or box.y1 > canvas.y1 - LAYOUT_CANVAS_SAFETY_PX
                ):
                    panel_issues.append(f"{part} clips the figure canvas")

            title_top = axis_box.y1
            title = ax.title
            if title.get_visible() and title.get_text().strip():
                title_box = title.get_window_extent(renderer)
                title_top = max(title_top, title_box.y1)
                for part, box in boxes.items():
                    if _intersection_area(title_box, box) > 1.0:
                        panel_issues.append(f"{part}/own-title overlap")
            if min(box.y0 for box in boxes.values()) < title_top - 1.0:
                panel_issues.append("header does not clear owning axis/title")

            header_union = _bbox_union(boxes.values())
            owner_pos = ax.get_position()
            for other_index, other in enumerate(main_axes):
                if other is ax:
                    continue
                other_pos = other.get_position()
                horizontal_overlap = (
                    min(owner_pos.x1, other_pos.x1)
                    - max(owner_pos.x0, other_pos.x0) > 1.0e-4
                )
                if not (
                    horizontal_overlap
                    and other_pos.y0 >= owner_pos.y1 - 1.0e-4
                ):
                    continue
                upper_axis_box = other.get_window_extent(renderer)
                for part, header_box in boxes.items():
                    upper_area = float(_intersection_area(header_box, upper_axis_box))
                    upper_panel_intersection_max_px2 = max(
                        upper_panel_intersection_max_px2, upper_area,
                    )
                    if upper_area > 1.0:
                        panel_issues.append(
                            f"{part} intrudes into upper-panel plotting area "
                            f"on axes {other_index}"
                        )
                upper_artists = [
                    other.title,
                    other.xaxis.label,
                    other.yaxis.label,
                    other.xaxis.get_offset_text(),
                    other.yaxis.get_offset_text(),
                    *other.get_xticklabels(),
                    *other.get_yticklabels(),
                    *other.texts,
                    *[
                        child for child in other.get_children()
                        if isinstance(child, Legend)
                    ],
                ]
                for part, header_box in boxes.items():
                    for upper_artist in upper_artists:
                        if not upper_artist.get_visible():
                            continue
                        if (
                            isinstance(upper_artist, Text)
                            and not upper_artist.get_text().strip()
                        ):
                            continue
                        upper_box = upper_artist.get_window_extent(renderer)
                        upper_area = float(_intersection_area(header_box, upper_box))
                        upper_panel_intersection_max_px2 = max(
                            upper_panel_intersection_max_px2, upper_area,
                        )
                        if upper_area > 1.0:
                            panel_issues.append(
                                f"{part} overlaps an upper-panel decoration "
                                f"on axes {other_index}"
                            )
                            break
            if suptitle_box is not None and _intersection_area(
                header_union, suptitle_box,
            ) > 1.0:
                panel_issues.append("header/suptitle overlap")

        axis_index = int(owner.replace("axis", ""))
        panel_status.append({
            "owner": owner,
            "axis_index": axis_index,
            "status": "PASS" if not panel_issues else "FAIL",
            "cms_internal_gap_pt": (
                None if not np.isfinite(gap_pt) else float(gap_pt)
            ),
            "header_lines": int(
                getattr(fig, "_layout_header_lines", {}).get(owner, 1)
            ),
            "cms_left_delta_px": (
                None if not np.isfinite(cms_left_delta_px)
                else float(cms_left_delta_px)
            ),
            "lumi_right_delta_px": (
                None if not np.isfinite(lumi_right_delta_px)
                else float(lumi_right_delta_px)
            ),
            "upper_panel_intersection_max_px2": float(
                upper_panel_intersection_max_px2
            ),
            "issues": panel_issues,
        })
        issues.extend(f"{owner}: {item}" for item in panel_issues)

    return {
        "cms_spacing_status": "PASS" if not issues else "FAIL",
        "cms_gap_pt": float(max(gaps)) if gaps else np.nan,
        "cms_gap_min_pt": float(min(gaps)) if gaps else np.nan,
        "cms_gap_max_pt": float(max(gaps)) if gaps else np.nan,
        "cms_issues": issues,
        "cms_header_mode": str(getattr(fig, "_layout_header_mode", "unknown")),
        "cms_header_lines": int(max(
            getattr(fig, "_layout_header_lines", {"none": 1}).values(),
            default=1,
        )),
        "cms_panel_count": int(len(tagged)),
        "cms_expected_panel_count": int(len(expected)),
        "cms_panel_status": panel_status,
    }


def _unique_intervals(intervals, tolerance=1.0e-4):
    unique = []
    for interval in sorted(intervals):
        if not any(abs(interval[0] - old[0]) < tolerance and abs(interval[1] - old[1]) < tolerance for old in unique):
            unique.append(interval)
    return unique


def _subplot_spacing_audit(fig):
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    axes = _main_plot_axes(fig)
    colorbar_count = len(fig.axes) - len(axes)
    positions = [ax.get_position() for ax in axes]
    if not positions:
        return {
            "panel_spacing_status": "FAIL", "panel_issues": ["no main plotting axes"],
            "horizontal_gap_ratio_max": np.nan, "vertical_gap_ratio_max": np.nan,
            "axes_area_occupancy": np.nan,
        }
    union = _bbox_union(positions)
    area_sum = float(sum(box.width * box.height for box in positions))
    union_area = max(float(union.width * union.height), 1.0e-12)
    occupancy = area_sum / union_area
    x_intervals = _unique_intervals([(box.x0, box.x1) for box in positions])
    y_intervals = _unique_intervals([(box.y0, box.y1) for box in positions])
    h_ratios = []
    for left, right in zip(x_intervals[:-1], x_intervals[1:]):
        gap = right[0] - left[1]
        mean_width = 0.5 * ((left[1] - left[0]) + (right[1] - right[0]))
        if gap >= -1.0e-4:
            h_ratios.append(gap / max(mean_width, 1.0e-12))
    v_ratios = []
    for lower, upper in zip(y_intervals[:-1], y_intervals[1:]):
        gap = upper[0] - lower[1]
        mean_height = 0.5 * ((lower[1] - lower[0]) + (upper[1] - upper[0]))
        if gap >= -1.0e-4:
            v_ratios.append(gap / max(mean_height, 1.0e-12))
    max_h = max(h_ratios, default=0.0)
    max_v = max(v_ratios, default=0.0)
    height_values = [box.height for box in positions]
    is_ratio_layout = len(axes) == 2 and max(height_values) / max(min(height_values), 1.0e-12) > 1.8
    issues = []
    if is_ratio_layout:
        left_edges = [box.x0 for box in positions]
        right_edges = [box.x1 for box in positions]
        if max(left_edges) - min(left_edges) > 1.0e-4 or max(right_edges) - min(right_edges) > 1.0e-4:
            issues.append(
                "main/ratio horizontal misalignment: "
                f"left spread={max(left_edges)-min(left_edges):.6f}, "
                f"right spread={max(right_edges)-min(right_edges):.6f}"
            )
        main_index = int(np.argmax(height_values))
        visible_main_xlabels = [
            tick for tick in axes[main_index].get_xticklabels()
            if tick.get_visible() and tick.get_text().strip()
        ]
        if visible_main_xlabels:
            issues.append("main panel repeats x tick labels owned by the ratio panel")
    if len(axes) > 1:
        gap_limit = COLORBAR_GRID_MAX_GAP_RATIO if colorbar_count else REGULAR_GRID_MAX_GAP_RATIO
        if max_h > gap_limit:
            issues.append(f"large horizontal panel gap ratio {max_h:.3f} > {gap_limit:.3f}")
        if max_v > REGULAR_GRID_MAX_GAP_RATIO and not is_ratio_layout:
            issues.append(f"large vertical panel gap ratio {max_v:.3f} > {REGULAR_GRID_MAX_GAP_RATIO:.3f}")
        if occupancy < REGULAR_GRID_MIN_OCCUPANCY and not is_ratio_layout:
            issues.append(
                f"low axes-area occupancy {occupancy:.3f} < {REGULAR_GRID_MIN_OCCUPANCY:.3f}"
            )

        tight_boxes = [ax.get_tightbbox(renderer) for ax in axes]
        for first_index in range(len(axes)):
            for second_index in range(first_index + 1, len(axes)):
                first_pos = positions[first_index]
                second_pos = positions[second_index]
                separated = (
                    first_pos.x1 <= second_pos.x0 + 1.0e-4
                    or second_pos.x1 <= first_pos.x0 + 1.0e-4
                    or first_pos.y1 <= second_pos.y0 + 1.0e-4
                    or second_pos.y1 <= first_pos.y0 + 1.0e-4
                )
                if (
                    separated
                    and not is_ratio_layout
                    and _intersection_area(tight_boxes[first_index], tight_boxes[second_index]) > 4.0
                ):
                    issues.append(
                        f"subplot decoration collision between axes {first_index} and {second_index}"
                    )

    outer = {
        "left": float(union.x0),
        "right": float(1.0 - union.x1),
        "bottom": float(union.y0),
        "top": float(1.0 - union.y1),
    }
    if max(outer.values()) > 0.30:
        issues.append(
            "excessive axes outer margin "
            + ", ".join(f"{key}={value:.3f}" for key, value in outer.items())
        )
    return {
        "panel_spacing_status": "PASS" if not issues else "FAIL",
        "panel_issues": issues,
        "horizontal_gap_ratio_max": float(max_h),
        "vertical_gap_ratio_max": float(max_v),
        "axes_area_occupancy": float(occupancy),
        "main_axes_count": int(len(axes)),
        "colorbar_axes_count": int(colorbar_count),
        "ratio_layout": bool(is_ratio_layout),
        "axes_outer_left": outer["left"],
        "axes_outer_right": outer["right"],
        "axes_outer_top": outer["top"],
        "axes_outer_bottom": outer["bottom"],
    }


def _raster_whitespace_audit(path):
    from PIL import Image

    with Image.open(path) as image:
        rgb = np.asarray(image.convert("RGB"), dtype=np.uint8)
    content = np.any(rgb < RASTER_NEAR_WHITE_THRESHOLD, axis=2)
    issues = []
    if not np.any(content):
        margins = {name: 1.0 for name in ("left", "right", "top", "bottom")}
        issues.append("saved PNG contains no detected non-background content")
    else:
        ys, xs = np.nonzero(content)
        height, width = content.shape
        margins = {
            "left": float(xs.min() / width),
            "right": float((width - 1 - xs.max()) / width),
            "top": float(ys.min() / height),
            "bottom": float((height - 1 - ys.max()) / height),
        }
        for side, fraction in margins.items():
            if fraction > RASTER_MARGIN_MAX_FRACTION:
                issues.append(
                    f"large raster {side} whitespace fraction {fraction:.3f} > "
                    f"{RASTER_MARGIN_MAX_FRACTION:.3f}"
                )
    return {
        "raster_whitespace_status": "PASS" if not issues else "FAIL",
        "raster_issues": issues,
        **{f"raster_margin_{key}": value for key, value in margins.items()},
    }


def validate_figure_layout(fig, stem):
    """Collect semantic, per-panel CMS, subplot and explicit diagnostics."""
    semantic = _semantic_layout_audit(fig, stem)
    cms = _cms_spacing_audit(fig)
    panels = _subplot_spacing_audit(fig)
    explicit = list(getattr(fig, "_layout_explicit_overlap_audits", []))
    explicit_issues = []
    for check in explicit:
        area = float(check.get("intersection_area_px2", np.nan))
        if not np.isfinite(area) or area > 1.0e-9:
            explicit_issues.append(
                f"{check.get('name', 'explicit overlap')}: "
                f"intersection area {area:.6g} px^2"
            )
    record = {
        "stem": str(stem), **semantic, **cms, **panels,
        "explicit_overlap_status": "PASS" if not explicit_issues else "FAIL",
        "explicit_overlap_audits": explicit,
        "explicit_overlap_issues": explicit_issues,
    }
    record["issues"] = (
        semantic["bbox_issues"] + semantic["semantic_issues"]
        + cms["cms_issues"] + panels["panel_issues"] + explicit_issues
    )
    return record


def _source_cell_from_caller():
    import inspect

    frame = inspect.currentframe()
    try:
        caller = frame.f_back.f_back
        filename = str(caller.f_code.co_filename)
        match = re.search(r"cell-(\d+)$", filename)
        return int(match.group(1)) if match else filename
    finally:
        del frame


def _write_layout_audit(outdir):
    if not FIGURE_LAYOUT_AUDIT:
        return
    serializable = []
    for row in FIGURE_LAYOUT_AUDIT:
        flat = dict(row)
        flat["issues"] = " | ".join(row.get("issues", []))
        for key in (
            "bbox_issues", "semantic_issues", "cms_issues", "panel_issues",
            "raster_issues", "explicit_overlap_issues",
        ):
            if key in flat:
                flat[key] = " | ".join(flat[key])
        for key in ("cms_panel_status", "explicit_overlap_audits"):
            if key in flat:
                flat[key] = json.dumps(flat[key], sort_keys=True)
        serializable.append(flat)
    pd.DataFrame(serializable).to_csv(outdir / "figure_layout_audit.csv", index=False)
    with open(outdir / "figure_layout_audit.json", "w") as handle:
        json.dump(FIGURE_LAYOUT_AUDIT, handle, indent=2)


def save_and_show(fig, stem, outdir=None):
    outdir = Path(outdir or OUTDIR)
    outdir.mkdir(parents=True, exist_ok=True)
    tagged = estimator_stem(stem)
    presentation = _is_presentation_stem(tagged)
    record = None
    if presentation:
        _draw_cms_headers(fig, lumi_fb=LUMI_RUN2_FB_INV, label="Internal")
        fig.canvas.draw()
        record = validate_figure_layout(fig, tagged)
        record["source_cell"] = _source_cell_from_caller()
    paths = {}
    for extension in ("png", "pdf"):
        path = outdir / f"{tagged}.{extension}"
        fig.savefig(path, facecolor="white", dpi=180 if extension == "png" else None)
        paths[extension] = path
        print("[plot] wrote", path)
    if presentation:
        raster = _raster_whitespace_audit(paths["png"])
        record.update(raster)
        record["issues"] = record["issues"] + raster["raster_issues"]
        record["overall_status"] = "PASS" if not record["issues"] else "FAIL"
        FIGURE_LAYOUT_AUDIT.append(record)
        _write_layout_audit(outdir)
        if record["issues"]:
            diagnostic = "; ".join(record["issues"][:12])
            message = f"{tagged}: layout audit issues: {diagnostic}"
            if LAYOUT_AUDIT_STRICT:
                print("[layout audit] FAIL (strict):", message)
                plt.show()
                raise RuntimeError(message)
            warnings.warn(message)
            print("[layout audit] COLLECTED:", message)
        else:
            print(
                f"[layout audit] PASS: {tagged}; CMS gap="
                f"{record['cms_gap_pt']:.2f} pt; max gaps="
                f"({record['horizontal_gap_ratio_max']:.3f}, "
                f"{record['vertical_gap_ratio_max']:.3f}); occupancy="
                f"{record['axes_area_occupancy']:.3f}"
            )
    plt.show()
    plt.close(fig)
    return record


def create_presentation_contact_sheet(outdir, records, output_path):
    """Create a filename-labelled sheet from every audited presentation PNG."""
    from PIL import Image, ImageDraw, ImageFont, ImageOps
    import textwrap

    unique = []
    seen = set()
    for record in records:
        stem = str(record["stem"])
        if stem not in seen:
            seen.add(stem)
            unique.append(stem)
    if not unique:
        raise RuntimeError("No presentation figures were audited")
    columns = 4
    thumb_width, thumb_height = 480, 340
    label_height, gutter = 76, 18
    rows = int(math.ceil(len(unique) / columns))
    canvas = Image.new(
        "RGB",
        (
            columns * (thumb_width + gutter) + gutter,
            rows * (thumb_height + label_height + gutter) + gutter,
        ),
        "white",
    )
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 17)
    except OSError:
        font = ImageFont.load_default()
    for index, stem in enumerate(unique):
        path = Path(outdir) / f"{stem}.png"
        if not path.is_file():
            raise FileNotFoundError(path)
        with Image.open(path) as image:
            tile = ImageOps.contain(image.convert("RGB"), (thumb_width, thumb_height))
        column = index % columns
        row = index // columns
        x0 = gutter + column * (thumb_width + gutter)
        y0 = gutter + row * (thumb_height + label_height + gutter)
        label = "\n".join(textwrap.wrap(path.name, width=50, break_long_words=True))
        draw.multiline_text((x0, y0), label, fill="black", font=font, spacing=3)
        image_x = x0 + (thumb_width - tile.width) // 2
        image_y = y0 + label_height + (thumb_height - tile.height) // 2
        canvas.paste(tile, (image_x, image_y))
    output_path = Path(output_path)
    canvas.save(output_path)
    print("[layout audit] contact sheet:", output_path)
    return output_path


print("Layout audit mode:", "STRICT" if LAYOUT_AUDIT_STRICT else "COLLECTION")


def safe_divide(num, den, default=0.0):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    out = np.full(np.broadcast(num, den).shape, default, dtype=float)
    return np.divide(num, den, out=out,
                     where=np.isfinite(den) & (den != 0.0))


def weighted_effective_entries(weights):
    weights = np.asarray(weights, dtype=float)
    weights = weights[np.isfinite(weights)]
    if weights.size == 0:
        return 0.0
    s1 = float(np.sum(weights))
    s2 = float(np.sum(weights**2))
    return float(s1*s1/s2) if s2 > 0.0 else 0.0


def nearest_psd(matrix, relative_floor=1.0e-12):
    matrix = np.asarray(matrix, dtype=float)
    matrix = 0.5*(matrix + matrix.T)
    eigval, eigvec = np.linalg.eigh(matrix)
    scale = max(float(np.max(np.abs(eigval))), 1.0e-20)
    floor = relative_floor*scale
    eigval = np.maximum(eigval, floor)
    return eigvec @ np.diag(eigval) @ eigvec.T


def rms_covariance_about_nominal(samples, nominal):
    """Second-moment covariance around a nominal state; useful for approved variations."""
    arr = np.asarray(samples, dtype=float)
    nominal = np.asarray(nominal, dtype=float)
    if arr.ndim != 2 or len(arr) == 0:
        return np.zeros((nominal.size, nominal.size), dtype=float)
    delta = arr - nominal[None, :]
    return 0.5*((delta.T @ delta)/float(len(delta)) + ((delta.T @ delta)/float(len(delta))).T)


def sample_covariance_centered(samples, n_dim=None):
    """Ordinary sample covariance around the ensemble mean."""
    arr = np.asarray(samples, dtype=float)
    if arr.ndim != 2 or len(arr) < 2:
        n = int(n_dim or (arr.shape[1] if arr.ndim == 2 else 1))
        return np.zeros((n, n), dtype=float)
    cov = np.cov(arr, rowvar=False, ddof=1)
    cov = np.atleast_2d(np.asarray(cov, dtype=float))
    return 0.5*(cov+cov.T)


configure_plot_style()
print("uproot:", uproot.__version__)
print("iminuit imported")
print("mplhep available:", HAS_MPLHEP)


## 1. Configuration and propagation policy

Edit the primary analysis flags in the next cell. The **template estimator itself is not a flag here**: the nominal estimator is fixed to binned HIST templates.

- `FREE_CC_FRACTIONS=False`: nominal one-parameter fit in $f_{\mathrm{DPS}}$; the $c\bar c/b\bar b$ SPS and DPS compositions are fixed from the central fiducial MC composition.
- `FREE_CC_FRACTIONS=True`: optional three-parameter diagnostic fit in $f_{\mathrm{DPS}}$, $f_{cc}^{\mathrm{SPS}}$, and $f_{cc}^{\mathrm{DPS}}$.
- `USE_DPHI_PI=True`: nominal folded range $0\leq|\Delta\phi|\leq\pi$.
- `USE_DPHI_PI=False`: alternative range $0\leq|\Delta\phi|\leq3.0$.
- `USE_6X6_BINNING=True`: nominal legacy `6x6` configuration; the explicit edge arrays below define the actual number of bins.
- `INCLUDE_AN143_INTEGRATED_SYSTEMATICS`: controls fully correlated absolute-normalization bookkeeping; the $f_{\mathrm{DPS}}$ likelihood remains shape-only.
- `INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS`: includes the validated differential efficiency-response covariance when compatible with the nominal configuration.
- `AUTO_DD_REGION=True`: nominal automatic DPS-enriched-region scan subject to the predeclared support/closure gates.
- `AUTO_DD_REGION=False`: use the frozen fixed comparison region defined below.
- `APPLY_DD_PURITY_SCALE=True`: multiply the raw mixed-event data-driven extraction by the post-fit DPS purity in the selected enriched region.
- `RUN_SLIDE_DIAGNOSTICS=True`: draw the convener-facing diagnostic figures inline and save them.

The full Run-2 data luminosity is 126.28 fb$^{-1}$. Until the 2016 efficiency campaign is available, the validated 2017+2018 detector response is retained as the current Run-2 response proxy.

In [ ]:
# ================================================================
# PRIMARY ANALYSIS FLAGS
# ================================================================
# Nominal choice: one physics parameter, with the ccbar/bbbar composition
# fixed to the central fiducial MC prediction.
#
# False -> only fDPS is free; ccbar/bbbar fractions are fixed from the
#          central fiducial MC composition (1 free parameter).
# True -> fDPS, f_cc_SPS and f_cc_DPS are fitted.
# Nominal central template estimator: fixed binned histograms.
# The central estimator is fixed and has no alternative branch in this notebook.
ESTIMATOR_TAG = "Hist"

# One switch centralizes quick-development versus convener-production statistics.
# Nominal default is production/high-statistics mode.
HIGH_STATISTICS_MODE = True   # final / production
# HIGH_STATISTICS_MODE = False  # quick / development (non-reportable)
_statistics_mode_override = os.environ.get(
    "JPSIDSTAR_HIGH_STATISTICS_MODE"
)
if _statistics_mode_override is not None:
    HIGH_STATISTICS_MODE = bool(int(_statistics_mode_override))
_STATISTICS_DEFAULTS = {
    False: {
        "model": 500, "source": 1200, "prediction_data": 500,
        "prediction_model": 500, "dd_data": 500, "dd_model": 250,
        "gof": 1000, "closure": 500, "mixed": 1_000_000,
        "dd_visualization": 5_000,
    },
    True: {
        "model": 5000, "source": 12000, "prediction_data": 5000,
        "prediction_model": 5000, "dd_data": 5000, "dd_model": 2500,
        "gof": 10000, "closure": 5000, "mixed": 2_000_000,
        "dd_visualization": 50_000,
    },
}
_RUN_DEFAULTS = _STATISTICS_DEFAULTS[HIGH_STATISTICS_MODE]

FREE_CC_FRACTIONS = False

# Optional Gaussian constraints on the ccbar/bbbar compositions.
# The central values are obtained automatically from the same fiducial MC
# composition used in the 1-parameter fit.
USE_CC_FRACTION_CONSTRAINTS = True

# Absolute uncertainty on the fractions.
# Exploratory values for now; must be physically justified for the nominal result.
CC_SPS_CONSTRAINT_SIGMA = 0.02
CC_DPS_CONSTRAINT_SIGMA = 0.02

# False -> fit |Delta phi| in [0, 3.0], as in the current FINAL notebook.
# True  -> fit the full folded range [0, pi].
USE_DPHI_PI = True

# False -> 3x3 fit.
# True  -> use the legacy '6x6' key.  Its explicit edge arrays currently
#          define 4x4=16 bins; the key is retained to avoid a broad rewrite.
USE_6X6_BINNING = True

# False -> do not add the integrated AN-21-143 systematic budget to the
#          absolute-spectrum data uncertainty.
# True  -> include it as a fully correlated normalization uncertainty.
#
# IMPORTANT: this is a shape-only fDPS likelihood. A fully correlated global
# normalization uncertainty cancels from the normalized fit and from Data/MC.
# Therefore this flag changes the absolute-spectrum uncertainty (and bookkeeping),
# but it does not artificially broaden the fDPS profile. Only genuine differential
# variations can change the shape likelihood.
INCLUDE_AN143_INTEGRATED_SYSTEMATICS = True

# Validated differential efficiency-response systematics from the dedicated
# 2026-09-04 study.  These are normalized-shape covariance matrices and are
# therefore propagated directly inside the profile likelihood, GOF toys,
# projected pulls and post-fit prediction bands.
INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS = True

# The covariance bundle was generated for the nominal pi-range 4x4 fit
# (legacy key "6x6").  If those fit-range/binning flags are deliberately
# changed, disable this flag and rerun the dedicated systematic study for the
# new configuration rather than reusing an incompatible matrix.
REQUIRE_EFFICIENCY_RESPONSE_SYSTEMATICS = True

# Optional: run the non-nominal binning in the same notebook as an audit cross-check.
RUN_ALTERNATE_BINNING_CROSSCHECK = False

# Data-driven DPS-enriched normalization region.
#
# True:
#   scan the allowed (|Delta y|, |Delta phi|) rectangles and select the
#   maximum-purity region after the predeclared support/closure gates.
#
# False:
#   use the fixed comparison region
#       |Delta y| >= 2.2
#       0 <= |Delta phi| <= 4pi/5
#
# This choice affects only the data-driven cross-check.  It does not change
# the nominal 2D profile-likelihood fit.
AUTO_DD_REGION = True

# Apply the DPS purity from the template fit as a scale factor
# to the data-driven extraction.
#
# True:
#   fDPS_DD = f_raw_DD * P_DPS(post-fit)
#
# False:
#   fDPS_DD = f_raw_DD
#
# The post-fit purity is still calculated and kept as a diagnostic,
# but it is not applied to the data-driven result when False.
APPLY_DD_PURITY_SCALE = True


# The available MC does not describe the normalized 2D shape within its known
# finite-sample uncertainties.
USE_MODEL_INADEQUACY = True

MODEL_INADEQUACY_REL = float(os.environ.get("JPSIDSTAR_MODEL_INADEQUACY_REL", "0.25"))
MODEL_INADEQUACY_SOURCE = (
    "single global K=I relative 2D shape floor, calibrated on nominal residuals"
)
MODEL_INADEQUACY_CALIBRATED_ON_SAME_DATA = True
MODEL_INADEQUACY_EXTERNALLY_VALIDATED = False
if not (0.0 <= MODEL_INADEQUACY_REL <= 0.50):
    raise ValueError("MODEL_INADEQUACY_REL must be between 0 and 0.50")

# ================================================================
# Paths
# ================================================================
PROJECT_DIR = Path(os.environ.get(
    "JPSIDSTAR_PROJECT_DIR",
    "/eos/home-j/jgomespi/SWAN_projects/Associated_Production_JPsi_Dstar",
)).expanduser()
DATA_DIR = Path(
    os.environ.get("JPSIDSTAR_DATA_DIR", str(PROJECT_DIR / "Template_fit"))
).expanduser()
EFFICIENCY_DIR = Path(
    os.environ.get(
        "JPSIDSTAR_EFF_DIR",
        "/afs/cern.ch/work/j/jgomespi/private/workspace/JPsiDstar/"
        "Efficiency_Quarkonia_OpenCharm/output/efficiency/final",
    )
).expanduser()

EFFICIENCY_RESPONSE_SYSTEMATICS_DIR = Path(
    os.environ.get(
        "JPSIDSTAR_EFF_RESPONSE_SYST_DIR",
        str(DATA_DIR / "fdps_systematics_EfficiencyYearBinning_20260904"),
    )
).expanduser()

DPHI_MAX = float(np.pi if USE_DPHI_PI else 3.0)
NOMINAL_BINNING = "6x6" if USE_6X6_BINNING else "3x3"
FIT_PARAMETER_MODE = "3par" if FREE_CC_FRACTIONS else "1par"
DPHI_RANGE_TAG = "pi" if USE_DPHI_PI else "3p0"
AN_TAG = "ANon" if INCLUDE_AN143_INTEGRATED_SYSTEMATICS else "ANoff"
EFFSYST_TAG = (
    "EffRespSystOn"
    if INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS
    else "EffRespSystOff"
)
FLOOR_TAG = (
    f"MCfloor{int(round(100*MODEL_INADEQUACY_REL)):02d}"
    if USE_MODEL_INADEQUACY else "MCfloorOff"
)

DD_REGION_TAG = "DDauto" if AUTO_DD_REGION else "DDfixed"
DD_SCALE_TAG = "DDscaleOn" if APPLY_DD_PURITY_SCALE else "DDscaleOff"

DATA_PERIOD = "Run2"
EFFICIENCY_SCOPE_TAG = "Eff1718"
RUN_SCOPE_TAG = f"{DATA_PERIOD}Data_{EFFICIENCY_SCOPE_TAG}"

CONFIG_TAG = (
    f"{RUN_SCOPE_TAG}_{FIT_PARAMETER_MODE}_{NOMINAL_BINNING}_"
    f"dphi{DPHI_RANGE_TAG}_{AN_TAG}_{EFFSYST_TAG}_{FLOOR_TAG}_"
    f"{DD_REGION_TAG}_{DD_SCALE_TAG}"
)

OUTPUT_ROOT = Path(os.environ.get(
    "JPSIDSTAR_OUTPUT_ROOT",
    str(DATA_DIR / "FINAL_NOMINAL_RUN2_JPsiDstar_TemplateFit_DataDriven_HIST_CONVENERS_20260908_outputs"),
)).expanduser()
OUTDIR = OUTPUT_ROOT
OUTDIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = "Charmonium_all_vtx0p05_sigma_eff_jpsi_pt_bin1_25_100.root"
SWEIGHT_FILE = (
    "Charmonium_all_vtx0p05_sigma_eff_jpsi_pt_bin1_25_100_"
    "3Dfit_splot_signal_weights.root"
)
MC_TREE_WEIGHT_BRANCH = "weight"

# Frozen entry-count contract for the historical full-Run-2 data/sPlot pair.
REFERENCE_RAW_RUN2_ENTRIES = 36208
REFERENCE_SPLOT_ENTRIES = 25554

# ================================================================
# Physics normalization / fiducial phase space
# ================================================================
LUMI_RUN2_FB_INV = 126.28
LUMI_RUN2_PB_INV = 1000.0 * LUMI_RUN2_FB_INV
BR_JPSI_TO_MUMU = 0.05961
BR_DSTAR_TO_D0PI = 0.677
BR_D0_TO_KPI = 0.03947
BRANCHING_PRODUCT = BR_JPSI_TO_MUMU * BR_DSTAR_TO_D0PI * BR_D0_TO_KPI

JPSI_PT_MIN, JPSI_PT_MAX = 25.0, 100.0
JPSI_ABSY_MAX = 1.2
JPSI_MASS_MIN, JPSI_MASS_MAX = 2.95, 3.25
JPSI_DL_MIN, JPSI_DL_MAX = -0.5, 1.4
DSTAR_PT_MIN, DSTAR_PT_MAX = 4.0, 60.0
DSTAR_ABSY_MAX = 2.1

# ================================================================
# 2D binnings
# ================================================================
BINNING = {
    "3x3": {
        "dphi": np.array([
            0.0,
            DPHI_MAX / 3.0,
            2.0 * DPHI_MAX / 3.0,
            DPHI_MAX,
        ]),
        "dy": np.array([
            0.0,
            0.9,
            1.8,
            3.6,
        ]),
    },

    "6x6": {
        "dphi": np.array([
            0.0,
            DPHI_MAX / 4.0,
            2.0 * DPHI_MAX / 4.0,
            3.0 * DPHI_MAX / 4.0,
            DPHI_MAX,
        ]),
        "dy": np.array([
            0.0,
            0.733333333333333333333333,#0.75
            1.466666666666666666666666, # 1.5
            2.2, # 2.25
            4.0, #
        ]),
    },
}
ALTERNATE_BINNING = "3x3" if NOMINAL_BINNING == "6x6" else "6x6"
ACTUAL_N_DPHI_BINS = len(BINNING[NOMINAL_BINNING]["dphi"]) - 1
ACTUAL_N_DY_BINS = len(BINNING[NOMINAL_BINNING]["dy"]) - 1
MIN_TEMPLATE_NEFF_REPORTABLE = 10.0

# ================================================================
# Run-2 data period and currently available efficiency response
# ================================================================
# The data and sWeights are the historical FULL Run-2 inputs.
# They are intentionally kept intact.  The validated efficiency campaign
# currently contains 2017 and 2018; 2016APV/2016 will be incorporated later.
LUMI_BY_YEAR_FB = {
    "2016APV": 16.52,
    "2016": 8.53,
    "2017": 41.48,
    "2018": 59.75,
}

EFFICIENCY_YEARS = ("2017", "2018")
EFFICIENCY_YEAR_WEIGHTS = {
    year: float(LUMI_BY_YEAR_FB[year])
    for year in EFFICIENCY_YEARS
}
LUMI_EFFICIENCY_RESPONSE_FB_INV = float(
    sum(EFFICIENCY_YEAR_WEIGHTS.values())
)

# This flag is a scope label, not an event-level selection:
# full Run-2 data are corrected with the luminosity-weighted 2017+2018
# detector response until the 2016 maps are available.
USE_PARTIAL_RUN2_EFFICIENCY_PROXY = True
USE_2017_MC_AS_RUN2_PROXY = False  # compatibility/audit field only

EFFICIENCY_FLOOR = 1.0e-6
ASSOCIATION_EFF_MODE = "pt"

# Exactly one association representation is used in the nominal product.
EFFICIENCY_FACTORS_USED = [
    "acc_dimu",
    "acc_dstar",
    "eff_cuts_dimu",
    "eff_cuts_dstar",
    "eff_trigger",
    "eff_asso_pt",
]
EFFICIENCY_REQUIRED = EFFICIENCY_FACTORS_USED + ["eff_asso_rap"]
EFFICIENCY_SOURCE_FOR_COMPONENT = {
    "DPS-ccbar": "DPS-ccbar",
    "DPS-bbbar": "DPS-bbbar",
    "SPS-ccbar": "SPS-ccbar",
    "SPS-bbbar": "SPS-bbbar",
}

# Final efficiency ROOTs were validated with N_eff >= 25 for every map/bin
# entering the nominal product.
MIN_EFFICIENCY_MAP_NEFF = 25.0

# The current ROOT format stores per-factor/bin statistics but not the
# cross-factor event-level covariance. Independent factor fluctuations are
# therefore an explicit approximation, not a hidden exact statement.
EFFICIENCY_CROSS_FACTOR_COVARIANCE_AVAILABLE = False
# ================================================================
# Statistical ensemble / fit controls
# ================================================================
N_MODEL_STAT_REPLICAS = int(os.environ.get(
    "JPSIDSTAR_MODEL_STAT_REPLICAS", str(_RUN_DEFAULTS["model"])
))
N_SOURCE_AUDIT_REPLICAS = int(os.environ.get(
    "JPSIDSTAR_SOURCE_AUDIT_REPLICAS", str(_RUN_DEFAULTS["source"])
))
N_PREDICTION_DATA_TOYS = int(os.environ.get(
    "JPSIDSTAR_PREDICTION_DATA_TOYS", str(_RUN_DEFAULTS["prediction_data"])
))
N_PREDICTION_MODEL_REPLICAS = int(os.environ.get(
    "JPSIDSTAR_PREDICTION_MODEL_REPLICAS", str(_RUN_DEFAULTS["prediction_model"])
))
N_DD_DATA_REPLICAS = int(os.environ.get(
    "JPSIDSTAR_DD_DATA_REPLICAS", str(_RUN_DEFAULTS["dd_data"])
))
N_DD_MODEL_REPLICAS = int(os.environ.get(
    "JPSIDSTAR_DD_MODEL_REPLICAS", str(_RUN_DEFAULTS["dd_model"])
))
N_DD_VISUALIZATION_TOYS = int(os.environ.get(
    "JPSIDSTAR_DD_VISUALIZATION_TOYS",
    str(_RUN_DEFAULTS["dd_visualization"]),
))
MIN_DD_SUCCESSFUL_REPLICAS = 100
MIN_SUCCESSFUL_REPLICAS = 100
MIN_MODEL_REPLICA_SUCCESS_FRACTION = 0.90
MIN_DATA_REPLICA_SUCCESS_FRACTION = 0.95
MAX_RELEVANT_SIGMA_MCSE_FRACTION = 0.05
N_PROFILE_SCAN = 81
N_GOF_TOYS = int(os.environ.get(
    "JPSIDSTAR_PROFILE_GOF_TOYS", str(_RUN_DEFAULTS["gof"])
))
RANDOM_SEED = 12345
GOF_PVALUE_MIN_REPORTABLE = 0.01

# Explicit validation flag. Set only after a reviewed closure test.
SPLOT_CLOSURE_VALIDATED = bool(int(os.environ.get("JPSIDSTAR_SPLOT_CLOSURE_VALIDATED", "1")))

# ================================================================
# Differential systematic inputs
# ================================================================
SHAPE_SYSTEMATICS_DIR = Path(
    os.environ.get(
        "JPSIDSTAR_SHAPE_SYST_DIR",
        str(DATA_DIR / "Data" / "shape_systematics"),
    )
).expanduser()

# Signal-extraction/sPlot variants can be placed as ROOT files here.
SPLOT_VARIATION_DIR = SHAPE_SYSTEMATICS_DIR / "signal_yield_fit_variants"

# ================================================================
# Data-driven DPS-enriched region and mixing
# ================================================================
# AUTO_DD_REGION is defined above with the primary analysis flags.

# The DD method is always normalized in the explicitly clipped domain,
# even if a future template-fit cross-check uses the interval up to pi.
#DD_DPHI_MAX = min(3.0, DPHI_MAX)
DD_DPHI_MAX = DPHI_MAX

# Scan used only when AUTO_DD_REGION = True.
DD_SCAN_DY_MIN = np.arange(1.2, 3.01, 0.2, dtype=float)
DD_SCAN_DPHI_EDGES = np.arange(0.0, DD_DPHI_MAX + 0.001, 0.5, dtype=float)
DD_SCAN_MIN_DPHI_WIDTH = 1.0
DD_PURITY_PLATEAU = 0.01
DD_MIN_DPS_ACCEPTANCE = 0.02
DD_MAX_MIXED_DPS_CLOSURE_DEVIATION = 0.20
MIN_DATA_REGION_NEFF = 50.0
MIN_MIXED_REGION_NEFF = 1000.0
MIN_SPS_REGION_NEFF = 50.0
MIN_DPS_MC_REGION_NEFF = 50.0

# Fixed region used when AUTO_DD_REGION = False.
FIXED_DPS_REGION = {
    "dy_min": 2.2,
    "dphi_min": 0.0,
    "dphi_max": 4*DD_DPHI_MAX/5.0,
}

# In automatic mode this is only the initial seed and is overwritten
# after the nominal fit.  In fixed mode it remains the final DD region.
CENTRAL_DPS_REGION = FIXED_DPS_REGION.copy()

MIXING_INPUT_DIR = Path(os.environ.get("JPSIDSTAR_MIXING_INPUT_DIR", str(PROJECT_DIR / "condor_data_preparation" / "output_noHLT_preassoc_v1"))).expanduser()
MIXING_INPUT_GLOB = "Run2/*/*/*/output_mixing_*.coffea"
REQUIRE_DIFFERENT_EVENT_FOR_MIXING = True
MAX_JPSI_MIX_POOL = int(os.environ.get("JPSIDSTAR_MAX_JPSI_POOL", "750000"))
MAX_DSTAR_MIX_POOL = int(os.environ.get("JPSIDSTAR_MAX_DSTAR_POOL", "250000"))
TARGET_MIXED_PAIRS = int(os.environ.get(
    "JPSIDSTAR_TARGET_MIXED", str(_RUN_DEFAULTS["mixed"])
))
MIXING_RNG_SEED = 12345
MIX_CHUNK = int(os.environ.get("JPSIDSTAR_MIX_EFF_CHUNK", "100000"))

# ================================================================
# AN2021-143 v6 integrated uncertainty budget
# ================================================================
# These are integrated cross-section uncertainties. They are retained only
# as a fully correlated absolute-normalization reference unless an approved
# differential variation exists.
AN143_REFERENCE_RELATIVE_SYSTEMATICS = {
    "luminosity": 0.0242,
    "branching_ratio": 0.0205,
    "signal_yield": 0.0100,
    "tracking": 0.0925,
    "mc_sample_statistics": 0.0010,
    "pileup_reweighting": 0.1633,
    "yield_instability_pileup": 0.2000,
    "trigger_efficiency_model": 0.0340,
    "jpsi_polarization": 0.0,
}
AN143_TABLE_TOTAL_REL = 0.2784
AN143_REFERENCE_QUADRATURE_REL = math.sqrt(
    sum(v*v for v in AN143_REFERENCE_RELATIVE_SYSTEMATICS.values())
)
AN143_COMMON_NORMALIZATION_SOURCES = tuple(
    name for name in AN143_REFERENCE_RELATIVE_SYSTEMATICS
    if name != "mc_sample_statistics"
)
AN143_COMMON_REL = math.sqrt(sum(
    AN143_REFERENCE_RELATIVE_SYSTEMATICS[name]**2
    for name in AN143_COMMON_NORMALIZATION_SOURCES
))

_STATISTICS_MODE_LABEL = (
    "HIGH / PRODUCTION" if HIGH_STATISTICS_MODE else "LOW / DEVELOPMENT"
)
print("Statistics mode:", _STATISTICS_MODE_LABEL)
if not HIGH_STATISTICS_MODE:
    print("WARNING: LOW / DEVELOPMENT results are non-reportable.")
print("template estimator            =", ESTIMATOR_TAG)


print("AN143 table total             =", f"{100*AN143_TABLE_TOTAL_REL:.3f}%")
print("AN143 quadrature cross-check =", f"{100*AN143_REFERENCE_QUADRATURE_REL:.3f}%")
print("AN143 upper normalization ref =", f"{100*AN143_COMMON_REL:.3f}%")
print("model-stat replicas           =", N_MODEL_STAT_REPLICAS)
print("prediction data toys          =", N_PREDICTION_DATA_TOYS)
print("prediction model replicas     =", N_PREDICTION_MODEL_REPLICAS)
print("data-driven data replicas     =", N_DD_DATA_REPLICAS)
print("data-driven model replicas    =", N_DD_MODEL_REPLICAS)
print("DD visualization toys         =", N_DD_VISUALIZATION_TOYS)
print("GOF toys                      =", N_GOF_TOYS)
print("source-audit replicas         =", N_SOURCE_AUDIT_REPLICAS)
print("mixed-event pairs             =", TARGET_MIXED_PAIRS)
print("fit parameter mode             =", FIT_PARAMETER_MODE)
print("|Delta phi| maximum            =", DPHI_MAX)
print("nominal binning                =", NOMINAL_BINNING)
print("actual nominal grid            =", f"{ACTUAL_N_DPHI_BINS}x{ACTUAL_N_DY_BINS}")
print("alternate-binning cross-check  =", RUN_ALTERNATE_BINNING_CROSSCHECK)
print("data-driven region mode        =", "automatic maximum-purity" if AUTO_DD_REGION else "fixed")
print("AN143 integrated syst enabled  =", INCLUDE_AN143_INTEGRATED_SYSTEMATICS)
print("eff-response syst enabled      =", INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS)
print("eff-response syst directory    =", EFFICIENCY_RESPONSE_SYSTEMATICS_DIR)
print("empirical MC-shape floor       =", f"{100*MODEL_INADEQUACY_REL:.1f}%" if USE_MODEL_INADEQUACY else "OFF")
print("sPlot closure validated       =", SPLOT_CLOSURE_VALIDATED)
print("data-driven purity scale       =", APPLY_DD_PURITY_SCALE)
print("data period                    =", DATA_PERIOD)
print("full Run-2 luminosity          =", f"{LUMI_RUN2_FB_INV:.2f} fb^-1")
print("efficiency-response years      =", list(EFFICIENCY_YEARS))
print("efficiency-response luminosity =", f"{LUMI_EFFICIENCY_RESPONSE_FB_INV:.2f} fb^-1")
print("2016 efficiency pending        =", USE_PARTIAL_RUN2_EFFICIENCY_PROXY)
print("OUTDIR:", OUTDIR.resolve())

ANALYSIS_CONFIGURATION = {
    "TEMPLATE_MODE": "hist_nominal",
    "ESTIMATOR_TAG": ESTIMATOR_TAG,
    "HIGH_STATISTICS_MODE": bool(HIGH_STATISTICS_MODE),
    "FREE_CC_FRACTIONS": bool(FREE_CC_FRACTIONS),
    "N_FREE_PARAMETERS": int(3 if FREE_CC_FRACTIONS else 1),
    "USE_DPHI_PI": bool(USE_DPHI_PI),
    "DPHI_MAX": float(DPHI_MAX),
    "USE_6X6_BINNING": bool(USE_6X6_BINNING),
    "NOMINAL_BINNING": NOMINAL_BINNING,
    "ACTUAL_N_DPHI_BINS": int(ACTUAL_N_DPHI_BINS),
    "ACTUAL_N_DY_BINS": int(ACTUAL_N_DY_BINS),
    "INCLUDE_AN143_INTEGRATED_SYSTEMATICS": bool(INCLUDE_AN143_INTEGRATED_SYSTEMATICS),
    "INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS": bool(INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS),
    "REQUIRE_EFFICIENCY_RESPONSE_SYSTEMATICS": bool(REQUIRE_EFFICIENCY_RESPONSE_SYSTEMATICS),
    "EFFICIENCY_RESPONSE_SYSTEMATICS_DIR": str(EFFICIENCY_RESPONSE_SYSTEMATICS_DIR),
    "RUN_ALTERNATE_BINNING_CROSSCHECK": bool(RUN_ALTERNATE_BINNING_CROSSCHECK),
    "USE_MODEL_INADEQUACY": bool(USE_MODEL_INADEQUACY),
    "MODEL_INADEQUACY_REL": float(MODEL_INADEQUACY_REL if USE_MODEL_INADEQUACY else 0.0),
    "MODEL_INADEQUACY_SOURCE": MODEL_INADEQUACY_SOURCE,
    "MODEL_INADEQUACY_CALIBRATED_ON_SAME_DATA": bool(MODEL_INADEQUACY_CALIBRATED_ON_SAME_DATA),
    "MODEL_INADEQUACY_EXTERNALLY_VALIDATED": bool(MODEL_INADEQUACY_EXTERNALLY_VALIDATED),
    "N_MODEL_STAT_REPLICAS": int(N_MODEL_STAT_REPLICAS),
    "N_PREDICTION_DATA_TOYS": int(N_PREDICTION_DATA_TOYS),
    "N_PREDICTION_MODEL_REPLICAS": int(N_PREDICTION_MODEL_REPLICAS),
    "N_DD_DATA_REPLICAS": int(N_DD_DATA_REPLICAS),
    "N_DD_MODEL_REPLICAS": int(N_DD_MODEL_REPLICAS),
    "N_DD_VISUALIZATION_TOYS": int(N_DD_VISUALIZATION_TOYS),
    "N_GOF_TOYS": int(N_GOF_TOYS),
    "N_SOURCE_AUDIT_REPLICAS": int(N_SOURCE_AUDIT_REPLICAS),
    "TARGET_MIXED_PAIRS": int(TARGET_MIXED_PAIRS),
    "DD_MAX_MIXED_DPS_CLOSURE_DEVIATION": float(DD_MAX_MIXED_DPS_CLOSURE_DEVIATION),
    "CONFIG_TAG": CONFIG_TAG,
    "AUTO_DD_REGION": bool(AUTO_DD_REGION),
    "DD_REGION_MODE": "automatic_maximum_purity" if AUTO_DD_REGION else "fixed",
    "FIXED_DPS_REGION": {
        key: float(value) for key, value in FIXED_DPS_REGION.items()
    },
    "APPLY_DD_PURITY_SCALE": bool(APPLY_DD_PURITY_SCALE),
    "DATA_PERIOD": DATA_PERIOD,
    "LUMI_RUN2_FB_INV": float(LUMI_RUN2_FB_INV),
    "EFFICIENCY_YEARS": list(EFFICIENCY_YEARS),
    "LUMI_EFFICIENCY_RESPONSE_FB_INV": float(LUMI_EFFICIENCY_RESPONSE_FB_INV),
    "USE_PARTIAL_RUN2_EFFICIENCY_PROXY": bool(USE_PARTIAL_RUN2_EFFICIENCY_PROXY),
    "ASSOCIATION_EFF_MODE": ASSOCIATION_EFF_MODE,
}
with open(OUTDIR/"analysis_configuration.json", "w") as handle:
    json.dump(ANALYSIS_CONFIGURATION, handle, indent=2)


In [ ]:
# ================================================================
# SLIDE / CONVENER DIAGNOSTIC FLAGS
# ================================================================
#
# These flags control only extra presentation diagnostics added in
# FINAL_SLIDES. They do not alter the nominal physics result.
# ================================================================

RUN_SLIDE_DIAGNOSTICS = bool(
    int(os.environ.get("JPSIDSTAR_RUN_SLIDE_DIAGNOSTICS", "1"))
)

RUN_TEMPLATE_FIT_CLOSURE = bool(
    int(os.environ.get("JPSIDSTAR_RUN_TEMPLATE_CLOSURE", "1"))
)

SLIDE_SHAPE_NBINS = int(
    os.environ.get("JPSIDSTAR_SLIDE_SHAPE_NBINS", "12")
)

TEMPLATE_FIT_CLOSURE_N_TOYS = int(os.environ.get(
    "JPSIDSTAR_TEMPLATE_FIT_CLOSURE_N_TOYS",
    os.environ.get("JPSIDSTAR_SLIDE_CLOSURE_N_TOYS", str(_RUN_DEFAULTS["closure"]))
))
# Backward-compatible local alias used by the frozen closure cell.
SLIDE_CLOSURE_N_TOYS = TEMPLATE_FIT_CLOSURE_N_TOYS

SLIDE_CLOSURE_INJECTIONS = np.array(
    [0.20, 0.35, 0.50, 0.65, 0.80],
    dtype=float,
)

if SLIDE_SHAPE_NBINS < 4:
    raise ValueError("SLIDE_SHAPE_NBINS must be >= 4")

if SLIDE_CLOSURE_N_TOYS < 50 and RUN_TEMPLATE_FIT_CLOSURE:
    warnings.warn(
        "Template-fit closure is configured with fewer than 50 toys "
        "per injected point; this is adequate only for a quick code check."
    )

print(
    "[slides] diagnostics:",
    "ON" if RUN_SLIDE_DIAGNOSTICS else "OFF",
)
print(
    "[slides] template-fit closure:",
    "ON" if RUN_TEMPLATE_FIT_CLOSURE else "OFF",
    f"({SLIDE_CLOSURE_N_TOYS} toys / injected point)",
)


### 1a. Input preflight

This gate uses the exact locations established by the lxplus audit on 2026-09-04. The eight selected-MC ROOT files are the preferred inputs; the missing legacy CSV fallbacks are therefore not a blocker. The missing `shape_systematics` directory is allowed for the nominal/statistical run and is recorded later as an incomplete differential-systematic input.

In [ ]:
# ================================================================
# INPUT PREFLIGHT
# ================================================================
#
# Contract:
#
#   data:
#       full Run-2 data + full Run-2 sWeights
#
#   MC:
#       the eight selected-MC ROOT files already used by the analysis
#
#   efficiencies:
#       final validated 2017 + 2018 ROOT files
#
#   mixing:
#       full Run-2 coffea pool
#
# The final efficiency ROOT files are preferentially read from EOS.
# An AFS location is retained only as a secondary fallback.
# ================================================================

from pathlib import Path
import warnings


# ----------------------------------------------------------------
# Main analysis inputs
# ----------------------------------------------------------------

data_root_dir = DATA_DIR / "Data"

required_data_files = [
    DATA_FILE,
    SWEIGHT_FILE,
]

required_mc_files = [
    "dps_ccbar_9to30_usual.root",
    "dps_ccbar_30to50_usual.root",
    "dps_ccbar_50to100_usual.root",
    "dps_bbbar_9to30_usual.root",
    "dps_bbbar_30to50_usual.root",
    "dps_bbbar_50to100_usual.root",
    "sps_3fs_ccbar_25to100_usual.root",
    "sps_3fs_bbbar_25to100_usual.root",
]


# ----------------------------------------------------------------
# Efficiency inputs
# ----------------------------------------------------------------

required_efficiency_files = [
    f"efficiencies_{component}_differential_final_jpsi_{year}.root"
    for year in EFFICIENCY_YEARS
    for component in (
        "DPS-ccbar",
        "DPS-bbbar",
        "SPS-ccbar",
        "SPS-bbbar",
    )
]

required_efficiency_systematic_files = [
    "efficiency_systematics_for_final.json",
    "year_response_covariance.npy",
    "efficiency_binning_covariance.npy",
    "efficiency_response_systematic_covariance_total.npy",
]


# Prefer the EOS copy because this notebook is normally executed
# in SWAN/Jupyter and EOS visibility is stable there.
EFFICIENCY_DIR_CANDIDATES = [
    EFFICIENCY_DIR,
    DATA_DIR / "Data" / "efficiency_final_2017_2018",

    Path(
        "/eos/home-j/jgomespi/SWAN_projects/"
        "Associated_Production_JPsi_Dstar/"
        "Template_fit/Data/"
        "efficiency_final_2017_2018"
    ),

    Path(
        "/afs/cern.ch/work/j/jgomespi/private/workspace/"
        "JPsiDstar/Efficiency_Quarkonia_OpenCharm/"
        "output/efficiency/final"
    ),
]


def directory_contains_all_files(directory, filenames):
    directory = Path(directory)

    if not directory.is_dir():
        return False

    return all(
        (directory / filename).is_file()
        for filename in filenames
    )


EFFICIENCY_DIR = None

for candidate in EFFICIENCY_DIR_CANDIDATES:

    if directory_contains_all_files(
        candidate,
        required_efficiency_files,
    ):
        EFFICIENCY_DIR = candidate
        break


if EFFICIENCY_DIR is None:

    details = []

    for candidate in EFFICIENCY_DIR_CANDIDATES:

        missing_here = [
            filename
            for filename in required_efficiency_files
            if not (candidate / filename).is_file()
        ]

        details.append(
            "\nDirectory:\n"
            f"  {candidate}\n"
            "Missing:\n  "
            + "\n  ".join(missing_here)
        )

    raise FileNotFoundError(
        "Could not find a complete validated 2017+2018 "
        "efficiency set in any allowed location.\n"
        + "\n".join(details)
    )


# ----------------------------------------------------------------
# Validate dedicated efficiency-response systematic bundle
# ----------------------------------------------------------------

missing_eff_syst = []
if INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS:
    missing_eff_syst = [
        str(EFFICIENCY_RESPONSE_SYSTEMATICS_DIR / filename)
        for filename in required_efficiency_systematic_files
        if not (EFFICIENCY_RESPONSE_SYSTEMATICS_DIR / filename).is_file()
    ]

    if missing_eff_syst and REQUIRE_EFFICIENCY_RESPONSE_SYSTEMATICS:
        raise FileNotFoundError(
            "Missing mandatory efficiency-response systematic products:\n  "
            + "\n  ".join(missing_eff_syst)
        )


# ----------------------------------------------------------------
# Validate data and selected MC
# ----------------------------------------------------------------

missing_main = [
    str(data_root_dir / filename)
    for filename in (
        required_data_files
        + required_mc_files
    )
    if not (
        data_root_dir / filename
    ).is_file()
]


if missing_main:

    raise FileNotFoundError(
        "Missing mandatory data/MC inputs:\n  "
        + "\n  ".join(missing_main)
    )


# ----------------------------------------------------------------
# Validate final efficiency ROOTs
# ----------------------------------------------------------------

missing_eff = [
    str(EFFICIENCY_DIR / filename)
    for filename in required_efficiency_files
    if not (
        EFFICIENCY_DIR / filename
    ).is_file()
]


if missing_eff:

    raise FileNotFoundError(
        "Missing mandatory 2017+2018 efficiency inputs:\n  "
        + "\n  ".join(missing_eff)
    )


# ----------------------------------------------------------------
# Validate full Run-2 mixing inputs
# ----------------------------------------------------------------

mixing_files = sorted(
    MIXING_INPUT_DIR.glob(
        MIXING_INPUT_GLOB
    )
)


if not mixing_files:

    raise FileNotFoundError(
        "No mixing inputs found under:\n  "
        f"{MIXING_INPUT_DIR / MIXING_INPUT_GLOB}"
    )


# ----------------------------------------------------------------
# Hard sanity checks
# ----------------------------------------------------------------

if len(required_data_files) != 2:
    raise RuntimeError(
        "Internal preflight error: expected exactly "
        "data ROOT + sWeight ROOT"
    )

if len(required_mc_files) != 8:
    raise RuntimeError(
        "Internal preflight error: expected exactly "
        "8 selected-MC ROOT files"
    )

if len(required_efficiency_files) != 8:
    raise RuntimeError(
        "Internal preflight error: expected exactly "
        "8 final efficiency ROOT files"
    )

if set(EFFICIENCY_YEARS) != {
    "2017",
    "2018",
}:
    raise RuntimeError(
        "This notebook expects exactly the validated "
        "2017+2018 efficiency response. "
        f"Found EFFICIENCY_YEARS={EFFICIENCY_YEARS}"
    )


# ----------------------------------------------------------------
# Report
# ----------------------------------------------------------------

print("=" * 72)
print("INPUT PREFLIGHT: PASS")
print("=" * 72)

print()
print("Data directory:")
print(" ", data_root_dir)

print()
print("Efficiency directory selected:")
print(" ", EFFICIENCY_DIR)

print()
print(
    "data/sWeight ROOTs:",
    len(required_data_files),
)

print(
    "selected-MC ROOTs:",
    len(required_mc_files),
)

print(
    "final efficiency ROOTs:",
    len(required_efficiency_files),
)

print(
    "eff-response systematic files:",
    (len(required_efficiency_systematic_files) if INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS else 0),
)

print(
    "mixing coffea files:",
    len(mixing_files),
)


# ----------------------------------------------------------------
# Informational checks
# ----------------------------------------------------------------

print()
print("Efficiency response years:")
print(" ", list(EFFICIENCY_YEARS))

print(
    "Efficiency-response luminosity [fb^-1]:",
    sum(
        LUMI_BY_YEAR_FB[year]
        for year in EFFICIENCY_YEARS
    ),
)

print(
    "Full Run-2 data luminosity [fb^-1]:",
    LUMI_RUN2_FB_INV,
)


if not SHAPE_SYSTEMATICS_DIR.is_dir():

    warnings.warn(
        "Differential shape-systematics directory is absent. "
        "This does NOT block the nominal/statistical fit. "
        "The final systematic-completeness audit must remain "
        "non-final until those inputs are supplied."
    )


print()
print("Preflight contract:")
print("  full Run-2 data                : READY")
print("  full Run-2 sWeights            : READY")
print("  selected SPS/DPS MC            : READY")
print("  2017 final efficiency response : READY")
print("  2018 final efficiency response : READY")
print(
    "  efficiency-response syst.      : "
    + (
        "READY"
        if INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS and not missing_eff_syst
        else "DISABLED" if not INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS
        else "MISSING"
    )
)
print("  full Run-2 mixed-event pool    : READY")

if SHAPE_SYSTEMATICS_DIR.is_dir():
    print(
        "  differential shape syst.      : READY"
    )
else:
    print(
        "  differential shape syst.      : NOT AVAILABLE "
        "(non-blocking for nominal fit)"
    )

print()
print("=" * 72)


## 2. Input data and sPlot closure diagnostic
¶
The physics distribution is built from the supplied signal sWeights after reproducing the signal-fit phase-space cuts. The correlation audit between the sPlot discriminants and $(|\Delta\phi|,|\Delta y|)$ is retained exactly as in the FINAL workflow.

In [ ]:
BRANCH_CANDIDATES = {
    "jpsi_mass": ["jpsi_mass", "Jpsi_mass", "dimu_mass", "pair_data_jpsi_mass"],
    "dstar_deltam": ["dstar_deltam", "dstar_deltamr", "dstar_mass", "pair_data_dstar_deltamr"],
    "jpsi_dl": ["jpsi_dl", "Jpsi_dl", "dimu_dl", "pair_data_jpsi_dl"],
    "delta_phi": ["jpsi_dstar_deltaphi", "delta_phi", "deltaphi", "pair_data_delta_phi"],
    "delta_y": ["jpsi_dstar_deltarap", "jpsi_dstar_deltay", "delta_y", "deltarap", "pair_data_delta_y"],
    "jpsi_rap": ["jpsi_rap", "jpsi_y", "dimu_rap", "pair_data_jpsi_rap"],
    "jpsi_phi": ["jpsi_phi", "dimu_phi", "pair_data_jpsi_phi"],
    "jpsi_pt": ["jpsi_pt", "dimu_pt", "pair_data_jpsi_pt"],
    "dstar_rap": ["dstar_rap", "dstar_y", "pair_data_dstar_rap"],
    "dstar_phi": ["dstar_phi", "pair_data_dstar_phi"],
    "dstar_pt": ["dstar_pt", "pair_data_dstar_pt"],
    "sweight": ["sweight", "sWeight", "signal_weight", "sw", "weight"],
}


def canonical_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def fold_delta_phi(values):
    values = np.asarray(values, dtype=float)
    return np.abs(np.arctan2(np.sin(values), np.cos(values)))


def find_file(filename, roots=None):
    roots = list(roots or []) + [
        DATA_DIR / "Data",
        DATA_DIR,
        Path.cwd(),
        Path.cwd()/"Template_fit"/"Data",
        Path("/mnt/data"),
    ]
    seen = set()
    for raw_root in roots:
        root = Path(raw_root).expanduser()
        try:
            root = root.resolve()
        except Exception:
            pass
        if root in seen or not root.exists():
            continue
        seen.add(root)
        direct = root / filename
        if direct.exists():
            return direct
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    return None


def open_first_ttree(path, preferred=("asso", "splot", "Events", "tree")):
    root_file = uproot.open(path)
    classnames = root_file.classnames()
    for preferred_name in preferred:
        for key, classname in classnames.items():
            if key.split(";")[0] == preferred_name and "TTree" in classname:
                return root_file, root_file[key]
    for key, classname in classnames.items():
        if "TTree" in classname:
            return root_file, root_file[key]
    root_file.close()
    raise RuntimeError(f"No TTree found in {path}")


def resolve_branch(tree, logical, required=True):
    branches = list(tree.keys())
    canonical = {canonical_name(branch): branch for branch in branches}
    for candidate in BRANCH_CANDIDATES.get(logical, [logical]):
        if candidate in branches:
            return candidate
        key = canonical_name(candidate)
        if key in canonical:
            return canonical[key]
    if required:
        raise KeyError(f"Could not resolve {logical}; available={branches[:80]}")
    return None


def read_logical_dataframe(path, logical_names, fixed=None):
    root_file, tree = open_first_ttree(path)
    aliases = {logical: resolve_branch(tree, logical) for logical in logical_names}
    for logical, branch in (fixed or {}).items():
        if branch not in tree.keys():
            root_file.close()
            raise KeyError(f"Missing fixed branch {branch!r} in {path}")
        aliases[logical] = branch
    arrays = tree.arrays(sorted(set(aliases.values())), library="pd")
    root_file.close()
    inverse = {branch: logical for logical, branch in aliases.items()}
    return arrays.rename(columns=inverse).reset_index(drop=True), aliases


data_path = find_file(DATA_FILE)
sweight_path = find_file(SWEIGHT_FILE)
if data_path is None or sweight_path is None:
    raise FileNotFoundError("Could not locate the Run-2 data and sWeight ROOT files")

logical_data = [
    "jpsi_mass", "dstar_deltam", "jpsi_dl", "delta_phi", "delta_y",
    "jpsi_rap", "jpsi_phi", "jpsi_pt", "dstar_rap", "dstar_phi", "dstar_pt",
]
data_raw, _ = read_logical_dataframe(data_path, logical_data)
sweight_raw, _ = read_logical_dataframe(sweight_path, ["sweight"])

if len(data_raw) != REFERENCE_RAW_RUN2_ENTRIES:
    raise RuntimeError(
        f"Unexpected raw Run-2 entry count: {len(data_raw)} "
        f"(expected {REFERENCE_RAW_RUN2_ENTRIES})"
    )
if len(sweight_raw) != REFERENCE_SPLOT_ENTRIES:
    raise RuntimeError(
        f"Unexpected sWeight entry count: {len(sweight_raw)} "
        f"(expected {REFERENCE_SPLOT_ENTRIES})"
    )

data_raw["delta_phi"] = fold_delta_phi(data_raw["delta_phi"])
data_raw["delta_y"] = np.abs(data_raw["delta_y"].to_numpy(float))

splot_mask = (
    (data_raw["jpsi_mass"] > JPSI_MASS_MIN)
    & (data_raw["jpsi_mass"] < JPSI_MASS_MAX)
    & (data_raw["jpsi_dl"] > JPSI_DL_MIN)
    & (data_raw["jpsi_dl"] < JPSI_DL_MAX)
    & (data_raw["dstar_deltam"] > 0.13957)
    & (data_raw["dstar_deltam"] < 0.16000)
)
data_sw = data_raw.loc[splot_mask].reset_index(drop=True).copy()
sweights = sweight_raw["sweight"].to_numpy(float)
if len(data_sw) != len(sweights):
    raise ValueError(
        f"Selected data ({len(data_sw)}) and sWeights ({len(sweights)}) differ"
    )
if len(data_sw) != REFERENCE_SPLOT_ENTRIES:
    raise RuntimeError(
        f"Unexpected sPlot-selected data count: {len(data_sw)} "
        f"(expected {REFERENCE_SPLOT_ENTRIES})"
    )
data_sw["sweight"] = sweights

# Enforce the same fiducial kinematics used for MC without breaking the
# event-by-event sWeight alignment. The current input is already fully
# fiducial; fail loudly if a future input is not.
data_fiducial_mask = (
    (data_sw["jpsi_pt"] > JPSI_PT_MIN)
    & (data_sw["jpsi_pt"] < JPSI_PT_MAX)
    & (np.abs(data_sw["jpsi_rap"]) < JPSI_ABSY_MAX)
    & (data_sw["dstar_pt"] > DSTAR_PT_MIN)
    & (data_sw["dstar_pt"] < DSTAR_PT_MAX)
    & (np.abs(data_sw["dstar_rap"]) < DSTAR_ABSY_MAX)
)
if not bool(np.all(data_fiducial_mask)):
    raise RuntimeError(
        f"{np.count_nonzero(~data_fiducial_mask)} sPlot-matched data events "
        "fall outside the MC fiducial phase space; rebuild aligned sWeights."
    )

selection_summary = pd.DataFrame([
    ("Raw Run-2 entries", len(data_raw)),
    ("sWeight entries", len(sweights)),
    ("sPlot-matched entries", len(data_sw)),
    ("Data outside MC fiducial phase space", int(np.count_nonzero(~data_fiducial_mask))),
    ("Sum of sWeights", float(np.sum(sweights))),
    ("sqrt(sum(sWeight^2))", float(np.sqrt(np.sum(sweights**2)))),
    ("Negative-sWeight fraction", float(np.mean(sweights < 0.0))),
    ("sWeight effective entries", weighted_effective_entries(sweights)),
], columns=["quantity", "value"])
print(selection_summary.to_string(index=False))
selection_summary.to_csv(OUTDIR/"splot_matching_summary.csv", index=False)

# Necessary diagnostic for using the sPlot projection in the physics variables.
rows = []
for discriminant in ["jpsi_mass", "dstar_deltam", "jpsi_dl"]:
    for observable in ["delta_phi", "delta_y"]:
        rho, pvalue = spearmanr(
            data_sw[discriminant].to_numpy(float),
            data_sw[observable].to_numpy(float),
            nan_policy="omit",
        )
        rows.append({
            "discriminant": discriminant,
            "observable": observable,
            "spearman_rho": float(rho),
            "pvalue": float(pvalue),
        })
splot_correlation_table = pd.DataFrame(rows)
splot_correlation_table.to_csv(OUTDIR/"splot_observable_correlations.csv", index=False)

rho_matrix = (
    splot_correlation_table.pivot(
        index="discriminant", columns="observable", values="spearman_rho"
    )
    .reindex(index=["jpsi_mass", "dstar_deltam", "jpsi_dl"],
             columns=["delta_phi", "delta_y"])
)
fig, ax = plt.subplots(figsize=(6.8, 4.8), layout="constrained")
im = ax.imshow(rho_matrix.to_numpy(float), vmin=-0.15, vmax=0.15, aspect="auto", cmap="coolwarm")
ax.set_xticks(np.arange(2), [r"$|\Delta\phi|$", r"$|\Delta y|$"])
ax.set_yticks(np.arange(3), [r"$m(J/\psi)$", r"$\Delta m(D^*)$", r"$\ell_{J/\psi}$"])
for iy in range(3):
    for ix in range(2):
        ax.text(ix, iy, f"{rho_matrix.iloc[iy, ix]:+.3f}", ha="center", va="center", fontsize=11)
cbar = fig.colorbar(im, ax=ax, shrink=0.86)
cbar.set_label("Spearman $\\rho$")
ax.set_title("sPlot discriminant–physics-variable correlation audit", pad=12)
save_and_show(fig, "splot_correlation_audit", OUTDIR)
if np.nanmax(np.abs(splot_correlation_table["spearman_rho"])) >= 0.10:
    warnings.warn(
        "At least one signal-fit discriminant has |Spearman rho| >= 0.10 with a physics observable. "
        "The sPlot factorization assumption needs an explicit closure validation before this fit is reportable."
    )


### Slide diagnostic — data before and after sPlot

The comparison below uses **exactly the same 25,554 candidates** on both curves. Therefore it isolates the effect of the signal sWeights rather than mixing the sPlot selection with the sWeight application.

For each observable:

- **before sPlot weighting:** $w_i=1$;
- **after sPlot weighting:** $w_i=w_i^{\rm sPlot}$.

Both histograms are normalized independently to unit area. The weighted uncertainties use $\sum_i w_i^2$ and propagate the normalization constraint.

In [ ]:
# ================================================================
# SLIDE DIAGNOSTIC: DATA BEFORE / AFTER sPLOT WEIGHTING
# Robust version for SWAN / Matplotlib
# ================================================================

plt.close("all")


def normalized_weighted_hist(values, weights, edges):
    """
    Normalized weighted histogram with the covariance induced by
    the normalization constraint.

    For each raw bin:
        H_k = sum_i w_i
        Var(H_k) = sum_i w_i^2

    After normalization:
        p_k = H_k / sum_j H_j

    The full covariance is propagated with the Jacobian.
    """

    values = np.asarray(values, dtype=float).reshape(-1)
    weights = np.asarray(weights, dtype=float).reshape(-1)
    edges = np.asarray(edges, dtype=float).reshape(-1)

    if len(values) != len(weights):
        raise RuntimeError(
            f"values/weights size mismatch: "
            f"{len(values)} vs {len(weights)}"
        )

    finite = (
        np.isfinite(values)
        & np.isfinite(weights)
    )

    values = values[finite]
    weights = weights[finite]

    h = np.histogram(
        values,
        bins=edges,
        weights=weights,
    )[0].astype(float)

    h2 = np.histogram(
        values,
        bins=edges,
        weights=weights**2,
    )[0].astype(float)

    total = float(np.sum(h))

    if (not np.isfinite(total)) or total <= 0.0:
        raise RuntimeError(
            "Cannot normalize weighted histogram: "
            f"integral = {total}"
        )

    shape = h / total

    # ------------------------------------------------------------
    # Propagate p_k = H_k / sum_j H_j
    # ------------------------------------------------------------

    nbin = len(h)

    J = (
        np.eye(nbin, dtype=float) / total
        - h[:, None] / total**2
    )

    cov_raw = np.diag(h2)

    cov = J @ cov_raw @ J.T
    cov = 0.5 * (cov + cov.T)

    err = np.sqrt(
        np.clip(
            np.diag(cov),
            0.0,
            None,
        )
    )

    return {
        "shape": shape,
        "error": err,
        "cov": cov,
        "raw": h,
        "raw_sumw2": h2,
        "integral": total,
    }


if RUN_SLIDE_DIAGNOSTICS:

    # ============================================================
    # Observable binning
    # ============================================================

    dphi_edges_slide = np.linspace(
        0.0,
        float(DPHI_MAX),
        int(SLIDE_SHAPE_NBINS) + 1,
    )

    dy_nominal_edges = np.asarray(
        BINNING[NOMINAL_BINNING]["dy"],
        dtype=float,
    ).reshape(-1)

    dy_edges_slide = np.linspace(
        float(dy_nominal_edges[0]),
        float(dy_nominal_edges[-1]),
        int(SLIDE_SHAPE_NBINS) + 1,
    )


    # ============================================================
    # EXACTLY the same candidates before/after weighting
    # ============================================================

    n_candidates = int(len(data_sw))

    before_weights = np.ones(
        n_candidates,
        dtype=float,
    )

    after_weights = np.asarray(
        data_sw["sweight"].to_numpy(),
        dtype=float,
    ).reshape(-1)

    if len(after_weights) != n_candidates:
        raise RuntimeError(
            "sWeight/data alignment failure: "
            f"{len(after_weights)} weights for "
            f"{n_candidates} candidates"
        )


    # ============================================================
    # Figure
    #
    # Do NOT use constrained_layout here:
    # the SWAN Matplotlib stack showed a text-layout failure.
    # ============================================================

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11.8, 4.9),
    )

    fig.subplots_adjust(
        left=0.09,
        right=0.985,
        bottom=0.17,
        top=0.77,
        wspace=0.22,
    )


    observables = [
        {
            "column": "delta_phi",
            "edges": dphi_edges_slide,
            "xlabel": XLABEL_DPHI,
        },
        {
            "column": "delta_y",
            "edges": dy_edges_slide,
            "xlabel": XLABEL_DY,
        },
    ]

    slide_histograms = {}


    # ============================================================
    # Fill and plot
    # ============================================================

    for ax, spec in zip(axes, observables):

        column = spec["column"]
        edges = np.asarray(
            spec["edges"],
            dtype=float,
        ).reshape(-1)

        values = np.asarray(
            data_sw[column].to_numpy(),
            dtype=float,
        ).reshape(-1)

        before_result = normalized_weighted_hist(
            values=values,
            weights=before_weights,
            edges=edges,
        )

        after_result = normalized_weighted_hist(
            values=values,
            weights=after_weights,
            edges=edges,
        )

        before = before_result["shape"]
        before_err = before_result["error"]

        after = after_result["shape"]
        after_err = after_result["error"]

        centers = 0.5 * (
            edges[:-1]
            + edges[1:]
        )

        widths = np.diff(edges)


        # --------------------------------------------------------
        # Before sPlot weighting
        # --------------------------------------------------------

        ax.errorbar(
            centers,
            before,
            xerr=0.5 * widths,
            yerr=before_err,
            fmt="o",
            markersize=5.0,
            capsize=2.5,
            elinewidth=1.2,
            linewidth=1.2,
            label="Before sPlot weighting",
        )


        # --------------------------------------------------------
        # After nominal signal sWeights
        # --------------------------------------------------------

        ax.errorbar(
            centers,
            after,
            xerr=0.5 * widths,
            yerr=after_err,
            fmt="s",
            markersize=5.0,
            capsize=2.5,
            elinewidth=1.2,
            linewidth=1.2,
            label="After sPlot weighting",
        )


        # --------------------------------------------------------
        # Cosmetics
        # --------------------------------------------------------

        ax.set_xlim(
            float(edges[0]),
            float(edges[-1]),
        )

        ax.set_xlabel(
            spec["xlabel"],
            fontsize=AXIS_LABEL_FONT_SIZE,
        )

        ax.set_ylabel(
            YLABEL_NORMALIZED,
            fontsize=AXIS_LABEL_FONT_SIZE,
        )

        ax.tick_params(
            axis="both",
            labelsize=11.5,
        )

        ax.grid(
            alpha=0.20,
            linewidth=0.7,
        )

        ax.axhline(
            0.0,
            linewidth=0.8,
            alpha=0.6,
        )


        # Do not blindly force ymin=0:
        # an sWeighted bin is allowed to become negative.
        ymin_candidate = min(
            np.min(before - before_err),
            np.min(after - after_err),
        )

        ymax_candidate = max(
            np.max(before + before_err),
            np.max(after + after_err),
        )

        if ymin_candidate >= 0.0:
            ymin = 0.0
        else:
            ymin = 1.15 * float(ymin_candidate)

        ymax = 1.18 * float(ymax_candidate)

        ax.set_ylim(
            ymin,
            ymax,
        )

        ax.legend(
            loc="best",
            fontsize=22,
            frameon=False,
        )


        slide_histograms[column] = {
            "edges": edges,
            "centers": centers,
            "before": before_result,
            "after": after_result,
        }



    #fig.suptitle(
    #    "Effect of the signal sPlot weighting on the data shapes",
    #    fontsize=15.5,
    #    y=0.965,
    #)


    save_and_show(fig, "data_splot_before_after_CMS", OUTDIR)


    # ============================================================
    # Numerical summary for audit / slides
    # ============================================================

    sum_sweights = float(
        np.sum(after_weights)
    )

    sum_sweights2 = float(
        np.sum(after_weights**2)
    )

    negative_fraction = float(
        np.mean(after_weights < 0.0)
    )

    neff_sweights = float(
        weighted_effective_entries(
            after_weights
        )
    )

    DATA_SPLOT_SLIDE_SUMMARY = pd.DataFrame(
        [
            {
                "quantity":
                    "selected_candidates",
                "value":
                    float(n_candidates),
            },
            {
                "quantity":
                    "sum_sweights",
                "value":
                    sum_sweights,
            },
            {
                "quantity":
                    "sum_sweights_squared",
                "value":
                    sum_sweights2,
            },
            {
                "quantity":
                    "negative_sweight_fraction",
                "value":
                    negative_fraction,
            },
            {
                "quantity":
                    "sweight_effective_entries",
                "value":
                    neff_sweights,
            },
        ]
    )

    summary_path = (
        OUTDIR
        / "data_splot_before_after_summary.csv"
    )

    DATA_SPLOT_SLIDE_SUMMARY.to_csv(
        summary_path,
        index=False,
    )


    # ============================================================
    # Additional bin-by-bin audit table
    # ============================================================

    audit_rows = []

    for column, result in slide_histograms.items():

        edges = result["edges"]

        for ibin in range(
            len(edges) - 1
        ):

            audit_rows.append(
                {
                    "observable": column,
                    "bin": ibin,
                    "low": float(
                        edges[ibin]
                    ),
                    "high": float(
                        edges[ibin + 1]
                    ),
                    "before":
                        float(
                            result[
                                "before"
                            ]["shape"][ibin]
                        ),
                    "before_error":
                        float(
                            result[
                                "before"
                            ]["error"][ibin]
                        ),
                    "after":
                        float(
                            result[
                                "after"
                            ]["shape"][ibin]
                        ),
                    "after_error":
                        float(
                            result[
                                "after"
                            ]["error"][ibin]
                        ),
                }
            )

    DATA_SPLOT_BIN_AUDIT = pd.DataFrame(
        audit_rows
    )

    bin_audit_path = (
        OUTDIR
        / "data_splot_before_after_bins.csv"
    )

    DATA_SPLOT_BIN_AUDIT.to_csv(
        bin_audit_path,
        index=False,
    )


    print()
    print("=" * 72)
    print("DATA BEFORE / AFTER sPLOT: SLIDE DIAGNOSTIC")
    print("=" * 72)

    print(
        DATA_SPLOT_SLIDE_SUMMARY.to_string(
            index=False
        )
    )

    print()
    print(
        "Same candidates on both curves:",
        n_candidates,
    )

    print(
        "Before weighting integral:",
        float(np.sum(before_weights)),
    )

    print(
        "After weighting integral :",
        sum_sweights,
    )

    print(
        "Negative sWeight fraction:",
        f"{100.0 * negative_fraction:.3f}%",
    )

    print(
        "sWeight effective entries:",
        f"{neff_sweights:.2f}",
    )

    print()
    print(
        "[audit] wrote",
        summary_path,
    )

    print(
        "[audit] wrote",
        bin_audit_path,
    )

    print("=" * 72)


else:

    print(
        "[slides] data before/after "
        "sPlot diagnostic disabled"
    )


## 3. SPS/DPS Monte Carlo and DPS $p_T$ stitching
¶
The four physics components are kept separately: DPS-$c\bar c$, DPS-$b\bar b$, SPS-$c\bar c$, and SPS-$b\bar b$. DPS $p_T(J/\psi)$ stitching is rederived inside the finite-simulation replicas.

In [ ]:
SAMPLE_INFO = {
    "DPS-ccbar-25to30": {
        "component": "DPS-ccbar", "kind": "DPS", "flavour": "ccbar",
        "pt_low": 25.0, "pt_high": 30.0,
        "root_file": "dps_ccbar_9to30_usual.root", "csv_file": "data_dps_ccbar_25to30.csv",
        "nevt": 4895653*(4.9/4.12), "xsec_barn": 373.3e-12,
    },
    "DPS-ccbar-30to50": {
        "component": "DPS-ccbar", "kind": "DPS", "flavour": "ccbar",
        "pt_low": 30.0, "pt_high": 50.0,
        "root_file": "dps_ccbar_30to50_usual.root", "csv_file": "data_dps_ccbar_30to50.csv",
        "nevt": 1095994*(1.0/1.3), "xsec_barn": 0.1351e-12,
    },
    "DPS-ccbar-50to100": {
        "component": "DPS-ccbar", "kind": "DPS", "flavour": "ccbar",
        "pt_low": 50.0, "pt_high": 100.0,
        "root_file": "dps_ccbar_50to100_usual.root", "csv_file": "data_dps_ccbar_50to100.csv",
        "nevt": 477085*(4.77/5.5), "xsec_barn": 0.001466e-12,
    },
    "DPS-bbbar-25to30": {
        "component": "DPS-bbbar", "kind": "DPS", "flavour": "bbbar",
        "pt_low": 25.0, "pt_high": 30.0,
        "root_file": "dps_bbbar_9to30_usual.root", "csv_file": "data_dps_bbbar_25to30.csv",
        "nevt": 502277*(502.28/752.0), "xsec_barn": 0.625e-12,
    },
    "DPS-bbbar-30to50": {
        "component": "DPS-bbbar", "kind": "DPS", "flavour": "bbbar",
        "pt_low": 30.0, "pt_high": 50.0,
        "root_file": "dps_bbbar_30to50_usual.root", "csv_file": "data_dps_bbbar_30to50.csv",
        "nevt": 239513*(239.51/233.0), "xsec_barn": 0.000702e-12,
    },
    "DPS-bbbar-50to100": {
        "component": "DPS-bbbar", "kind": "DPS", "flavour": "bbbar",
        "pt_low": 50.0, "pt_high": 100.0,
        "root_file": "dps_bbbar_50to100_usual.root", "csv_file": "data_dps_bbbar_50to100.csv",
        "nevt": 105746*(105.75/130.0), "xsec_barn": 0.00000771e-12,
    },
    "SPS-ccbar-25to100": {
        "component": "SPS-ccbar", "kind": "SPS", "flavour": "ccbar",
        "pt_low": 25.0, "pt_high": 100.0,
        "root_file": "sps_3fs_ccbar_25to100_usual.root", "csv_file": "data_sps_3fs_ccbar.csv",
        "nevt": 554631.676300578, "xsec_barn": 0.008139e-12,
    },
    "SPS-bbbar-25to100": {
        "component": "SPS-bbbar", "kind": "SPS", "flavour": "bbbar",
        "pt_low": 25.0, "pt_high": 100.0,
        "root_file": "sps_3fs_bbbar_25to100_usual.root", "csv_file": "data_sps_3fs_bbbar.csv",
        "nevt": 99168.64, "xsec_barn": 0.00570e-12,
    },
}

FIXED_DPS_PT_STITCH = {
    "DPS-ccbar-25to30": 1.0,
    "DPS-ccbar-30to50": 17.207198481273394,
    "DPS-ccbar-50to100": 276.6850750445472,
    "DPS-bbbar-25to30": 1.0,
    "DPS-bbbar-30to50": 12.015539297981329,
    "DPS-bbbar-50to100": 143.82641208838336,
    "SPS-ccbar-25to100": 1.0,
    "SPS-bbbar-25to100": 1.0,
}
STITCH_MODE = "derive_from_jpsi_pt"
STITCH_EDGE_WIDTH_GEV = 1.0


def standardize_csv_columns(frame):
    rename = {}
    for logical, candidates in BRANCH_CANDIDATES.items():
        for candidate in candidates:
            if candidate in frame.columns:
                rename[candidate] = logical
                break
    out = frame.rename(columns=rename).copy()
    if "tree_weight" not in out.columns:
        out["tree_weight"] = 1.0
    return out


def mc_fiducial_mask(frame):
    return (
        (frame["jpsi_pt"] > JPSI_PT_MIN) & (frame["jpsi_pt"] < JPSI_PT_MAX)
        & (np.abs(frame["jpsi_rap"]) < JPSI_ABSY_MAX)
        & (frame["dstar_pt"] > DSTAR_PT_MIN) & (frame["dstar_pt"] < DSTAR_PT_MAX)
        & (np.abs(frame["dstar_rap"]) < DSTAR_ABSY_MAX)
    )


def load_one_mc_sample(sample_name, info):
    root_path = find_file(info["root_file"])
    csv_path = find_file(info["csv_file"])
    logical = ["delta_phi", "delta_y", "jpsi_pt", "jpsi_rap", "dstar_pt", "dstar_rap"]
    if root_path is not None:
        frame, _ = read_logical_dataframe(
            root_path, logical, fixed={"tree_weight": MC_TREE_WEIGHT_BRANCH}
        )
        input_path = root_path
    elif csv_path is not None:
        frame = standardize_csv_columns(pd.read_csv(csv_path))
        input_path = csv_path
    else:
        raise FileNotFoundError(f"Missing ROOT/CSV input for {sample_name}")

    missing = [name for name in logical + ["tree_weight"] if name not in frame]
    if missing:
        raise KeyError(f"{sample_name} missing {missing} in {input_path}")

    frame["delta_phi"] = fold_delta_phi(frame["delta_phi"])
    frame["delta_y"] = np.abs(frame["delta_y"].to_numpy(float))
    before = len(frame)
    frame = frame.loc[mc_fiducial_mask(frame)].reset_index(drop=True)
    frame["sample"] = sample_name
    frame["component"] = info["component"]
    frame["kind"] = info["kind"]
    frame["flavour"] = info["flavour"]
    frame["pt_low"] = info["pt_low"]
    frame["pt_high"] = info["pt_high"]
    base = info["xsec_barn"]*(LUMI_RUN2_FB_INV/1.0e-15)/info["nevt"]
    frame["mc_weight_xsec_only"] = frame["tree_weight"].to_numpy(float)*base
    frame["pt_stitch_scale"] = 1.0
    frame["mc_weight"] = frame["mc_weight_xsec_only"]
    print(f"[MC] {sample_name:22s} fiducial={len(frame):7d}/{before:7d} input={input_path}")
    return frame


def apply_fixed_stitching(mc):
    out = mc.copy()
    out["pt_stitch_scale"] = out["sample"].map(FIXED_DPS_PT_STITCH).fillna(1.0)
    out["mc_weight"] = out["mc_weight_xsec_only"]*out["pt_stitch_scale"]
    table = pd.DataFrame([
        {"component": SAMPLE_INFO[name]["component"], "sample": name,
         "boundary": np.nan, "pt_stitch_scale": scale,
         "pt_stitch_rel_unc": np.nan, "method": "fixed"}
        for name, scale in FIXED_DPS_PT_STITCH.items()
    ])
    return out, table


def derive_stitching(mc, edge_width=1.0):
    out = mc.copy()
    records = []
    for component in ["DPS-ccbar", "DPS-bbbar"]:
        ordered = sorted(
            [name for name, info in SAMPLE_INFO.items() if info["component"] == component],
            key=lambda name: SAMPLE_INFO[name]["pt_low"],
        )
        scales = {ordered[0]: 1.0}
        rel_unc = {ordered[0]: 0.0}
        records.append({"component": component, "sample": ordered[0], "boundary": np.nan,
                        "pt_stitch_scale": 1.0, "pt_stitch_rel_unc": 0.0, "method": "anchor"})
        for left_name, right_name in zip(ordered[:-1], ordered[1:]):
            boundary = SAMPLE_INFO[right_name]["pt_low"]
            left = out[out["sample"] == left_name]
            right = out[out["sample"] == right_name]
            lm = (left["jpsi_pt"] >= boundary-edge_width) & (left["jpsi_pt"] < boundary)
            rm = (right["jpsi_pt"] >= boundary) & (right["jpsi_pt"] < boundary+edge_width)
            lw = left.loc[lm, "mc_weight_xsec_only"].to_numpy(float)
            rw = right.loc[rm, "mc_weight_xsec_only"].to_numpy(float)
            lbase = float(np.sum(lw))
            rsum = float(np.sum(rw))
            lsum = lbase*scales[left_name]
            if rsum > 0.0 and np.isfinite(lsum) and lbase != 0.0:
                scale = lsum/rsum
                method = "boundary_density_match"
                lstat = math.sqrt(float(np.sum(lw**2)))/abs(lbase)
                rstat = math.sqrt(float(np.sum(rw**2)))/abs(rsum)
                uncertainty = math.sqrt(rel_unc[left_name]**2 + lstat**2 + rstat**2)
            else:
                scale = FIXED_DPS_PT_STITCH[right_name]
                method = "fallback_fixed"
                uncertainty = np.nan
            scales[right_name] = scale
            rel_unc[right_name] = uncertainty
            records.append({"component": component, "sample": right_name, "boundary": boundary,
                            "pt_stitch_scale": scale, "pt_stitch_rel_unc": uncertainty,
                            "method": method})
        for sample_name, scale in scales.items():
            out.loc[out["sample"] == sample_name, "pt_stitch_scale"] = scale
    out.loc[out["kind"] == "SPS", "pt_stitch_scale"] = 1.0
    out["mc_weight"] = out["mc_weight_xsec_only"]*out["pt_stitch_scale"]
    return out, pd.DataFrame(records)


mc_raw = pd.concat(
    [load_one_mc_sample(name, info) for name, info in SAMPLE_INFO.items()],
    ignore_index=True, sort=False,
)
if STITCH_MODE == "derive_from_jpsi_pt":
    mc_df, stitch_table = derive_stitching(mc_raw, STITCH_EDGE_WIDTH_GEV)
else:
    mc_df, stitch_table = apply_fixed_stitching(mc_raw)

stitch_audit = (
    mc_df.groupby(["component", "sample"], sort=False)["mc_weight"]
    .agg(sumw="sum", sumw2=lambda values: float(np.sum(np.asarray(values, float)**2)))
    .reset_index()
)
stitch_audit["effective_entries"] = safe_divide(
    stitch_audit["sumw"]**2, stitch_audit["sumw2"], default=0.0
)
stitch_audit["component_fraction"] = stitch_audit["sumw"] / stitch_audit.groupby(
    "component"
)["sumw"].transform("sum")

print("\nNominal DPS stitching:")
for _, row in stitch_table.iterrows():
    if str(row.get("component","")).startswith("DPS"):
        unc = row.get("pt_stitch_rel_unc", np.nan)
        suffix = f" ± {100*unc:.1f}% (stat)" if np.isfinite(unc) else ""
        print(f"  {row['sample']:22s}: {row['pt_stitch_scale']:.6g}{suffix}")
stitch_table.to_csv(OUTDIR/"dps_pt_stitching_used.csv", index=False)
stitch_audit.to_csv(OUTDIR/"dps_pt_stitching_audit.csv", index=False)


## 4. Efficiency maps

Eight final ROOT files are mandatory: four SPS/DPS components for 2017 and 2018. The six nominal factors are evaluated event-by-event **within each year first**, and the complete yearly products are then luminosity-weighted.

The nominal product contains exactly one association efficiency, `eff_asso_pt`. `eff_asso_rap` is loaded only as an alternative diagnostic representation.

The final-map contract is checked in this notebook: compatible axes, finite central values, stored asymmetric errors, stored $N_{\mathrm{eff}}$, and $N_{\mathrm{eff}}\ge25$ for every map entering the nominal correction.

In [ ]:
class Hist2DMap:
    def __init__(self, values, xedges, yedges, name):
        self.values = np.asarray(values, dtype=float)
        self.xedges = np.asarray(xedges, dtype=float)
        self.yedges = np.asarray(yedges, dtype=float)
        self.name = str(name)
        if self.values.shape != (len(self.xedges)-1, len(self.yedges)-1):
            raise ValueError(f"Bad map shape for {name}: {self.values.shape}")

    @classmethod
    def from_uproot(cls, obj, name):
        values, xedges, yedges = obj.to_numpy(flow=False)
        return cls(values, xedges, yedges, name)

    def compatible(self, other):
        return np.allclose(self.xedges, other.xedges) and np.allclose(self.yedges, other.yedges)

    def bin_indices(self, x, y):
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        ix = np.searchsorted(self.xedges, x, side="right") - 1
        iy = np.searchsorted(self.yedges, y, side="right") - 1
        ix = np.where(x == self.xedges[-1], self.values.shape[0]-1, ix)
        iy = np.where(y == self.yedges[-1], self.values.shape[1]-1, iy)
        valid = (
            np.isfinite(x) & np.isfinite(y)
            & (ix >= 0) & (ix < self.values.shape[0])
            & (iy >= 0) & (iy < self.values.shape[1])
        )
        if not np.all(valid):
            raise ValueError(
                f"[eff] {self.name}: {np.count_nonzero(~valid)}/{valid.size} entries outside map coverage"
            )
        return ix.astype(np.int32), iy.astype(np.int32)

    def eval(self, x, y, values_override=None):
        ix, iy = self.bin_indices(x, y)
        values = self.values if values_override is None else np.asarray(values_override, dtype=float)
        return values[ix, iy]


EFFICIENCY_SEARCH_DIRS = [
    EFFICIENCY_DIR,
    EFFICIENCY_DIR/"final",
    DATA_DIR/"Data"/"efficiency",
    DATA_DIR/"Data"/"efficiency"/"final",
    Path.cwd()/"efficiency",
    Path.cwd()/"output"/"efficiency",
]
EFFICIENCY_SOURCE_CACHE = {}
EFFICIENCY_FILE_AUDIT = []
EFFICIENCY_BIN_AUDIT = []


def efficiency_filename(source, year):
    return f"efficiencies_{source}_differential_final_jpsi_{year}.root"


def find_efficiency_file(source, year):
    filename = efficiency_filename(source, year)
    for root in EFFICIENCY_SEARCH_DIRS:
        root = Path(root).expanduser()
        for candidate in (root/filename, root/"final"/filename):
            if candidate.is_file():
                return candidate.resolve()
    for root in EFFICIENCY_SEARCH_DIRS:
        root = Path(root).expanduser()
        if root.is_dir():
            matches = sorted(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    return None



def validate_efficiency_campaign():
    """Require the complete validated 2017+2018 response for all four components."""
    sources = sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values()))
    missing = [
        f"{source}:{year}"
        for source in sources
        for year in EFFICIENCY_YEARS
        if find_efficiency_file(source, year) is None
    ]
    if missing:
        raise FileNotFoundError(
            "Mandatory 2017+2018 efficiency maps are missing: "
            + ", ".join(missing)
        )

    print("[eff] data period: full Run 2")
    print("[eff] response years:", list(EFFICIENCY_YEARS))
    print(
        "[eff] response luminosity:",
        f"{LUMI_EFFICIENCY_RESPONSE_FB_INV:.2f} fb^-1",
    )
    print("[eff] 2016 response pending: yes")
    return True


EFFICIENCY_CAMPAIGN_COMPLETE_1718 = validate_efficiency_campaign()


def _read_hist_if_present(root_file, key):
    if key not in root_file:
        return None
    obj = root_file[key]
    out = obj.to_numpy(flow=False)
    if len(out) != 3:
        return None
    return Hist2DMap.from_uproot(obj, key)


def load_efficiency_file(path):
    result = {}
    with uproot.open(path) as root_file:
        keys = {key.split(";")[0] for key in root_file.keys(recursive=True)}
        for factor in EFFICIENCY_REQUIRED:
            central = _read_hist_if_present(root_file, factor)
            if central is None:
                continue
            info = {"central": central}
            for label, key in [
                ("err_up", f"{factor}_err_up"),
                ("err_down", f"{factor}_err_down"),
                ("n_eff", f"{factor}_n_eff"),
            ]:
                h = _read_hist_if_present(root_file, key)
                if h is not None:
                    info[label] = h
            raw = {}
            for suffix in ("num_sumw", "num_sumw2", "den_sumw", "den_sumw2"):
                key = f"raw/{factor}_{suffix}"
                h = _read_hist_if_present(root_file, key)
                if h is not None:
                    raw[suffix] = h
            info["raw"] = raw
            result[factor] = info
    return result


def _weighted_ratio_variance_from_raw(info):
    raw = info.get("raw", {})
    needed = {"num_sumw", "num_sumw2", "den_sumw", "den_sumw2"}
    if not needed.issubset(raw):
        return None
    e = info["central"].values
    D = raw["den_sumw"].values
    Qn = raw["num_sumw2"].values
    Qd = raw["den_sumw2"].values
    # Delta-method variance for a weighted subset ratio N/D.
    # numerator events are a subset of denominator events.
    qfail = np.maximum(Qd-Qn, 0.0)
    var = safe_divide((1.0-e)**2*Qn + e**2*qfail, D**2, default=np.nan)
    return var


def load_efficiency_source(source):
    if source in EFFICIENCY_SOURCE_CACHE:
        return EFFICIENCY_SOURCE_CACHE[source]
    years = {}
    for year in EFFICIENCY_YEARS:
        path = find_efficiency_file(source, year)
        if path is None:
            raise FileNotFoundError(
                f"Mandatory efficiency ROOT missing: {source}, {year}"
            )
        maps = load_efficiency_file(path)
        missing = [name for name in EFFICIENCY_REQUIRED if name not in maps]
        if missing:
            raise KeyError(
                f"{source}/{year}: missing mandatory efficiency maps {missing}"
            )
        EFFICIENCY_FILE_AUDIT.append({
            "source": source, "year": year, "file_found": True,
            "path": str(path), "missing": "",
        })
        years[year] = {"path": path, "maps": maps}
        print(f"[eff] loaded {source}, {year}: {path}")

        for factor, info in maps.items():
            raw_var = _weighted_ratio_variance_from_raw(info)
            enc = None
            if "err_up" in info and "err_down" in info:
                enc = 0.5*(info["err_up"].values + info["err_down"].values)
            for index in np.ndindex(info["central"].values.shape):
                row = {
                    "source": source,
                    "year": year,
                    "factor": factor,
                    "bin": str(index),
                    "eff": float(info["central"].values[index]),
                    "n_eff": float(info["n_eff"].values[index]) if "n_eff" in info else np.nan,
                    "encoded_sigma_sym": float(enc[index]) if enc is not None else np.nan,
                    "raw_delta_sigma": float(np.sqrt(max(raw_var[index], 0.0))) if raw_var is not None and np.isfinite(raw_var[index]) else np.nan,
                }
                EFFICIENCY_BIN_AUDIT.append(row)

    if set(years) != set(EFFICIENCY_YEARS):
        raise RuntimeError(
            f"{source}: loaded years {sorted(years)} do not match "
            f"required years {list(EFFICIENCY_YEARS)}"
        )
    EFFICIENCY_SOURCE_CACHE[source] = years
    return years


def factor_coordinates(frame, factor):
    jpt = frame["jpsi_pt"].to_numpy(float)
    jy = np.abs(frame["jpsi_rap"].to_numpy(float))
    dpt = frame["dstar_pt"].to_numpy(float)
    dy = np.abs(frame["dstar_rap"].to_numpy(float))
    if factor in {"acc_dimu", "eff_cuts_dimu", "eff_trigger"}:
        return jpt, jy
    if factor in {"acc_dstar", "eff_cuts_dstar"}:
        return dpt, dy
    if factor == "eff_asso_pt":
        return jpt, dpt
    if factor == "eff_asso_rap":
        return jy, dy
    raise KeyError(factor)


def sample_factor_map(info, z, factor):
    central = info["central"].values
    up = info.get("err_up")
    down = info.get("err_down")
    if up is None or down is None:
        return central.copy()

    z = np.asarray(z, dtype=float)
    if z.shape != central.shape:
        raise ValueError("Efficiency-replica z shape mismatch")

    varied = central + np.where(z >= 0.0, z*up.values, z*down.values)

    # acc_dimu and acc_dstar are weighted response ratios, not strict
    # binomial probabilities. Values slightly above one are legitimate.
    if factor in {"acc_dimu", "acc_dstar"}:
        return np.maximum(varied, EFFICIENCY_FLOOR)

    return np.clip(varied, EFFICIENCY_FLOOR, 1.0)


def draw_efficiency_z(rng):
    replica = {}
    for source in sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values())):
        years = load_efficiency_source(source)
        for year, bundle in years.items():
            for factor in EFFICIENCY_FACTORS_USED:
                if factor not in bundle["maps"]:
                    continue
                shape = bundle["maps"][factor]["central"].values.shape
                replica[(source, year, factor)] = rng.normal(size=shape)
    return replica


def evaluate_efficiency_product_year(
    frame,
    source,
    year,
    z_replica=None,
    association_mode=None,
):
    """Evaluate the six-factor efficiency product for one response year."""
    year = str(year)
    if year not in EFFICIENCY_YEARS:
        raise ValueError(
            f"Efficiency year {year} is outside {list(EFFICIENCY_YEARS)}"
        )

    years = load_efficiency_source(source)
    bundle = years[year]
    maps = bundle["maps"]

    association_mode = (
        ASSOCIATION_EFF_MODE if association_mode is None else association_mode
    )
    factors = [
        "acc_dimu",
        "acc_dstar",
        "eff_cuts_dimu",
        "eff_cuts_dstar",
        "eff_trigger",
        "eff_asso_pt" if association_mode == "pt" else "eff_asso_rap",
    ]

    total_eff = np.ones(len(frame), dtype=float)
    for factor in factors:
        if factor not in maps:
            raise KeyError(f"Missing mandatory map {source}/{year}/{factor}")
        info = maps[factor]
        x, y = factor_coordinates(frame, factor)
        values = info["central"].values
        key = (source, year, factor)
        if z_replica is not None and key in z_replica:
            values = sample_factor_map(info, z_replica[key], factor)
        total_eff *= info["central"].eval(x, y, values_override=values)

    if np.any(~np.isfinite(total_eff)) or np.any(total_eff <= 0.0):
        raise ValueError(f"Invalid efficiency product for {source}/{year}")
    return np.maximum(total_eff, EFFICIENCY_FLOOR)


def evaluate_efficiency_product(
    frame,
    source,
    z_replica=None,
    association_mode=None,
):
    """Luminosity-weighted 2017+2018 detector response for full Run-2 inputs.

    The product is formed separately within each year and the two yearly
    products are then averaged.  This avoids multiplying luminosity-averaged
    factors, which is not equivalent.
    """
    weighted = np.zeros(len(frame), dtype=float)
    total_lumi = 0.0

    for year in EFFICIENCY_YEARS:
        lumi = float(EFFICIENCY_YEAR_WEIGHTS[year])
        weighted += lumi * evaluate_efficiency_product_year(
            frame,
            source,
            year,
            z_replica=z_replica,
            association_mode=association_mode,
        )
        total_lumi += lumi

    if total_lumi <= 0.0:
        raise RuntimeError("Non-positive efficiency-response luminosity")

    combined = weighted / total_lumi
    if np.any(~np.isfinite(combined)) or np.any(combined <= 0.0):
        raise ValueError(f"Invalid combined efficiency product for {source}")
    return np.maximum(combined, EFFICIENCY_FLOOR)


def validate_loaded_efficiency_contract():
    """Hard gate on axes, finite values and N_eff for the nominal maps."""
    rows = []

    for source in sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values())):
        years = load_efficiency_source(source)
        ref_maps = years[EFFICIENCY_YEARS[0]]["maps"]

        for year in EFFICIENCY_YEARS:
            maps = years[year]["maps"]

            for factor in EFFICIENCY_REQUIRED:
                info = maps[factor]
                central = info["central"]

                if not np.all(np.isfinite(central.values)):
                    raise ValueError(f"{source}/{year}/{factor}: non-finite central values")
                if np.any(central.values <= 0.0):
                    raise ValueError(f"{source}/{year}/{factor}: non-positive central values")

                if factor not in {"acc_dimu", "acc_dstar"} and np.any(central.values > 1.0 + 1e-10):
                    raise ValueError(
                        f"{source}/{year}/{factor}: strict efficiency above one"
                    )

                for label in ("err_up", "err_down", "n_eff"):
                    if label not in info:
                        raise KeyError(f"{source}/{year}/{factor}: missing {label}")
                    if not central.compatible(info[label]):
                        raise ValueError(
                            f"{source}/{year}/{factor}: {label} axes differ from central"
                        )

                if year != EFFICIENCY_YEARS[0]:
                    ref = ref_maps[factor]["central"]
                    if not ref.compatible(central):
                        raise ValueError(
                            f"{source}/{factor}: 2017/2018 axes are incompatible"
                        )

                min_neff = float(np.nanmin(info["n_eff"].values))
                rows.append({
                    "source": source,
                    "year": year,
                    "factor": factor,
                    "min_neff": min_neff,
                })

                if factor in EFFICIENCY_FACTORS_USED and min_neff < MIN_EFFICIENCY_MAP_NEFF:
                    raise RuntimeError(
                        f"{source}/{year}/{factor}: min N_eff={min_neff:.3f} "
                        f"< {MIN_EFFICIENCY_MAP_NEFF}"
                    )

    audit = pd.DataFrame(rows)
    audit.to_csv(OUTDIR/"efficiency_final_contract_audit.csv", index=False)
    return audit


# Force loading now so the audit is produced before the fit.
for _source in sorted(set(EFFICIENCY_SOURCE_FOR_COMPONENT.values())):
    load_efficiency_source(_source)

EFFICIENCY_FILE_AUDIT_DF = pd.DataFrame(EFFICIENCY_FILE_AUDIT)
EFFICIENCY_BIN_AUDIT_DF = pd.DataFrame(EFFICIENCY_BIN_AUDIT)
found = int(EFFICIENCY_FILE_AUDIT_DF.get("file_found", pd.Series(dtype=bool)).fillna(False).sum()) if len(EFFICIENCY_FILE_AUDIT_DF) else 0
print(f"[eff] loaded {found} efficiency ROOT file entries")
if len(EFFICIENCY_BIN_AUDIT_DF):
    finite_neff = EFFICIENCY_BIN_AUDIT_DF["n_eff"].to_numpy(float)
    finite_neff = finite_neff[np.isfinite(finite_neff)]
    if len(finite_neff):
        print(f"[eff] encoded map-bin N_eff range: {np.min(finite_neff):.2f} -- {np.max(finite_neff):.2f}")
EFFICIENCY_FILE_AUDIT_DF.to_csv(OUTDIR/"efficiency_file_audit.csv", index=False)
EFFICIENCY_BIN_AUDIT_DF.to_csv(OUTDIR/"efficiency_bin_statistical_audit.csv", index=False)

if found != 8:
    raise RuntimeError(f"Expected exactly 8 final efficiency ROOT entries, found {found}")

EFFICIENCY_CONTRACT_AUDIT = validate_loaded_efficiency_contract()
print(
    "[eff] final 2017+2018 contract: PASS; "
    f"minimum nominal-map N_eff={EFFICIENCY_CONTRACT_AUDIT.loc[EFFICIENCY_CONTRACT_AUDIT['factor'].isin(EFFICIENCY_FACTORS_USED), 'min_neff'].min():.2f}"
)


In [ ]:
# ================================================================
# INDEPENDENT EFFICIENCY CLOSURE AUDIT
# ================================================================
# Diagnostic only: this cell does not modify efficiency maps, templates,
# covariance matrices, fits, fDPS, sigma_eff, or reportability.

_closure_required_components = [
    "SPS-ccbar",
    "SPS-bbbar",
    "DPS-ccbar",
    "DPS-bbbar",
]

_closure_efficiency_paths = sorted({
    str(Path(path).expanduser().resolve())
    for path in EFFICIENCY_FILE_AUDIT_DF.get("path", pd.Series(dtype=str)).dropna()
})

_closure_efficiency_root = Path(EFFICIENCY_DIR).expanduser().resolve().parent
_closure_map_construction_products = sorted(
    str(path.resolve())
    for path in (_closure_efficiency_root / "cache").glob("*_merged.coffea")
    if path.is_file()
)
_closure_map_construction_manifests = sorted(
    str(path.resolve())
    for path in (_closure_efficiency_root / "input_manifests").glob("*.txt")
    if path.is_file()
)

_closure_nominal_template_inputs = []
for _closure_sample_name, _closure_info in SAMPLE_INFO.items():
    _closure_input = (
        find_file(_closure_info["root_file"])
        or find_file(_closure_info["csv_file"])
    )
    if _closure_input is not None:
        _closure_nominal_template_inputs.append(str(Path(_closure_input).resolve()))
_closure_nominal_template_inputs = sorted(set(_closure_nominal_template_inputs))

_closure_search_locations = [
    DATA_DIR / "Data" / "independent_efficiency_closure",
    DATA_DIR / "Data" / "efficiency_closure_test",
    _closure_efficiency_root / "independent_closure",
    _closure_efficiency_root / "closure_test",
]
_closure_declared_path = os.environ.get(
    "JPSIDSTAR_INDEPENDENT_EFFICIENCY_CLOSURE_SAMPLE",
    "",
).strip()
if _closure_declared_path:
    _closure_search_locations.append(Path(_closure_declared_path).expanduser())

_closure_discovered_candidates = sorted({
    str(path.resolve())
    for path in _closure_search_locations
    if path.exists()
})

# The existing Coffea caches/manifests are the map-construction sample, not a
# holdout. The nominal MC inputs are selected-level template inputs and do not
# provide an independent truth/reco pair. No approved independent-sample
# manifest or event-disjoint truth/test input is present at the searched paths.
_closure_message = (
    "Independent efficiency closure not evaluated: no demonstrably independent "
    "truth/test sample was found. No physics systematic has been invented."
)

INDEPENDENT_EFFICIENCY_CLOSURE_RESULT = {
    "status": "PENDING_INDEPENDENT_SAMPLE",
    "sample": None,
    "path": None,
    "provenance": {
        "efficiency_map_inputs": _closure_efficiency_paths,
        "map_construction_caches": _closure_map_construction_products,
        "map_construction_manifests": _closure_map_construction_manifests,
        "nominal_selected_template_inputs": _closure_nominal_template_inputs,
        "searched_locations": [str(path) for path in _closure_search_locations],
        "discovered_candidate_locations": _closure_discovered_candidates,
    },
    "independent": False,
    "components_tested": [],
    "required_components": _closure_required_components,
    "integrated_closure_values": {},
    "normalized_shape_closure_metric": {},
    "notes": [
        "The final efficiency maps and merged Coffea caches use the map-construction events.",
        "The nominal SPS/DPS MC inputs are selected-level template samples, not an independent truth/reco closure sample.",
        "2017 versus 2018 is not treated as an independent closure.",
        "No residual has been converted into a covariance or physics systematic.",
        _closure_message,
    ],
}

INDEPENDENT_EFFICIENCY_CLOSURE_TABLE = pd.DataFrame([
    {
        "status": INDEPENDENT_EFFICIENCY_CLOSURE_RESULT["status"],
        "independent": False,
        "component": None,
        "sample": None,
        "path": None,
        "generator_fiducial_yield": np.nan,
        "efficiency_corrected_selected_yield": np.nan,
        "integrated_closure_ratio": np.nan,
        "max_abs_normalized_shape_deviation": np.nan,
        "quadratic_residual": np.nan,
        "note": _closure_message,
    }
])

if "OUTDIR" in globals() and OUTDIR is not None:
    _closure_outdir = Path(OUTDIR)
    INDEPENDENT_EFFICIENCY_CLOSURE_TABLE.to_csv(
        _closure_outdir / "independent_efficiency_closure_audit.csv",
        index=False,
    )
    with (_closure_outdir / "independent_efficiency_closure_audit.json").open(
        "w",
        encoding="utf-8",
    ) as _closure_handle:
        json.dump(
            INDEPENDENT_EFFICIENCY_CLOSURE_RESULT,
            _closure_handle,
            indent=2,
            sort_keys=True,
        )

print(_closure_message)
display(INDEPENDENT_EFFICIENCY_CLOSURE_TABLE)


## 5. Joint finite-simulation response ensemble
¶
One event bootstrap is shared by template shapes, stitching and, in the one-parameter mode, the MC-derived $c\bar c/b\bar b$ composition. Efficiency-map fluctuations are applied within the same replica.
In the three-parameter mode, $f_{cc}^{\mathrm{SPS}}$ and $f_{cc}^{\mathrm{DPS}}$ are physics parameters of the fit rather than MC-derived composition numbers.

In [ ]:
COMPONENTS = ["DPS-ccbar", "DPS-bbbar", "SPS-ccbar", "SPS-bbbar"]
COMPONENT_INDEX = {name: i for i, name in enumerate(COMPONENTS)}


def make_spec(name):
    edges = BINNING[name]
    return {
        "name": name,
        "dphi_edges": np.asarray(edges["dphi"], dtype=float),
        "dy_edges": np.asarray(edges["dy"], dtype=float),
    }


def assign_bins(frame, spec):
    x = frame["delta_phi"].to_numpy(float)
    y = frame["delta_y"].to_numpy(float)
    xe = spec["dphi_edges"]
    ye = spec["dy_edges"]
    ix = np.searchsorted(xe, x, side="right") - 1
    iy = np.searchsorted(ye, y, side="right") - 1
    ix = np.where(x == xe[-1], len(xe)-2, ix)
    iy = np.where(y == ye[-1], len(ye)-2, iy)
    valid = (
        np.isfinite(x) & np.isfinite(y)
        & (ix >= 0) & (ix < len(xe)-1)
        & (iy >= 0) & (iy < len(ye)-1)
    )
    ny = len(ye)-1
    flat = ix*ny + iy
    return flat.astype(np.int32), valid, (len(xe)-1)*ny


def helmert_contrast(n):
    if n < 2:
        raise ValueError("Need at least two bins")
    H = np.zeros((n-1, n), dtype=float)
    for i in range(1, n):
        H[i-1, :i] = 1.0/math.sqrt(i*(i+1.0))
        H[i-1, i] = -i/math.sqrt(i*(i+1.0))
    return H


def data_selected_contributions(frame):
    return frame["sweight"].to_numpy(float)/(LUMI_RUN2_PB_INV*BRANCHING_PRODUCT)


def central_stitch_scales():
    table = (
        mc_df[["sample", "pt_stitch_scale"]]
        .drop_duplicates("sample")
        .set_index("sample")["pt_stitch_scale"]
    )
    return {str(k): float(v) for k, v in table.items()}


CENTRAL_STITCH_SCALES = central_stitch_scales()
MC_SAMPLE_CACHE = {
    sample: mc_raw.loc[mc_raw["sample"] == sample].reset_index(drop=True)
    for sample in SAMPLE_INFO
}


def derive_replica_stitch_scales(multipliers):
    if STITCH_MODE != "derive_from_jpsi_pt":
        return CENTRAL_STITCH_SCALES.copy()
    scales = {}
    for component in ["DPS-ccbar", "DPS-bbbar"]:
        ordered = sorted(
            [name for name, info in SAMPLE_INFO.items() if info["component"] == component],
            key=lambda name: SAMPLE_INFO[name]["pt_low"],
        )
        scales[ordered[0]] = 1.0
        for left_name, right_name in zip(ordered[:-1], ordered[1:]):
            boundary = SAMPLE_INFO[right_name]["pt_low"]
            left = MC_SAMPLE_CACHE[left_name]
            right = MC_SAMPLE_CACHE[right_name]
            lm = (left["jpsi_pt"].to_numpy(float) >= boundary-STITCH_EDGE_WIDTH_GEV) & (left["jpsi_pt"].to_numpy(float) < boundary)
            rm = (right["jpsi_pt"].to_numpy(float) >= boundary) & (right["jpsi_pt"].to_numpy(float) < boundary+STITCH_EDGE_WIDTH_GEV)
            lw = left["mc_weight_xsec_only"].to_numpy(float) * multipliers[left_name]
            rw = right["mc_weight_xsec_only"].to_numpy(float) * multipliers[right_name]
            lsum = float(np.sum(lw[lm])) * scales[left_name]
            rsum = float(np.sum(rw[rm]))
            if rsum > 0.0 and np.isfinite(lsum):
                scales[right_name] = lsum/rsum
            else:
                scales[right_name] = CENTRAL_STITCH_SCALES[right_name]
    for sample, info in SAMPLE_INFO.items():
        if info["kind"] == "SPS":
            scales[sample] = 1.0
    return scales


def _component_from_histograms(selected_hist, fid_hist, fid_total):
    if fid_total <= 0.0 or not np.isfinite(fid_total):
        raise ValueError("Non-positive fiducial total")
    response = np.asarray(selected_hist, float)/fid_total
    fid_shape = np.asarray(fid_hist, float)/fid_total
    return response, fid_shape, float(fid_total)


def _build_component_states(problem, multipliers=None, stitch_scales=None, z_replica=None,
                            bootstrap_hist=True, region=None):
    spec = problem["spec"]
    region = CENTRAL_DPS_REGION if region is None else region
    n_bins = problem["n_bins"]
    selected_h = {c: np.zeros(n_bins) for c in COMPONENTS}
    fid_h = {c: np.zeros(n_bins) for c in COMPONENTS}
    fid_total = {c: 0.0 for c in COMPONENTS}
    fid_region = {c: 0.0 for c in COMPONENTS}
    selected_h2 = {c: np.zeros(n_bins) for c in COMPONENTS}

    if multipliers is None:
        multipliers = {sample: np.ones(len(frame)) for sample, frame in MC_SAMPLE_CACHE.items()}
    if stitch_scales is None:
        stitch_scales = CENTRAL_STITCH_SCALES

    for sample, frame in MC_SAMPLE_CACHE.items():
        component = SAMPLE_INFO[sample]["component"]
        idx, valid, _ = assign_bins(frame, spec)
        mult = multipliers[sample] if bootstrap_hist else np.ones(len(frame), dtype=float)
        selected = (
            frame["mc_weight_xsec_only"].to_numpy(float)
            * float(stitch_scales.get(sample, 1.0))
            * mult
            / (LUMI_RUN2_PB_INV*BRANCHING_PRODUCT)
        )
        eff = evaluate_efficiency_product(
            frame,
            EFFICIENCY_SOURCE_FOR_COMPONENT[component],
            z_replica=z_replica,
        )
        fid = selected/eff
        ii = idx[valid]
        sw = selected[valid]
        fw = fid[valid]
        selected_h[component] += np.bincount(ii, weights=sw, minlength=n_bins)
        selected_h2[component] += np.bincount(ii, weights=sw**2, minlength=n_bins)
        fid_h[component] += np.bincount(ii, weights=fw, minlength=n_bins)
        fid_total[component] += float(np.sum(fw))
        regmask = (
            valid
            & (frame["delta_y"].to_numpy(float) >= region["dy_min"])
            & (frame["delta_phi"].to_numpy(float) >= region["dphi_min"])
            & (frame["delta_phi"].to_numpy(float) <= region["dphi_max"])
        )
        fid_region[component] += float(np.sum(fid[regmask]))

    states = {}
    for component in COMPONENTS:
        response, fid_shape, total = _component_from_histograms(
            selected_h[component], fid_h[component], fid_total[component]
        )
        neff = safe_divide(selected_h[component]**2, selected_h2[component], default=0.0)
        states[component] = {
            "response": response,
            "fid_shape": fid_shape,
            "fid_total": total,
            "region_fraction": float(fid_region[component]/total),
            "neff": neff,
        }
    return states



# ----------------------------------------------------------------
# Nominal central 2D template estimator: exact binned HIST response
# ----------------------------------------------------------------

def build_profile_problem(name):
    spec = make_spec(name)
    data_idx, data_valid, n_bins = assign_bins(data_sw, spec)
    data_weight = data_selected_contributions(data_sw)
    data_hist = np.bincount(
        data_idx[data_valid], weights=data_weight[data_valid], minlength=n_bins
    ).astype(float)
    data_var = np.bincount(
        data_idx[data_valid], weights=data_weight[data_valid]**2, minlength=n_bins
    ).astype(float)
    data_total = float(np.sum(data_hist))
    if data_total <= 0.0:
        raise ValueError("Non-positive sWeighted data integral")
    data_shape = data_hist/data_total
    J = np.eye(n_bins)/data_total - data_hist[:, None]/data_total**2
    data_cov = J @ np.diag(data_var) @ J.T
    H = helmert_contrast(n_bins)
    problem = {
        "name": name,
        "spec": spec,
        "n_bins": n_bins,
        "H": H,
        "data_idx": data_idx,
        "data_valid": data_valid,
        "data_weight": data_weight,
        "data_hist": data_hist,
        "data_var": data_var,
        "data_shape": data_shape,
        "data_cov": 0.5*(data_cov+data_cov.T),
        "data_reduced": H @ data_shape,
    }
    problem["central_states"] = _build_component_states(problem)
    problem["min_neff"] = min(
        float(np.min(problem["central_states"][c]["neff"])) for c in COMPONENTS
    )
    return problem


def build_replica_bank(problem, n, mode="joint", seed=RANDOM_SEED+100):
    if n <= 0:
        return None
    rng = np.random.default_rng(seed)
    R = np.zeros((n, len(COMPONENTS), problem["n_bins"]), dtype=float)
    F = np.zeros_like(R)
    T = np.zeros((n, len(COMPONENTS)), dtype=float)
    G = np.zeros((n, len(COMPONENTS)), dtype=float)
    ok = np.zeros(n, dtype=bool)
    z_success = []

    iterator = range(n)
    if tqdm is not None:
        iterator = tqdm(iterator, desc=f"stat replicas {problem['name']}:{mode}")

    for ir in iterator:
        try:
            # One event bootstrap is shared by template, composition and stitching.
            draw_mult = mode in {"joint", "mc_template", "stitching"}
            multipliers = {
                sample: rng.poisson(1.0, size=len(frame)).astype(float)
                if draw_mult else np.ones(len(frame), dtype=float)
                for sample, frame in MC_SAMPLE_CACHE.items()
            }
            fluctuate_stitch = mode in {"joint", "stitching"}
            stitch_scales = (
                derive_replica_stitch_scales(multipliers)
                if fluctuate_stitch else CENTRAL_STITCH_SCALES
            )
            fluctuate_eff = mode in {"joint", "efficiency"}
            z_replica = draw_efficiency_z(rng) if fluctuate_eff else None
            bootstrap_hist = mode in {"joint", "mc_template"}
            states = _build_component_states(
                problem,
                multipliers=multipliers,
                stitch_scales=stitch_scales,
                z_replica=z_replica,
                bootstrap_hist=bootstrap_hist,
            )
            for component, ic in COMPONENT_INDEX.items():
                R[ir, ic] = states[component]["response"]
                F[ir, ic] = states[component]["fid_shape"]
                T[ir, ic] = states[component]["fid_total"]
                G[ir, ic] = states[component]["region_fraction"]
            ok[ir] = True
            z_success.append(z_replica)
        except Exception as exc:
            if ir < 5:
                warnings.warn(f"Replica {ir} failed ({mode}): {exc}")

    R, F, T, G = R[ok], F[ok], T[ok], G[ok]
    required = max(
        MIN_SUCCESSFUL_REPLICAS,
        int(math.ceil(MIN_MODEL_REPLICA_SUCCESS_FRACTION * int(n))),
    )
    if len(R) < required:
        raise RuntimeError(
            f"Only {len(R)}/{int(n)} successful {mode} replicas for "
            f"{problem['name']}; need at least {required}."
        )
    return {"response": R, "fid_shape": F, "fid_total": T, "region_fraction": G,
            "z_replica": z_success if mode in {"joint", "efficiency"} else [],
            "n": len(R), "mode": mode}


def _derived_composition(fid_total):
    arr = np.asarray(fid_total, dtype=float)
    squeeze = arr.ndim == 1
    if squeeze:
        arr = arr[None, :]
    ic_scc = COMPONENT_INDEX["SPS-ccbar"]
    ic_sbb = COMPONENT_INDEX["SPS-bbbar"]
    ic_dcc = COMPONENT_INDEX["DPS-ccbar"]
    ic_dbb = COMPONENT_INDEX["DPS-bbbar"]
    fcc_sps = safe_divide(
        arr[:, ic_scc], arr[:, ic_scc] + arr[:, ic_sbb], default=0.5
    )
    fcc_dps = safe_divide(
        arr[:, ic_dcc], arr[:, ic_dcc] + arr[:, ic_dbb], default=0.5
    )
    if squeeze:
        return float(fcc_sps[0]), float(fcc_dps[0])
    return fcc_sps, fcc_dps


def mixture_from_arrays(
    response, fid_shape, fid_total, f_dps,
    f_cc_sps=None, f_cc_dps=None,
):
    """Build SPS/DPS mixture in fiducial-fraction parameterization.

    In 1-parameter mode the ccbar fractions are inferred from the fiducial MC
    totals separately for every finite-simulation replica, reproducing the
    current FINAL behavior.

    In 3-parameter mode explicit f_cc_sps and f_cc_dps values are supplied by
    the fit. They are then held at the tested parameter point while the
    component response shapes fluctuate across finite-simulation replicas.
    """
    f = float(f_dps)
    ic_scc = COMPONENT_INDEX["SPS-ccbar"]
    ic_sbb = COMPONENT_INDEX["SPS-bbbar"]
    ic_dcc = COMPONENT_INDEX["DPS-ccbar"]
    ic_dbb = COMPONENT_INDEX["DPS-bbbar"]

    response = np.asarray(response, dtype=float)
    fid_shape = np.asarray(fid_shape, dtype=float)
    fid_total = np.asarray(fid_total, dtype=float)

    squeeze = response.ndim == 2
    if squeeze:
        response = response[None, ...]
        fid_shape = fid_shape[None, ...]
        fid_total = fid_total[None, ...]

    nrep = response.shape[0]
    derived_sps, derived_dps = _derived_composition(fid_total)

    if f_cc_sps is None:
        fcc_sps = np.asarray(derived_sps, dtype=float)
    else:
        fcc_sps = np.full(nrep, float(f_cc_sps), dtype=float)

    if f_cc_dps is None:
        fcc_dps = np.asarray(derived_dps, dtype=float)
    else:
        fcc_dps = np.full(nrep, float(f_cc_dps), dtype=float)

    if np.any((fcc_sps <= 0.0) | (fcc_sps >= 1.0)):
        raise ValueError("f_cc_sps outside physical interval")
    if np.any((fcc_dps <= 0.0) | (fcc_dps >= 1.0)):
        raise ValueError("f_cc_dps outside physical interval")

    r_sps = fcc_sps[:, None]*response[:, ic_scc] + (1.0-fcc_sps)[:, None]*response[:, ic_sbb]
    r_dps = fcc_dps[:, None]*response[:, ic_dcc] + (1.0-fcc_dps)[:, None]*response[:, ic_dbb]
    t_sps = fcc_sps[:, None]*fid_shape[:, ic_scc] + (1.0-fcc_sps)[:, None]*fid_shape[:, ic_sbb]
    t_dps = fcc_dps[:, None]*fid_shape[:, ic_dcc] + (1.0-fcc_dps)[:, None]*fid_shape[:, ic_dbb]

    unnorm = (1.0-f)*r_sps + f*r_dps
    shape = safe_divide(unnorm, np.sum(unnorm, axis=1)[:, None], default=np.nan)
    t_sps = safe_divide(t_sps, np.sum(t_sps, axis=1)[:, None], default=np.nan)
    t_dps = safe_divide(t_dps, np.sum(t_dps, axis=1)[:, None], default=np.nan)

    out = {
        "shape": shape,
        "sps_response": r_sps,
        "dps_response": r_dps,
        "sps_fid_shape": t_sps,
        "dps_fid_shape": t_dps,
        "f_cc_sps": fcc_sps,
        "f_cc_dps": fcc_dps,
    }
    if squeeze:
        return {
            "shape": out["shape"][0],
            "sps_response": out["sps_response"][0],
            "dps_response": out["dps_response"][0],
            "sps_fid_shape": out["sps_fid_shape"][0],
            "dps_fid_shape": out["dps_fid_shape"][0],
            "f_cc_sps": float(out["f_cc_sps"][0]),
            "f_cc_dps": float(out["f_cc_dps"][0]),
        }
    return out


def central_component_arrays(problem):
    response = np.stack([problem["central_states"][c]["response"] for c in COMPONENTS])
    fid_shape = np.stack([problem["central_states"][c]["fid_shape"] for c in COMPONENTS])
    fid_total = np.array([problem["central_states"][c]["fid_total"] for c in COMPONENTS])
    return response, fid_shape, fid_total


def fixed_composition_for_problem(problem):
    _, _, fid_total = central_component_arrays(problem)
    fcc_sps, fcc_dps = _derived_composition(fid_total)
    return {"f_cc_sps": float(fcc_sps), "f_cc_dps": float(fcc_dps)}


def cc_fraction_constraint_penalty(problem, f_cc_sps, f_cc_dps):
    """
    Gaussian constraint contribution to -2 log L.

    The parameters remain free. The constraint only penalizes departures
    from the central fiducial MC composition.
    """
    if not (FREE_CC_FRACTIONS and USE_CC_FRACTION_CONSTRAINTS):
        return 0.0

    nominal = fixed_composition_for_problem(problem)

    pull_sps = (
        float(f_cc_sps) - nominal["f_cc_sps"]
    ) / float(CC_SPS_CONSTRAINT_SIGMA)

    pull_dps = (
        float(f_cc_dps) - nominal["f_cc_dps"]
    ) / float(CC_DPS_CONSTRAINT_SIGMA)

    return float(pull_sps**2 + pull_dps**2)


def starting_parameters(problem, start=None):
    composition = fixed_composition_for_problem(problem)
    values = {
        "f_dps": 0.35,
        "f_cc_sps": composition["f_cc_sps"],
        "f_cc_dps": composition["f_cc_dps"],
    }
    if start:
        for key in values:
            if key in start and np.isfinite(start[key]):
                values[key] = float(start[key])
    return values


def active_parameter_names():
    return ["f_dps", "f_cc_sps", "f_cc_dps"] if FREE_CC_FRACTIONS else ["f_dps"]


def mixture_parameter_kwargs(params):
    if FREE_CC_FRACTIONS:
        return {
            "f_cc_sps": float(params["f_cc_sps"]),
            "f_cc_dps": float(params["f_cc_dps"]),
        }
    return {"f_cc_sps": None, "f_cc_dps": None}


def central_mixture(problem, params):
    if np.isscalar(params):
        params = {"f_dps": float(params)}
    response, fid_shape, fid_total = central_component_arrays(problem)
    kwargs = mixture_parameter_kwargs(params)
    return mixture_from_arrays(
        response, fid_shape, fid_total, float(params["f_dps"]), **kwargs
    )


def bank_mixture(problem, params, bank_key="joint"):
    if np.isscalar(params):
        params = {"f_dps": float(params)}
    bank = problem["stat_banks"][bank_key]
    kwargs = mixture_parameter_kwargs(params)
    return mixture_from_arrays(
        bank["response"], bank["fid_shape"], bank["fid_total"],
        float(params["f_dps"]), **kwargs,
    )


def sample_covariance(arr, n_dim):
    arr = np.asarray(arr, dtype=float)
    if arr.ndim != 2 or len(arr) < 2:
        return np.zeros((n_dim, n_dim), dtype=float)
    cov = np.cov(arr, rowvar=False, ddof=1)
    cov = np.atleast_2d(cov).astype(float)
    return 0.5*(cov+cov.T)


PROBLEMS = {NOMINAL_BINNING: build_profile_problem(NOMINAL_BINNING)}
if RUN_ALTERNATE_BINNING_CROSSCHECK:
    PROBLEMS[ALTERNATE_BINNING] = build_profile_problem(ALTERNATE_BINNING)

for i, problem in enumerate(PROBLEMS.values()):
    problem["stat_banks"] = {}
    problem["stat_banks"]["joint"] = build_replica_bank(
        problem, N_MODEL_STAT_REPLICAS, mode="joint", seed=RANDOM_SEED+1000+50*i
    )
    # Smaller source-isolation ensembles are audit diagnostics only.
    for j, mode in enumerate(("mc_template", "stitching", "efficiency")):
        problem["stat_banks"][mode] = build_replica_bank(
            problem, N_SOURCE_AUDIT_REPLICAS, mode=mode,
            seed=RANDOM_SEED+2000+200*i+20*j,
        )

quality_rows=[]
for name, problem in PROBLEMS.items():
    row={"binning":name,"n_bins":problem["n_bins"],"min_Neff":problem["min_neff"]}
    for c in COMPONENTS:
        row[f"min_Neff_{c}"] = float(np.min(problem["central_states"][c]["neff"]))
    row["joint_stat_replicas"] = int(problem["stat_banks"]["joint"]["n"])
    row["template_stats_reportable"] = bool(problem["min_neff"] >= MIN_TEMPLATE_NEFF_REPORTABLE)
    fixed = fixed_composition_for_problem(problem)
    row["fixed_f_cc_sps"] = fixed["f_cc_sps"]
    row["fixed_f_cc_dps"] = fixed["f_cc_dps"]
    quality_rows.append(row)
BINNING_QUALITY = pd.DataFrame(quality_rows)
for _, row in BINNING_QUALITY.iterrows():
    print(
        f"[{row['binning']}] template bins={int(row['n_bins'])}, "
        f"minimum N_eff={row['min_Neff']:.3f}, "
        f"joint replicas={int(row['joint_stat_replicas'])}, "
        f"fixed f_cc(SPS)={row['fixed_f_cc_sps']:.4f}, "
        f"fixed f_cc(DPS)={row['fixed_f_cc_dps']:.4f}"
    )
    
BINNING_QUALITY.to_csv(OUTDIR/"binning_template_statistics.csv", index=False)

## 5a. Nominal HIST template diagnostics

Before any fit is performed, inspect the four efficiency-folded central MC components. These are the actual binned response templates entering the nominal likelihood. The figure is normalized to unit area component by component so that it shows **shape**, not cross-section normalization.

In [ ]:
def _normalized_component_shape(problem, component):
    values = np.asarray(
        problem["central_states"][component]["response"],
        dtype=float,
    )
    total = float(np.sum(values))

    if total <= 0.0 or not np.isfinite(total):
        raise RuntimeError(
            f"Non-positive central HIST response for {component}"
        )

    return values / total

def _draw_component_header(ax, component):
    """
    Draw only the component name above the centralized CMS header.

    CMS / Internal / luminosity are added automatically by
    save_and_show(), so they must NOT be drawn here.
    """

    # Important: do not use ax.set_title(component).
    # The centralized CMS-header code explicitly uses the Matplotlib
    # title position as its anchor and would therefore put CMS above it.
    ax.set_title("")

    # Fixed offset in points above the top of the plotting axis.
    # This is more robust than using y=1.14, 1.18, etc.
    ax.annotate(
        component,
        xy=(0.50, 1.0),
        xycoords="axes fraction",
        xytext=(0.0, 27.0),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=PANEL_TITLE_FONT_SIZE,
        annotation_clip=False,
    )
    
if RUN_SLIDE_DIAGNOSTICS:

    problem = PROBLEMS[NOMINAL_BINNING]

    xe = np.asarray(
        problem["spec"]["dphi_edges"],
        dtype=float,
    )

    ye = np.asarray(
        problem["spec"]["dy_edges"],
        dtype=float,
    )

    nx = len(xe) - 1
    ny = len(ye) - 1

    # ============================================================
    # Diagnostic table
    # ============================================================

    rows = []

    for component in COMPONENTS:

        state = problem["central_states"][component]

        rows.append({
            "component": component,
            "response_integral": float(
                np.sum(state["response"])
            ),
            "fiducial_total": float(
                state["fid_total"]
            ),
            "minimum_N_eff": float(
                np.min(state["neff"])
            ),
        })

    HIST_TEMPLATE_DIAGNOSTIC = pd.DataFrame(rows)

    display(HIST_TEMPLATE_DIAGNOSTIC)

    HIST_TEMPLATE_DIAGNOSTIC.to_csv(
        OUTDIR
        / "nominal_HIST_component_template_diagnostic.csv",
        index=False,
    )

    # ============================================================
    # Figure
    # ============================================================

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(11.2, 8.4),
    )

    # KEEP THE ORIGINAL PANEL GEOMETRY
    fig.subplots_adjust(
        left=0.09,
        right=0.91,
        bottom=0.13,
        top=0.77,
        wspace=0.24,
        hspace=0.48,
    )

    # ============================================================
    # Normalized component templates
    # ============================================================

    shapes = {
        component:
        _normalized_component_shape(
            problem,
            component,
        ).reshape(nx, ny)

        for component in COMPONENTS
    }

    vmax = max(
        float(np.max(shape))
        for shape in shapes.values()
    )

    # ============================================================
    # Panels
    # ============================================================

    for panel_index, (ax, component) in enumerate(
        zip(axes.flat, COMPONENTS)
    ):

        mesh = ax.pcolormesh(
            xe,
            ye,
            shapes[component].T,
            shading="auto",
            vmin=0.0,
            vmax=vmax,
        )

        # --------------------------------------------------------
        # Axis labels
        # --------------------------------------------------------

        ax.set_xlabel(
            XLABEL_DPHI
            if panel_index >= 2
            else ""
        )

        ax.tick_params(
            axis="x",
            labelbottom=(panel_index >= 2),
        )

        ax.set_ylabel(
            XLABEL_DY
        )

        # --------------------------------------------------------
        # Header
        #
        #       component
        # CMS Internal       luminosity
        # -----------------------------
        #            plot
        # --------------------------------------------------------

        _draw_component_header(
            ax,
            component,
        )

    # ============================================================
    # Shared colorbar
    # ============================================================

    cbar = fig.colorbar(
        mesh,
        ax=axes.ravel().tolist(),
        shrink=0.82,
        pad=0.02,
    )

    cbar.set_label(
        CBAR_NORMALIZED_2D,
        fontsize=11,
        labelpad=6,
    )

    cbar.ax.tick_params(
        labelsize=10,
    )

    # ============================================================
    # Save
    # ============================================================

    save_and_show(
        fig,
        "prefit_component_HIST_templates_2D_CMS",
        OUTDIR,
    )

else:

    print(
        "[templates] nominal HIST component diagnostic disabled"
    )

### Slide diagnostic — each MC sample before and after efficiency correction

The ROOT samples already correspond to the **selected/reconstructed** MC used by the analysis. Therefore the slide comparison is

$$w_i^{\rm selected}
\quad\longrightarrow\quad
w_i^{\rm fid}
=
\frac{w_i^{\rm selected}}{\epsilon_i},$$

with the full nominal response

$$\epsilon_i=
A_{\mu\mu}\,
A_{D^*}\,
\epsilon_{\mu\mu}^{\rm cuts}\,
\epsilon_{D^*}^{\rm cuts}\,
\epsilon_{\rm trigger}\,
\epsilon_{\rm asso}^{p_T}.$$

The 2017 and 2018 products are formed separately and only then luminosity-combined, exactly as in the nominal fit.

In [ ]:
# ================================================================
# SLIDE DIAGNOSTIC: ALL 8 MC SAMPLES BEFORE / AFTER EFFICIENCIES
# ================================================================

SAMPLE_SLIDE_LABELS = {
    "DPS-ccbar-25to30":  r"DPS $c\bar c$, $25<p_T^{J/\psi}<30$ GeV",
    "DPS-ccbar-30to50":  r"DPS $c\bar c$, $30<p_T^{J/\psi}<50$ GeV",
    "DPS-ccbar-50to100": r"DPS $c\bar c$, $50<p_T^{J/\psi}<100$ GeV",
    "DPS-bbbar-25to30":  r"DPS $b\bar b$, $25<p_T^{J/\psi}<30$ GeV",
    "DPS-bbbar-30to50":  r"DPS $b\bar b$, $30<p_T^{J/\psi}<50$ GeV",
    "DPS-bbbar-50to100": r"DPS $b\bar b$, $50<p_T^{J/\psi}<100$ GeV",
    "SPS-ccbar-25to100": r"SPS $c\bar c$, $25<p_T^{J/\psi}<100$ GeV",
    "SPS-bbbar-25to100": r"SPS $b\bar b$, $25<p_T^{J/\psi}<100$ GeV",
}


def _mc_selected_and_fiducial_weights(sample, frame):
    component = SAMPLE_INFO[sample]["component"]

    selected = (
        frame["mc_weight_xsec_only"].to_numpy(float)
        * float(CENTRAL_STITCH_SCALES.get(sample, 1.0))
        / (LUMI_RUN2_PB_INV * BRANCHING_PRODUCT)
    )

    efficiency = evaluate_efficiency_product(
        frame,
        EFFICIENCY_SOURCE_FOR_COMPONENT[component],
    )

    if np.any(~np.isfinite(efficiency)) or np.any(efficiency <= 0.0):
        raise RuntimeError(
            f"{sample}: invalid nominal efficiency product"
        )

    fiducial = selected / efficiency
    return selected, fiducial, efficiency


def _unit_area_hist(values, weights, edges):
    hist = np.histogram(
        np.asarray(values, float),
        bins=np.asarray(edges, float),
        weights=np.asarray(weights, float),
    )[0].astype(float)

    total = float(np.sum(hist))
    if not np.isfinite(total) or total <= 0.0:
        raise RuntimeError("Non-positive histogram integral")

    return hist / total


def plot_mc_efficiency_before_after(observable, edges, xlabel):
    sample_order = list(SAMPLE_INFO.keys())

    fig, axes = plt.subplots(
        4, 2,
        # Fixed-size labels need enough physical space on the dense 4x2 grid.
        figsize=(12.4, 18.0),
    )
    fig.subplots_adjust(
        left=0.09, right=0.985, bottom=0.07, top=0.83,
        wspace=0.20, hspace=0.48,
    )
    axes = np.asarray(axes).reshape(-1)

    audit_rows = []

    for iax, sample in enumerate(sample_order):
        ax = axes[iax]
        frame = MC_SAMPLE_CACHE[sample]
        selected, fiducial, efficiency = _mc_selected_and_fiducial_weights(
            sample,
            frame,
        )

        values = frame[observable].to_numpy(float)

        selected_shape = _unit_area_hist(
            values,
            selected,
            edges,
        )
        fiducial_shape = _unit_area_hist(
            values,
            fiducial,
            edges,
        )

        ax.stairs(
            selected_shape,
            edges,
            linewidth=1.7,
            label="Selected MC",
        )
        ax.stairs(
            fiducial_shape,
            edges,
            linewidth=1.7,
            linestyle="--",
            label="Efficiency-corrected MC",
        )

        # Keep the native Matplotlib title empty so that the centralized
        # CMS header remains immediately above the plotting axis.
        ax.set_title("")

        # Sample title goes above the CMS / Internal / luminosity line.
        ax.annotate(
            SAMPLE_SLIDE_LABELS.get(sample, sample),
            xy=(0.50, 1.0),
            xycoords="axes fraction",
            xytext=(0.0, 27.0),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=13.5,
            annotation_clip=False,
        )

        # Keep the dense 4x2 canvas compact: the bottom row owns the shared x label.
        ax.set_xlabel(
            xlabel if iax >= len(sample_order) - 2 else "",
            fontsize=AXIS_LABEL_FONT_SIZE,
        )
        # CMS style anchors y labels at the top by default; center these long labels.
        ax.set_ylabel(
            YLABEL_NORMALIZED, fontsize=0.9*AXIS_LABEL_FONT_SIZE,
            labelpad=7, loc="center",
        )
        ax.tick_params(axis="both", labelsize=9.5)
        ax.set_ylim(bottom=0.0)

        if iax == 0:
            ax.legend(fontsize=9.7)

        audit_rows.append(
            {
                "sample": sample,
                "component": SAMPLE_INFO[sample]["component"],
                "n_selected_events": int(len(frame)),
                "sumw_selected": float(np.sum(selected)),
                "sumw_fiducial": float(np.sum(fiducial)),
                "effective_efficiency_sumw_ratio": float(
                    np.sum(selected) / np.sum(fiducial)
                ),
                "efficiency_min": float(np.min(efficiency)),
                "efficiency_max": float(np.max(efficiency)),
            }
        )

    #cms_label(axes[0], "Internal", LUMI_RUN2_FB_INV)

    #fig.suptitle(
    #    "Selected vs efficiency-corrected MC shapes",
    #    fontsize=18,
    #)

    stem = (
        "mc_efficiency_before_after_dphi_CMS"
        if observable == "delta_phi"
        else "mc_efficiency_before_after_dy_CMS"
    )

    save_and_show(fig, stem, OUTDIR)

    return pd.DataFrame(audit_rows)


if RUN_SLIDE_DIAGNOSTICS:
    dphi_edges_mc_slide = np.linspace(
        0.0,
        float(DPHI_MAX),
        SLIDE_SHAPE_NBINS + 1,
    )

    dy_nominal_edges = np.asarray(
        BINNING[NOMINAL_BINNING]["dy"],
        dtype=float,
    )

    dy_edges_mc_slide = np.linspace(
        float(dy_nominal_edges[0]),
        float(dy_nominal_edges[-1]),
        SLIDE_SHAPE_NBINS + 1,
    )

    audit_phi = plot_mc_efficiency_before_after(
        "delta_phi",
        dphi_edges_mc_slide,
        XLABEL_DPHI,
    )

    audit_dy = plot_mc_efficiency_before_after(
        "delta_y",
        dy_edges_mc_slide,
        XLABEL_DY,
    )

    MC_EFFICIENCY_SLIDE_AUDIT = audit_phi.copy()

    MC_EFFICIENCY_SLIDE_AUDIT.to_csv(
        OUTDIR / "mc_efficiency_before_after_summary.csv",
        index=False,
    )

    print(
        MC_EFFICIENCY_SLIDE_AUDIT[
            [
                "sample",
                "effective_efficiency_sumw_ratio",
                "efficiency_min",
                "efficiency_max",
            ]
        ].to_string(index=False)
    )
else:
    print("[slides] MC efficiency before/after plots disabled")

### Pre-fit template-fit inputs — suggested convener plot

This is the clearest pre-fit figure to show before presenting the fitted $f_{\mathrm{DPS}}$.

No fitted DPS fraction is used. The signal-sWeighted data are compared with the two **pure hypotheses** that enter the likelihood:

- $f_{\mathrm{DPS}}=0$: pure SPS response;
- $f_{\mathrm{DPS}}=1$: pure DPS response.

The left panels show the normalized 2D shapes, while the projections show where the discrimination comes from in $|\Delta\phi|$ and $|\Delta y|$. Because this is pre-fit, there is no arbitrary SPS+DPS normalization or fitted mixture on this figure.

In [ ]:
if RUN_SLIDE_DIAGNOSTICS:

    # ============================================================
    # Temporarily use a slightly more compact CMS header
    # for these relatively narrow panels.
    # Original global values are restored at the end.
    # ============================================================

    _CMS_FONT_SIZE_ORIGINAL = CMS_FONT_SIZE
    _CMS_EXTRA_FONT_SIZE_ORIGINAL = CMS_EXTRA_FONT_SIZE
    _LUMI_FONT_SIZE_ORIGINAL = LUMI_FONT_SIZE

    CMS_FONT_SIZE = 18.0
    CMS_EXTRA_FONT_SIZE = 13.0
    LUMI_FONT_SIZE = 12.0

    try:

        problem = PROBLEMS[NOMINAL_BINNING]

        xe = np.asarray(
            problem["spec"]["dphi_edges"],
            dtype=float
        )

        ye = np.asarray(
            problem["spec"]["dy_edges"],
            dtype=float
        )

        nx, ny = len(xe) - 1, len(ye) - 1

        data_shape = np.asarray(
            problem["data_shape"],
            dtype=float
        )

        data_cov = np.asarray(
            problem["data_cov"],
            dtype=float
        )

        sps_shape = np.asarray(
            central_mixture(
                problem,
                {"f_dps": 0.0}
            )["shape"],
            dtype=float
        )

        dps_shape = np.asarray(
            central_mixture(
                problem,
                {"f_dps": 1.0}
            )["shape"],
            dtype=float
        )

        if not all(
            np.isclose(
                np.sum(v),
                1.0,
                atol=1e-12
            )
            for v in (
                data_shape,
                sps_shape,
                dps_shape
            )
        ):
            raise RuntimeError(
                "Pre-fit normalized-shape contract failed"
            )


        # ========================================================
        # 2D input shapes
        # ========================================================

        maps = {
            "Data":
                data_shape.reshape(nx, ny),

            "SPS template":
                sps_shape.reshape(nx, ny),

            "DPS template":
                dps_shape.reshape(nx, ny),
        }

        vmax = max(
            float(np.max(v))
            for v in maps.values()
        )

        fig, axes = plt.subplots(
            1,
            3,
            figsize=(14.6, 5.1)
        )

        fig.subplots_adjust(
            left=0.065,
            right=0.91,
            bottom=0.19,
            top=0.73,
            wspace=0.24
        )

        for ax, (title, values) in zip(
            axes,
            maps.items()
        ):

            mesh = ax.pcolormesh(
                xe,
                ye,
                values.T,
                shading="auto",
                vmin=0.0,
                vmax=vmax
            )

            ax.set_xlabel(
                XLABEL_DPHI
            )

            ax.set_ylabel(
                XLABEL_DY
            )

            # ----------------------------------------------------
            # IMPORTANT:
            #
            # Do NOT use ax.set_title(title), because the
            # centralized CMS-header routine detects the native
            # Matplotlib title and places CMS above it.
            #
            # Instead:
            #
            #          Data / SPS template / DPS template
            # CMS Internal                  luminosity
            # -----------------------------------------
            #                    plot
            # ----------------------------------------------------

            ax.set_title("")

            ax.annotate(
                title,
                xy=(0.50, 1.0),
                xycoords="axes fraction",
                xytext=(0.0, 25.0),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=13.0,
                annotation_clip=False,
            )


        cbar = fig.colorbar(
            mesh,
            ax=axes.ravel().tolist(),
            shrink=0.82,
            pad=0.02
        )

        cbar.set_label(
            CBAR_NORMALIZED_2D,
            fontsize=11,
            labelpad=6
        )

        cbar.ax.tick_params(
            labelsize=10
        )

        # fig.suptitle(
        #     "Pre-fit inputs to the nominal 2D template likelihood",
        #     fontsize=16,
        #     y=0.96
        # )

        save_and_show(
            fig,
            "prefit_data_SPS_DPS_2D_inputs_CMS",
            OUTDIR
        )


        # ========================================================
        # Projections with correctly propagated
        # normalized-data covariance
        # ========================================================

        P_phi = np.zeros(
            (nx, nx * ny),
            dtype=float
        )

        P_y = np.zeros(
            (ny, nx * ny),
            dtype=float
        )

        for ix in range(nx):
            P_phi[
                ix,
                ix * ny:(ix + 1) * ny
            ] = 1.0

        for iy in range(ny):
            P_y[
                iy,
                iy::ny
            ] = 1.0


        fig, axes = plt.subplots(
            1,
            2,
            figsize=(11.8, 4.9)
        )

        # No suptitle in this canvas, so do not reserve 27%
        # of the figure above the axes.
        fig.subplots_adjust(
            left=0.10,
            right=0.985,
            bottom=0.20,
            top=0.80,
            wspace=0.23
        )


        for ax, P, edges, xlabel in [

            (
                axes[0],
                P_phi,
                xe,
                XLABEL_DPHI
            ),

            (
                axes[1],
                P_y,
                ye,
                XLABEL_DY
            ),

        ]:

            data_proj = (
                P @ data_shape
            )

            sps_proj = (
                P @ sps_shape
            )

            dps_proj = (
                P @ dps_shape
            )

            cov_proj = (
                P
                @ data_cov
                @ P.T
            )

            err_proj = np.sqrt(
                np.clip(
                    np.diag(
                        cov_proj
                    ),
                    0.0,
                    None
                )
            )

            centers = (
                0.5
                * (
                    edges[:-1]
                    + edges[1:]
                )
            )

            widths = np.diff(
                edges
            )


            ax.errorbar(
                centers,
                data_proj,
                xerr=0.5 * widths,
                yerr=err_proj,
                fmt="o",
                color="black",
                markersize=5.0,
                capsize=2.5,
                label="Data",
                zorder=5,
            )


            ax.stairs(
                sps_proj,
                edges,
                linewidth=2.0,
                label="SPS template"
            )


            ax.stairs(
                dps_proj,
                edges,
                linewidth=2.0,
                linestyle="--",
                label="DPS template"
            )


            ax.set_xlabel(
                xlabel
            )

            ax.set_ylabel(
                YLABEL_NORMALIZED
            )


            ax.legend(
                loc=(0.6, 0.65),
                frameon=False,
                fontsize=12.5
            )


            ax.grid(
                axis="y",
                alpha=0.16
            )


        # fig.suptitle(
        #     "Pre-fit separation power of the SPS and DPS templates",
        #     fontsize=16,
        #     y=0.96
        # )

        save_and_show(
            fig,
            "prefit_SPS_DPS_projection_separation_CMS",
            OUTDIR
        )


    finally:

        # ========================================================
        # Restore the standard CMS sizes for every subsequent plot
        # ========================================================

        CMS_FONT_SIZE = (
            _CMS_FONT_SIZE_ORIGINAL
        )

        CMS_EXTRA_FONT_SIZE = (
            _CMS_EXTRA_FONT_SIZE_ORIGINAL
        )

        LUMI_FONT_SIZE = (
            _LUMI_FONT_SIZE_ORIGINAL
        )

else:

    print(
        "[prefit] slide diagnostic disabled"
    )

## 6. Approved differential systematic variations

Only controlled differential inputs are converted into shape covariances. Signal-extraction/sWeight variants are loaded automatically when present. The validated efficiency-response study contributes two additional normalized-shape covariance terms: response-year dependence and efficiency-map binning stability. Integrated AN percentages are never converted into artificial bin-by-bin variations.

In [ ]:
def _sweight_variant_files():
    if not SPLOT_VARIATION_DIR.is_dir():
        return []
    return sorted([p for p in SPLOT_VARIATION_DIR.rglob("*.root") if p.is_file()])

def build_splot_systematic(problem):
    files = _sweight_variant_files()
    shapes=[]
    hists=[]
    weights=[]
    records=[]
    for path in files:
        try:
            frame, _ = read_logical_dataframe(path, ["sweight"])
            w = frame["sweight"].to_numpy(float)
            if len(w) != len(data_sw):
                raise ValueError(f"length {len(w)} != {len(data_sw)}")
            idx = problem["data_idx"]
            valid = problem["data_valid"]
            wpb = w/(LUMI_RUN2_PB_INV*BRANCHING_PRODUCT)
            hist = np.bincount(idx[valid], weights=wpb[valid], minlength=problem["n_bins"]).astype(float)
            total = float(np.sum(hist))
            if total <= 0.0:
                raise ValueError("non-positive weighted integral")
            shape = hist/total
            shapes.append(shape)
            hists.append(hist)
            weights.append(w.copy())
            records.append({"file":str(path),"status":"loaded"})
        except Exception as exc:
            records.append({"file":str(path),"status":f"failed: {exc}"})
    if not shapes:
        return {
            "cov": np.zeros((problem["n_bins"], problem["n_bins"])),
            "shapes": np.empty((0, problem["n_bins"])),
            "hists": np.empty((0, problem["n_bins"])),
            "weights": np.empty((0, len(data_sw))),
            "records": records,
            "available": False,
        }
    delta = np.asarray(shapes)-problem["data_shape"]
    # RMS covariance of the approved fit-variant ensemble around nominal.
    cov = delta.T @ delta / float(len(delta))
    cov = 0.5*(cov+cov.T)
    return {
        "cov": cov,
        "shapes": np.asarray(shapes),
        "hists": np.asarray(hists),
        "weights": np.asarray(weights),
        "records": records,
        "available": True,
    }


for name, problem in PROBLEMS.items():
    problem["splot_systematic"] = build_splot_systematic(problem)

SPLOT_VARIATION_AUDIT = pd.DataFrame(
    PROBLEMS[NOMINAL_BINNING]["splot_systematic"]["records"]
)
if len(SPLOT_VARIATION_AUDIT):
    print(f"[sPlot syst] {sum(SPLOT_VARIATION_AUDIT['status'].eq('loaded'))} approved variation files loaded")
SPLOT_VARIATION_AUDIT.to_csv(OUTDIR/"splot_variant_audit.csv", index=False)

# Double-counting guard for the absolute upper-panel reference band.
DIFFERENTIAL_AN143_SOURCES = set()
if PROBLEMS[NOMINAL_BINNING]["splot_systematic"]["available"]:
    DIFFERENTIAL_AN143_SOURCES.add("signal_yield")
AN143_COMMON_REL_AVAILABLE = math.sqrt(sum(
    AN143_REFERENCE_RELATIVE_SYSTEMATICS[name]**2
    for name in AN143_COMMON_NORMALIZATION_SOURCES
    if name not in DIFFERENTIAL_AN143_SOURCES
))
AN143_COMMON_REL_EFFECTIVE = (
    AN143_COMMON_REL_AVAILABLE if INCLUDE_AN143_INTEGRATED_SYSTEMATICS else 0.0
)
print("AN143 available common normalization reference =", f"{100*AN143_COMMON_REL_AVAILABLE:.3f}%")
print("AN143 common normalization used in spectra     =", f"{100*AN143_COMMON_REL_EFFECTIVE:.3f}%")
print("AN143 sources removed because propagated differentially =", sorted(DIFFERENTIAL_AN143_SOURCES))


# ================================================================
# VALIDATED EFFICIENCY-RESPONSE SYSTEMATIC COVARIANCE
# ================================================================
# Produced by JPsiDstar_Systematics_EfficiencyYearBinning_20260904.
# The matrices live in the full normalized 2D-shape basis.  Their
# normalization mode must vanish before projection to the Helmert basis.

EFFICIENCY_RESPONSE_SYSTEMATIC_FILES = {
    "summary": "efficiency_systematics_for_final.json",
    "year": "year_response_covariance.npy",
    "binning": "efficiency_binning_covariance.npy",
    "total": "efficiency_response_systematic_covariance_total.npy",
}


def _sanitize_normalized_shape_covariance(covariance, label, n_bins):
    cov = np.asarray(covariance, dtype=float)
    if cov.shape != (int(n_bins), int(n_bins)):
        raise RuntimeError(
            f"{label}: covariance shape {cov.shape} != {(int(n_bins), int(n_bins))}"
        )
    if not np.all(np.isfinite(cov)):
        raise RuntimeError(f"{label}: covariance contains non-finite values")
    cov = 0.5 * (cov + cov.T)

    # Explicitly remove the normalization mode.  For the stored matrices this
    # should only remove floating-point residue because every source variation
    # was constructed from normalized shapes.
    one = np.ones(int(n_bins), dtype=float)
    P = np.eye(int(n_bins)) - np.outer(one, one) / float(n_bins)
    cov = P @ cov @ P
    cov = 0.5 * (cov + cov.T)

    eigval, eigvec = np.linalg.eigh(cov)
    scale = max(float(np.max(np.abs(eigval))), 1.0e-20)
    if float(np.min(eigval)) < -1.0e-10 * scale:
        raise RuntimeError(
            f"{label}: covariance is not positive semidefinite; "
            f"min eigenvalue={float(np.min(eigval)):.3e}"
        )
    eigval = np.maximum(eigval, 0.0)
    cov = eigvec @ np.diag(eigval) @ eigvec.T
    cov = P @ cov @ P
    cov = 0.5 * (cov + cov.T)

    normalization_residual = float(np.max(np.abs(cov @ one)))
    if normalization_residual > max(1.0e-12, 1.0e-8 * scale):
        raise RuntimeError(
            f"{label}: non-zero normalization mode after sanitization: "
            f"{normalization_residual:.3e}"
        )
    return cov


def load_efficiency_response_systematic_bundle():
    n_nominal = int(ACTUAL_N_DPHI_BINS * ACTUAL_N_DY_BINS)

    if not INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS:
        z = np.zeros((n_nominal, n_nominal), dtype=float)
        return {
            "enabled": False,
            "available": False,
            "summary": {},
            "year_cov": z.copy(),
            "binning_cov": z.copy(),
            "cov": z.copy(),
            "source_directory": str(EFFICIENCY_RESPONSE_SYSTEMATICS_DIR),
        }

    # The dedicated study was run for the current nominal configuration:
    # full folded Delta-phi range and the legacy '6x6' key whose explicit
    # arrays define the 4x4 grid.  Reusing the matrix under a different shape
    # basis would be invalid.
    compatible_nominal_basis = (
        bool(USE_DPHI_PI)
        and NOMINAL_BINNING == "6x6"
        and ACTUAL_N_DPHI_BINS == 4
        and ACTUAL_N_DY_BINS == 4
    )
    if not compatible_nominal_basis:
        raise RuntimeError(
            "The stored efficiency-response covariance was validated only for "
            "USE_DPHI_PI=True and the nominal 4x4 grid (legacy key '6x6'). "
            "Disable INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS and rerun the "
            "dedicated systematic study before changing this fit basis."
        )

    paths = {
        key: EFFICIENCY_RESPONSE_SYSTEMATICS_DIR / filename
        for key, filename in EFFICIENCY_RESPONSE_SYSTEMATIC_FILES.items()
    }
    missing = [str(path) for path in paths.values() if not path.is_file()]
    if missing:
        if REQUIRE_EFFICIENCY_RESPONSE_SYSTEMATICS:
            raise FileNotFoundError(
                "Missing efficiency-response systematic products:\n  "
                + "\n  ".join(missing)
            )
        warnings.warn(
            "Efficiency-response systematic products are incomplete; "
            "the source is disabled for this run."
        )
        z = np.zeros((n_nominal, n_nominal), dtype=float)
        return {
            "enabled": True,
            "available": False,
            "summary": {},
            "year_cov": z.copy(),
            "binning_cov": z.copy(),
            "cov": z.copy(),
            "source_directory": str(EFFICIENCY_RESPONSE_SYSTEMATICS_DIR),
        }

    with open(paths["summary"]) as handle:
        summary = json.load(handle)

    if summary.get("efficiency_nominal_response") != "luminosity-weighted 2017+2018":
        raise RuntimeError(
            "Efficiency systematic summary does not describe the expected "
            "luminosity-weighted 2017+2018 nominal response"
        )

    year_cov = _sanitize_normalized_shape_covariance(
        np.load(paths["year"]), "year-response systematic", n_nominal
    )
    binning_cov = _sanitize_normalized_shape_covariance(
        np.load(paths["binning"]), "efficiency-binning systematic", n_nominal
    )
    stored_total = _sanitize_normalized_shape_covariance(
        np.load(paths["total"]), "stored total efficiency-response systematic", n_nominal
    )
    total_cov = _sanitize_normalized_shape_covariance(
        year_cov + binning_cov, "reconstructed total efficiency-response systematic", n_nominal
    )

    scale = max(float(np.max(np.abs(total_cov))), 1.0e-20)
    if not np.allclose(stored_total, total_cov, rtol=1.0e-8, atol=1.0e-12 * max(1.0, scale)):
        raise RuntimeError(
            "Stored combined efficiency-response covariance does not equal "
            "year-response + efficiency-binning covariance"
        )

    return {
        "enabled": True,
        "available": True,
        "summary": summary,
        "year_cov": year_cov,
        "binning_cov": binning_cov,
        "cov": total_cov,
        "source_directory": str(EFFICIENCY_RESPONSE_SYSTEMATICS_DIR),
        "year_fdps_envelope": float(
            summary.get("year_response", {}).get("delta_fDPS_envelope", np.nan)
        ),
        "year_fdps_rms": float(
            summary.get("year_response", {}).get("delta_fDPS_rms", np.nan)
        ),
        "binning_fdps_envelope": float(
            summary.get("efficiency_binning", {}).get("delta_fDPS_envelope", np.nan)
        ),
        "binning_fdps_rms": float(
            summary.get("efficiency_binning", {}).get("delta_fDPS_rms", np.nan)
        ),
        "source_nominal_fdps": float(summary.get("nominal_fDPS", np.nan)),
        "year_response_temporary_until_2016": bool(
            summary.get("year_response", {}).get("temporary_until_2016_available", True)
        ),
    }


EFFICIENCY_RESPONSE_SYSTEMATICS = load_efficiency_response_systematic_bundle()


def efficiency_response_systematic_for_problem(problem):
    n_bins = int(problem["n_bins"])
    if not EFFICIENCY_RESPONSE_SYSTEMATICS["available"]:
        z = np.zeros((n_bins, n_bins), dtype=float)
        return {
            "available": False,
            "cov": z,
            "year_cov": z.copy(),
            "binning_cov": z.copy(),
            "reason": "disabled or unavailable",
        }

    # The stored covariance is only for the nominal 4x4 shape basis.  An
    # optional alternate-binning diagnostic remains allowed, but does not
    # silently inherit an incompatible systematic matrix.
    if problem["name"] != NOMINAL_BINNING:
        z = np.zeros((n_bins, n_bins), dtype=float)
        return {
            "available": False,
            "cov": z,
            "year_cov": z.copy(),
            "binning_cov": z.copy(),
            "reason": "systematic bundle validated only for nominal binning",
        }

    if EFFICIENCY_RESPONSE_SYSTEMATICS["cov"].shape != (n_bins, n_bins):
        raise RuntimeError(
            f"Efficiency systematic dimension mismatch for {problem['name']}: "
            f"{EFFICIENCY_RESPONSE_SYSTEMATICS['cov'].shape} vs {(n_bins, n_bins)}"
        )

    return {
        "available": True,
        "cov": EFFICIENCY_RESPONSE_SYSTEMATICS["cov"].copy(),
        "year_cov": EFFICIENCY_RESPONSE_SYSTEMATICS["year_cov"].copy(),
        "binning_cov": EFFICIENCY_RESPONSE_SYSTEMATICS["binning_cov"].copy(),
        "reason": "validated dedicated systematic study",
    }


for name, problem in PROBLEMS.items():
    problem["efficiency_response_systematic"] = (
        efficiency_response_systematic_for_problem(problem)
    )


def differential_shape_systematic_covariance(problem):
    """Total approved normalized-shape systematic covariance for the fit."""
    cov = np.zeros((problem["n_bins"], problem["n_bins"]), dtype=float)
    if problem["splot_systematic"]["available"]:
        cov = cov + problem["splot_systematic"]["cov"]
    eff_syst = problem.get("efficiency_response_systematic", {})
    if eff_syst.get("available", False):
        cov = cov + eff_syst["cov"]
    return 0.5 * (cov + cov.T)


if EFFICIENCY_RESPONSE_SYSTEMATICS["available"]:
    print("[eff-response syst] validated covariance bundle loaded")
    print(
        "[eff-response syst] year-response |delta fDPS| envelope =",
        f"{EFFICIENCY_RESPONSE_SYSTEMATICS['year_fdps_envelope']:.6f}",
    )
    print(
        "[eff-response syst] binning |delta fDPS| envelope      =",
        f"{EFFICIENCY_RESPONSE_SYSTEMATICS['binning_fdps_envelope']:.6f}",
    )
    print(
        "[eff-response syst] scalar-envelope quadrature         =",
        f"{math.hypot(EFFICIENCY_RESPONSE_SYSTEMATICS['year_fdps_envelope'], EFFICIENCY_RESPONSE_SYSTEMATICS['binning_fdps_envelope']):.6f}",
    )


## SLIDE MATERIAL 1 — How the 2D template fit works

### 1. Build SPS and DPS templates

The selected MC is efficiency-corrected to the fiducial level and grouped into the four physics components

$$\mathrm{SPS}\text{-}c\bar c,\qquad
\mathrm{SPS}\text{-}b\bar b,\qquad
\mathrm{DPS}\text{-}c\bar c,\qquad
\mathrm{DPS}\text{-}b\bar b.$$

The flavour mixtures are

$$T_{\rm SPS}
=
f_{c\bar c}^{\rm SPS}\,T_{{\rm SPS},c\bar c}
+
\left(1-f_{c\bar c}^{\rm SPS}\right)T_{{\rm SPS},b\bar b},$$

$$T_{\rm DPS}
=
f_{c\bar c}^{\rm DPS}\,T_{{\rm DPS},c\bar c}
+
\left(1-f_{c\bar c}^{\rm DPS}\right)T_{{\rm DPS},b\bar b}.$$

### 2. Fit the DPS fraction

In each bin of the two-dimensional distribution
$(|\Delta\phi|,|\Delta y|)$,

$$M_i(f_{\rm DPS})
=
(1-f_{\rm DPS})\,T_{{\rm SPS},i}
+
f_{\rm DPS}\,T_{{\rm DPS},i}.$$

The nominal fit has **one free physics parameter**, $f_{\rm DPS}$.  
The $c\bar c/b\bar b$ fractions are fixed from the fiducial MC composition in the nominal configuration.

**Slide takeaway:** the data shape is decomposed into an SPS-like and a DPS-like component using their different two-dimensional kinematic correlations.

## SLIDE MATERIAL 2 — Likelihood and uncertainty treatment

The signal-weighted data histogram and the MC prediction are normalized in the full 2D fiducial region. Because the data contain **sWeights, including negative weights**, the fit uses a Gaussian likelihood rather than a Poisson likelihood.

After removing the trivial normalization mode, the objective is

$$-2\ln\mathcal L(f_{\rm DPS})
=
(\mathbf d-\mathbf m)^T
V^{-1}
(\mathbf d-\mathbf m)
+
\ln |V|
+\mathrm{const.}$$

where

- $\mathbf d$: normalized sWeighted data shape;
- $\mathbf m(f_{\rm DPS})$: normalized SPS+DPS prediction;
- $V$: covariance matrix of the fitted shape.

The covariance contains the data statistical uncertainty, finite-MC/template uncertainty, efficiency-map statistical uncertainty, DPS-stitching uncertainty, the approved differential efficiency-response systematics, and the adopted normalized MC-shape component.

The best-fit value minimizes $-2\ln\mathcal L$. The 68% confidence interval is obtained from the profile-likelihood condition

$$\Delta(-2\ln\mathcal L)=1.$$

The toy-calibrated goodness-of-fit is a separate question: it asks whether the **size of the residuals at the fitted point** is compatible with pseudoexperiments generated from the fitted model.

In [ ]:
# ================================================================
# SLIDE NUMBERS: CURRENT NOMINAL TEMPLATE CONFIGURATION
# ================================================================
PARAMETER_NAMES = active_parameter_names()

_slide_problem = PROBLEMS[NOMINAL_BINNING]
_slide_fixed = fixed_composition_for_problem(_slide_problem)

TEMPLATE_FIT_SLIDE_NUMBERS = {
    "binning_label": NOMINAL_BINNING,
    "n_2d_bins": int(_slide_problem["n_bins"]),
    "n_free_parameters": int(len(PARAMETER_NAMES)),
    "f_cc_sps": float(_slide_fixed["f_cc_sps"]),
    "f_cc_dps": float(_slide_fixed["f_cc_dps"]),
    "dphi_max": float(DPHI_MAX),
    "model_inadequacy_rel": float(MODEL_INADEQUACY_REL),
}

print("=" * 72)
print("TEMPLATE-FIT SLIDE NUMBERS")
print("=" * 72)
print(
    f"Nominal 2D binning: {NOMINAL_BINNING} "
    f"({TEMPLATE_FIT_SLIDE_NUMBERS['n_2d_bins']} populated template bins)"
)
print(
    f"Free physics parameters: {TEMPLATE_FIT_SLIDE_NUMBERS['n_free_parameters']}"
)
print(
    "Fixed MC flavour composition:"
)
print(
    f"  f_cc(SPS) = {TEMPLATE_FIT_SLIDE_NUMBERS['f_cc_sps']:.4f}"
)
print(
    f"  f_cc(DPS) = {TEMPLATE_FIT_SLIDE_NUMBERS['f_cc_dps']:.4f}"
)
print(
    f"|Delta phi| range: [0, {TEMPLATE_FIT_SLIDE_NUMBERS['dphi_max']:.4f}]"
)
print("=" * 72)


## 7. Nominal likelihood and $f_{\mathrm{DPS}}$ interval

The fit is performed in the $N-1$ dimensional Helmert contrast space of the normalized 2D histogram. The covariance contains the sWeighted-data covariance, finite-model covariance, approved sPlot variations when available, the validated efficiency-response systematic covariance, and the adopted Monte Carlo shape-inadequacy term.

With `FREE_CC_FRACTIONS=False`, MINOS profiles only $f_{\mathrm{DPS}}$. With `FREE_CC_FRACTIONS=True`, the MINOS interval for $f_{\mathrm{DPS}}$ profiles over $f_{cc}^{\mathrm{SPS}}$ and $f_{cc}^{\mathrm{DPS}}$.

In [ ]:
PARAMETER_NAMES = active_parameter_names()


def model_stat_covariance(problem, params, bank_key="joint"):
    bank_model = bank_mixture(problem, params, bank_key=bank_key)
    return sample_covariance(bank_model["shape"], problem["n_bins"])


def model_inadequacy_shape_covariance(shape, relative_floor):
    """Normalized-shape covariance from one global relative 2D floor."""
    p = np.asarray(shape, dtype=float)
    p = p / float(np.sum(p))
    A = np.diag(p) - np.outer(p, p)
    cov = float(relative_floor)**2 * (A @ A.T)
    cov = 0.5 * (cov + cov.T)
    if not np.allclose(np.sum(cov, axis=0), 0.0, atol=1e-12, rtol=0.0):
        raise AssertionError("model-inadequacy covariance contains a normalization mode")
    if float(np.min(np.linalg.eigvalsh(cov))) < -1e-12:
        raise AssertionError("model-inadequacy covariance is not positive semidefinite")
    return cov


def effective_model_inadequacy_rel(include_systematics, requested=None):
    if not include_systematics or not USE_MODEL_INADEQUACY:
        return 0.0
    value = MODEL_INADEQUACY_REL if requested is None else float(requested)
    return float(value)


def q_likelihood(problem, params, data_reduced=None, include_systematics=True,
                 bank_key="joint", model_inadequacy_rel=None):
    if np.isscalar(params):
        params = {"f_dps": float(params)}
    f = float(params["f_dps"])
    if not (1.0e-6 < f < 1.0-1.0e-6):
        return 1.0e30, 1.0e30, None, None, None, np.inf
    if FREE_CC_FRACTIONS:
        for key in ("f_cc_sps", "f_cc_dps"):
            if key not in params or not (1.0e-6 < float(params[key]) < 1.0-1.0e-6):
                return 1.0e30, 1.0e30, None, None, None, np.inf

    model = central_mixture(problem, params)
    H = problem["H"]
    y = problem["data_reduced"] if data_reduced is None else np.asarray(data_reduced, dtype=float)
    residual = y - H @ model["shape"]
    cov = problem["data_cov"] + model_stat_covariance(problem, params, bank_key=bank_key)
    if include_systematics:
        cov = cov + differential_shape_systematic_covariance(problem)
    floor_rel = effective_model_inadequacy_rel(
        include_systematics, model_inadequacy_rel
    )
    if floor_rel > 0.0:
        cov = cov + model_inadequacy_shape_covariance(model["shape"], floor_rel)
    V = H @ cov @ H.T
    V = nearest_psd(V, 1.0e-12)
    quadratic = float(residual @ np.linalg.solve(V, residual))
    sign, logdet = np.linalg.slogdet(V)
    if sign <= 0 or not np.isfinite(logdet):
        return 1.0e30, 1.0e30, model, V, cov, np.inf
    # V depends on the physics parameters through the finite-model ensemble.
    # Therefore the Gaussian normalization term is part of the likelihood.

    constraint_penalty = 0.0

    if FREE_CC_FRACTIONS:
        constraint_penalty = cc_fraction_constraint_penalty(
            problem,
            params["f_cc_sps"],
            params["f_cc_dps"],
        )

    n2ll = float(
        quadratic
        + logdet
        + constraint_penalty
    )

    return n2ll, quadratic, model, V, cov, float(logdet)

def make_minuit(problem, include_systematics=True, data_reduced=None, start=None,
                model_inadequacy_rel=None):
    start_values = starting_parameters(problem, start=start)

    if FREE_CC_FRACTIONS:
        def objective(f_dps, f_cc_sps, f_cc_dps):
            params = {
                "f_dps": f_dps,
                "f_cc_sps": f_cc_sps,
                "f_cc_dps": f_cc_dps,
            }
            n2ll, _, _, _, _, _ = q_likelihood(
                problem, params, data_reduced=data_reduced,
                include_systematics=include_systematics,
                model_inadequacy_rel=model_inadequacy_rel,
            )
            return 0.5*n2ll

        m = Minuit(
            objective,
            f_dps=start_values["f_dps"],
            f_cc_sps=start_values["f_cc_sps"],
            f_cc_dps=start_values["f_cc_dps"],
        )
        for name in ("f_dps", "f_cc_sps", "f_cc_dps"):
            m.limits[name] = (1.0e-6, 1.0-1.0e-6)
    else:
        def objective(f_dps):
            params = {"f_dps": f_dps}
            n2ll, _, _, _, _, _ = q_likelihood(
                problem, params, data_reduced=data_reduced,
                include_systematics=include_systematics,
                model_inadequacy_rel=model_inadequacy_rel,
            )
            return 0.5*n2ll

        m = Minuit(objective, f_dps=start_values["f_dps"])
        m.limits["f_dps"] = (1.0e-6, 1.0-1.0e-6)

    m.errordef = 0.5
    m.print_level = 0
    return m


def fit_problem(problem, include_systematics=True, data_reduced=None,
                run_minos=True, start=None, model_inadequacy_rel=None):
    floor_rel = effective_model_inadequacy_rel(
        include_systematics, model_inadequacy_rel
    )
    m = make_minuit(
        problem, include_systematics, data_reduced, start=start,
        model_inadequacy_rel=floor_rel,
    )
    m.migrad()
    m.hesse()
    if run_minos:
        try:
            m.minos("f_dps")
        except Exception as exc:
            warnings.warn(f"MINOS failed for {problem['name']}: {exc}")

    values = {name: float(m.values[name]) for name in PARAMETER_NAMES}
    n2ll, quadratic, model, V, full_cov, logdet = q_likelihood(
        problem, values, data_reduced=data_reduced,
        include_systematics=include_systematics,
        model_inadequacy_rel=floor_rel,
    )

    if "f_dps" in m.merrors:
        e = m.merrors["f_dps"]
        down, up = float(-e.lower), float(e.upper)
        source = "MINOS"
    else:
        down = up = float(m.errors["f_dps"])
        source = "HESSE"

    errors = {name: float(m.errors[name]) for name in PARAMETER_NAMES}
    fixed_comp = fixed_composition_for_problem(problem)
    if not FREE_CC_FRACTIONS:
        values = {**values, **fixed_comp}
        errors = {**errors, "f_cc_sps": 0.0, "f_cc_dps": 0.0}

    return {
        "problem": problem,
        "include_systematics": include_systematics,
        "model_inadequacy_rel": float(floor_rel),
        "minuit": m,
        "valid": bool(m.valid),
        "values": values,
        "errors": errors,
        "model": model,
        "q": n2ll,
        "quadratic": quadratic,
        "logdet": logdet,
        "V_reduced": V,
        "full_cov": full_cov,
        "f_error_down": down,
        "f_error_up": up,
        "f_error_source": source,
        "data_reduced": problem["data_reduced"] if data_reduced is None else np.asarray(data_reduced,float),
    }


FITS={}
for name, problem in PROBLEMS.items():
    print("\n"+"="*76+f"\nFIT {name} ({len(PARAMETER_NAMES)} free parameter(s))\n"+"="*76)
    stat_fit = fit_problem(problem, include_systematics=False, run_minos=True)
    baseline_fit = fit_problem(
        problem, include_systematics=True, run_minos=True,
        start=stat_fit["values"], model_inadequacy_rel=0.0,
    )
    full_fit = fit_problem(
        problem, include_systematics=True, run_minos=True,
        start=baseline_fit["values"],
        model_inadequacy_rel=MODEL_INADEQUACY_REL,
    )
    FITS[name] = {"stat":stat_fit, "baseline":baseline_fit, "total":full_fit}
    print("valid:", full_fit["valid"])
    print("fDPS:", full_fit["values"]["f_dps"])
    if FREE_CC_FRACTIONS:
        print("f_cc(SPS):", full_fit["values"]["f_cc_sps"], "+/-", full_fit["errors"]["f_cc_sps"])
        print("f_cc(DPS):", full_fit["values"]["f_cc_dps"], "+/-", full_fit["errors"]["f_cc_dps"])
    else:
        print("fixed f_cc(SPS):", full_fit["values"]["f_cc_sps"])
        print("fixed f_cc(DPS):", full_fit["values"]["f_cc_dps"])
    print("statistical profile interval:", (-stat_fit["f_error_down"], +stat_fit["f_error_up"]))
    print("baseline fDPS (floor=0):    ", baseline_fit["values"]["f_dps"])
    print("full profile interval:       ", (-full_fit["f_error_down"], +full_fit["f_error_up"]))
    print("empirical shape floor:       ", f"{100*full_fit['model_inadequacy_rel']:.1f}%")
    print("differential sPlot syst input:", problem["splot_systematic"]["available"])
    print(
        "efficiency-response syst input:",
        problem["efficiency_response_systematic"]["available"],
    )


### Slide table — post-fit uncertainty budget

The nominal notebook contains three nested fits:

1. **statistical:** data + finite simulated response only;
2. **baseline:** statistical + approved differential systematic covariance, with the adopted MC-shape component disabled;
3. **total:** complete nominal covariance.

The table below converts these nested profile intervals into a **quadrature-equivalent impact decomposition**. This is intended for the presentation; the authoritative total interval remains the full profile-likelihood result.

In [ ]:
# ================================================================
# POST-FIT fDPS UNCERTAINTY TABLE
# ================================================================
_slide_stat = FITS[NOMINAL_BINNING]["stat"]
_slide_baseline = FITS[NOMINAL_BINNING]["baseline"]
_slide_total = FITS[NOMINAL_BINNING]["total"]


def _quadrature_increment(total, previous):
    return float(np.sqrt(max(float(total) ** 2 - float(previous) ** 2, 0.0)))


stat_down = float(_slide_stat["f_error_down"])
stat_up = float(_slide_stat["f_error_up"])
baseline_down = float(_slide_baseline["f_error_down"])
baseline_up = float(_slide_baseline["f_error_up"])
total_down = float(_slide_total["f_error_down"])
total_up = float(_slide_total["f_error_up"])
approved_diff_down = _quadrature_increment(baseline_down, stat_down)
approved_diff_up = _quadrature_increment(baseline_up, stat_up)
model_shape_down = _quadrature_increment(total_down, baseline_down)
model_shape_up = _quadrature_increment(total_up, baseline_up)
syst_equiv_down = _quadrature_increment(total_down, stat_down)
syst_equiv_up = _quadrature_increment(total_up, stat_up)

year_envelope = float(EFFICIENCY_RESPONSE_SYSTEMATICS.get(
    "year_fdps_envelope", np.nan
))
binning_envelope = float(EFFICIENCY_RESPONSE_SYSTEMATICS.get(
    "binning_fdps_envelope", np.nan
))
splot_available = bool(PROBLEMS[NOMINAL_BINNING]["splot_systematic"]["available"])

POSTFIT_UNCERTAINTY_TABLE = pd.DataFrame([
    {
        "contribution": "Statistical profile (data + finite MC/response statistics)",
        "down": stat_down, "up": stat_up, "quantity_type": "profile interval",
        "status_note": "finite MC, efficiency-map replicas, stitching and composition are jointly covaried",
    },
    {
        "contribution": "Finite MC / response statistics",
        "down": np.nan, "up": np.nan, "quantity_type": "covariance component",
        "status_note": "included in statistical profile; not reduced to an independent scalar",
    },
    {
        "contribution": "Efficiency response systematics (all approved)",
        "down": approved_diff_down, "up": approved_diff_up,
        "quantity_type": "quadrature-equivalent profile increment",
        "status_note": "year + efficiency-binning covariances; sPlot variants when available",
    },
    {
        "contribution": "Year response",
        "down": year_envelope, "up": year_envelope,
        "quantity_type": "dedicated variation envelope",
        "status_note": "temporary until 2016APV/2016 maps are available",
    },
    {
        "contribution": "Efficiency binning",
        "down": binning_envelope, "up": binning_envelope,
        "quantity_type": "dedicated variation envelope",
        "status_note": "propagated through normalized-shape covariance",
    },
    {
        "contribution": "Signal-yield / sPlot fit variants",
        "down": np.nan, "up": np.nan,
        "quantity_type": "differential covariance" if splot_available else "pending",
        "status_note": "implemented" if splot_available else "pending input; not invented",
    },
    {
        "contribution": "Adopted MC-shape component",
        "down": model_shape_down, "up": model_shape_up,
        "quantity_type": "quadrature-equivalent profile increment",
        "status_note": f"fixed adopted {100*MODEL_INADEQUACY_REL:.0f}% normalized-shape floor",
    },
    {
        "contribution": "Total profile",
        "down": total_down, "up": total_up, "quantity_type": "profile interval",
        "status_note": "nominal HIST likelihood prescription",
    },
])
POSTFIT_UNCERTAINTY_TABLE["down_percent_points"] = 100.0 * POSTFIT_UNCERTAINTY_TABLE["down"]
POSTFIT_UNCERTAINTY_TABLE["up_percent_points"] = 100.0 * POSTFIT_UNCERTAINTY_TABLE["up"]
POSTFIT_UNCERTAINTY_TABLE.to_csv(
    OUTDIR / f"postfit_fdps_uncertainty_table_{ESTIMATOR_TAG}_SLIDES.csv", index=False
)

fdps_slide_central = float(_slide_total["values"]["f_dps"])
FDPS_SLIDE_RESULT = {
    "central": fdps_slide_central,
    "stat_down": stat_down, "stat_up": stat_up,
    "syst_down": syst_equiv_down, "syst_up": syst_equiv_up,
    "total_down": total_down, "total_up": total_up,
}

display(POSTFIT_UNCERTAINTY_TABLE)
print("Quadrature-equivalent stat/syst decomposition (presentation only):")
print(
    f"f_DPS = {fdps_slide_central:.4f} "
    f"+{stat_up:.4f}/-{stat_down:.4f} (stat) "
    f"+{syst_equiv_up:.4f}/-{syst_equiv_down:.4f} (syst)"
)
print(f"Total profile: +{total_up:.4f}/-{total_down:.4f}")


## 8. Profile scan and toy-calibrated goodness of fit
¶
For the three-parameter configuration, every point of the $f_{\mathrm{DPS}}$ profile scan re-minimizes the two composition parameters. The toy GOF also refits the active parameter set for every pseudoexperiment.

In [ ]:
# ================================================================
# PROFILE-LIKELIHOOD SCANS + GOODNESS-OF-FIT TOYS
# SWAN-safe CMS labels
# ================================================================

# Important after a failed mplhep drawing:
# remove any malformed figures still registered by matplotlib.
plt.close("all")


# ----------------------------------------------------------------
# SWAN-safe CMS label
# ----------------------------------------------------------------



# ================================================================
# PROFILE LIKELIHOOD
# ================================================================

def profile_at_f(fit, f_value):
    problem = fit["problem"]
    f_value = float(f_value)

    if FREE_CC_FRACTIONS:

        m = make_minuit(
            problem,
            include_systematics=fit["include_systematics"],
            data_reduced=fit["data_reduced"],
            start=fit["values"],
            model_inadequacy_rel=fit["model_inadequacy_rel"],
        )

        m.values["f_dps"] = f_value
        m.fixed["f_dps"] = True

        m.migrad()

        params = {
            name: float(m.values[name])
            for name in PARAMETER_NAMES
        }

        n2ll, quadratic, model, _, _, _ = q_likelihood(
            problem,
            params,
            data_reduced=fit["data_reduced"],
            include_systematics=fit["include_systematics"],
            model_inadequacy_rel=fit["model_inadequacy_rel"],
        )

        valid = bool(
            m.valid
            and np.isfinite(n2ll)
        )

        return {
            "f": f_value,
            "q": float(n2ll),
            "quadratic": float(quadratic),
            "model": model,
            "params": params,
            "valid": valid,
        }

    # ------------------------------------------------------------
    # One-parameter case
    # ------------------------------------------------------------

    params = {
        "f_dps": f_value
    }

    n2ll, quadratic, model, _, _, _ = q_likelihood(
        problem,
        params,
        data_reduced=fit["data_reduced"],
        include_systematics=fit["include_systematics"],
        model_inadequacy_rel=fit["model_inadequacy_rel"],
    )

    return {
        "f": f_value,
        "q": float(n2ll),
        "quadratic": float(quadratic),
        "model": model,
        "params": params,
        "valid": bool(np.isfinite(n2ll)),
    }


def make_profile_scan(fit, n=N_PROFILE_SCAN):

    f0 = float(
        fit["values"]["f_dps"]
    )

    span = max(
        4.0 * float(fit["f_error_down"]),
        4.0 * float(fit["f_error_up"]),
        0.12,
    )

    grid = np.linspace(
        max(1e-5, f0 - span),
        min(1.0 - 1e-5, f0 + span),
        int(n),
    )

    rows = []

    for f_value in grid:

        p = profile_at_f(
            fit,
            f_value,
        )

        row = {
            "fDPS": float(f_value),
            "Q": float(p["q"]),
            "valid": bool(p["valid"]),
        }

        if (
            FREE_CC_FRACTIONS
            and p["valid"]
        ):
            row[
                "profiled_f_cc_sps"
            ] = p["params"]["f_cc_sps"]

            row[
                "profiled_f_cc_dps"
            ] = p["params"]["f_cc_dps"]

        rows.append(row)

    df = pd.DataFrame(rows)

    # ------------------------------------------------------------
    # Likelihood-ratio statistic:
    #
    # Q(f) - Q_min
    # =
    # -2 ln [ L(f, theta_hat_hat) / L(f_hat, theta_hat) ]
    # =
    # -2 ln lambda(f)
    # ------------------------------------------------------------

    q_min = float(fit["q"])

    df["deltaQ"] = (
        df["Q"] - q_min
    )

    # protect against tiny negative numerical values
    df.loc[
        np.abs(df["deltaQ"]) < 1e-10,
        "deltaQ",
    ] = 0.0

    return df


def plot_profile_scan(name, fit):

    scan = make_profile_scan(fit)

    scan.to_csv(
        OUTDIR
        / f"profile_scan_{name}.csv",
        index=False,
    )

    x = scan[
        "fDPS"
    ].to_numpy(float)

    y = scan[
        "deltaQ"
    ].to_numpy(float)

    valid = (
        scan["valid"].to_numpy(bool)
        & np.isfinite(x)
        & np.isfinite(y)
    )

    if np.count_nonzero(valid) < 3:
        raise RuntimeError(
            f"{name}: insufficient valid points "
            "in profile-likelihood scan"
        )

    fig, ax = plt.subplots(
        figsize=(7.0, 5.4)
    )
    fig.subplots_adjust(left=0.18, right=0.96, bottom=0.20, top=0.82)

    ax.plot(
        x[valid],
        y[valid],
        linewidth=1.7,
        label="Profile likelihood",
    )

    # ------------------------------------------------------------
    # For one parameter:
    #   68.3% CL -> Delta(-2 ln L) = 1
    #   95% CL   -> Delta(-2 ln L) = 3.84
    # ------------------------------------------------------------

    ax.axhline(
        1.0,
        linestyle="--",
        linewidth=1.0,
        label=r"$68.3\%$ CL",
    )

    ax.axhline(
        3.84,
        linestyle=":",
        linewidth=1.0,
        label=r"$95\%$ CL",
    )

    f_best = float(
        fit["values"]["f_dps"]
    )

    ax.axvline(
        f_best,
        linestyle="--",
        linewidth=1.0,
    )

    ax.set_xlabel(
        LABEL_FDPS,
        fontsize=AXIS_LABEL_FONT_SIZE,
    )

    # Correct statistical meaning of the vertical axis
    ax.set_ylabel(
        r"$-2\ln\mathcal{L}(f_{\mathrm{DPS}})$",
        fontsize=AXIS_LABEL_FONT_SIZE,
    )

    ax.set_ylim(
        bottom=0.0
    )


    subtitle = (
        r"profiled over "
        r"$f_{c\bar c}^{\mathrm{SPS}}$ and "
        r"$f_{c\bar c}^{\mathrm{DPS}}$"
        if FREE_CC_FRACTIONS
        else
        r"1-parameter profile"
    )

    ax.text(
        0.97,
        0.65,
        subtitle,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=14,
        bbox=dict(
            facecolor="white",
            alpha=0.88,
            edgecolor="none",
            pad=2.2,
        ),
    )

    ax.legend(
        loc="upper right",
        fontsize=14.5,
    )

    save_and_show(
        fig,
        f"profile_likelihood_scan_{name}",
        OUTDIR,
    )

    return scan


# Produce all profile scans
PROFILE_SCANS = {}

for name, bundle in FITS.items():

    print(
        f"[profile] {name}"
    )

    PROFILE_SCANS[name] = (
        plot_profile_scan(
            name,
            bundle["total"],
        )
    )


# ================================================================
# GOODNESS-OF-FIT PSEUDOEXPERIMENTS
# ================================================================

def gof_toys(
    fit,
    n_toys=N_GOF_TOYS,
    seed=RANDOM_SEED + 91,
):

    n_toys = int(n_toys)

    if n_toys <= 0:

        return {
            "pvalue": np.nan,
            "q_toys": np.array([]),
            "n_ge": 0,
            "n_requested": 0,
            "pvalue_mcse": np.nan,
            "pvalue_ci95": [
                np.nan,
                np.nan,
            ],
        }

    rng = np.random.default_rng(
        seed
    )

    problem = fit[
        "problem"
    ]

    params = fit[
        "values"
    ]

    _, _, model, V, _, _ = q_likelihood(
        problem,
        params,
        include_systematics=
            fit["include_systematics"],
        model_inadequacy_rel=
            fit["model_inadequacy_rel"],
    )

    mean = (
        problem["H"]
        @ model["shape"]
    )

    V_psd = nearest_psd(
        V,
        1e-12,
    )

    try:
        L = np.linalg.cholesky(
            V_psd
        )

    except np.linalg.LinAlgError:
        # additional extremely small stabilization
        eps = (
            1e-10
            * max(
                float(
                    np.max(
                        np.diag(V_psd)
                    )
                ),
                1.0,
            )
        )

        L = np.linalg.cholesky(
            V_psd
            + eps
            * np.eye(
                V_psd.shape[0]
            )
        )

    qtoys = []

    iterator = range(
        n_toys
    )

    if tqdm is not None:

        iterator = tqdm(
            iterator,
            desc=(
                f"GOF toys "
                f"{problem['name']}"
            ),
        )

    for _ in iterator:

        toy = (
            mean
            + L
            @ rng.normal(
                size=len(mean)
            )
        )

        tf = fit_problem(
            problem,
            include_systematics=
                fit["include_systematics"],
            data_reduced=toy,
            run_minos=False,
            start=fit["values"],
            model_inadequacy_rel=
                fit["model_inadequacy_rel"],
        )

        if (
            tf["valid"]
            and np.isfinite(
                tf["quadratic"]
            )
        ):

            qtoys.append(
                float(
                    tf["quadratic"]
                )
            )

    qtoys = np.asarray(
        qtoys,
        dtype=float,
    )

    required = int(
        math.ceil(
            MIN_DATA_REPLICA_SUCCESS_FRACTION
            * n_toys
        )
    )

    if len(qtoys) < required:

        raise RuntimeError(
            f"Only {len(qtoys)}/{n_toys} "
            "successful goodness-of-fit toys; "
            f"need at least {required}."
        )

    # ------------------------------------------------------------
    # GOF statistic:
    #
    # residual quadratic at the profiled best-fit point.
    # The determinant contribution belongs to parameter inference,
    # not to this residual-size diagnostic.
    # ------------------------------------------------------------

    observed_q = float(
        fit["quadratic"]
    )

    nge = int(
        np.count_nonzero(
            qtoys >= observed_q
        )
    )

    nt = int(
        len(qtoys)
    )

    # finite-MC corrected p value
    p = float(
        (nge + 1.0)
        / (nt + 1.0)
    )

    ci_low = (
        0.0
        if nge == 0
        else float(
            beta_distribution.ppf(
                0.025,
                nge,
                nt - nge + 1,
            )
        )
    )

    ci_high = (
        1.0
        if nge == nt
        else float(
            beta_distribution.ppf(
                0.975,
                nge + 1,
                nt - nge,
            )
        )
    )

    p_mcse = float(
        math.sqrt(
            max(
                p * (1.0 - p),
                0.0,
            )
            / (nt + 1.0)
        )
    )

    return {
        "pvalue": p,
        "q_toys": qtoys,
        "n_ge": nge,
        "n_requested": n_toys,
        "pvalue_mcse": p_mcse,
        "pvalue_ci95": [
            ci_low,
            ci_high,
        ],
    }


# ================================================================
# RUN GOF TOYS
# ================================================================

GOF = {}

for i, (name, bundle) in enumerate(
    FITS.items()
):

    result = gof_toys(
        bundle["total"],
        seed=(
            RANDOM_SEED
            + 91
            + 100 * i
        ),
    )

    GOF[name] = result

    print(
        name,
        "toy goodness-of-fit p-value =",
        f"{result['pvalue']:.3f}",
        "+/-",
        f"{result['pvalue_mcse']:.3f}",
        (
            f"(95% interval "
            f"{result['pvalue_ci95'][0]:.3f}"
            "--"
            f"{result['pvalue_ci95'][1]:.3f}; "
            f"{result['n_ge']} exceedances / "
            f"{len(result['q_toys'])} "
            "successful toys)"
        ),
    )

    np.savetxt(
        OUTDIR
        / f"gof_toys_{name}.txt",
        result["q_toys"],
    )


# ================================================================
# GOF DISTRIBUTION PLOTS
# ================================================================

def plot_gof_distribution(
    name,
    bundle,
):

    result = GOF[name]

    qtoys = np.asarray(
        result["q_toys"],
        dtype=float,
    )

    if len(qtoys) == 0:
        return

    observed = float(
        bundle["total"][
            "quadratic"
        ]
    )

    fig, ax = plt.subplots(
        figsize=(7.3, 5.2)
    )
    fig.subplots_adjust(left=0.19, right=0.97, bottom=0.21, top=0.82)

    nbins = max(
        18,
        min(
            40,
            int(
                np.sqrt(
                    len(qtoys)
                )
                * 1.4
            ),
        ),
    )

    ax.hist(
        qtoys,
        bins=nbins,
        histtype="stepfilled",
        alpha=0.35,
        label="Pseudoexperiments",
    )

    ax.axvline(
        observed,
        linewidth=2.0,
        linestyle="--",
        label="Observed",
    )

    ax.set_xlabel(
        r"Profiled residual quadratic "
        r"$Q_{\rm res}$",
        fontsize=AXIS_LABEL_FONT_SIZE,
    )

    ax.set_ylabel(
        "Number of pseudoexperiments",
        fontsize=0.9*AXIS_LABEL_FONT_SIZE,
    )


    ax.legend(
        loc="upper right",
        fontsize=14.5,
    )

    p = float(
        result["pvalue"]
    )

    ax.text(
        0.97,
        0.68,
        (
            "Goodness-of-fit: "
            + rf"$p={p:.2f}$"
            + "\n"
            + rf"$95\%\ \mathrm{{interval}}="
              rf"[{result['pvalue_ci95'][0]:.2f},"
              rf"\,{result['pvalue_ci95'][1]:.2f}]$"
            + "\n"
            + rf"$N_{{\mathrm{{toys}}}}="
              rf"{len(qtoys)}$"
        ),
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=15.13,
        bbox=dict(
            facecolor="white",
            alpha=0.88,
            edgecolor="none",
            pad=2.5,
        ),
    )

    save_and_show(
        fig,
        f"gof_toys_{name}_CMS",
        OUTDIR,
    )


for _name, _bundle in FITS.items():

    plot_gof_distribution(
        _name,
        _bundle,
    )


print(
    "\n[profile/GOF] completed successfully"
)

### Template-fit closure — injected $f_{\rm DPS}$

The existing toy GOF tests whether the fitted model describes the observed residual size. The closure test below addresses a different question:

> If a known DPS fraction is injected into pseudo-data generated with the same likelihood model, does the fit recover it without bias?

Five injected values are tested. At each point, pseudoexperiments are generated in the reduced normalized-shape space using the covariance of the nominal likelihood and are refitted with the full template-fit machinery.

The slide-level closure criteria are:

$$\langle f_{\rm DPS}^{\rm fit}\rangle
\simeq
f_{\rm DPS}^{\rm injected},$$

$$\langle {\rm pull}\rangle \simeq 0,
\qquad
\sigma({\rm pull})\simeq 1,$$

with

$${\rm pull}
=
\frac{
f_{\rm DPS}^{\rm fit}-f_{\rm DPS}^{\rm injected}
}{
\sigma(f_{\rm DPS}^{\rm fit})
}.$$

In [ ]:
# ================================================================
# TEMPLATE-FIT CLOSURE / LINEARITY TEST FOR CONVENER SLIDES
# ================================================================

def run_template_fit_closure(
    problem,
    reference_fit,
    injected_values,
    n_toys,
    seed,
):
    rng = np.random.default_rng(seed)

    toy_rows = []
    summary_rows = []

    for iinj, f_injected in enumerate(
        np.asarray(injected_values, dtype=float)
    ):
        params_injected = dict(reference_fit["values"])
        params_injected["f_dps"] = float(f_injected)

        (
            _n2ll,
            _quadratic,
            injected_model,
            V_injected,
            _full_cov,
            _logdet,
        ) = q_likelihood(
            problem,
            params_injected,
            include_systematics=reference_fit["include_systematics"],
            model_inadequacy_rel=reference_fit["model_inadequacy_rel"],
        )

        if injected_model is None:
            raise RuntimeError(
                f"Could not construct injected model for fDPS={f_injected}"
            )

        mean_reduced = (
            problem["H"]
            @ np.asarray(injected_model["shape"], dtype=float)
        )

        L = np.linalg.cholesky(
            nearest_psd(
                V_injected,
                1.0e-12,
            )
        )

        iterator = range(int(n_toys))
        if tqdm is not None:
            iterator = tqdm(
                iterator,
                desc=(
                    "template closure "
                    f"fDPS={f_injected:.2f}"
                ),
            )

        fitted = []
        fitted_err = []
        pulls = []

        for itoy in iterator:
            toy_reduced = (
                mean_reduced
                + L @ rng.normal(
                    size=len(mean_reduced)
                )
            )

            toy_fit = fit_problem(
                problem,
                include_systematics=reference_fit["include_systematics"],
                data_reduced=toy_reduced,
                run_minos=False,
                start=params_injected,
                model_inadequacy_rel=reference_fit["model_inadequacy_rel"],
            )

            if not toy_fit["valid"]:
                continue

            f_fit = float(
                toy_fit["values"]["f_dps"]
            )

            sigma_fit = float(
                toy_fit["errors"]["f_dps"]
            )

            if (
                not np.isfinite(f_fit)
                or not np.isfinite(sigma_fit)
                or sigma_fit <= 0.0
            ):
                continue

            pull = (
                f_fit - float(f_injected)
            ) / sigma_fit

            fitted.append(f_fit)
            fitted_err.append(sigma_fit)
            pulls.append(pull)

            toy_rows.append(
                {
                    "f_injected": float(f_injected),
                    "toy": int(itoy),
                    "f_fit": f_fit,
                    "sigma_fit_hesse": sigma_fit,
                    "pull": pull,
                }
            )

        fitted = np.asarray(fitted, dtype=float)
        fitted_err = np.asarray(fitted_err, dtype=float)
        pulls = np.asarray(pulls, dtype=float)

        required = int(
            math.ceil(
                MIN_DATA_REPLICA_SUCCESS_FRACTION
                * int(n_toys)
            )
        )

        if len(fitted) < required:
            raise RuntimeError(
                f"Template closure fDPS={f_injected:.2f}: "
                f"only {len(fitted)}/{n_toys} successful fits; "
                f"need at least {required}."
            )

        mean_fit = float(np.mean(fitted))
        std_fit = float(
            np.std(fitted, ddof=1)
        )

        mean_error = float(
            np.mean(fitted_err)
        )

        pull_mean = float(
            np.mean(pulls)
        )

        pull_width = float(
            np.std(pulls, ddof=1)
        )

        bias = mean_fit - float(f_injected)

        coverage68 = float(
            np.mean(
                np.abs(
                    fitted - float(f_injected)
                )
                <= fitted_err
            )
        )

        summary_rows.append(
            {
                "f_injected": float(f_injected),
                "n_success": int(len(fitted)),
                "mean_f_fit": mean_fit,
                "bias": bias,
                "rms_f_fit": std_fit,
                "error_on_mean": (
                    std_fit
                    / math.sqrt(len(fitted))
                ),
                "mean_hesse_error": mean_error,
                "pull_mean": pull_mean,
                "pull_width": pull_width,
                "coverage68_hesse": coverage68,
            }
        )

    return (
        pd.DataFrame(toy_rows),
        pd.DataFrame(summary_rows),
    )


if RUN_TEMPLATE_FIT_CLOSURE:
    closure_problem = PROBLEMS[NOMINAL_BINNING]
    closure_reference_fit = FITS[NOMINAL_BINNING]["total"]

    (
        TEMPLATE_CLOSURE_TOYS,
        TEMPLATE_CLOSURE_SUMMARY,
    ) = run_template_fit_closure(
        closure_problem,
        closure_reference_fit,
        SLIDE_CLOSURE_INJECTIONS,
        SLIDE_CLOSURE_N_TOYS,
        RANDOM_SEED + 41001,
    )

    TEMPLATE_CLOSURE_SUMMARY["bias_significance"] = np.abs(
        TEMPLATE_CLOSURE_SUMMARY["bias"] / TEMPLATE_CLOSURE_SUMMARY["error_on_mean"]
    )
    TEMPLATE_CLOSURE_SUMMARY["closure_status"] = np.where(
        (TEMPLATE_CLOSURE_SUMMARY["bias_significance"] <= 3.0)
        & (np.abs(TEMPLATE_CLOSURE_SUMMARY["pull_mean"]) <= 0.25)
        & TEMPLATE_CLOSURE_SUMMARY["pull_width"].between(0.80, 1.20)
        & TEMPLATE_CLOSURE_SUMMARY["coverage68_hesse"].between(0.60, 0.76),
        "PASS", "WARNING",
    )
    print("[closure] quantitative status:",
          TEMPLATE_CLOSURE_SUMMARY["closure_status"].value_counts().to_dict())

    TEMPLATE_CLOSURE_TOYS.to_csv(
        OUTDIR / f"templatefit_closure_toys_{ESTIMATOR_TAG}.csv",
        index=False,
    )

    TEMPLATE_CLOSURE_SUMMARY.to_csv(
        OUTDIR / f"templatefit_closure_summary_{ESTIMATOR_TAG}.csv",
        index=False,
    )

    print()
    print("=" * 88)
    print("TEMPLATE-FIT CLOSURE SUMMARY")
    print("=" * 88)
    print(
        TEMPLATE_CLOSURE_SUMMARY.to_string(
            index=False,
            formatters={
                "f_injected": lambda x: f"{x:.3f}",
                "mean_f_fit": lambda x: f"{x:.4f}",
                "bias": lambda x: f"{x:+.4f}",
                "rms_f_fit": lambda x: f"{x:.4f}",
                "error_on_mean": lambda x: f"{x:.4f}",
                "mean_hesse_error": lambda x: f"{x:.4f}",
                "pull_mean": lambda x: f"{x:+.3f}",
                "pull_width": lambda x: f"{x:.3f}",
                "coverage68_hesse": lambda x: f"{x:.3f}",
            },
        )
    )
    print("=" * 88)

    # ------------------------------------------------------------
    # Linearity plot
    # ------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(7.2, 6.2),
    )
    fig.subplots_adjust(left=0.18, right=0.96, bottom=0.18, top=0.82)

    x = TEMPLATE_CLOSURE_SUMMARY[
        "f_injected"
    ].to_numpy(float)

    y = TEMPLATE_CLOSURE_SUMMARY[
        "mean_f_fit"
    ].to_numpy(float)

    yerr = TEMPLATE_CLOSURE_SUMMARY[
        "error_on_mean"
    ].to_numpy(float)

    lo = min(
        float(np.min(x)),
        float(np.min(y)),
    ) - 0.05

    hi = max(
        float(np.max(x)),
        float(np.max(y)),
    ) + 0.05

    ax.plot(
        [lo, hi],
        [lo, hi],
        linestyle="--",
        linewidth=1.4,
        label="Perfect closure",
    )

    ax.errorbar(
        x,
        y,
        yerr=yerr,
        fmt="o",
        markersize=6.0,
        capsize=3.0,
        linewidth=1.4,
        label="Pseudoexperiment mean",
    )

    ax.set_xlabel(
        f"Injected {LABEL_FDPS}"
    )

    ax.set_ylabel(
        f"Mean fitted {LABEL_FDPS}"
    )

    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.legend(fontsize=19)


    save_and_show(
        fig,
        "templatefit_closure_linearity_CMS",
        OUTDIR,
    )

    # ------------------------------------------------------------
    # Pull mean and width
    # ------------------------------------------------------------

    fig, axes = plt.subplots(
        1, 2,
        figsize=(10.8, 4.9),
    )
    fig.subplots_adjust(left=0.13, right=0.985, bottom=0.20, top=0.77, wspace=0.24)

    axes[0].axhline(
        0.0,
        linestyle="--",
        linewidth=1.3,
    )

    axes[0].plot(
        x,
        TEMPLATE_CLOSURE_SUMMARY[
            "pull_mean"
        ].to_numpy(float),
        "o-",
        linewidth=1.4,
    )

    axes[0].set_xlabel(
        f"Injected {LABEL_FDPS}"
    )
    axes[0].set_ylabel("Pull mean")

    axes[1].axhline(
        1.0,
        linestyle="--",
        linewidth=1.3,
    )

    axes[1].plot(
        x,
        TEMPLATE_CLOSURE_SUMMARY[
            "pull_width"
        ].to_numpy(float),
        "o-",
        linewidth=1.4,
    )

    axes[1].set_xlabel(
        f"Injected {LABEL_FDPS}"
    )
    axes[1].set_ylabel("Pull width")


    save_and_show(
        fig,
        "templatefit_closure_pull_CMS",
        OUTDIR,
    )

else:
    TEMPLATE_CLOSURE_TOYS = pd.DataFrame()
    TEMPLATE_CLOSURE_SUMMARY = pd.DataFrame()
    print("[slides] template-fit closure disabled")


## 9. Quality gates and provenance

The reportability gate checks fit validity, finite-template support, toy GOF, the sPlot closure flag, efficiency campaign scope, and physical-boundary behavior of every active fit parameter. The fixed Monte Carlo shape-inadequacy covariance is an adopted analysis component and is recorded in the audit, but is not itself a reportability veto. Audit products are saved to the configuration-specific output directory.

In [ ]:
def reportability(name):
    bundle=FITS[name]; fit=bundle["total"]; problem=fit["problem"]
    reasons=[]
    if not fit["valid"]:
        reasons.append("invalid Minuit fit")
    if problem["min_neff"] < MIN_TEMPLATE_NEFF_REPORTABLE:
        reasons.append(f"minimum template N_eff={problem['min_neff']:.2f} < {MIN_TEMPLATE_NEFF_REPORTABLE}")
    if problem["stat_banks"]["joint"]["n"] < MIN_SUCCESSFUL_REPLICAS:
        reasons.append("too few successful joint statistical replicas")

    for par in PARAMETER_NAMES:
        value = float(fit["values"][par])
        if value <= 1e-3 or value >= 1-1e-3:
            reasons.append(f"{par} on/near physical boundary ({value:.4f})")

    p=GOF[name]["pvalue"]
    if not np.isfinite(p) or p < GOF_PVALUE_MIN_REPORTABLE:
        reasons.append(f"toy GOF p={p}")
    if not SPLOT_CLOSURE_VALIDATED:
        reasons.append("sPlot closure not validated")
    # The fixed model-inadequacy covariance is an adopted nominal component.
    # It remains fully documented below but is not a reportability veto.
    scope_note = (
        "full Run-2 data use the luminosity-weighted 2017+2018 efficiency "
        "response while 2016 efficiencies are pending"
        if USE_PARTIAL_RUN2_EFFICIENCY_PROXY
        else ""
    )
    return {
        "binning": name,
        "reportable": len(reasons) == 0,
        "reasons": "; ".join(reasons),
        "preliminary_efficiency_scope": bool(USE_PARTIAL_RUN2_EFFICIENCY_PROXY),
        "scope_note": scope_note,
    }

REPORTABILITY=pd.DataFrame([reportability(name) for name in FITS])
REPORTABILITY.to_csv(OUTDIR/"profile_reportability.csv",index=False)

missing_candidates=[]
if not PROBLEMS[NOMINAL_BINNING]["splot_systematic"]["available"]:
    missing_candidates.append("signal_yield_fit_variants")

# A true independent detector-efficiency closure still requires an independent
# truth/test sample.  The raw-sum reconstruction closure in the dedicated
# systematic notebook is technical validation only and is not assigned as a
# physics uncertainty.
missing_candidates.append("independent_efficiency_closure")

composition_propagation = (
    "f_cc_sps and f_cc_dps profiled as free physics parameters"
    if FREE_CC_FRACTIONS
    else "recomputed from same bootstrap fiducial totals"
)
composition_class = "profiled parameter" if FREE_CC_FRACTIONS else "statistical"

an_status = (
    "ENABLED_ABSOLUTE_ONLY" if INCLUDE_AN143_INTEGRATED_SYSTEMATICS
    else "DISABLED_BY_FLAG"
)

eff_syst_available = bool(
    PROBLEMS[NOMINAL_BINNING]["efficiency_response_systematic"]["available"]
)
year_env = float(EFFICIENCY_RESPONSE_SYSTEMATICS.get("year_fdps_envelope", np.nan))
binning_env = float(EFFICIENCY_RESPONSE_SYSTEMATICS.get("binning_fdps_envelope", np.nan))

UNCERTAINTY_INVENTORY = pd.DataFrame([
    {"source":"sWeighted data","class":"statistical","propagation":"analytic covariance + event bootstrap in post-fit replicas","status":"IMPLEMENTED"},
    {"source":"finite SPS/DPS template MC","class":"statistical","propagation":"Poisson event bootstrap","status":"IMPLEMENTED"},
    {"source":"DPS stitching boundary statistics","class":"statistical","propagation":"rederived inside same event bootstrap","status":"IMPLEMENTED"},
    {"source":"ccbar/bbbar composition","class":composition_class,"propagation":composition_propagation,"status":"IMPLEMENTED"},
    {"source":"efficiency-map finite MC","class":"statistical","propagation":"coherent bin-wise asymmetric map replicas for 2017 and 2018 before luminosity combination","status":"IMPLEMENTED_WITH_CROSS_FACTOR_FALLBACK" if not EFFICIENCY_CROSS_FACTOR_COVARIANCE_AVAILABLE else "IMPLEMENTED"},
    {"source":"2016 efficiency response","class":"analysis scope","propagation":"pending dedicated 2016APV/2016 maps; current result uses 2017+2018 luminosity-weighted response","status":"PENDING_PRELIMINARY"},
    {"source":"response-year dependence","class":"systematic differential","propagation":f"validated normalized-shape covariance; temporary until 2016; |delta fDPS| envelope={year_env:.6f}","status":"IMPLEMENTED_TEMPORARY_UNTIL_2016" if eff_syst_available else "MISSING_INPUT"},
    {"source":"efficiency-map binning stability","class":"systematic differential","propagation":f"validated normalized-shape covariance from exact raw-sum rebinning; |delta fDPS| envelope={binning_env:.6f}","status":"IMPLEMENTED" if eff_syst_available else "MISSING_INPUT"},
    {"source":"independent efficiency closure","class":"systematic differential","propagation":"requires an independent truth/test sample; technical raw-map reconstruction closure is not used as a physics uncertainty","status":"PENDING_INDEPENDENT_SAMPLE"},
    {"source":"signal-yield/sPlot fit variants","class":"systematic differential","propagation":"empirical normalized-shape covariance from alternate sWeights","status":"IMPLEMENTED" if PROBLEMS[NOMINAL_BINNING]["splot_systematic"]["available"] else "MISSING_INPUT"},
    {"source":"AN143 integrated budget","class":"systematic normalization","propagation":"fully correlated upper absolute spectrum only; cancels from shape likelihood and Data/MC","status":an_status},
    {"source":"MC-shape inadequacy covariance","class":"fixed model component","propagation":"one global relative K=I 2D covariance, normalization mode projected out; same covariance in fit, toys, projections and pulls","status":"IMPLEMENTED_FIXED_ANALYSIS_COMPONENT" if USE_MODEL_INADEQUACY else "DISABLED_BY_FLAG"},
])
UNCERTAINTY_INVENTORY.to_csv(OUTDIR/"uncertainty_inventory.csv",index=False)

for _, row in REPORTABILITY.iterrows():
    state = "REPORTABLE CANDIDATE" if row["reportable"] else "DIAGNOSTIC ONLY"
    print(f"[{row['binning']}] {state}")
    if row["reasons"]:
        for reason in str(row["reasons"]).split("; "):
            print("   -", reason)
if not EFFICIENCY_CROSS_FACTOR_COVARIANCE_AVAILABLE:
    print("   - note: efficiency cross-factor covariance is unavailable; independent-factor map-stat fallback is used")

if USE_PARTIAL_RUN2_EFFICIENCY_PROXY:
    print(
        "   - scope: full Run-2 data with 2017+2018 luminosity-weighted "
        "efficiency response; 2016 efficiency maps are pending"
    )


## 10. Post-fit prediction covariance and convener-facing spectra
¶
The post-fit prediction keeps the same separation of data statistics, model/efficiency/stitching statistics, fitted-parameter propagation, and available differential systematics. The upper-panel data uncertainty optionally includes the integrated AN normalization according to the configuration flag; the Data/MC ratio does not.

In [ ]:
# ================================================================
# POST-FIT PREDICTION + PROJECTIONS + UNCERTAINTY BREAKDOWN
# SWAN / Matplotlib-safe version
# ================================================================

plt.close("all")


# ----------------------------------------------------------------
# SWAN-safe CMS label
# ----------------------------------------------------------------



# ----------------------------------------------------------------
# SWAN-safe saver
# ----------------------------------------------------------------



# ================================================================
# PROJECTION MATRIX
# ================================================================

def projection_matrix(problem, axis):
    nx = len(problem["spec"]["dphi_edges"]) - 1
    ny = len(problem["spec"]["dy_edges"]) - 1

    if axis == "dphi":
        P = np.zeros((nx, nx * ny))

        for ix in range(nx):
            P[ix, ix * ny:(ix + 1) * ny] = 1.0

        return P, problem["spec"]["dphi_edges"]

    if axis == "dy":
        P = np.zeros((ny, nx * ny))

        for ix in range(nx):
            for iy in range(ny):
                P[iy, ix * ny + iy] = 1.0

        return P, problem["spec"]["dy_edges"]

    raise ValueError(axis)


# ================================================================
# FLAT POST-FIT STATE
# ================================================================

def flat_state(
    problem,
    params,
    data_hist,
    response,
    fid_shape,
    fid_total,
):
    if np.isscalar(params):
        params = {
            "f_dps": float(params)
        }

    f_dps = float(
        params["f_dps"]
    )

    mix = mixture_from_arrays(
        response,
        fid_shape,
        fid_total,
        f_dps,
        **mixture_parameter_kwargs(params),
    )

    selected = (
        (1.0 - f_dps) * mix["sps_response"]
        + f_dps * mix["dps_response"]
    )

    fid = (
        (1.0 - f_dps) * mix["sps_fid_shape"]
        + f_dps * mix["dps_fid_shape"]
    )

    eff = safe_divide(
        selected,
        fid,
        default=np.nan,
    )

    if (
        np.any(~np.isfinite(eff))
        or np.any(eff <= 0.0)
    ):
        raise ValueError(
            "Invalid effective efficiency"
        )

    data_fid = (
        np.asarray(data_hist, float)
        / eff
    )

    sigma_fid = float(
        np.sum(data_fid)
    )

    dps = (
        sigma_fid
        * f_dps
        * mix["dps_fid_shape"]
    )

    sps = (
        sigma_fid
        * (1.0 - f_dps)
        * mix["sps_fid_shape"]
    )

    return {
        "data": data_fid,
        "dps": dps,
        "sps": sps,
        "model": dps + sps,
        "efficiency": eff,
        "sigma_fid_pb": sigma_fid,
        "f_dps": f_dps,
        "f_cc_sps": float(
            mix["f_cc_sps"]
        ),
        "f_cc_dps": float(
            mix["f_cc_dps"]
        ),
    }


def project_flat_state(
    problem,
    state,
    axis,
):
    P, edges = projection_matrix(
        problem,
        axis,
    )

    return {
        "edges": np.asarray(
            edges,
            float,
        ),
        "data": P @ state["data"],
        "dps": P @ state["dps"],
        "sps": P @ state["sps"],
        "model": P @ state["model"],
        "P": P,
    }


# ================================================================
# EXPLICIT OBJECTIVE
# ================================================================

def _explicit_objective_setup(
    problem,
    data_shape=None,
    y_reduced=None,
):
    H = problem["H"]

    if y_reduced is None:
        y = (
            H
            @ np.asarray(
                data_shape,
                float,
            )
        )
    else:
        y = np.asarray(
            y_reduced,
            float,
        )

    return H, y


def _explicit_shape_objective(
    problem,
    H,
    y,
    model_shape,
):
    cov = (
        problem["data_cov"]
        + differential_shape_systematic_covariance(
            problem
        )
    )

    if (
        USE_MODEL_INADEQUACY
        and MODEL_INADEQUACY_REL > 0.0
    ):
        cov = (
            cov
            + model_inadequacy_shape_covariance(
                model_shape,
                MODEL_INADEQUACY_REL,
            )
        )

    V = nearest_psd(
        H @ cov @ H.T,
        1e-12,
    )

    residual = (
        y
        - H
        @ np.asarray(
            model_shape,
            float,
        )
    )

    sign, logdet = (
        np.linalg.slogdet(V)
    )

    if (
        sign <= 0
        or not np.isfinite(logdet)
    ):
        return 1e30

    return float(
        residual
        @ np.linalg.solve(
            V,
            residual,
        )
        + logdet
    )


# ================================================================
# FIT EXPLICIT MODEL
# ================================================================

def fit_parameters_against_explicit_model(
    problem,
    data_shape,
    response,
    fid_shape,
    fid_total,
    start=None,
):
    H, y = _explicit_objective_setup(
        problem,
        data_shape=data_shape,
    )

    if not FREE_CC_FRACTIONS:

        def objective(f):
            if not (
                1e-6
                < f
                < 1.0 - 1e-6
            ):
                return 1e30

            model = mixture_from_arrays(
                response,
                fid_shape,
                fid_total,
                f,
            )

            return _explicit_shape_objective(
                problem,
                H,
                y,
                model["shape"],
            )

        result = minimize_scalar(
            objective,
            bounds=(
                1e-6,
                1.0 - 1e-6,
            ),
            method="bounded",
            options={
                "xatol": 2e-6
            },
        )

        return (
            {
                "f_dps":
                    float(result.x)
            },
            bool(
                result.success
                and np.isfinite(
                    result.fun
                )
            ),
        )

    seed = starting_parameters(
        problem,
        start=start,
    )

    if start is None:
        try:
            dsps, ddps = (
                _derived_composition(
                    fid_total
                )
            )

            seed["f_cc_sps"] = float(
                dsps
            )

            seed["f_cc_dps"] = float(
                ddps
            )

        except Exception:
            pass

    def objective(
        f_dps,
        f_cc_sps,
        f_cc_dps,
    ):
        if not all(
            1e-6 < x < 1.0 - 1e-6
            for x in (
                f_dps,
                f_cc_sps,
                f_cc_dps,
            )
        ):
            return 1e30

        model = mixture_from_arrays(
            response,
            fid_shape,
            fid_total,
            f_dps,
            f_cc_sps=f_cc_sps,
            f_cc_dps=f_cc_dps,
        )

        constraint_penalty = (
            cc_fraction_constraint_penalty(
                problem,
                f_cc_sps,
                f_cc_dps,
            )
        )

        return float(
            _explicit_shape_objective(
                problem,
                H,
                y,
                model["shape"],
            )
            + constraint_penalty
        )

    m = Minuit(
        objective,
        f_dps=seed["f_dps"],
        f_cc_sps=seed["f_cc_sps"],
        f_cc_dps=seed["f_cc_dps"],
    )

    m.errordef = 1.0

    for name in (
        "f_dps",
        "f_cc_sps",
        "f_cc_dps",
    ):
        m.limits[name] = (
            1e-6,
            1.0 - 1e-6,
        )

    m.print_level = 0
    m.migrad()

    params = {
        name:
            float(m.values[name])
        for name
        in PARAMETER_NAMES
    }

    return (
        params,
        bool(
            m.valid
            and np.isfinite(
                m.fval
            )
        ),
    )


# ================================================================
# FIT DATA-ONLY REDUCED MODEL
# ================================================================

def fit_parameters_data_only_reduced(
    problem,
    y_reduced,
    start=None,
):
    H, y = _explicit_objective_setup(
        problem,
        y_reduced=y_reduced,
    )

    response, fid_shape, fid_total = (
        central_component_arrays(
            problem
        )
    )

    if not FREE_CC_FRACTIONS:

        def objective(f):
            if not (
                1e-6
                < f
                < 1.0 - 1e-6
            ):
                return 1e30

            model = mixture_from_arrays(
                response,
                fid_shape,
                fid_total,
                f,
            )

            return _explicit_shape_objective(
                problem,
                H,
                y,
                model["shape"],
            )

        result = minimize_scalar(
            objective,
            bounds=(
                1e-6,
                1.0 - 1e-6,
            ),
            method="bounded",
            options={
                "xatol": 2e-6
            },
        )

        return (
            {
                "f_dps":
                    float(result.x)
            },
            bool(
                result.success
                and np.isfinite(
                    result.fun
                )
            ),
        )

    seed = starting_parameters(
        problem,
        start=start,
    )

    def objective(
        f_dps,
        f_cc_sps,
        f_cc_dps,
    ):
        if not all(
            1e-6 < x < 1.0 - 1e-6
            for x in (
                f_dps,
                f_cc_sps,
                f_cc_dps,
            )
        ):
            return 1e30

        model = mixture_from_arrays(
            response,
            fid_shape,
            fid_total,
            f_dps,
            f_cc_sps=f_cc_sps,
            f_cc_dps=f_cc_dps,
        )

        constraint_penalty = (
            cc_fraction_constraint_penalty(
                problem,
                f_cc_sps,
                f_cc_dps,
            )
        )

        return float(
            _explicit_shape_objective(
                problem,
                H,
                y,
                model["shape"],
            )
            + constraint_penalty
        )

    m = Minuit(
        objective,
        f_dps=seed["f_dps"],
        f_cc_sps=seed["f_cc_sps"],
        f_cc_dps=seed["f_cc_dps"],
    )

    m.errordef = 1.0

    for name in (
        "f_dps",
        "f_cc_sps",
        "f_cc_dps",
    ):
        m.limits[name] = (
            1e-6,
            1.0 - 1e-6,
        )

    m.print_level = 0
    m.migrad()

    params = {
        name:
            float(m.values[name])
        for name
        in PARAMETER_NAMES
    }

    return (
        params,
        bool(
            m.valid
            and np.isfinite(
                m.fval
            )
        ),
    )


# ================================================================
# MODEL-STAT REFIT ENSEMBLE
# ================================================================

def build_model_stat_refit_ensemble(
    problem,
    n=N_PREDICTION_MODEL_REPLICAS,
):
    bank = (
        problem["stat_banks"][
            "joint"
        ]
    )

    n = min(
        int(n),
        int(bank["n"]),
    )

    if n <= 0:
        return {
            "n": 0
        }

    choose = np.unique(
        np.linspace(
            0,
            bank["n"] - 1,
            n,
            dtype=int,
        )
    )

    out = {
        "f": [],
        "f_cc_sps": [],
        "f_cc_dps": [],
        "model": [],
        "dps": [],
        "sps": [],
    }

    if tqdm is not None:
        iterator = tqdm(
            choose,
            desc=(
                "prediction model-stat refits "
                f"{problem['name']}"
            ),
        )
    else:
        iterator = choose

    start = (
        FITS[
            problem["name"]
        ]["total"]["values"]
    )

    for k in iterator:
        try:
            response = (
                bank["response"][k]
            )

            fid_shape = (
                bank["fid_shape"][k]
            )

            fid_total = (
                bank["fid_total"][k]
            )

            params, ok = (
                fit_parameters_against_explicit_model(
                    problem,
                    problem["data_shape"],
                    response,
                    fid_shape,
                    fid_total,
                    start=start,
                )
            )

            if not ok:
                continue

            state = flat_state(
                problem,
                params,
                problem["data_hist"],
                response,
                fid_shape,
                fid_total,
            )

            out["f"].append(
                params["f_dps"]
            )

            out["f_cc_sps"].append(
                state["f_cc_sps"]
            )

            out["f_cc_dps"].append(
                state["f_cc_dps"]
            )

            out["model"].append(
                state["model"]
            )

            out["dps"].append(
                state["dps"]
            )

            out["sps"].append(
                state["sps"]
            )

        except Exception:
            continue

    for key in out:
        out[key] = np.asarray(
            out[key],
            float,
        )

    out["n"] = len(
        out["f"]
    )

    required = int(
        math.ceil(
            MIN_MODEL_REPLICA_SUCCESS_FRACTION
            * len(choose)
        )
    )

    if out["n"] < required:
        raise RuntimeError(
            f"Only {out['n']}/{len(choose)} "
            "successful prediction-model refits; "
            f"need at least {required}."
        )

    return out


# ================================================================
# DATA-STAT TOYS
# ================================================================

def build_f_from_data_stat_ensemble(
    problem,
    n=N_PREDICTION_DATA_TOYS,
    seed=RANDOM_SEED + 811,
):
    H = problem["H"]

    Vd = nearest_psd(
        H
        @ problem["data_cov"]
        @ H.T,
        1e-12,
    )

    L = np.linalg.cholesky(
        Vd
    )

    response, fid_shape, fid_total = (
        central_component_arrays(
            problem
        )
    )

    rng = np.random.default_rng(
        seed
    )

    out = {
        "f": [],
        "f_cc_sps": [],
        "f_cc_dps": [],
        "model": [],
        "dps": [],
        "sps": [],
    }

    iterator = range(
        int(n)
    )

    if tqdm is not None:
        iterator = tqdm(
            iterator,
            desc=(
                "prediction f(data-stat) toys "
                f"{problem['name']}"
            ),
        )

    start = (
        FITS[
            problem["name"]
        ]["total"]["values"]
    )

    for _ in iterator:

        y = (
            problem["data_reduced"]
            + L
            @ rng.normal(
                size=H.shape[0]
            )
        )

        params, ok = (
            fit_parameters_data_only_reduced(
                problem,
                y,
                start=start,
            )
        )

        if not ok:
            continue

        state = flat_state(
            problem,
            params,
            problem["data_hist"],
            response,
            fid_shape,
            fid_total,
        )

        out["f"].append(
            params["f_dps"]
        )

        out["f_cc_sps"].append(
            state["f_cc_sps"]
        )

        out["f_cc_dps"].append(
            state["f_cc_dps"]
        )

        out["model"].append(
            state["model"]
        )

        out["dps"].append(
            state["dps"]
        )

        out["sps"].append(
            state["sps"]
        )

    for key in out:
        out[key] = np.asarray(
            out[key],
            float,
        )

    out["n"] = len(
        out["f"]
    )

    required = int(
        math.ceil(
            MIN_DATA_REPLICA_SUCCESS_FRACTION
            * int(n)
        )
    )

    if out["n"] < required:
        raise RuntimeError(
            f"Only {out['n']}/{int(n)} "
            "successful prediction-data toys; "
            f"need at least {required}."
        )

    return out


# ================================================================
# SPLOT VARIANTS
# ================================================================

def build_splot_prediction_variants(
    problem
):
    response, fid_shape, fid_total = (
        central_component_arrays(
            problem
        )
    )

    out = {
        "f": [],
        "f_cc_sps": [],
        "f_cc_dps": [],
        "data": [],
        "model": [],
        "dps": [],
        "sps": [],
    }

    syst = (
        problem[
            "splot_systematic"
        ]
    )

    if not syst["available"]:

        for key in out:
            if key in (
                "data",
                "model",
                "dps",
                "sps",
            ):
                out[key] = np.empty(
                    (
                        0,
                        problem["n_bins"],
                    )
                )

            else:
                out[key] = np.empty(0)

        out["n"] = 0

        return out

    start = (
        FITS[
            problem["name"]
        ]["total"]["values"]
    )

    for hist, shape in zip(
        syst["hists"],
        syst["shapes"],
    ):

        params, ok = (
            fit_parameters_against_explicit_model(
                problem,
                shape,
                response,
                fid_shape,
                fid_total,
                start=start,
            )
        )

        if not ok:
            continue

        state = flat_state(
            problem,
            params,
            hist,
            response,
            fid_shape,
            fid_total,
        )

        out["f"].append(
            params["f_dps"]
        )

        out["f_cc_sps"].append(
            state["f_cc_sps"]
        )

        out["f_cc_dps"].append(
            state["f_cc_dps"]
        )

        for key in (
            "data",
            "model",
            "dps",
            "sps",
        ):
            out[key].append(
                state[key]
            )

    for key in out:
        out[key] = np.asarray(
            out[key],
            float,
        )

    out["n"] = len(
        out["f"]
    )

    return out


# ================================================================
# BUILD POST-FIT PREDICTION
# ================================================================

POSTFIT_PREDICTION = {}

for i, (
    name,
    bundle,
) in enumerate(
    FITS.items()
):

    problem = (
        bundle["total"][
            "problem"
        ]
    )

    response, fid_shape, fid_total = (
        central_component_arrays(
            problem
        )
    )

    params_hat = (
        bundle["total"][
            "values"
        ]
    )

    central = flat_state(
        problem,
        params_hat,
        problem["data_hist"],
        response,
        fid_shape,
        fid_total,
    )

    model_stat = (
        build_model_stat_refit_ensemble(
            problem
        )
    )

    f_data_stat = (
        build_f_from_data_stat_ensemble(
            problem,
            seed=(
                RANDOM_SEED
                + 811
                + 100 * i
            ),
        )
    )

    splot_var = (
        build_splot_prediction_variants(
            problem
        )
    )

    V_modelstat = (
        sample_covariance_centered(
            model_stat.get(
                "model",
                np.empty(
                    (
                        0,
                        problem["n_bins"],
                    )
                ),
            ),
            problem["n_bins"],
        )
    )

    V_fdatastat = (
        sample_covariance_centered(
            f_data_stat.get(
                "model",
                np.empty(
                    (
                        0,
                        problem["n_bins"],
                    )
                ),
            ),
            problem["n_bins"],
        )
    )

    V_splot_model_syst = (
        rms_covariance_about_nominal(
            splot_var.get(
                "model",
                np.empty(
                    (
                        0,
                        problem["n_bins"],
                    )
                ),
            ),
            central["model"],
        )
    )

    model_norm = float(
        np.sum(
            central["model"]
        )
    )

    eff_syst = (
        problem[
            "efficiency_response_systematic"
        ]
    )

    if eff_syst["available"]:

        V_eff_year_syst = (
            model_norm**2
            * eff_syst[
                "year_cov"
            ]
        )

        V_eff_binning_syst = (
            model_norm**2
            * eff_syst[
                "binning_cov"
            ]
        )

    else:

        V_eff_year_syst = np.zeros(
            (
                problem["n_bins"],
                problem["n_bins"],
            )
        )

        V_eff_binning_syst = np.zeros(
            (
                problem["n_bins"],
                problem["n_bins"],
            )
        )

    V_eff_response_syst = (
        V_eff_year_syst
        + V_eff_binning_syst
    )

    V_model_syst = (
        V_splot_model_syst
        + V_eff_response_syst
    )

    V_data_syst = (
        rms_covariance_about_nominal(
            splot_var.get(
                "data",
                np.empty(
                    (
                        0,
                        problem["n_bins"],
                    )
                ),
            ),
            central["data"],
        )
    )

    V_data_stat = np.diag(
        safe_divide(
            problem["data_var"],
            central["efficiency"]**2,
            default=0.0,
        )
    )

    floor_rel = float(
        bundle["total"][
            "model_inadequacy_rel"
        ]
    )

    V_model_inadequacy = (
        model_norm**2
        * model_inadequacy_shape_covariance(
            central["model"],
            floor_rel,
        )
    )

    V_prediction_known = (
        V_modelstat
        + V_fdatastat
        + V_model_syst
    )

    V_prediction = (
        V_prediction_known
        + V_model_inadequacy
    )

    model_bias = np.zeros(
        problem["n_bins"]
    )

    if model_stat.get(
        "n",
        0,
    ) >= 2:

        model_bias = (
            np.mean(
                model_stat["model"],
                axis=0,
            )
            - central["model"]
        )

    pred_sigma_known = np.sqrt(
        np.clip(
            np.diag(
                V_prediction_known
            ),
            0,
            None,
        )
    )

    if len(
        pred_sigma_known
    ):
        bias_sigma_ratio = np.max(
            np.abs(
                safe_divide(
                    model_bias,
                    pred_sigma_known,
                    default=0.0,
                )
            )
        )

    else:
        bias_sigma_ratio = np.nan

    POSTFIT_PREDICTION[
        name
    ] = {
        "central":
            central,

        "model_stat":
            model_stat,

        "f_data_stat":
            f_data_stat,

        "splot_variants":
            splot_var,

        "V_modelstat":
            V_modelstat,

        "V_fdatastat":
            V_fdatastat,

        "V_splot_model_syst":
            V_splot_model_syst,

        "V_eff_year_syst":
            V_eff_year_syst,

        "V_eff_binning_syst":
            V_eff_binning_syst,

        "V_eff_response_syst":
            V_eff_response_syst,

        "V_model_syst":
            V_model_syst,

        "V_data_stat":
            V_data_stat,

        "V_data_syst":
            V_data_syst,

        "V_model_inadequacy":
            V_model_inadequacy,

        "V_prediction_known":
            V_prediction_known,

        "V_prediction":
            V_prediction,

        "model_bias":
            model_bias,

        "max_model_bootstrap_bias_sigma":
            float(
                bias_sigma_ratio
            ),
    }

    print(
        f"[{name}] prediction ensemble: "
        f"model-refit={model_stat.get('n',0)}, "
        f"f(data-stat)={f_data_stat.get('n',0)}, "
        f"sPlot variants={splot_var.get('n',0)}, "
        f"max bootstrap centering="
        f"{bias_sigma_ratio:.2f} sigma"
    )


# ================================================================
# PROJECT COVARIANCES
# ================================================================

def projection_covariance(
    P,
    covariance,
):
    projected = (
        P
        @ covariance
        @ P.T
    )

    return 0.5 * (
        projected
        + projected.T
    )


def make_projection_result(
    name,
    bundle,
    axis,
):
    problem = (
        bundle["total"][
            "problem"
        ]
    )

    pred = (
        POSTFIT_PREDICTION[
            name
        ]
    )

    central = project_flat_state(
        problem,
        pred["central"],
        axis,
    )

    P = central["P"]

    V_data_stat = projection_covariance(
        P,
        pred["V_data_stat"],
    )

    V_data_syst = projection_covariance(
        P,
        pred["V_data_syst"],
    )

    V_modelstat = projection_covariance(
        P,
        pred["V_modelstat"],
    )

    V_fdatastat = projection_covariance(
        P,
        pred["V_fdatastat"],
    )

    V_splot_model_syst = (
        projection_covariance(
            P,
            pred[
                "V_splot_model_syst"
            ],
        )
    )

    V_eff_year_syst = (
        projection_covariance(
            P,
            pred[
                "V_eff_year_syst"
            ],
        )
    )

    V_eff_binning_syst = (
        projection_covariance(
            P,
            pred[
                "V_eff_binning_syst"
            ],
        )
    )

    V_eff_response_syst = (
        projection_covariance(
            P,
            pred[
                "V_eff_response_syst"
            ],
        )
    )

    V_model_syst = (
        projection_covariance(
            P,
            pred[
                "V_model_syst"
            ],
        )
    )

    V_model_inadequacy = (
        projection_covariance(
            P,
            pred[
                "V_model_inadequacy"
            ],
        )
    )

    V_prediction_known = (
        projection_covariance(
            P,
            pred[
                "V_prediction_known"
            ],
        )
    )

    V_prediction = (
        projection_covariance(
            P,
            pred[
                "V_prediction"
            ],
        )
    )

    data_shape_sigma = np.sqrt(
        np.clip(
            np.diag(
                V_data_stat
                + V_data_syst
            ),
            0,
            None,
        )
    )

    data_global_sigma = (
        AN143_COMMON_REL_EFFECTIVE
        * np.abs(
            central["data"]
        )
    )

    data_upper_total_sigma = (
        np.sqrt(
            data_shape_sigma**2
            + data_global_sigma**2
        )
    )

    pred_sigma_known = np.sqrt(
        np.clip(
            np.diag(
                V_prediction_known
            ),
            0,
            None,
        )
    )

    pred_sigma = np.sqrt(
        np.clip(
            np.diag(
                V_prediction
            ),
            0,
            None,
        )
    )

    ratio = safe_divide(
        central["data"],
        central["model"],
        default=np.nan,
    )

    ratio_data_sigma = safe_divide(
        data_shape_sigma,
        central["model"],
        default=np.nan,
    )

    ratio_pred_sigma_known = (
        safe_divide(
            pred_sigma_known,
            central["model"],
            default=0.0,
        )
    )

    ratio_pred_sigma = (
        safe_divide(
            pred_sigma,
            central["model"],
            default=0.0,
        )
    )

    marginal_pull = safe_divide(
        (
            central["data"]
            - central["model"]
        ),
        np.sqrt(
            data_shape_sigma**2
            + pred_sigma**2
        ),
        default=np.nan,
    )

    return {
        **central,

        "V_data_stat":
            V_data_stat,

        "V_data_syst":
            V_data_syst,

        "V_modelstat":
            V_modelstat,

        "V_fdatastat":
            V_fdatastat,

        "V_splot_model_syst":
            V_splot_model_syst,

        "V_eff_year_syst":
            V_eff_year_syst,

        "V_eff_binning_syst":
            V_eff_binning_syst,

        "V_eff_response_syst":
            V_eff_response_syst,

        "V_model_syst":
            V_model_syst,

        "V_model_inadequacy":
            V_model_inadequacy,

        "V_prediction_known":
            V_prediction_known,

        "V_prediction":
            V_prediction,

        "data_shape_sigma":
            data_shape_sigma,

        "data_upper_total_sigma":
            data_upper_total_sigma,

        "pred_sigma_known":
            pred_sigma_known,

        "pred_sigma":
            pred_sigma,

        "ratio":
            ratio,

        "ratio_data_sigma":
            ratio_data_sigma,

        "ratio_pred_sigma_known":
            ratio_pred_sigma_known,

        "ratio_pred_sigma":
            ratio_pred_sigma,

        "marginal_pull":
            marginal_pull,
    }


# ================================================================
# STEP HELPER
# ================================================================

def step_extend(values):
    values = np.asarray(
        values,
        float,
    )

    if len(values) == 0:
        return values

    return np.r_[
        values,
        values[-1],
    ]


# ================================================================
# UNCERTAINTY BREAKDOWN PLOT
# ================================================================

def plot_prediction_breakdown(
    name,
    bundle,
    axis,
    result,
):
    edges = np.asarray(
        result["edges"],
        float,
    )

    centers = 0.5 * (
        edges[:-1]
        + edges[1:]
    )

    model = np.asarray(
        result["model"],
        float,
    )

    sources = [
        (
            "Finite model, efficiency, stitching and composition",
            result["V_modelstat"],
        ),
        (
            f"{LABEL_FDPS} response to data statistics",
            result["V_fdatastat"],
        ),
        (
            "Signal-extraction / sPlot variations",
            result["V_splot_model_syst"],
        ),
        (
            "Efficiency response: year dependence",
            result["V_eff_year_syst"],
        ),
        (
            "Efficiency response: map binning",
            result["V_eff_binning_syst"],
        ),
        (
            "All available differential systematics",
            result["V_model_syst"],
        ),
        (
            "Monte Carlo shape inadequacy",
            result["V_model_inadequacy"],
        ),
        (
            "Subtotal before the empirical term",
            result["V_prediction_known"],
        ),
        (
            "Total prediction uncertainty",
            result["V_prediction"],
        ),
    ]

    # No constrained_layout
    fig, ax = plt.subplots(
        figsize=(9.0, 5.6)
    )

    fig.subplots_adjust(
        left=0.15,
        right=0.96,
        bottom=0.20,
        top=0.84,
    )

    largest_fractional_uncertainty = 0.0

    for label, cov in sources:

        sigma = np.sqrt(
            np.clip(
                np.diag(cov),
                0,
                None,
            )
        )

        frac = (
            100.0
            * safe_divide(
                sigma,
                model,
                default=0.0,
            )
        )

        finite = frac[
            np.isfinite(frac)
        ]

        if len(finite):
            largest_fractional_uncertainty = max(
                largest_fractional_uncertainty,
                float(
                    np.max(finite)
                ),
            )

        ax.step(
            edges,
            step_extend(frac),
            where="post",
            linewidth=1.8,
            label=label,
        )

        ax.plot(
            centers,
            frac,
            marker="o",
            linestyle="none",
            markersize=3.6,
        )

    ax.set_ylabel(
        "Prediction uncertainty [%]"
    )

    ax.set_xlabel(
        XLABEL_DPHI
        if axis == "dphi"
        else
        XLABEL_DY
    )

    ax.set_ylim(
        0.0,
        max(
            1.0,
            1.85
            * largest_fractional_uncertainty,
        ),
    )


    ax.legend(
        loc="upper center",
        bbox_to_anchor=(
            0.5,
            0.94,
        ),
        ncol=2,
        fontsize=8.4,
        handlelength=2.0,
        columnspacing=1.0,
        borderpad=0.45,
    )

    save_and_show(
        fig,
        (
            f"prediction_uncertainty_breakdown_"
            f"{name}_{axis}"
        ),
        OUTDIR,
    )


# ================================================================
# FINAL CMS PROJECTION
# ================================================================

def plot_cms_projection(
    name,
    bundle,
    axis,
):
    result = make_projection_result(
        name,
        bundle,
        axis,
    )

    edges = np.asarray(
        result["edges"],
        float,
    )

    widths = np.diff(
        edges
    )

    centers = 0.5 * (
        edges[:-1]
        + edges[1:]
    )

    data = (
        result["data"]
        / widths
    )

    dps = (
        result["dps"]
        / widths
    )

    sps = (
        result["sps"]
        / widths
    )

    model = (
        result["model"]
        / widths
    )

    data_upper = (
        result[
            "data_upper_total_sigma"
        ]
        / widths
    )

    pred_sigma = (
        result["pred_sigma"]
        / widths
    )

    # ------------------------------------------------------------
    # No constrained_layout
    # ------------------------------------------------------------

    fig = plt.figure(
        figsize=(8.8, 7.2)
    )

    gs = fig.add_gridspec(
        2,
        1,
        height_ratios=[
            3.2,
            1.0,
        ],
        hspace=0.045,
        left=0.18,
        right=0.97,
        bottom=0.16,
        top=0.86,
    )

    ax = fig.add_subplot(
        gs[0, 0]
    )

    rax = fig.add_subplot(
        gs[1, 0],
        sharex=ax,
    )

    # ------------------------------------------------------------
    # DPS
    # ------------------------------------------------------------

    ax.bar(
        centers,
        dps,
        width=widths,
        align="center",
        color="#4c78a8",
        edgecolor="black",
        linewidth=0.55,
        label="Double-parton scattering",
    )

    # ------------------------------------------------------------
    # SPS
    # ------------------------------------------------------------

    ax.bar(
        centers,
        sps,
        bottom=dps,
        width=widths,
        align="center",
        color="#f2a65a",
        edgecolor="black",
        linewidth=0.55,
        label="Single-parton scattering",
    )

    # ------------------------------------------------------------
    # Total prediction uncertainty
    # ------------------------------------------------------------

    ax.fill_between(
        edges,
        step_extend(
            np.maximum(
                0.0,
                model
                - pred_sigma,
            )
        ),
        step_extend(
            model
            + pred_sigma
        ),
        step="post",
        facecolor="0.88",
        alpha=0.55,
        edgecolor="0.30",
        hatch="////",
        linewidth=0.9,
        linestyle="--",
        label="Total uncertainty",
        zorder=4,
    )

    # ------------------------------------------------------------
    # Data
    # ------------------------------------------------------------

    ax.errorbar(
        centers,
        data,
        xerr=(
            0.5
            * widths
        ),
        yerr=data_upper,
        fmt="o",
        markersize=5.3,
        color="black",
        ecolor="black",
        capsize=2.2,
        linewidth=1.25,
        label="Data",
        zorder=6,
    )

    if axis == "dphi":
        projection_ylabel = YLABEL_DPHI_XSEC
        projection_ylim = (0.0, 600.0)
    elif axis == "dy":
        projection_ylabel = YLABEL_DY_XSEC
        projection_ylim = (0.0, 700.0)
    else:
        raise ValueError(axis)

    ax.set_ylabel(
        projection_ylabel
    )

    ax.set_ylim(
        *projection_ylim
    )


    plt.setp(
        ax.get_xticklabels(),
        visible=False,
    )

    # ------------------------------------------------------------
    # Ratio uncertainty
    # ------------------------------------------------------------

    rax.fill_between(
        edges,
        step_extend(
            1.0
            - result[
                "ratio_pred_sigma"
            ]
        ),
        step_extend(
            1.0
            + result[
                "ratio_pred_sigma"
            ]
        ),
        step="post",
        facecolor="0.88",
        alpha=0.55,
        edgecolor="0.30",
        hatch="////",
        linewidth=0.9,
        linestyle="--",
        zorder=1,
    )

    # ------------------------------------------------------------
    # Data / prediction
    # ------------------------------------------------------------

    rax.errorbar(
        centers,
        result["ratio"],
        xerr=(
            0.5
            * widths
        ),
        yerr=result[
            "ratio_data_sigma"
        ],
        fmt="o",
        markersize=4.6,
        color="black",
        ecolor="black",
        capsize=2.0,
        linewidth=1.0,
        zorder=3,
    )

    rax.axhline(
        1.0,
        linestyle="--",
        color="black",
        linewidth=1.2,
        zorder=2,
    )

    rax.set_ylabel(
        YLABEL_DATA_PREDICTION
    )

    rax.set_xlabel(
        XLABEL_DPHI
        if axis == "dphi"
        else
        XLABEL_DY
    )

    rax.set_ylim(
        0.0,
        2.0,
    )

    # ------------------------------------------------------------
    # Fit information
    # ------------------------------------------------------------

    fit = bundle["total"]

    gof = float(
        GOF[name][
            "pvalue"
        ]
    )

    info_text = (
        rf"$f_{{\mathrm{{DPS}}}}="
        rf"{100*fit['values']['f_dps']:.1f}"
        rf"^{{+{100*fit['f_error_up']:.1f}}}"
        rf"_{{-{100*fit['f_error_down']:.1f}}}$ %"
        "\n"
        + "Goodness-of-fit: "
        + rf"$p={gof:.2f}$"
    )

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(
            0.5,
            0.88,
        ),
        ncol=2,
        fontsize=16.13,
        handlelength=1.8,
        columnspacing=1.0,
        borderpad=0.45,
    )

    ax.text(
        0.98,
        0.60,
        info_text,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=20.0,
        linespacing=1.25,
        bbox=dict(
            facecolor="white",
            alpha=0.90,
            edgecolor="none",
            pad=2.8,
        ),
        zorder=10,
    )

    stem = (
        f"profile_{name}_{axis}_CMS_FINAL"
    )

    save_and_show(
        fig,
        stem,
        OUTDIR,
    )

    # ------------------------------------------------------------
    # Breakdown plot
    # ------------------------------------------------------------

    plot_prediction_breakdown(
        name,
        bundle,
        axis,
        result,
    )

    # ------------------------------------------------------------
    # Save numerical budget
    # ------------------------------------------------------------

    budget = pd.DataFrame(
        {
            "bin_low":
                edges[:-1],

            "bin_high":
                edges[1:],

            "data":
                result["data"],

            "model":
                result["model"],

            "ratio":
                result["ratio"],

            "data_shape_sigma":
                result[
                    "data_shape_sigma"
                ],

            "data_upper_total_sigma":
                result[
                    "data_upper_total_sigma"
                ],

            "prediction_sigma":
                result[
                    "pred_sigma"
                ],

            "prediction_known_sigma":
                result[
                    "pred_sigma_known"
                ],

            "prediction_model_inadequacy_sigma":
                np.sqrt(
                    np.clip(
                        np.diag(
                            result[
                                "V_model_inadequacy"
                            ]
                        ),
                        0,
                        None,
                    )
                ),

            "prediction_modelstat_sigma":
                np.sqrt(
                    np.clip(
                        np.diag(
                            result[
                                "V_modelstat"
                            ]
                        ),
                        0,
                        None,
                    )
                ),

            "prediction_fdatastat_sigma":
                np.sqrt(
                    np.clip(
                        np.diag(
                            result[
                                "V_fdatastat"
                            ]
                        ),
                        0,
                        None,
                    )
                ),

            "prediction_diff_syst_sigma":
                np.sqrt(
                    np.clip(
                        np.diag(
                            result[
                                "V_model_syst"
                            ]
                        ),
                        0,
                        None,
                    )
                ),

            "marginal_pull_data_minus_mc":
                result[
                    "marginal_pull"
                ],
        }
    )

    budget.to_csv(
        OUTDIR
        / (
            f"prediction_budget_"
            f"{name}_{axis}.csv"
        ),
        index=False,
    )

    return result


# ================================================================
# RUN ALL PROJECTIONS
# ================================================================

PROJECTION_RESULTS = {}

for _name, _bundle in FITS.items():

    PROJECTION_RESULTS[
        (_name, "dphi")
    ] = plot_cms_projection(
        _name,
        _bundle,
        "dphi",
    )

    PROJECTION_RESULTS[
        (_name, "dy")
    ] = plot_cms_projection(
        _name,
        _bundle,
        "dy",
    )


print(
    "\n[postfit prediction] completed successfully"
)


## 11. Local 2D residual map
¶
The two marginal-residual maps show the baseline known uncertainties and the same fit after adding the single empirical normalized-shape floor. They are local diagnostics; the global compatibility statement remains the toy-calibrated GOF.

In [ ]:
def _local_marginal_pull(problem, fit, include_floor):
    params = fit["values"]
    model = central_mixture(problem, params)["shape"]
    cov = problem["data_cov"] + model_stat_covariance(
        problem, params, bank_key="joint"
    )
    if fit["include_systematics"]:
        cov = cov + differential_shape_systematic_covariance(problem)
    if include_floor and fit["model_inadequacy_rel"] > 0.0:
        cov = cov + model_inadequacy_shape_covariance(
            model, fit["model_inadequacy_rel"]
        )
    sigma = np.sqrt(np.clip(np.diag(cov), 0, None))
    return safe_divide(problem["data_shape"] - model, sigma, default=np.nan)


def plot_local_pull_map(name, bundle):
    problem = bundle["total"]["problem"]
    baseline_pull = _local_marginal_pull(problem, bundle["baseline"], False)
    adjusted_pull = _local_marginal_pull(problem, bundle["total"], True)

    nx = len(problem["spec"]["dphi_edges"]) - 1
    ny = len(problem["spec"]["dy_edges"]) - 1
    matrices = [baseline_pull.reshape(nx, ny), adjusted_pull.reshape(nx, ny)]
    titles = [
        r"$\mathrm{Known\ uncertainties}$",
        rf"$\mathrm{{Total\ uncertainty}}\;"
        rf"(\delta_{{\mathrm{{model}}}}^{{\mathrm{{shape}}}}={100*bundle['total']['model_inadequacy_rel']:.0f}\%)$",
    ]
    vmax = max(3.0, min(8.0, float(np.nanmax(np.abs(baseline_pull)))))

    fig, axes = plt.subplots(1, 2, figsize=(11.8, 5.2))
    fig.subplots_adjust(left=0.08, right=0.91, bottom=0.16, top=0.78, wspace=0.20)
    mesh = None
    for ax, matrix, title in zip(axes, matrices, titles):
        mesh = ax.pcolormesh(
            problem["spec"]["dphi_edges"], problem["spec"]["dy_edges"],
            matrix.T, shading="auto", cmap="coolwarm", vmin=-vmax, vmax=vmax,
        )
        for ix in range(nx):
            for iy in range(ny):
                xc = 0.5*(problem["spec"]["dphi_edges"][ix]+problem["spec"]["dphi_edges"][ix+1])
                yc = 0.5*(problem["spec"]["dy_edges"][iy]+problem["spec"]["dy_edges"][iy+1])
                ax.text(xc, yc, f"{matrix[ix,iy]:+.1f}", ha="center", va="center", fontsize=9)
        ax.text(
            0.5, 0.97, title, transform=ax.transAxes, ha="center", va="top",
            fontsize=11.0,
            bbox=dict(facecolor="white", alpha=0.84, edgecolor="none", pad=2.0),
            zorder=10,
        )
        ax.set_xlabel(XLABEL_DPHI, fontsize=AXIS_LABEL_FONT_SIZE)
        ax.set_ylabel(XLABEL_DY, fontsize=AXIS_LABEL_FONT_SIZE)
        ax.tick_params(labelsize=11)
    cbar = fig.colorbar(mesh, ax=axes, shrink=0.88)
    cbar.set_label(r"$(\mathrm{Data}-\mathrm{MC})/\sigma_{\mathrm{marg}}$", fontsize=12)
    cbar.ax.tick_params(labelsize=10)
    save_and_show(fig, f"profile_{name}_2D_local_pull_FINAL", OUTDIR)
    return {
        "baseline": matrices[0],
        "adjusted": matrices[1],
        "max_abs_baseline": float(np.nanmax(np.abs(baseline_pull))),
        "max_abs_adjusted": float(np.nanmax(np.abs(adjusted_pull))),
    }


LOCAL_PULL_MAPS = {name: plot_local_pull_map(name, bundle) for name, bundle in FITS.items()}


In [ ]:
# ============================================================
# POST-FIT CONDITIONAL TEMPLATE DIAGNOSTIC
#
# Differential fiducial cross section in |Delta phi|
# after a configurable |Delta y| cut.
#
# IMPORTANT:
#   this is a CONDITIONAL distribution:
#
#       d sigma_fid / d|Delta phi|
#       for |Delta y| >= DPS_CUT
#
# Therefore its integral is NOT the full fiducial cross section.
# ============================================================


DPS_CUT = 2.2

# Number of Delta-phi bins in this conditional diagnostic.
N_DPHI_DIAG_BINS = 4


problem_diag = FITS[NOMINAL_BINNING]["total"]["problem"]
params_diag = FITS[NOMINAL_BINNING]["total"]["values"].copy()

postfit_diag = central_mixture(
    problem_diag,
    params_diag,
)

f_dps_diag = float(params_diag["f_dps"])
f_cc_sps_diag = float(postfit_diag["f_cc_sps"])
f_cc_dps_diag = float(postfit_diag["f_cc_dps"])


print(
    "=== POST-FIT CONDITIONAL DELTA-PHI DIAGNOSTIC ==="
)
print(
    f"|Delta y| >= {DPS_CUT:.3f}"
)
print(
    f"post-fit fDPS      = {f_dps_diag:.6f}"
)
print(
    f"post-fit f_cc(SPS) = {f_cc_sps_diag:.6f}"
)
print(
    f"post-fit f_cc(DPS) = {f_cc_dps_diag:.6f}"
)


# ------------------------------------------------------------
# 1. Effective post-fit efficiency in every fitted 2D bin
# ------------------------------------------------------------

selected_flat_diag = (
    (1.0 - f_dps_diag)
    * postfit_diag["sps_response"]
    +
    f_dps_diag
    * postfit_diag["dps_response"]
)

fid_flat_diag = (
    (1.0 - f_dps_diag)
    * postfit_diag["sps_fid_shape"]
    +
    f_dps_diag
    * postfit_diag["dps_fid_shape"]
)

eff_flat_diag = safe_divide(
    selected_flat_diag,
    fid_flat_diag,
    default=np.nan,
)

if (
    np.any(~np.isfinite(eff_flat_diag))
    or
    np.any(eff_flat_diag <= 0.0)
):
    raise RuntimeError(
        "Invalid effective efficiency in post-fit diagnostic."
    )


# ------------------------------------------------------------
# 2. Efficiency-corrected sWeighted data
#
# data_selected_contributions already contains:
#
#       sWeight / (L * BR)
#
# so after division by efficiency these weights are already
# fiducial cross-section contributions in pb.
# ------------------------------------------------------------

data_idx_diag, data_valid_diag, _ = assign_bins(
    data_sw,
    problem_diag["spec"],
)

data_weight_diag = np.zeros(
    len(data_sw),
    dtype=float,
)

base_data_weight_diag = (
    data_selected_contributions(data_sw)
)

data_weight_diag[data_valid_diag] = (
    base_data_weight_diag[data_valid_diag]
    /
    eff_flat_diag[
        data_idx_diag[data_valid_diag]
    ]
)


phi_data_diag = (
    data_sw["delta_phi"].to_numpy(float)
)

dy_data_diag = (
    data_sw["delta_y"].to_numpy(float)
)


data_domain_diag = (
    data_valid_diag
    & np.isfinite(phi_data_diag)
    & np.isfinite(dy_data_diag)
    & (phi_data_diag >= 0.0)
    & (phi_data_diag <= DPHI_MAX)
)

data_cut_diag = (
    data_domain_diag
    & (dy_data_diag >= DPS_CUT)
)


# ------------------------------------------------------------
# 3. Delta-phi binning
# ------------------------------------------------------------

phi_edges_diag = np.linspace(
    0.0,
    DPHI_MAX,
    N_DPHI_DIAG_BINS + 1,
)

phi_centers_diag = (
    0.5
    * (
        phi_edges_diag[:-1]
        + phi_edges_diag[1:]
    )
)

phi_widths_diag = np.diff(
    phi_edges_diag
)


# ------------------------------------------------------------
# 4. Fiducial MC shapes
#
# Each physics component is first normalized over the COMPLETE
# fiducial domain. The Delta-y cut is only applied afterwards.
#
# Therefore the fitted global fDPS retains its original meaning.
# ------------------------------------------------------------

component_hist_diag = {}


for component in COMPONENTS:

    h_component = np.zeros(
        len(phi_edges_diag) - 1,
        dtype=float,
    )

    total_component = 0.0


    for sample, frame in MC_SAMPLE_CACHE.items():

        if (
            SAMPLE_INFO[sample]["component"]
            != component
        ):
            continue


        _, mc_valid_diag, _ = assign_bins(
            frame,
            problem_diag["spec"],
        )


        selected_weight = (
            frame[
                "mc_weight_xsec_only"
            ].to_numpy(float)
            *
            CENTRAL_STITCH_SCALES.get(
                sample,
                1.0,
            )
            /
            (
                LUMI_RUN2_PB_INV
                * BRANCHING_PRODUCT
            )
        )


        component_eff = (
            evaluate_efficiency_product(
                frame,
                EFFICIENCY_SOURCE_FOR_COMPONENT[
                    component
                ],
            )
        )


        fid_weight = (
            selected_weight
            /
            np.maximum(
                component_eff,
                EFFICIENCY_FLOOR,
            )
        )


        phi_mc = (
            frame[
                "delta_phi"
            ].to_numpy(float)
        )

        dy_mc = (
            frame[
                "delta_y"
            ].to_numpy(float)
        )


        mc_domain = (
            mc_valid_diag
            & np.isfinite(phi_mc)
            & np.isfinite(dy_mc)
            & np.isfinite(fid_weight)
            & (phi_mc >= 0.0)
            & (phi_mc <= DPHI_MAX)
        )


        # Full fiducial normalization of the component
        total_component += float(
            np.sum(
                fid_weight[mc_domain]
            )
        )


        # Conditional region used only for the projection
        mc_cut = (
            mc_domain
            & (dy_mc >= DPS_CUT)
        )


        h_component += np.histogram(
            phi_mc[mc_cut],
            bins=phi_edges_diag,
            weights=fid_weight[mc_cut],
        )[0]


    if total_component <= 0.0:
        raise RuntimeError(
            f"Non-positive normalization for {component}"
        )


    component_hist_diag[component] = (
        h_component
        / total_component
    )


# ------------------------------------------------------------
# 5. Post-fit SPS and DPS decomposition
# ------------------------------------------------------------

h_sps_shape_diag = (
    f_cc_sps_diag
    * component_hist_diag["SPS-ccbar"]
    +
    (1.0 - f_cc_sps_diag)
    * component_hist_diag["SPS-bbbar"]
)

h_dps_shape_diag = (
    f_cc_dps_diag
    * component_hist_diag["DPS-ccbar"]
    +
    (1.0 - f_cc_dps_diag)
    * component_hist_diag["DPS-bbbar"]
)


# FULL fiducial data normalization
sigma_data_diag = float(
    np.sum(
        data_weight_diag[
            data_domain_diag
        ]
    )
)


h_sps_diag = (
    sigma_data_diag
    * (1.0 - f_dps_diag)
    * h_sps_shape_diag
)

h_dps_diag = (
    sigma_data_diag
    * f_dps_diag
    * h_dps_shape_diag
)

h_model_diag = (
    h_sps_diag
    + h_dps_diag
)


# ------------------------------------------------------------
# 6. Data histogram
#
# h_data_diag is sigma per bin [pb].
# ------------------------------------------------------------

h_data_diag = np.histogram(
    phi_data_diag[data_cut_diag],
    bins=phi_edges_diag,
    weights=data_weight_diag[
        data_cut_diag
    ],
)[0]


h_data_var_diag = np.histogram(
    phi_data_diag[data_cut_diag],
    bins=phi_edges_diag,
    weights=(
        data_weight_diag[
            data_cut_diag
        ]
        ** 2
    ),
)[0]


h_data_err_diag = np.sqrt(
    np.clip(
        h_data_var_diag,
        0.0,
        None,
    )
)


# ------------------------------------------------------------
# 7. Convert EVERYTHING into differential cross section
#
#     d sigma / d|Delta phi| = sigma_bin / Delta phi_bin
# ------------------------------------------------------------

data_diff_diag = (
    h_data_diag
    / phi_widths_diag
)

data_diff_err_diag = (
    h_data_err_diag
    / phi_widths_diag
)

model_diff_diag = (
    h_model_diag
    / phi_widths_diag
)

dps_diff_diag = (
    h_dps_diag
    / phi_widths_diag
)

sps_diff_diag = (
    h_sps_diag
    / phi_widths_diag
)


# Purity is dimensionless and unchanged by dividing by bin width.
purity_phi_diag = safe_divide(
    h_dps_diag,
    h_model_diag,
    default=np.nan,
)


# ------------------------------------------------------------
# 8. Integrated cross-section checks
# ------------------------------------------------------------

sigma_conditional_data_diag = float(
    np.sum(
        data_diff_diag
        * phi_widths_diag
    )
)

sigma_conditional_model_diag = float(
    np.sum(
        model_diff_diag
        * phi_widths_diag
    )
)


print(
    f"Full fiducial data cross section       = "
    f"{sigma_data_diag:.3f} pb"
)

print(
    f"Data cross section for |Delta y| >= "
    f"{DPS_CUT:.2f} = "
    f"{sigma_conditional_data_diag:.3f} pb"
)

print(
    f"Post-fit cross section for |Delta y| >= "
    f"{DPS_CUT:.2f} = "
    f"{sigma_conditional_model_diag:.3f} pb"
)

print(
    f"Data fraction with |Delta y| >= "
    f"{DPS_CUT:.2f}: "
    f"{sigma_conditional_data_diag/sigma_data_diag:.4f}"
)

print(
    f"Post-fit MC fraction with |Delta y| >= "
    f"{DPS_CUT:.2f}: "
    f"{sigma_conditional_model_diag/sigma_data_diag:.4f}"
)


# ------------------------------------------------------------
# 9. Plot
# ------------------------------------------------------------

fig, (ax, axp) = plt.subplots(
    2,
    1,
    figsize=(9.0, 7.6),
    gridspec_kw={
        "height_ratios": [3.2, 1.0],
        "hspace": 0.06,
    },
    sharex=True,
)
fig.subplots_adjust(left=0.16, right=0.97, bottom=0.14, top=0.83, hspace=0.07)


# Data
ax.errorbar(
    phi_centers_diag,
    data_diff_diag,
    yerr=data_diff_err_diag,
    xerr=0.5 * phi_widths_diag,
    fmt="o",
    markersize=4.5,
    linewidth=1.0,
    capsize=0,
    label="Data",
    zorder=5,
)


# Post-fit decomposition
ax.stairs(
    model_diff_diag,
    phi_edges_diag,
    linewidth=2.0,
    label="Post-fit SPS + DPS",
)

ax.stairs(
    dps_diff_diag,
    phi_edges_diag,
    linewidth=1.7,
    linestyle="--",
    label="Post-fit DPS",
)

ax.stairs(
    sps_diff_diag,
    phi_edges_diag,
    linewidth=1.7,
    linestyle=":",
    label="Post-fit SPS",
)


# DPS-enriched candidate boundaries
for axis in (ax, axp):

    axis.axvline(
        np.pi / 3.0,
        linestyle="--",
        linewidth=1.0,
        alpha=0.55,
    )

    axis.axvline(
        np.pi / 2.0,
        linestyle="-.",
        linewidth=1.2,
        alpha=0.70,
    )

    axis.axvline(
        2.0 * np.pi / 3.0,
        linestyle="--",
        linewidth=1.0,
        alpha=0.55,
    )


# CMS labels



ax.text(
    0.98,
    0.84,
    rf"$|\Delta y| \geq {DPS_CUT:.2f}$"
    + "\n"
    + rf"$f_{{\mathrm{{DPS}}}}^{{\mathrm{{post-fit}}}}="
      rf"{f_dps_diag:.3f}$",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=11.5,
)


ax.set_ylabel(
    YLABEL_DPHI_XSEC,
    fontsize=AXIS_LABEL_FONT_SIZE,
)


ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.50, 0.82),
    frameon=False,
    ncol=2,
    fontsize=10.5,
)


# Purity panel
axp.stairs(
    purity_phi_diag,
    phi_edges_diag,
    linewidth=1.8,
)

axp.set_ylim(
    0.7,
    0.9,
)

axp.set_ylabel(
    LABEL_PDPS,
    fontsize=20,
)

axp.set_xlabel(
    XLABEL_DPHI,
    fontsize=20,
)

axp.grid(
    axis="y",
    alpha=0.20,
)


save_and_show(
    fig,
    f"templatefit_postfit_dphi_after_dy_"
    f"{DPS_CUT:.2f}_differential",
    OUTDIR,
)


## 12. Configurable data-driven DPS-enriched cross-check
¶
The mixed-event data-driven method can use either an automatically optimized
DPS-enriched normalization region or a predeclared fixed region.
With
AUTO_DD_REGION=True
, rectangular regions in
$(|\Delta\phi|,|\Delta y|)$ are scanned and the maximum post-fit DPS-purity
candidate passing the predeclared statistical-support and mixed-event closure
requirements is selected.
With
AUTO_DD_REGION=False
, the normalization region is fixed to
$|\Delta y|\geq1.8$ and
$\pi/3\leq|\Delta\phi|\leq2\pi/3$.
The data-driven result is never used to select the automatic region.

In [ ]:
# ================================================================
# LEGACY COFFEA MIXING INPUT LOADER
# Robust compatibility with old Coffea 0.7.x .coffea files
# ================================================================

import gc
import pickle
import collections
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


# ----------------------------------------------------------------
# Legacy Coffea accumulator classes
# ----------------------------------------------------------------

class _LegacyCoffeaObject:
    """
    Generic placeholder for old Coffea objects that are irrelevant
    for the mixing extraction, e.g. coffea.hist objects.
    """

    def __new__(cls, *args, **kwargs):
        return object.__new__(cls)

    def __init__(self, *args, **kwargs):
        pass

    def __setstate__(self, state):
        if isinstance(state, dict):
            self.__dict__.update(state)

        elif (
            isinstance(state, tuple)
            and len(state) == 2
        ):
            dict_state, slot_state = state

            if isinstance(dict_state, dict):
                self.__dict__.update(dict_state)

            if isinstance(slot_state, dict):
                self.__dict__.update(slot_state)

        else:
            self.__dict__["_legacy_state"] = state


class _LegacyColumnAccumulator(_LegacyCoffeaObject):

    def __init__(self, value=None):
        if value is not None:
            self._value = np.asarray(value)

    @property
    def value(self):
        if hasattr(self, "_value"):
            return self._value

        if "value" in self.__dict__:
            return self.__dict__["value"]

        raise AttributeError(
            "Legacy column_accumulator has no stored value"
        )


class _LegacyValueAccumulator(_LegacyCoffeaObject):

    def __init__(
        self,
        default_factory=None,
        initial=None,
    ):
        self.default_factory = default_factory

        if initial is not None:
            self._value = initial
        elif callable(default_factory):
            self._value = default_factory()

    @property
    def value(self):
        if hasattr(self, "_value"):
            return self._value

        if "value" in self.__dict__:
            return self.__dict__["value"]

        return None


class _LegacyDictAccumulator(dict):
    pass


class _LegacyListAccumulator(list):
    pass


class _LegacySetAccumulator(set):
    pass


class _LegacyDefaultDictAccumulator(collections.defaultdict):
    pass


_LEGACY_ACCUMULATOR_CLASSES = {
    "AccumulatorABC": _LegacyCoffeaObject,
    "column_accumulator": _LegacyColumnAccumulator,
    "value_accumulator": _LegacyValueAccumulator,
    "dict_accumulator": _LegacyDictAccumulator,
    "list_accumulator": _LegacyListAccumulator,
    "set_accumulator": _LegacySetAccumulator,
    "defaultdict_accumulator": _LegacyDefaultDictAccumulator,
}


# ----------------------------------------------------------------
# Dynamically generated dummy classes for old coffea.hist objects
# ----------------------------------------------------------------

_LEGACY_DUMMY_CLASSES = {}


def _legacy_dummy_class(module, name):

    key = (module, name)

    if key not in _LEGACY_DUMMY_CLASSES:

        cls = type(
            str(name),
            (_LegacyCoffeaObject,),
            {},
        )

        cls.__module__ = __name__

        _LEGACY_DUMMY_CLASSES[key] = cls

    return _LEGACY_DUMMY_CLASSES[key]


# ----------------------------------------------------------------
# Custom unpickler
# ----------------------------------------------------------------

class _LegacyCoffeaUnpickler(pickle.Unpickler):

    def find_class(self, module, name):

        # Old Coffea accumulator implementation
        if module == "coffea.processor.accumulator":

            return _LEGACY_ACCUMULATOR_CLASSES.get(
                name,
                _legacy_dummy_class(module, name),
            )

        # Old coffea.hist disappeared from recent Coffea versions.
        # We do not use these objects here, so they can be represented
        # by lightweight placeholders during unpickling.
        if module.startswith("coffea.hist"):

            return _legacy_dummy_class(module, name)

        # Try normal import first for everything else.
        try:
            return super().find_class(module, name)

        except (
            ModuleNotFoundError,
            ImportError,
            AttributeError,
        ):
            # Other obsolete Coffea internals are not needed for
            # MixingInput. Keep the file readable with placeholders.
            if module.startswith("coffea."):

                return _legacy_dummy_class(
                    module,
                    name,
                )

            raise


# ----------------------------------------------------------------
# Direct reader of .coffea = LZ4-compressed pickle
# ----------------------------------------------------------------

def load_legacy_coffea(path):

    try:
        import lz4.frame
    except Exception as exc:
        raise RuntimeError(
            "The Python package 'lz4' is required to read "
            "legacy .coffea files."
        ) from exc

    path = Path(path)

    with lz4.frame.open(
        str(path),
        mode="rb",
    ) as stream:

        return _LegacyCoffeaUnpickler(
            stream
        ).load()


# ----------------------------------------------------------------
# Convert old/new accumulator to flat NumPy array
# ----------------------------------------------------------------

def accumulator_to_numpy(
    obj,
    dtype=np.float32,
):

    if hasattr(obj, "value"):
        obj = obj.value

    elif hasattr(obj, "_value"):
        obj = obj._value

    if hasattr(obj, "to_numpy"):
        obj = obj.to_numpy()

    array = np.asarray(
        obj,
        dtype=dtype,
    )

    return array.reshape(-1)


# ----------------------------------------------------------------
# Bounded priority reservoir
# ----------------------------------------------------------------

def _priority_reservoir_update(
    reservoir,
    priorities,
    chunk,
    chunk_priorities,
    max_size,
):

    max_size = int(max_size)

    if max_size <= 0:
        raise ValueError(
            "Reservoir maximum size must be positive"
        )

    n_chunk = len(chunk_priorities)

    if n_chunk == 0:
        return reservoir, priorities

    if reservoir is None:

        reservoir = {
            key: np.asarray(value).copy()
            for key, value in chunk.items()
        }

        priorities = np.asarray(
            chunk_priorities,
            dtype=np.float32,
        ).copy()

    else:

        reservoir = {
            key: np.concatenate(
                [
                    reservoir[key],
                    np.asarray(chunk[key]),
                ]
            )
            for key in reservoir
        }

        priorities = np.concatenate(
            [
                priorities,
                np.asarray(
                    chunk_priorities,
                    dtype=np.float32,
                ),
            ]
        )

    if len(priorities) > max_size:

        keep = np.argpartition(
            priorities,
            -max_size,
        )[-max_size:]

        priorities = priorities[keep]

        reservoir = {
            key: value[keep]
            for key, value in reservoir.items()
        }

    return reservoir, priorities


# ----------------------------------------------------------------
# Load all mixing inputs
# ----------------------------------------------------------------

def load_mixing_input_reservoirs(
    input_dir,
    pattern,
    max_jpsi,
    max_dstar,
    seed,
):

    files = sorted(
        Path(input_dir).glob(pattern)
    )

    if not files:
        raise FileNotFoundError(
            f"No mixing inputs found: "
            f"{Path(input_dir) / pattern}"
        )

    print(
        f"[mixing] Found {len(files)} legacy "
        ".coffea input files"
    )

    rng = np.random.default_rng(seed)

    j_res = None
    d_res = None

    j_prio = None
    d_prio = None

    failed = []

    raw_j = 0
    raw_d = 0
    selected_j = 0
    selected_d = 0

    iterator = enumerate(files)

    if tqdm is not None:
        iterator = tqdm(
            iterator,
            total=len(files),
            desc="Bounded mixing reservoirs",
        )

    required_jpsi = [
        "jpsi_pre_pt",
        "jpsi_pre_phi",
        "jpsi_pre_rap",
        "jpsi_pre_mass",
        "jpsi_pre_dl",
    ]

    required_dstar = [
        "dstar_pre_pt",
        "dstar_pre_phi",
        "dstar_pre_rap",
        "dstar_pre_mass",
    ]

    for file_index, path in iterator:

        try:

            payload = load_legacy_coffea(path)

            if "MixingInput" not in payload:
                raise KeyError(
                    "Missing MixingInput"
                )

            mixing = payload["MixingInput"]

            missing = [
                key
                for key in (
                    required_jpsi
                    + required_dstar
                )
                if key not in mixing
            ]

            if missing:
                raise KeyError(
                    f"Missing MixingInput fields: "
                    f"{missing}"
                )

            j = {
                "jpsi_pt":
                    accumulator_to_numpy(
                        mixing["jpsi_pre_pt"]
                    ),

                "jpsi_phi":
                    accumulator_to_numpy(
                        mixing["jpsi_pre_phi"]
                    ),

                "jpsi_rap":
                    accumulator_to_numpy(
                        mixing["jpsi_pre_rap"]
                    ),

                "jpsi_mass":
                    accumulator_to_numpy(
                        mixing["jpsi_pre_mass"]
                    ),

                "jpsi_dl":
                    accumulator_to_numpy(
                        mixing["jpsi_pre_dl"]
                    ),
            }

            d = {
                "dstar_pt":
                    accumulator_to_numpy(
                        mixing["dstar_pre_pt"]
                    ),

                "dstar_phi":
                    accumulator_to_numpy(
                        mixing["dstar_pre_phi"]
                    ),

                "dstar_rap":
                    accumulator_to_numpy(
                        mixing["dstar_pre_rap"]
                    ),

                "dstar_mass":
                    accumulator_to_numpy(
                        mixing["dstar_pre_mass"]
                    ),
            }

            # --------------------------------------------
            # Internal length consistency
            # --------------------------------------------

            j_lengths = {
                len(v) for v in j.values()
            }

            d_lengths = {
                len(v) for v in d.values()
            }

            if len(j_lengths) != 1:
                raise RuntimeError(
                    f"Inconsistent J/psi arrays: "
                    f"{j_lengths}"
                )

            if len(d_lengths) != 1:
                raise RuntimeError(
                    f"Inconsistent D* arrays: "
                    f"{d_lengths}"
                )

            raw_j += len(j["jpsi_pt"])
            raw_d += len(d["dstar_pt"])

            # --------------------------------------------
            # J/psi selection
            # --------------------------------------------

            jmask = (
                np.isfinite(j["jpsi_pt"])
                & np.isfinite(j["jpsi_phi"])
                & np.isfinite(j["jpsi_rap"])
                & np.isfinite(j["jpsi_mass"])
                & np.isfinite(j["jpsi_dl"])

                & (
                    j["jpsi_pt"]
                    > JPSI_PT_MIN
                )
                & (
                    j["jpsi_pt"]
                    < JPSI_PT_MAX
                )

                & (
                    np.abs(j["jpsi_rap"])
                    < JPSI_ABSY_MAX
                )

                & (
                    j["jpsi_mass"]
                    > JPSI_MASS_MIN
                )
                & (
                    j["jpsi_mass"]
                    < JPSI_MASS_MAX
                )

                & (
                    j["jpsi_dl"]
                    > JPSI_DL_MIN
                )
                & (
                    j["jpsi_dl"]
                    < JPSI_DL_MAX
                )
            )

            # --------------------------------------------
            # D* selection
            # --------------------------------------------

            dmask = (
                np.isfinite(d["dstar_pt"])
                & np.isfinite(d["dstar_phi"])
                & np.isfinite(d["dstar_rap"])

                & (
                    d["dstar_pt"]
                    > DSTAR_PT_MIN
                )
                & (
                    d["dstar_pt"]
                    < DSTAR_PT_MAX
                )

                & (
                    np.abs(d["dstar_rap"])
                    < DSTAR_ABSY_MAX
                )
            )

            nj = int(
                np.count_nonzero(jmask)
            )

            nd = int(
                np.count_nonzero(dmask)
            )

            selected_j += nj
            selected_d += nd

            if nj > 0:

                jchunk = {
                    key: value[jmask]
                    for key, value in j.items()
                }

                jchunk["file_key"] = np.full(
                    nj,
                    file_index,
                    dtype=np.int32,
                )

                jp = rng.random(
                    nj,
                    dtype=np.float32,
                )

                j_res, j_prio = (
                    _priority_reservoir_update(
                        j_res,
                        j_prio,
                        jchunk,
                        jp,
                        max_jpsi,
                    )
                )

            if nd > 0:

                dchunk = {
                    key: value[dmask]
                    for key, value in d.items()
                }

                dchunk["file_key"] = np.full(
                    nd,
                    file_index,
                    dtype=np.int32,
                )

                dp = rng.random(
                    nd,
                    dtype=np.float32,
                )

                d_res, d_prio = (
                    _priority_reservoir_update(
                        d_res,
                        d_prio,
                        dchunk,
                        dp,
                        max_dstar,
                    )
                )

            # Release complete old Coffea payload immediately
            del payload
            del mixing
            del j
            del d

        except Exception as exc:

            failed.append(
                (
                    str(path),
                    f"{type(exc).__name__}: {exc}",
                )
            )

    # ----------------------------------------------------------------
    # Hard failure with ACTUAL diagnostics, rather than hiding 772 errors
    # ----------------------------------------------------------------

    if j_res is None or d_res is None:

        message = [
            "No mixing input was loaded successfully.",
            "",
            f"Files attempted: {len(files)}",
            f"Files failed:    {len(failed)}",
            "",
            "First failures:",
        ]

        for path, error in failed[:5]:
            message.append(
                f"  {path}\n"
                f"    -> {error}"
            )

        raise RuntimeError(
            "\n".join(message)
        )

    if failed:

        warnings.warn(
            f"{len(failed)}/{len(files)} mixing "
            "files could not be read. "
            f"First failures: {failed[:3]}"
        )

    summary = pd.DataFrame(
        [
            (
                "Mixing files found",
                len(files),
            ),
            (
                "Mixing files loaded",
                len(files) - len(failed),
            ),
            (
                "Mixing files failed",
                len(failed),
            ),
            (
                "Raw J/psi candidates seen",
                raw_j,
            ),
            (
                "Raw D* candidates seen",
                raw_d,
            ),
            (
                "Selected J/psi candidates seen",
                selected_j,
            ),
            (
                "Selected D* candidates seen",
                selected_d,
            ),
            (
                "J/psi reservoir retained",
                len(j_res["jpsi_pt"]),
            ),
            (
                "D* reservoir retained",
                len(d_res["dstar_pt"]),
            ),
        ],
        columns=[
            "quantity",
            "value",
        ],
    )

    return (
        pd.DataFrame(j_res),
        pd.DataFrame(d_res),
        summary,
    )


# ================================================================
# Remove duplicated D* fingerprints
# ================================================================

def drop_duplicate_dstar_fingerprints(
    dstar,
    decimals=8,
):

    columns = [
        "dstar_pt",
        "dstar_rap",
        "dstar_phi",
        "dstar_mass",
    ]

    scale = 10.0 ** decimals

    rounded = pd.DataFrame(
        {
            name:
                np.rint(
                    dstar[name].to_numpy(float)
                    * scale
                ).astype(np.int64)

            for name in columns
        }
    )

    fingerprint = (
        pd.util.hash_pandas_object(
            rounded,
            index=False,
        )
    )

    duplicate = (
        fingerprint
        .duplicated(keep="first")
        .to_numpy()
    )

    cleaned = (
        dstar.loc[~duplicate]
        .reset_index(drop=True)
    )

    return (
        cleaned,
        int(np.count_nonzero(duplicate)),
    )


# ================================================================
# Construct the mixed-event DPS template
# ================================================================

def build_mixed_template(
    jpsi,
    dstar,
    n_pairs,
    seed,
    require_different_event=True,
):

    if len(jpsi) == 0:
        raise RuntimeError(
            "J/psi mixing pool is empty"
        )

    if len(dstar) == 0:
        raise RuntimeError(
            "D* mixing pool is empty"
        )

    rng = np.random.default_rng(seed)

    ji = rng.integers(
        0,
        len(jpsi),
        size=int(n_pairs),
    )

    di = rng.integers(
        0,
        len(dstar),
        size=int(n_pairs),
    )

    rejected = 0
    veto_mode = "none"

    if (
        require_different_event
        and "file_key" in jpsi.columns
        and "file_key" in dstar.columns
    ):

        # There is no event/run/lumi identifier in these mixing
        # reservoirs, so use the conservative different-input-file veto.
        veto_mode = "file_key"

        jk = jpsi[
            "file_key"
        ].to_numpy()

        dk = dstar[
            "file_key"
        ].to_numpy()

        same = (
            jk[ji] == dk[di]
        )

        iteration = 0

        while np.any(same):

            count = int(
                np.count_nonzero(same)
            )

            rejected += count

            di[same] = rng.integers(
                0,
                len(dstar),
                size=count,
            )

            same = (
                jk[ji] == dk[di]
            )

            iteration += 1

            if iteration > 100:
                raise RuntimeError(
                    "Could not satisfy "
                    "different-source mixing veto"
                )

    jphi = (
        jpsi["jpsi_phi"]
        .to_numpy()[ji]
    )

    dphi = (
        dstar["dstar_phi"]
        .to_numpy()[di]
    )

    jy = (
        jpsi["jpsi_rap"]
        .to_numpy()[ji]
    )

    dy = (
        dstar["dstar_rap"]
        .to_numpy()[di]
    )

    mixed = pd.DataFrame(
        {
            "jpsi_pt":
                jpsi["jpsi_pt"]
                .to_numpy()[ji]
                .astype(np.float32),

            "jpsi_rap":
                jy.astype(np.float32),

            "jpsi_phi":
                jphi.astype(np.float32),

            "dstar_pt":
                dstar["dstar_pt"]
                .to_numpy()[di]
                .astype(np.float32),

            "dstar_rap":
                dy.astype(np.float32),

            "dstar_phi":
                dphi.astype(np.float32),

            "delta_phi":
                fold_delta_phi(
                    jphi - dphi
                ).astype(np.float32),

            "delta_y":
                np.abs(
                    jy - dy
                ).astype(np.float32),
        }
    )

    return (
        mixed,
        {
            "veto_mode": veto_mode,
            "rejected": rejected,
        },
    )


# ================================================================
# RUN — deterministic nominal mixed-event cache
# ================================================================
MIXING_CACHE_VERSION = 1
REBUILD_MIXING_CACHE = bool(int(os.environ.get(
    "JPSIDSTAR_REBUILD_MIXING_CACHE", "0"
)))
MIXING_CACHE_DIR = OUTPUT_ROOT / "shared_cache"
MIXING_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_mix_cache_key = (
    f"v{MIXING_CACHE_VERSION}_n{TARGET_MIXED_PAIRS}_"
    f"j{MAX_JPSI_MIX_POOL}_d{MAX_DSTAR_MIX_POOL}_seed{MIXING_RNG_SEED}"
)
MIXING_CACHE_PATH = MIXING_CACHE_DIR / f"mixed_template_{_mix_cache_key}.npz"
MIXING_CACHE_SUMMARY_PATH = MIXING_CACHE_DIR / f"mixed_template_{_mix_cache_key}.csv"

if MIXING_CACHE_PATH.is_file() and not REBUILD_MIXING_CACHE:
    with np.load(MIXING_CACHE_PATH) as payload:
        dps_mixed = pd.DataFrame({name: payload[name] for name in payload.files})
    mixing_summary = pd.read_csv(MIXING_CACHE_SUMMARY_PATH)
    mixing_summary = pd.concat([
        mixing_summary,
        pd.DataFrame([{"quantity": "Loaded from deterministic cache", "value": 1}]),
    ], ignore_index=True)
    print(f"[mixing] loaded deterministic cache: {MIXING_CACHE_PATH}")
else:
    jpsi_pool, dstar_pool, mixing_load_summary = load_mixing_input_reservoirs(
        MIXING_INPUT_DIR, MIXING_INPUT_GLOB,
        MAX_JPSI_MIX_POOL, MAX_DSTAR_MIX_POOL, MIXING_RNG_SEED,
    )
    dstar_pool, n_duplicates = drop_duplicate_dstar_fingerprints(dstar_pool)
    print("\nBuilding", f"{TARGET_MIXED_PAIRS:,}", "mixed pairs")
    dps_mixed, mixing_details = build_mixed_template(
        jpsi_pool, dstar_pool, TARGET_MIXED_PAIRS, MIXING_RNG_SEED + 17,
        require_different_event=REQUIRE_DIFFERENT_EVENT_FOR_MIXING,
    )
    mixing_summary = pd.concat([
        mixing_load_summary,
        pd.DataFrame([
            ("D* duplicates removed", n_duplicates),
            ("Mixed pairs", len(dps_mixed)),
            ("Mixing veto mode", mixing_details["veto_mode"]),
            ("Rejected same-source draws", mixing_details["rejected"]),
            ("Mixing cache version", MIXING_CACHE_VERSION),
        ], columns=["quantity", "value"]),
    ], ignore_index=True)
    np.savez_compressed(
        MIXING_CACHE_PATH,
        **{name: dps_mixed[name].to_numpy() for name in dps_mixed.columns},
    )
    mixing_summary.to_csv(MIXING_CACHE_SUMMARY_PATH, index=False)
    del jpsi_pool, dstar_pool
    gc.collect()
    print(f"[mixing] wrote deterministic cache: {MIXING_CACHE_PATH}")

print(mixing_summary.to_string(index=False))
mixing_summary.to_csv(OUTDIR / "mixed_template_summary.csv", index=False)
print("\n[mixing] mixed-event template ready.")


In [ ]:
def region_mask(frame,region):
    return (
        (frame["delta_y"].to_numpy(float)>=region["dy_min"])
        & (frame["delta_phi"].to_numpy(float)>=region["dphi_min"])
        & (frame["delta_phi"].to_numpy(float)<=region["dphi_max"])
    )

def weighted_region_fraction(weights,mask):
    w=np.asarray(weights,float); mask=np.asarray(mask,bool)
    total=float(np.sum(w)); reg=float(np.sum(w[mask]))
    total_var=float(np.sum(w**2)); reg_var=float(np.sum(w[mask]**2))
    if total<=0: raise ValueError("Non-positive weighted total")
    frac=reg/total
    var=reg_var/total**2+reg**2*total_var/total**4-2*reg*reg_var/total**3
    return {"fraction":float(frac),"variance":float(max(var,0)),"region_neff":weighted_effective_entries(w[mask]),"total_neff":weighted_effective_entries(w)}

def central_flat_model(problem,params):
    return central_mixture(problem,params)


def effective_efficiency_flat(problem,params):
    model=central_flat_model(problem,params)
    f=float(params["f_dps"])
    selected=(1-f)*model["sps_response"]+f*model["dps_response"]
    fid=(1-f)*model["sps_fid_shape"]+f*model["dps_fid_shape"]
    return safe_divide(selected,fid,default=np.nan)


def data_fiducial_event_weights(problem,params,sweights=None):
    idx,valid,_=assign_bins(data_sw,problem["spec"])
    eff=effective_efficiency_flat(problem,params)
    sw=data_sw["sweight"].to_numpy(float) if sweights is None else np.asarray(sweights,float)
    weights=np.zeros(len(data_sw),float)
    weights[valid]=sw[valid]/eff[idx[valid]]
    return weights,valid









def dd_purity_scale(purity):
    """
    Multiplicative scale applied to the raw data-driven estimator.
    """
    return float(purity) if APPLY_DD_PURITY_SCALE else 1.0

# ============================================================
# Automatic, auditable DPS-enriched-region selection
# ============================================================

NOMINAL_PARAMS = FITS[NOMINAL_BINNING]["total"]["values"].copy()
NOMINAL_F = float(NOMINAL_PARAMS["f_dps"])


def dd_domain_mask(frame, problem=None):
    """Common normalization domain for data, mixed events, and MC."""
    problem = problem or FITS[NOMINAL_BINNING]["total"]["problem"]
    phi = frame["delta_phi"].to_numpy(float)
    dy = frame["delta_y"].to_numpy(float)
    dy_edges = np.asarray(problem["spec"]["dy_edges"], float)
    return (
        np.isfinite(phi)
        & np.isfinite(dy)
        & (phi >= 0.0)
        & (phi <= DD_DPHI_MAX)
        & (dy >= dy_edges[0])
        & (dy <= dy_edges[-1])
    )


def _build_mixed_efficiency_cache():
    ecc = np.empty(len(dps_mixed), dtype=float)
    ebb = np.empty(len(dps_mixed), dtype=float)
    for start in range(0, len(dps_mixed), MIX_CHUNK):
        stop = min(len(dps_mixed), start + MIX_CHUNK)
        chunk = dps_mixed.iloc[start:stop]
        ecc[start:stop] = evaluate_efficiency_product(chunk, "DPS-ccbar")
        ebb[start:stop] = evaluate_efficiency_product(chunk, "DPS-bbbar")
    return ecc, ebb


MIXED_EFF_CC_CENTRAL, MIXED_EFF_BB_CENTRAL = _build_mixed_efficiency_cache()
MIXED_DOMAIN_MASK = dd_domain_mask(dps_mixed)


def _mixed_weights_central(params):
    problem = FITS[NOMINAL_BINNING]["total"]["problem"]
    fcc = float(central_mixture(problem, params)["f_cc_dps"])
    eps = fcc * MIXED_EFF_CC_CENTRAL + (1.0 - fcc) * MIXED_EFF_BB_CENTRAL
    return 1.0 / np.maximum(eps, EFFICIENCY_FLOOR)


def _make_dd_cache_entry(frame, weights, domain):
    domain = np.asarray(domain, bool)
    return {
        "phi": frame["delta_phi"].to_numpy(float)[domain],
        "dy": frame["delta_y"].to_numpy(float)[domain],
        "weights": np.asarray(weights, float)[domain],
    }


def _build_component_dd_cache(problem):
    parts = {component: {"phi": [], "dy": [], "weights": []} for component in COMPONENTS}
    for sample, frame in MC_SAMPLE_CACHE.items():
        component = SAMPLE_INFO[sample]["component"]
        selected = (
            frame["mc_weight_xsec_only"].to_numpy(float)
            * CENTRAL_STITCH_SCALES.get(sample, 1.0)
            / (LUMI_RUN2_PB_INV * BRANCHING_PRODUCT)
        )
        eff = evaluate_efficiency_product(
            frame, EFFICIENCY_SOURCE_FOR_COMPONENT[component]
        )
        fid_weight = selected / np.maximum(eff, EFFICIENCY_FLOOR)
        domain = dd_domain_mask(frame, problem)
        parts[component]["phi"].append(frame["delta_phi"].to_numpy(float)[domain])
        parts[component]["dy"].append(frame["delta_y"].to_numpy(float)[domain])
        parts[component]["weights"].append(fid_weight[domain])
    out = {}
    for component, values in parts.items():
        out[component] = {
            key: np.concatenate(chunks).astype(float, copy=False)
            for key, chunks in values.items()
        }
    return out


problem = FITS[NOMINAL_BINNING]["total"]["problem"]
DD_COMPONENT_CACHE = _build_component_dd_cache(problem)


def _cached_region_stats(entry, region):
    weights = np.asarray(entry["weights"], float)
    mask = (
        (entry["dy"] >= region["dy_min"])
        & (entry["phi"] >= region["dphi_min"])
        & (entry["phi"] <= region["dphi_max"])
    )
    out = weighted_region_fraction(weights, mask)
    out["region_sum"] = float(np.sum(weights[mask]))
    out["region_sum2"] = float(np.sum(weights[mask] ** 2))
    out["total_sum"] = float(np.sum(weights))
    out["total_sum2"] = float(np.sum(weights ** 2))
    return out


def mixed_region_fraction_central(params, region):
    entry = _make_dd_cache_entry(
        dps_mixed, _mixed_weights_central(params), MIXED_DOMAIN_MASK
    )
    return _cached_region_stats(entry, region)


def component_region_stats_central(problem, region):
    return {
        component: _cached_region_stats(DD_COMPONENT_CACHE[component], region)
        for component in COMPONENTS
    }


def component_region_fractions_central(problem, region):
    stats = component_region_stats_central(problem, region)
    return {component: stats[component]["fraction"] for component in COMPONENTS}


def _region_model_summary_from_stats(params, mixed_stats, component_stats):
    model = central_mixture(problem, params)
    f = float(params["f_dps"])
    fcs = float(model["f_cc_sps"])
    fcd = float(model["f_cc_dps"])
    qmix = float(mixed_stats["fraction"])
    fractions = {
        component: float(component_stats[component]["fraction"])
        for component in COMPONENTS
    }
    q_sps = (
        fcs * fractions["SPS-ccbar"]
        + (1.0 - fcs) * fractions["SPS-bbbar"]
    )
    q_dps_mc = (
        fcd * fractions["DPS-ccbar"]
        + (1.0 - fcd) * fractions["DPS-bbbar"]
    )
    dps_mc_contribution = f * q_dps_mc
    dps_mixed_contribution = f * qmix
    sps_contribution = (1.0 - f) * q_sps
    total_mc = dps_mc_contribution + sps_contribution
    total_mixed = dps_mixed_contribution + sps_contribution
    if (
        not np.isfinite(total_mc)
        or total_mc <= 0.0
        or total_mixed <= 0.0
        or qmix <= 0.0
        or q_dps_mc <= 0.0
    ):
        raise ValueError("Non-positive regional model normalization")
    return {
        "purity": float(dps_mc_contribution / total_mc),
        "mixed_purity": float(dps_mixed_contribution / total_mixed),
        "q_sps": float(q_sps),
        "q_dps_mc": float(q_dps_mc),
        "dps_mc_to_mixed_closure": float(q_dps_mc / qmix),
        "f_cc_sps": fcs,
        "f_cc_dps": fcd,
    }


def region_model_summary(problem, params, region):
    mixed_stats = mixed_region_fraction_central(params, region)
    component_stats = component_region_stats_central(problem, region)
    return _region_model_summary_from_stats(params, mixed_stats, component_stats)


def region_purity_central(problem, params, region):
    return region_model_summary(problem, params, region)["purity"]


def dd_point(params, region=None, sweights=None):
    if region is None:
        region = CENTRAL_DPS_REGION
    problem = FITS[NOMINAL_BINNING]["total"]["problem"]
    data_weights, fit_valid = data_fiducial_event_weights(
        problem, params, sweights=sweights
    )
    domain = fit_valid & dd_domain_mask(data_sw, problem)
    data_entry = _make_dd_cache_entry(data_sw, data_weights, domain)
    data_stats = _cached_region_stats(data_entry, region)
    mixed_stats = mixed_region_fraction_central(params, region)
    component_stats = component_region_stats_central(problem, region)
    summary = _region_model_summary_from_stats(
        params, mixed_stats, component_stats
    )
    qdata = float(data_stats["fraction"])
    qmix = float(mixed_stats["fraction"])
    raw = qdata / qmix
    raw_var = (
        data_stats["variance"] / qmix**2
        + qdata**2 * mixed_stats["variance"] / qmix**4
    )
    closure = float(summary["dps_mc_to_mixed_closure"])

    scale_factor = dd_purity_scale(summary["purity"])

    f_before_closure = raw * scale_factor
    f_corrected = f_before_closure / closure
    f_mixed_hybrid = raw * summary["mixed_purity"]
    solved_denominator = qmix - summary["q_sps"]
    f_solved = (
        (qdata - summary["q_sps"]) / solved_denominator
        if abs(solved_denominator) > 1.0e-12
        else np.nan
    )
    return {
        "f_raw": float(raw),
        "raw_stat": float(math.sqrt(max(raw_var, 0.0))),
        "purity": float(summary["purity"]),
        "mixed_purity": float(summary["mixed_purity"]),
        "f_before_closure": float(f_before_closure),
        "f_corrected": float(f_corrected),
        "f_mixed_hybrid_audit": float(f_mixed_hybrid),
        "f_solved_audit": float(f_solved),
        "data_region_fraction": qdata,
        "mixed_region_fraction": qmix,
        "sps_region_fraction": float(summary["q_sps"]),
        "dps_mc_region_fraction": float(summary["q_dps_mc"]),
        "dps_mc_to_mixed_closure": float(summary["dps_mc_to_mixed_closure"]),
        "data_region_neff": float(data_stats["region_neff"]),
        "mixed_region_neff": float(mixed_stats["region_neff"]),
        "scale_factor_applied": float(scale_factor),
    }


def build_dd_region_scan_cache(params):
    data_weights, fit_valid = data_fiducial_event_weights(problem, params)
    data_domain = fit_valid & dd_domain_mask(data_sw, problem)
    return {
        "data": _make_dd_cache_entry(data_sw, data_weights, data_domain),
        "mixed": _make_dd_cache_entry(
            dps_mixed, _mixed_weights_central(params), MIXED_DOMAIN_MASK
        ),
        "components": DD_COMPONENT_CACHE,
    }


def _combined_component_neff(component_stats, coefficients, names):
    sum_w = 0.0
    sum_w2 = 0.0
    for name in names:
        stats = component_stats[name]
        if stats["total_sum"] <= 0.0:
            continue
        scale = float(coefficients[name]) / stats["total_sum"]
        sum_w += scale * stats["region_sum"]
        sum_w2 += scale**2 * stats["region_sum2"]
    return float(sum_w**2 / sum_w2) if sum_w2 > 0.0 else 0.0


def evaluate_dd_candidate(params, region, cache):
    data_stats = _cached_region_stats(cache["data"], region)
    mixed_stats = _cached_region_stats(cache["mixed"], region)
    component_stats = {
        component: _cached_region_stats(cache["components"][component], region)
        for component in COMPONENTS
    }
    try:
        summary = _region_model_summary_from_stats(
            params, mixed_stats, component_stats
        )
    except ValueError:
        return None
    model = central_mixture(problem, params)
    f = float(params["f_dps"])
    fcs = float(model["f_cc_sps"])
    fcd = float(model["f_cc_dps"])
    coefficients = {
        "DPS-ccbar": f * fcd,
        "DPS-bbbar": f * (1.0 - fcd),
        "SPS-ccbar": (1.0 - f) * fcs,
        "SPS-bbbar": (1.0 - f) * (1.0 - fcs),
    }
    dps_mc_neff = _combined_component_neff(
        component_stats, coefficients, ["DPS-ccbar", "DPS-bbbar"]
    )
    sps_neff = _combined_component_neff(
        component_stats, coefficients, ["SPS-ccbar", "SPS-bbbar"]
    )
    qdata = float(data_stats["fraction"])
    qmix = float(mixed_stats["fraction"])
    if not np.isfinite(qdata) or not np.isfinite(qmix) or qmix <= 0.0:
        return None
    raw = qdata / qmix
    closure = float(summary["dps_mc_to_mixed_closure"])

    scale_factor = dd_purity_scale(summary["purity"])

    before_closure = raw * scale_factor
    corrected = before_closure / closure
    mixed_hybrid = raw * summary["mixed_purity"]
    reasons = []
    if not (0.0 < qdata < 1.0):
        reasons.append("invalid_data_fraction")
    if data_stats["region_sum"] <= 0.0:
        reasons.append("nonpositive_data_yield")
    if data_stats["region_neff"] < MIN_DATA_REGION_NEFF:
        reasons.append("data_neff")
    if mixed_stats["region_neff"] < MIN_MIXED_REGION_NEFF:
        reasons.append("mixed_neff")
    if sps_neff < MIN_SPS_REGION_NEFF:
        reasons.append("sps_mc_neff")
    if dps_mc_neff < MIN_DPS_MC_REGION_NEFF:
        reasons.append("dps_mc_neff")
    if qmix < DD_MIN_DPS_ACCEPTANCE:
        reasons.append("dps_acceptance")
    if abs(closure - 1.0) > DD_MAX_MIXED_DPS_CLOSURE_DEVIATION:
        reasons.append("mixed_dps_closure")
    if not np.isfinite(before_closure):
        reasons.append("nonfinite_estimator")
    return {
        **region,
        "purity": float(summary["purity"]),
        "mixed_purity": float(summary["mixed_purity"]),
        "f_raw": float(raw),
        "f_before_closure": float(before_closure),
        "f_corrected": float(corrected),
        "f_mixed_hybrid_audit": float(mixed_hybrid),
        "data_region_fraction": qdata,
        "mixed_region_fraction": qmix,
        "sps_region_fraction": float(summary["q_sps"]),
        "dps_mc_region_fraction": float(summary["q_dps_mc"]),
        "dps_mc_to_mixed_closure": float(summary["dps_mc_to_mixed_closure"]),
        "data_region_neff": float(data_stats["region_neff"]),
        "mixed_region_neff": float(mixed_stats["region_neff"]),
        "sps_region_neff": float(sps_neff),
        "dps_mc_region_neff": float(dps_mc_neff),
        "accepted": len(reasons) == 0,
        "rejection_reason": "|".join(reasons),
        "scale_factor_applied": float(scale_factor),
    }


def optimize_dps_enriched_region(params):
    cache = build_dd_region_scan_cache(params)
    phi_edges = np.unique(
        np.r_[0.0, np.asarray(DD_SCAN_DPHI_EDGES, float), DD_DPHI_MAX]
    )
    phi_edges = phi_edges[
        (phi_edges >= 0.0) & (phi_edges <= DD_DPHI_MAX + 1.0e-12)
    ]
    rows = []
    for dy_min in np.asarray(DD_SCAN_DY_MIN, float):
        for i in range(len(phi_edges) - 1):
            for j in range(i + 1, len(phi_edges)):
                dphi_min = float(phi_edges[i])
                dphi_max = float(phi_edges[j])
                if dphi_max - dphi_min < DD_SCAN_MIN_DPHI_WIDTH - 1.0e-12:
                    continue
                region = {
                    "dy_min": float(dy_min),
                    "dphi_min": dphi_min,
                    "dphi_max": dphi_max,
                }
                row = evaluate_dd_candidate(params, region, cache)
                if row is not None:
                    rows.append(row)
    scan = pd.DataFrame(rows)
    if scan.empty:
        raise RuntimeError("The automatic DD-region scan produced no finite candidate")
    accepted = scan.loc[scan["accepted"]].copy()
    if accepted.empty:
        counts = scan["rejection_reason"].value_counts().to_dict()
        raise RuntimeError(f"No DD region passes the predeclared support gates: {counts}")
    max_purity = float(accepted["purity"].max())
    accepted["min_mc_neff"] = np.minimum(
        accepted["sps_region_neff"], accepted["dps_mc_region_neff"]
    )
    # The user-facing central region is the literal maximum-purity candidate.
    # Statistical support is used only to break numerical ties.
    best = accepted.sort_values(
        ["purity", "mixed_region_fraction", "min_mc_neff", "data_region_neff"],
        ascending=[False, False, False, False],
    ).iloc[0]
    plateau = accepted.loc[
        accepted["purity"] >= max_purity - DD_PURITY_PLATEAU
    ].copy()
    plateau["min_mc_neff"] = np.minimum(
        plateau["sps_region_neff"], plateau["dps_mc_region_neff"]
    )
    plateau = plateau.sort_values(
        ["mixed_region_fraction", "min_mc_neff", "data_region_neff", "purity"],
        ascending=[False, False, False, False],
    )
    best_region = {
        "dy_min": float(best["dy_min"]),
        "dphi_min": float(best["dphi_min"]),
        "dphi_max": float(best["dphi_max"]),
    }
    scan["selected"] = (
        np.isclose(scan["dy_min"], best_region["dy_min"])
        & np.isclose(scan["dphi_min"], best_region["dphi_min"])
        & np.isclose(scan["dphi_max"], best_region["dphi_max"])
    )
    scan = scan.sort_values(
        ["accepted", "purity", "mixed_region_fraction"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    return best_region, scan, plateau, max_purity

DD_SCAN_CACHE = build_dd_region_scan_cache(NOMINAL_PARAMS)

if AUTO_DD_REGION:
    (
        CENTRAL_DPS_REGION,
        DD_REGION_SCAN,
        DD_REGION_PLATEAU,
        DD_MAX_PURITY,
    ) = optimize_dps_enriched_region(NOMINAL_PARAMS)

    DD_REGION_MODE_LABEL = "Maximum-purity optimized region"

else:
    # Explicitly restore the predeclared fixed region.
    CENTRAL_DPS_REGION = FIXED_DPS_REGION.copy()

    seed = evaluate_dd_candidate(
        NOMINAL_PARAMS,
        CENTRAL_DPS_REGION,
        DD_SCAN_CACHE,
    )

    if seed is None:
        raise RuntimeError(
            "The fixed data-driven region gives a non-finite normalization."
        )

    if not seed["accepted"]:
        warnings.warn(
            "The fixed data-driven region does not satisfy all automatic "
            f"support/closure gates: {seed['rejection_reason']}. "
            "The result is retained as a fixed-region diagnostic."
        )

    DD_REGION_SCAN = pd.DataFrame([seed])
    DD_REGION_SCAN["selected"] = True
    DD_REGION_PLATEAU = DD_REGION_SCAN.copy()
    DD_MAX_PURITY = float(seed["purity"])

    DD_REGION_MODE_LABEL = "Fixed comparison region"

DD_CENTRAL = dd_point(NOMINAL_PARAMS, CENTRAL_DPS_REGION)
DD_SELECTED_SCAN_ROW = DD_REGION_SCAN.loc[DD_REGION_SCAN["selected"]].iloc[0]
for _key in (
    "purity",
    "f_raw",
    "f_before_closure",
    "f_corrected",
    "data_region_fraction",
    "mixed_region_fraction",
    "data_region_neff",
    "mixed_region_neff",
    "dps_mc_to_mixed_closure",
):
    if not np.isclose(
        float(DD_SELECTED_SCAN_ROW[_key]), float(DD_CENTRAL[_key]), rtol=2.0e-10, atol=2.0e-12
    ):
        raise RuntimeError(f"DD scan/cache audit failed for {_key}")

DD_REGION_SCAN.to_csv(OUTDIR / "data_driven_region_scan.csv", index=False)
if abs(DD_CENTRAL["dps_mc_to_mixed_closure"] - 1.0) > DD_MAX_MIXED_DPS_CLOSURE_DEVIATION:
    warnings.warn(
        "DPS-MC/mixed regional closure differs from unity by more than 20%; "
        "the mixed-event cross-check remains diagnostic."
    )

    print(f"\n{DD_REGION_MODE_LABEL.upper()}")
print(
    f"  |dy| >= {CENTRAL_DPS_REGION['dy_min']:.3f}, "
    f"{CENTRAL_DPS_REGION['dphi_min']:.3f} <= |dphi| <= "
    f"{CENTRAL_DPS_REGION['dphi_max']:.3f}"
)
if AUTO_DD_REGION:
    print(
        f"  selected purity / scan max = {DD_CENTRAL['purity']:.4f} / "
        f"{DD_MAX_PURITY:.4f}"
    )
else:
    print(
        f"  fixed-region purity          = {DD_CENTRAL['purity']:.4f}"
    )
    
print(
    f"  N_eff data/mixed           = {DD_CENTRAL['data_region_neff']:.1f} / "
    f"{DD_CENTRAL['mixed_region_neff']:.1f}"
)
print(
    f"  DPS-MC / mixed closure      = "
    f"{DD_CENTRAL['dps_mc_to_mixed_closure']:.4f}"
)
print(
    f"  data-driven fDPS           = {DD_CENTRAL['f_before_closure']:.4f} "
    f"(raw {DD_CENTRAL['f_raw']:.4f})"
)
print(
    f"  closure-divided audit      = {DD_CENTRAL['f_corrected']:.4f} "
    "(template-dependent)"
)
print(
    f"  mixed-hybrid / solved audit= "
    f"{DD_CENTRAL['f_mixed_hybrid_audit']:.4f} / "
    f"{DD_CENTRAL['f_solved_audit']:.4f}"
)


In [ ]:
# ============================================================
# Exact 2D audit of the automatically selected region
# ============================================================

print(f"\n=== {DD_REGION_MODE_LABEL.upper()} AUDIT ===")
print(
    f"|dy| >= {CENTRAL_DPS_REGION['dy_min']:.3f}, "
    f"{CENTRAL_DPS_REGION['dphi_min']:.3f} <= |dphi| <= "
    f"{CENTRAL_DPS_REGION['dphi_max']:.3f}"
)

fresh = dd_point(NOMINAL_PARAMS, CENTRAL_DPS_REGION)
selected = DD_REGION_SCAN.loc[DD_REGION_SCAN["selected"]].iloc[0]
for key in (
    "purity",
    "f_raw",
    "f_before_closure",
    "f_corrected",
    "data_region_fraction",
    "mixed_region_fraction",
    "data_region_neff",
    "mixed_region_neff",
    "dps_mc_to_mixed_closure",
):
    np.testing.assert_allclose(
        float(fresh[key]), float(selected[key]), rtol=2.0e-10, atol=2.0e-12
    )

whole_domain = {
    "dy_min": float(problem["spec"]["dy_edges"][0]),
    "dphi_min": 0.0,
    "dphi_max": float(DD_DPHI_MAX),
}
whole = dd_point(NOMINAL_PARAMS, whole_domain)
np.testing.assert_allclose(whole["data_region_fraction"], 1.0, atol=1.0e-12)
np.testing.assert_allclose(whole["mixed_region_fraction"], 1.0, atol=1.0e-12)
expected_whole_domain_fdps = (
    NOMINAL_F
    if APPLY_DD_PURITY_SCALE
    else 1.0
)

np.testing.assert_allclose(
    whole["f_before_closure"],
    expected_whole_domain_fdps,
    rtol=2.0e-10,
    atol=2.0e-12,
)
if abs(fresh["dps_mc_to_mixed_closure"] - 1.0) > DD_MAX_MIXED_DPS_CLOSURE_DEVIATION:
    if AUTO_DD_REGION:
        raise RuntimeError(
            "Automatically selected region fails the predeclared mixed/DPS closure gate"
        )
    else:
        warnings.warn(
            "The fixed comparison region fails the predeclared mixed/DPS "
            "closure gate. The fixed-region result is diagnostic only."
        )
        
component_stats = component_region_stats_central(problem, CENTRAL_DPS_REGION)
model_summary = region_model_summary(problem, NOMINAL_PARAMS, CENTRAL_DPS_REGION)
print("\nComponent fractions and effective support:")
for component in COMPONENTS:
    stats = component_stats[component]
    print(
        f"{component:12s}: fraction={stats['fraction']:.6f}, "
        f"N_eff(region)={stats['region_neff']:.1f}"
    )

print("\n2D region quantities:")
print(f"data fraction             = {fresh['data_region_fraction']:.6f}")
print(f"mixed DPS fraction        = {fresh['mixed_region_fraction']:.6f}")
print(f"SPS model fraction        = {fresh['sps_region_fraction']:.6f}")
print(f"DPS MC fraction           = {fresh['dps_mc_region_fraction']:.6f}")
print(f"DPS MC / mixed closure    = {fresh['dps_mc_to_mixed_closure']:.6f}")
print(f"post-fit MC purity        = {model_summary['purity']:.6f}")
print(f"mixed-event purity audit = {model_summary['mixed_purity']:.6f}")
print(f"scan maximum purity       = {DD_MAX_PURITY:.6f}")
print("cache/unbinned audit      = PASSED")
print("whole-domain closure      = PASSED")


In [ ]:
# ============================================================
# DPS MC vs mixed-event closure:
# SELECTED level versus EFFICIENCY-CORRECTED FIDUCIAL level
#
# Purpose:
#   determine explicitly whether the MC/mixed non-closure
#   is caused by the efficiency correction or is already
#   present before applying 1/epsilon.
#
# IMPORTANT:
#
# Selected/reco level:
#   MC    -> reconstructed selected MC weights
#   mixed -> unit mixed-event weights
#
# Fiducial level:
#   MC    -> selected MC / epsilon_component
#   mixed -> 1 / epsilon_DPS(post-fit mixture)
#
# Data are NOT included in the MC/mixed closure itself because
# data contain SPS + DPS. The actual DD extraction corrects data
# separately using the post-fit effective efficiency.
# ============================================================


DPS_CLOSURE_DY_MIN = 1.8
N_DPHI_CLOSURE_BINS = 5

closure_phi_edges = np.linspace(
    0.0,
    DD_DPHI_MAX,
    N_DPHI_CLOSURE_BINS + 1,
)

closure_phi_centers = 0.5 * (
    closure_phi_edges[:-1]
    + closure_phi_edges[1:]
)


# ============================================================
# 1. Post-fit DPS composition
# ============================================================

problem_effdiag = FITS[
    NOMINAL_BINNING
]["total"]["problem"]

model_effdiag = central_mixture(
    problem_effdiag,
    NOMINAL_PARAMS,
)

fcc_dps_effdiag = float(
    model_effdiag["f_cc_dps"]
)

print(
    "=== DPS MC / MIXED EFFICIENCY AUDIT ==="
)

print(
    f"post-fit f_cc(DPS) = "
    f"{fcc_dps_effdiag:.6f}"
)


# ============================================================
# 2. Build MC event-level DPS samples explicitly
#
# We retain BOTH:
#
#   selected_weight
#
# and
#
#   fiducial_weight = selected_weight / epsilon
#
# so that the effect of the efficiency correction can be
# isolated directly.
# ============================================================

mc_effdiag_parts = {
    "DPS-ccbar": {
        "phi": [],
        "dy": [],
        "selected": [],
        "fiducial": [],
    },
    "DPS-bbbar": {
        "phi": [],
        "dy": [],
        "selected": [],
        "fiducial": [],
    },
}


for sample, frame in MC_SAMPLE_CACHE.items():

    component = SAMPLE_INFO[
        sample
    ]["component"]

    if component not in mc_effdiag_parts:
        continue

    selected_weight = (
        frame[
            "mc_weight_xsec_only"
        ].to_numpy(float)
        * CENTRAL_STITCH_SCALES.get(
            sample,
            1.0,
        )
        / (
            LUMI_RUN2_PB_INV
            * BRANCHING_PRODUCT
        )
    )

    efficiency = evaluate_efficiency_product(
        frame,
        EFFICIENCY_SOURCE_FOR_COMPONENT[
            component
        ],
    )

    efficiency = np.maximum(
        np.asarray(efficiency, float),
        EFFICIENCY_FLOOR,
    )

    fiducial_weight = (
        selected_weight
        / efficiency
    )

    domain = dd_domain_mask(
        frame,
        problem_effdiag,
    )

    phi = frame[
        "delta_phi"
    ].to_numpy(float)

    dy = frame[
        "delta_y"
    ].to_numpy(float)

    valid = (
        domain
        & np.isfinite(phi)
        & np.isfinite(dy)
        & np.isfinite(selected_weight)
        & np.isfinite(fiducial_weight)
    )

    mc_effdiag_parts[
        component
    ]["phi"].append(
        phi[valid]
    )

    mc_effdiag_parts[
        component
    ]["dy"].append(
        dy[valid]
    )

    mc_effdiag_parts[
        component
    ]["selected"].append(
        selected_weight[valid]
    )

    mc_effdiag_parts[
        component
    ]["fiducial"].append(
        fiducial_weight[valid]
    )


for component in mc_effdiag_parts:

    for key in [
        "phi",
        "dy",
        "selected",
        "fiducial",
    ]:

        mc_effdiag_parts[
            component
        ][key] = np.concatenate(
            mc_effdiag_parts[
                component
            ][key]
        )


# ============================================================
# 3. Build the post-fit ccbar/bbbar MC mixture correctly
#
# f_cc(DPS) is a FIDUCIAL fraction.
#
# Therefore each component must first be normalized to its
# own fiducial total. Only afterwards are ccbar and bbbar
# combined using f_cc(DPS).
#
# This is important at selected level because ccbar and bbbar
# need not have identical efficiencies.
# ============================================================

mc_sel_phi_parts = []
mc_sel_dy_parts = []
mc_sel_weight_parts = []

mc_fid_phi_parts = []
mc_fid_dy_parts = []
mc_fid_weight_parts = []


for component, fraction in [
    (
        "DPS-ccbar",
        fcc_dps_effdiag,
    ),
    (
        "DPS-bbbar",
        1.0 - fcc_dps_effdiag,
    ),
]:

    part = mc_effdiag_parts[
        component
    ]

    fid_total = float(
        np.sum(
            part["fiducial"]
        )
    )

    selected_total = float(
        np.sum(
            part["selected"]
        )
    )

    if fid_total <= 0.0:
        raise RuntimeError(
            f"Invalid fiducial total for "
            f"{component}"
        )

    mean_efficiency = (
        selected_total
        / fid_total
    )

    print(
        f"{component:10s}: "
        f"<epsilon> = "
        f"{mean_efficiency:.6f}"
    )

    # --------------------------------------------
    # SELECTED LEVEL
    #
    # fraction is fiducial, therefore normalize
    # selected response by fiducial component total.
    # --------------------------------------------

    mc_sel_phi_parts.append(
        part["phi"]
    )

    mc_sel_dy_parts.append(
        part["dy"]
    )

    mc_sel_weight_parts.append(
        fraction
        * part["selected"]
        / fid_total
    )


    # --------------------------------------------
    # FIDUCIAL LEVEL
    # --------------------------------------------

    mc_fid_phi_parts.append(
        part["phi"]
    )

    mc_fid_dy_parts.append(
        part["dy"]
    )

    mc_fid_weight_parts.append(
        fraction
        * part["fiducial"]
        / fid_total
    )


mc_sel_phi = np.concatenate(
    mc_sel_phi_parts
)

mc_sel_dy = np.concatenate(
    mc_sel_dy_parts
)

mc_sel_weight = np.concatenate(
    mc_sel_weight_parts
)


mc_fid_phi = np.concatenate(
    mc_fid_phi_parts
)

mc_fid_dy = np.concatenate(
    mc_fid_dy_parts
)

mc_fid_weight = np.concatenate(
    mc_fid_weight_parts
)


# ============================================================
# 4. Mixed-event sample
#
# Selected level:
#     every generated mixed pair has unit base weight
#
# Fiducial corrected level:
#     exactly the same 1/epsilon weight used by the
#     nominal data-driven extraction.
# ============================================================

mix_phi = dps_mixed.loc[
    MIXED_DOMAIN_MASK,
    "delta_phi",
].to_numpy(float)

mix_dy = dps_mixed.loc[
    MIXED_DOMAIN_MASK,
    "delta_y",
].to_numpy(float)


# BEFORE efficiency correction
mix_selected_weight = np.ones(
    np.count_nonzero(
        MIXED_DOMAIN_MASK
    ),
    dtype=float,
)


# AFTER efficiency correction
mix_fiducial_weight_all = (
    _mixed_weights_central(
        NOMINAL_PARAMS
    )
)

mix_fiducial_weight = (
    mix_fiducial_weight_all[
        MIXED_DOMAIN_MASK
    ]
)


mix_effective_mean_efficiency = (
    np.sum(
        mix_selected_weight
    )
    / np.sum(
        mix_fiducial_weight
    )
)

print(
    "mixed-event effective "
    f"<epsilon> = "
    f"{mix_effective_mean_efficiency:.6f}"
)


# ============================================================
# 5. Generic weighted fractions
# ============================================================

def _weighted_region_fraction(
    phi,
    dy,
    weights,
    region,
):

    phi = np.asarray(
        phi,
        float,
    )

    dy = np.asarray(
        dy,
        float,
    )

    weights = np.asarray(
        weights,
        float,
    )

    total = float(
        np.sum(weights)
    )

    if total <= 0.0:
        raise RuntimeError(
            "Non-positive normalization."
        )

    mask = (
        (dy >= region["dy_min"])
        & (
            phi
            >= region["dphi_min"]
        )
        & (
            phi
            <= region["dphi_max"]
        )
    )

    return float(
        np.sum(
            weights[mask]
        )
        / total
    )


def _weighted_projection_after_dy(
    phi,
    dy,
    weights,
    dy_min,
    edges,
):

    phi = np.asarray(
        phi,
        float,
    )

    dy = np.asarray(
        dy,
        float,
    )

    weights = np.asarray(
        weights,
        float,
    )

    total = float(
        np.sum(weights)
    )

    if total <= 0.0:
        raise RuntimeError(
            "Non-positive normalization."
        )

    mask = (
        np.isfinite(phi)
        & np.isfinite(dy)
        & np.isfinite(weights)
        & (dy >= dy_min)
    )

    hist = np.histogram(
        phi[mask],
        bins=edges,
        weights=weights[mask],
    )[0].astype(float)

    return (
        hist
        / total
    )


# ============================================================
# 6. Projections
# ============================================================

# ------------------
# selected/reco level
# ------------------

h_mc_selected = (
    _weighted_projection_after_dy(
        mc_sel_phi,
        mc_sel_dy,
        mc_sel_weight,
        DPS_CLOSURE_DY_MIN,
        closure_phi_edges,
    )
)

h_mix_selected = (
    _weighted_projection_after_dy(
        mix_phi,
        mix_dy,
        mix_selected_weight,
        DPS_CLOSURE_DY_MIN,
        closure_phi_edges,
    )
)


# ------------------
# fiducial level
# ------------------

h_mc_fiducial = (
    _weighted_projection_after_dy(
        mc_fid_phi,
        mc_fid_dy,
        mc_fid_weight,
        DPS_CLOSURE_DY_MIN,
        closure_phi_edges,
    )
)

h_mix_fiducial = (
    _weighted_projection_after_dy(
        mix_phi,
        mix_dy,
        mix_fiducial_weight,
        DPS_CLOSURE_DY_MIN,
        closure_phi_edges,
    )
)


ratio_selected = safe_divide(
    h_mc_selected,
    h_mix_selected,
    default=np.nan,
)

ratio_fiducial = safe_divide(
    h_mc_fiducial,
    h_mix_fiducial,
    default=np.nan,
)


# ============================================================
# 7. Exact integrated closures
# ============================================================

closure_regions = {

    "dy only": {
        "dy_min": DPS_CLOSURE_DY_MIN,
        "dphi_min": 0.0,
        "dphi_max": float(
            DD_DPHI_MAX
        ),
    },

    "ATLAS-like": {
        "dy_min": DPS_CLOSURE_DY_MIN,
        "dphi_min": 0.0,
        "dphi_max": float(
            np.pi / 2.0
        ),
    },

    "middle window": {
        "dy_min": DPS_CLOSURE_DY_MIN,
        "dphi_min": float(
            np.pi / 3.0
        ),
        "dphi_max": float(
            2.0 * np.pi / 3.0
        ),
    },
}


print()
print(
    "=== BEFORE vs AFTER EFFICIENCY CORRECTION ==="
)

print(
    "region          "
    "MCsel     MIXsel    closure_sel   "
    "MCfid     MIXfid    closure_fid"
)

print(
    "-" * 92
)


closure_results_effdiag = {}


for label, region in closure_regions.items():

    q_mc_selected = (
        _weighted_region_fraction(
            mc_sel_phi,
            mc_sel_dy,
            mc_sel_weight,
            region,
        )
    )

    q_mix_selected = (
        _weighted_region_fraction(
            mix_phi,
            mix_dy,
            mix_selected_weight,
            region,
        )
    )

    q_mc_fiducial = (
        _weighted_region_fraction(
            mc_fid_phi,
            mc_fid_dy,
            mc_fid_weight,
            region,
        )
    )

    q_mix_fiducial = (
        _weighted_region_fraction(
            mix_phi,
            mix_dy,
            mix_fiducial_weight,
            region,
        )
    )

    closure_selected = (
        q_mc_selected
        / q_mix_selected
    )

    closure_fiducial = (
        q_mc_fiducial
        / q_mix_fiducial
    )

    closure_results_effdiag[
        label
    ] = {
        "q_mc_selected":
            q_mc_selected,
        "q_mix_selected":
            q_mix_selected,
        "closure_selected":
            closure_selected,
        "q_mc_fiducial":
            q_mc_fiducial,
        "q_mix_fiducial":
            q_mix_fiducial,
        "closure_fiducial":
            closure_fiducial,
    }

    print(
        f"{label:14s} "
        f"{q_mc_selected:8.5f} "
        f"{q_mix_selected:8.5f} "
        f"{closure_selected:11.5f}   "
        f"{q_mc_fiducial:8.5f} "
        f"{q_mix_fiducial:8.5f} "
        f"{closure_fiducial:11.5f}"
    )


# ============================================================
# 8. Cross-check against the nominal DD cache
#
# The FIDUCIAL numbers above MUST reproduce the current
# production DD implementation.
# ============================================================

print()
print(
    "=== CROSS-CHECK WITH NOMINAL DD CACHE ==="
)


for label, region in closure_regions.items():

    cached_components = (
        component_region_stats_central(
            problem_effdiag,
            region,
        )
    )

    cached_q_mc = (
        fcc_dps_effdiag
        * cached_components[
            "DPS-ccbar"
        ]["fraction"]
        + (
            1.0
            - fcc_dps_effdiag
        )
        * cached_components[
            "DPS-bbbar"
        ]["fraction"]
    )

    cached_mix_stats = (
        mixed_region_fraction_central(
            NOMINAL_PARAMS,
            region,
        )
    )

    cached_q_mix = float(
        cached_mix_stats[
            "fraction"
        ]
    )

    explicit_q_mc = (
        closure_results_effdiag[
            label
        ][
            "q_mc_fiducial"
        ]
    )

    explicit_q_mix = (
        closure_results_effdiag[
            label
        ][
            "q_mix_fiducial"
        ]
    )

    print(
        f"{label:14s}: "
        f"Delta q_MC = "
        f"{explicit_q_mc-cached_q_mc:+.3e}, "
        f"Delta q_mix = "
        f"{explicit_q_mix-cached_q_mix:+.3e}"
    )


# ============================================================
# 9. Plot helper
# ============================================================

def _plot_efficiency_closure(
    h_mc,
    h_mix,
    ratio,
    level_label,
    filename,
    ylabel,
):

    fig, (
        ax,
        axr,
    ) = plt.subplots(
        2,
        1,
        figsize=(
            9.0,
            7.3,
        ),
        sharex=True,
        gridspec_kw={
            "height_ratios": [
                3.0,
                1.0,
            ],
            "hspace": 0.06,
        },
    )
    fig.subplots_adjust(left=0.18, right=0.98, bottom=0.15, top=0.84, hspace=0.07)


    ax.stairs(
        h_mc,
        closure_phi_edges,
        linewidth=2.0,
        label="DPS MC",
    )

    ax.stairs(
        h_mix,
        closure_phi_edges,
        linewidth=1.8,
        linestyle="--",
        label="Mixed-event DPS",
    )


    for axis in (
        ax,
        axr,
    ):

        axis.axvline(
            np.pi / 3.0,
            linestyle="--",
            linewidth=1.0,
            alpha=0.55,
        )

        axis.axvline(
            np.pi / 2.0,
            linestyle="-.",
            linewidth=1.2,
            alpha=0.70,
        )

        axis.axvline(
            2.0
            * np.pi
            / 3.0,
            linestyle="--",
            linewidth=1.0,
            alpha=0.55,
        )





    ax.text(
        0.98,
        0.84,
        rf"$|\Delta y|\geq"
        rf"{DPS_CLOSURE_DY_MIN:.2f}$"
        + "\n"
        + level_label,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=11.0,
    )


    ax.set_ylabel(
        ylabel,
        fontsize=0.8*AXIS_LABEL_FONT_SIZE,
    )

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(
            0.52,
            0.84,
        ),
        frameon=False,
        fontsize=10.5,
    )


    valid = np.isfinite(
        ratio
    )

    axr.plot(
        closure_phi_centers[
            valid
        ],
        ratio[
            valid
        ],
        "o",
        markersize=4.5,
    )

    axr.axhline(
        1.0,
        linestyle="--",
        linewidth=1.2,
    )

    axr.set_ylabel(
        "DPS MC / mixed-event DPS",
        fontsize=12,
    )

    axr.set_xlabel(
        XLABEL_DPHI,
        fontsize=20,
    )

    axr.grid(
        axis="y",
        alpha=0.20,
    )


    save_and_show(
        fig,
        filename,
        OUTDIR,
    )


# ============================================================
# 10. Plot BEFORE efficiency correction
# ============================================================

_plot_efficiency_closure(
    h_mc_selected,
    h_mix_selected,
    ratio_selected,
    "selected/reconstructed level",
    (
        "dps_mc_vs_mixed_"
        "SELECTED_before_efficiency"
    ),
    (
        "Fraction of selected-domain "
        "DPS / bin"
    ),
)


# ============================================================
# 11. Plot AFTER efficiency correction
# ============================================================

_plot_efficiency_closure(
    h_mc_fiducial,
    h_mix_fiducial,
    ratio_fiducial,
    r"fiducial level ($1/\epsilon$ corrected)",
    (
        "dps_mc_vs_mixed_"
        "FIDUCIAL_after_efficiency"
    ),
    (
        "Fraction of fiducial-domain "
        "DPS / bin"
    ),
)


In [ ]:
def mixed_region_fraction_replica(fcc_dps, z_replica, region):
    region_all = region_mask(dps_mixed, region)
    total = total_var = reg = reg_var = 0.0
    for start in range(0, len(dps_mixed), MIX_CHUNK):
        stop = min(len(dps_mixed), start + MIX_CHUNK)
        chunk = dps_mixed.iloc[start:stop]
        ecc = evaluate_efficiency_product(
            chunk, "DPS-ccbar", z_replica=z_replica
        )
        ebb = evaluate_efficiency_product(
            chunk, "DPS-bbbar", z_replica=z_replica
        )
        eps = fcc_dps * ecc + (1.0 - fcc_dps) * ebb
        weights = 1.0 / np.maximum(eps, EFFICIENCY_FLOOR)
        domain = MIXED_DOMAIN_MASK[start:stop]
        selected = region_all[start:stop] & domain
        wd = weights[domain]
        wr = weights[selected]
        total += float(np.sum(wd))
        total_var += float(np.sum(wd**2))
        reg += float(np.sum(wr))
        reg_var += float(np.sum(wr**2))
    if total <= 0.0 or reg <= 0.0:
        raise ValueError("Non-positive mixed-event replica normalization")
    fraction = reg / total
    variance = (
        reg_var / total**2
        + reg**2 * total_var / total**4
        - 2.0 * reg * reg_var / total**3
    )
    return float(fraction), float(max(variance, 0.0))


problem = FITS[NOMINAL_BINNING]["total"]["problem"]
response0, fid_shape0, fid_total0 = central_component_arrays(problem)
data_idx = problem["data_idx"]
data_valid = problem["data_valid"]
base_selected = problem["data_weight"]
data_dd_domain = data_valid & dd_domain_mask(data_sw, problem)
central_region_mask_all = (
    region_mask(data_sw, CENTRAL_DPS_REGION)
    & data_dd_domain
)
component_region_central = np.array(
    [
        component_region_fractions_central(problem, CENTRAL_DPS_REGION)[c]
        for c in COMPONENTS
    ],
    dtype=float,
)


def effective_efficiency_from_arrays(problem, params, response, fid_shape, fid_total):
    f = float(params["f_dps"])
    mix = mixture_from_arrays(
        response, fid_shape, fid_total, f, **mixture_parameter_kwargs(params)
    )
    selected = (1.0 - f) * mix["sps_response"] + f * mix["dps_response"]
    fid = (1.0 - f) * mix["sps_fid_shape"] + f * mix["dps_fid_shape"]
    return safe_divide(selected, fid, default=np.nan)


def region_metrics_from_fractions(
    response, fid_shape, fid_total, params, region_fraction, qmix
):
    f = float(params["f_dps"])
    mix = mixture_from_arrays(
        response, fid_shape, fid_total, f, **mixture_parameter_kwargs(params)
    )
    fcs = float(mix["f_cc_sps"])
    fcd = float(mix["f_cc_dps"])
    region_fraction = np.asarray(region_fraction, float)
    q_sps = (
        fcs * region_fraction[COMPONENT_INDEX["SPS-ccbar"]]
        + (1.0 - fcs) * region_fraction[COMPONENT_INDEX["SPS-bbbar"]]
    )
    q_dps_mc = (
        fcd * region_fraction[COMPONENT_INDEX["DPS-ccbar"]]
        + (1.0 - fcd) * region_fraction[COMPONENT_INDEX["DPS-bbbar"]]
    )
    qmix = float(qmix)
    dps_mc = f * float(q_dps_mc)
    dps_mixed = f * qmix
    sps = (1.0 - f) * float(q_sps)
    if min(qmix, q_dps_mc) <= 0.0 or min(dps_mc + sps, dps_mixed + sps) <= 0.0:
        raise ValueError("Non-positive region-model normalization")
    return {
        "purity": float(dps_mc / (dps_mc + sps)),
        "mixed_purity": float(dps_mixed / (dps_mixed + sps)),
        "closure": float(q_dps_mc / qmix),
        "q_dps_mc": float(q_dps_mc),
        "q_sps": float(q_sps),
    }


def dd_data_stat_ensemble(n=N_DD_DATA_REPLICAS, seed=RANDOM_SEED + 2201):
    rng = np.random.default_rng(seed)
    values, purities, fit_values = [], [], []
    iterator = range(int(n))
    if tqdm is not None:
        iterator = tqdm(iterator, desc="data-driven data-stat replicas")
    for _ in iterator:
        mult = rng.poisson(1.0, size=len(data_sw)).astype(float)
        hist = np.bincount(
            data_idx[data_valid],
            weights=(base_selected * mult)[data_valid],
            minlength=problem["n_bins"],
        ).astype(float)
        if np.sum(hist) <= 0.0:
            continue
        shape = hist / np.sum(hist)
        params, ok = fit_parameters_against_explicit_model(
            problem,
            shape,
            response0,
            fid_shape0,
            fid_total0,
            start=NOMINAL_PARAMS,
        )
        if not ok:
            continue
        eff = effective_efficiency_from_arrays(
            problem, params, response0, fid_shape0, fid_total0
        )
        if np.any(~np.isfinite(eff)) or np.any(eff <= 0.0):
            continue
        event_fid = np.zeros(len(data_sw), float)
        event_fid[data_valid] = (
            (base_selected * mult)[data_valid] / eff[data_idx[data_valid]]
        )
        denominator = float(np.sum(event_fid[data_dd_domain]))
        if denominator <= 0.0:
            continue
        qdata = float(np.sum(event_fid[central_region_mask_all]) / denominator)
        qmix = mixed_region_fraction_central(
            params, CENTRAL_DPS_REGION
        )["fraction"]
        metrics = region_metrics_from_fractions(
            response0,
            fid_shape0,
            fid_total0,
            params,
            component_region_central,
            qmix,
        )
        purity = metrics["purity"]
        value = (qdata / qmix) * dd_purity_scale(purity)
        if not (0.0 < qdata < 1.0) or not all(
            np.isfinite(x) for x in (qdata, qmix, purity, value)
        ):
            continue
        values.append(value)
        purities.append(purity)
        fit_values.append(float(params["f_dps"]))
    required = int(math.ceil(MIN_DATA_REPLICA_SUCCESS_FRACTION * int(n)))
    if len(values) < required:
        raise RuntimeError(
            f"Only {len(values)}/{int(n)} successful data-driven data replicas; "
            f"need at least {required}"
        )
    return (
        np.asarray(values, float),
        np.asarray(purities, float),
        np.asarray(fit_values, float),
    )


def dd_model_stat_ensemble(
    n=N_DD_MODEL_REPLICAS, seed=RANDOM_SEED + 3201
):
    rng = np.random.default_rng(seed)
    values, purities, fit_values = [], [], []
    failures = []
    iterator = range(int(n))
    if tqdm is not None:
        iterator = tqdm(iterator, desc="data-driven model-stat replicas")
    for replica_index in iterator:
        try:
            multipliers = {
                sample: rng.poisson(1.0, size=len(frame)).astype(float)
                for sample, frame in MC_SAMPLE_CACHE.items()
            }
            stitch_scales = derive_replica_stitch_scales(multipliers)
            zrep = draw_efficiency_z(rng)
            states = _build_component_states(
                problem,
                multipliers=multipliers,
                stitch_scales=stitch_scales,
                z_replica=zrep,
                bootstrap_hist=True,
                region=CENTRAL_DPS_REGION,
            )
            response = np.stack([states[c]["response"] for c in COMPONENTS])
            fid_shape = np.stack([states[c]["fid_shape"] for c in COMPONENTS])
            fid_total = np.array(
                [states[c]["fid_total"] for c in COMPONENTS], dtype=float
            )
            region_fraction = np.array(
                [states[c]["region_fraction"] for c in COMPONENTS], dtype=float
            )
            params, ok = fit_parameters_against_explicit_model(
                problem,
                problem["data_shape"],
                response,
                fid_shape,
                fid_total,
                start=NOMINAL_PARAMS,
            )
            if not ok:
                raise RuntimeError("replica refit failed")
            eff = effective_efficiency_from_arrays(
                problem, params, response, fid_shape, fid_total
            )
            if np.any(~np.isfinite(eff)) or np.any(eff <= 0.0):
                raise ValueError("invalid replica efficiency")
            event_fid = np.zeros(len(data_sw), float)
            event_fid[data_valid] = (
                base_selected[data_valid] / eff[data_idx[data_valid]]
            )
            denominator = float(np.sum(event_fid[data_dd_domain]))
            if denominator <= 0.0:
                raise ValueError("non-positive replica data denominator")
            qdata = float(
                np.sum(event_fid[central_region_mask_all]) / denominator
            )
            mix = mixture_from_arrays(
                response,
                fid_shape,
                fid_total,
                params["f_dps"],
                **mixture_parameter_kwargs(params),
            )
            fcc = float(mix["f_cc_dps"])
            qmix, _ = mixed_region_fraction_replica(
                fcc, zrep, CENTRAL_DPS_REGION
            )
            metrics = region_metrics_from_fractions(
                response,
                fid_shape,
                fid_total,
                params,
                region_fraction,
                qmix,
            )
            purity = metrics["purity"]
            value = (qdata / qmix) * dd_purity_scale(purity)
            if not (0.0 < qdata < 1.0) or not all(
                np.isfinite(x) for x in (qdata, qmix, purity, value)
            ):
                raise ValueError("non-finite replica result")
            values.append(value)
            purities.append(purity)
            fit_values.append(float(params["f_dps"]))
        except Exception as exc:
            failures.append((replica_index, repr(exc)))
    if failures:
        warnings.warn(
            f"{len(failures)}/{int(n)} data-driven model replicas failed; "
            f"first failures={failures[:3]}"
        )
    required = max(
        MIN_DD_SUCCESSFUL_REPLICAS,
        int(math.ceil(MIN_MODEL_REPLICA_SUCCESS_FRACTION * int(n))),
    )
    if len(values) < required:
        raise RuntimeError(
            f"Only {len(values)}/{int(n)} successful data-driven model replicas; "
            f"need at least {required}"
        )
    return (
        np.asarray(values, float),
        np.asarray(purities, float),
        np.asarray(fit_values, float),
    )


dd_data_values, dd_data_purity, dd_data_fit_values = dd_data_stat_ensemble()
dd_model_values, dd_model_purity, dd_model_fit_values = dd_model_stat_ensemble()


ENSEMBLE_DIAGNOSTIC_ROWS = []


def ensemble_uncertainty_summary(values, nominal, label):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    n = int(len(values))
    if n < 4:
        raise RuntimeError(f"Only {n} finite replicas for {label}")
    centered_std = float(np.std(values, ddof=1))
    bias = float(np.mean(values) - nominal)
    rms_nominal = float(np.sqrt(np.mean((values - nominal) ** 2)))
    sigma_mcse_fraction = float(1.0 / math.sqrt(2.0 * (n - 1)))
    mean_mcse = float(centered_std / math.sqrt(n))
    bias_significant = bool(
        abs(bias) > 2.0 * mean_mcse
        and abs(bias) > 0.20 * centered_std
    )
    odd = values[::2]
    even = values[1::2]
    odd_std = float(np.std(odd, ddof=1))
    even_std = float(np.std(even, ddof=1))
    split_se = math.sqrt(
        odd_std**2 / (2.0 * (len(odd) - 1))
        + even_std**2 / (2.0 * (len(even) - 1))
    )
    split_pull = float(abs(odd_std - even_std) / split_se) if split_se > 0.0 else 0.0
    row = {
        "ensemble": label,
        "replicas": n,
        "nominal": float(nominal),
        "centered_std": centered_std,
        "rms_about_nominal": rms_nominal,
        "bias": bias,
        "mean_mcse": mean_mcse,
        "sigma_mcse_fraction": sigma_mcse_fraction,
        "odd_std": odd_std,
        "even_std": even_std,
        "odd_even_sigma_pull": split_pull,
        "bias_significant": bias_significant,
    }
    ENSEMBLE_DIAGNOSTIC_ROWS.append(row)
    if sigma_mcse_fraction > MAX_RELEVANT_SIGMA_MCSE_FRACTION:
        warnings.warn(
            f"{label}: Monte Carlo uncertainty on sigma is "
            f"{100*sigma_mcse_fraction:.1f}%"
        )
    if split_pull > 2.0:
        warnings.warn(
            f"{label}: odd/even replica widths differ by {split_pull:.2f} sigma"
        )
    if bias_significant:
        warnings.warn(
            f"{label}: significant ensemble centering bias {bias:+.4f}; "
            "the quoted uncertainty uses RMS about the nominal value."
        )
    return row


dd_data_summary = ensemble_uncertainty_summary(
    dd_data_values, DD_CENTRAL["f_before_closure"], "data-driven: data statistics"
)
dd_model_summary = ensemble_uncertainty_summary(
    dd_model_values, DD_CENTRAL["f_before_closure"], "data-driven: model statistics"
)
purity_data_summary = ensemble_uncertainty_summary(
    dd_data_purity, DD_CENTRAL["purity"], "purity: data statistics"
)
purity_model_summary = ensemble_uncertainty_summary(
    dd_model_purity, DD_CENTRAL["purity"], "purity: model statistics"
)
dd_data_sigma = dd_data_summary["rms_about_nominal"]
dd_model_sigma = dd_model_summary["rms_about_nominal"]
dd_data_bias = dd_data_summary["bias"]
dd_model_bias = dd_model_summary["bias"]
purity_data_sigma = purity_data_summary["rms_about_nominal"]
purity_model_sigma = purity_model_summary["rms_about_nominal"]

# Finite mixed-pair statistics are independent of the data and template
# bootstraps. Propagate the analytic qmix variance into the mixed-event result.
central_mixed = mixed_region_fraction_central(
    NOMINAL_PARAMS, CENTRAL_DPS_REGION
)
mix_only_sigma = float(
    abs(DD_CENTRAL["f_before_closure"])
    * math.sqrt(max(central_mixed["variance"], 0.0))
    / central_mixed["fraction"]
)

# Retain the former purity-plateau quantity as an audit only.  It is not
# added to the explicit local region-definition systematic below.
plateau_values = DD_REGION_PLATEAU["f_before_closure"].to_numpy(float)
plateau_purities = DD_REGION_PLATEAU["purity"].to_numpy(float)
dd_region_plateau_audit = (
    float(np.sqrt(np.mean(
        (plateau_values - DD_CENTRAL["f_before_closure"]) ** 2
    ))) if len(plateau_values) else 0.0
)
purity_region_plateau_audit = (
    float(np.sqrt(np.mean(
        (plateau_purities - DD_CENTRAL["purity"]) ** 2
    ))) if len(plateau_purities) else 0.0
)

# ============================================================
# DD REGION-DEFINITION SYSTEMATIC
# Four small, predeclared one-at-a-time boundary variations.
# ============================================================

DD_PREDEFINED_REGION_VARIATIONS = (
    {
        "variation": r"|dy| > 2.1",
        "dy_min": 2.1,
        "dphi_min": 0.0,
        "dphi_max": 1.0,
    },
    {
        "variation": r"|dy| > 2.3",
        "dy_min": 2.3,
        "dphi_min": 0.0,
        "dphi_max": 1.0,
    },
    {
        "variation": r"|dphi| < 0.9",
        "dy_min": 2.2,
        "dphi_min": 0.0,
        "dphi_max": 0.9,
    },
    {
        "variation": r"|dphi| < 1.1",
        "dy_min": 2.2,
        "dphi_min": 0.0,
        "dphi_max": 1.1,
    },
)

_dd_region_rows = []
_dd_region_hard_failures = []
_dd_region_gate_warnings = []

_dd_f_nominal = float(
    DD_CENTRAL["f_before_closure"]
)

for _definition in DD_PREDEFINED_REGION_VARIATIONS:

    _region = {
        key: float(_definition[key])
        for key in (
            "dy_min",
            "dphi_min",
            "dphi_max",
        )
    }

    try:

        # --------------------------------------------------------
        # Actual DD result for the predefined region.
        # This is what enters the systematic variation.
        # --------------------------------------------------------

        _point = dd_point(
            NOMINAL_PARAMS,
            _region,
        )

        # --------------------------------------------------------
        # Reuse nominal optimization gates only as diagnostics.
        #
        # IMPORTANT:
        # these four regions were predefined as systematic
        # variations. They are NOT competing candidates for the
        # nominal optimized region.
        # --------------------------------------------------------

        _candidate = evaluate_dd_candidate(
            NOMINAL_PARAMS,
            _region,
            DD_SCAN_CACHE,
        )

    except Exception as _exc:

        _dd_region_hard_failures.append(
            f"{_definition['variation']}: "
            f"{type(_exc).__name__}: {_exc}"
        )

        continue

    # ------------------------------------------------------------
    # Nominal-region gate status: diagnostic only
    # ------------------------------------------------------------

    if _candidate is not None:

        _accepted_nominal_gate = bool(
            _candidate["accepted"]
        )

        _rejection_reason = str(
            _candidate["rejection_reason"]
        )

    else:

        _accepted_nominal_gate = False

        _rejection_reason = (
            "candidate_diagnostics_unavailable"
        )

    # ------------------------------------------------------------
    # Store result
    # ------------------------------------------------------------

    _row = {

        "variation":
            _definition["variation"],

        **_region,

        "purity":
            float(_point["purity"]),

        "data_region_neff":
            float(_point["data_region_neff"]),

        "mixed_region_neff":
            float(_point["mixed_region_neff"]),

        "closure":
            float(
                _point[
                    "dps_mc_to_mixed_closure"
                ]
            ),

        "fDPS_DD":
            float(
                _point[
                    "f_before_closure"
                ]
            ),

        "delta_fDPS_from_nominal":
            (
                float(
                    _point[
                        "f_before_closure"
                    ]
                )
                - _dd_f_nominal
            ),

        # Keep the nominal optimization-gate information
        # visible, but do NOT use it to select systematic
        # variations.
        "accepted":
            _accepted_nominal_gate,

        "rejection_reason":
            _rejection_reason,
    }

    # ------------------------------------------------------------
    # Hard validity test:
    # only non-finite/undefined results invalidate a predefined
    # systematic variation.
    # ------------------------------------------------------------

    _finite_fields = (
        "purity",
        "data_region_neff",
        "mixed_region_neff",
        "closure",
        "fDPS_DD",
        "delta_fDPS_from_nominal",
    )

    _nonfinite = [
        key
        for key in _finite_fields
        if not np.isfinite(_row[key])
    ]

    if _nonfinite:

        _dd_region_hard_failures.append(
            f"{_definition['variation']}: "
            f"non-finite fields {_nonfinite}"
        )

        continue

    # ------------------------------------------------------------
    # Gate failures are retained as audit information.
    #
    # Example:
    # |dy| > 2.3 has data N_eff below the threshold used by the
    # nominal region optimizer. Since this boundary shift was
    # predefined independently of its result, it remains in the
    # systematic RMS.
    # ------------------------------------------------------------

    if not _accepted_nominal_gate:

        _dd_region_gate_warnings.append(

            f"{_definition['variation']}: "
            "nominal region-selection gate not passed "
            f"({_rejection_reason}); retained as a "
            "predefined systematic boundary variation."

        )

    _dd_region_rows.append(
        _row
    )


# ============================================================
# Build table
# ============================================================

DD_REGION_VARIATIONS = pd.DataFrame(
    _dd_region_rows
)


# ============================================================
# Hard failures remain fatal.
# Gate failures do NOT.
# ============================================================

if (
    len(DD_REGION_VARIATIONS) != 4
    or _dd_region_hard_failures
):

    _diagnostic = (

        DD_REGION_VARIATIONS.to_string(
            index=False
        )

        if len(DD_REGION_VARIATIONS)

        else "<no finite region results>"

    )

    raise RuntimeError(

        "Predefined DD-region evaluation failed.\n"

        + "\n".join(
            _dd_region_hard_failures
        )

        + "\nDiagnostics:\n"

        + _diagnostic

    )


# ============================================================
# Print non-fatal nominal-gate warnings
# ============================================================

if _dd_region_gate_warnings:

    warnings.warn(

        "Predefined DD-region systematic variations: "

        + " | ".join(
            _dd_region_gate_warnings
        )

    )


# ============================================================
# Region systematic
#
# IMPORTANT:
# - RMS around NOMINAL DD result
# - nominal point itself is NOT included
# - exactly the four predefined variations enter
# ============================================================

dd_region_syst = float(

    np.sqrt(

        np.mean(

            DD_REGION_VARIATIONS[
                "delta_fDPS_from_nominal"
            ].to_numpy(float) ** 2

        )

    )

)


# Same construction for the purity uncertainty
purity_region_syst = float(

    np.sqrt(

        np.mean(

            (
                DD_REGION_VARIATIONS[
                    "purity"
                ].to_numpy(float)

                - float(
                    DD_CENTRAL[
                        "purity"
                    ]
                )

            ) ** 2

        )

    )

)


# ============================================================
# Compact audit
# ============================================================

print()
print(
    "PREDEFINED DD REGION VARIATIONS"
)

display(
    DD_REGION_VARIATIONS[
        [
            "variation",
            "dy_min",
            "dphi_max",
            "purity",
            "data_region_neff",
            "closure",
            "fDPS_DD",
            "delta_fDPS_from_nominal",
            "accepted",
            "rejection_reason",
        ]
    ]
)

print()
print(
    f"Nominal fDPS_DD = "
    f"{_dd_f_nominal:.6f}"
)

print(
    f"Region systematic RMS = "
    f"{dd_region_syst:.6f}"
)

# Signal-extraction variants, paired with their own template refits.
variant_dd = []
variant_purity = []
variant_fit = []
if problem["splot_systematic"]["available"]:
    for sw_variant, shape in zip(
        problem["splot_systematic"]["weights"],
        problem["splot_systematic"]["shapes"],
    ):
        params_var, ok = fit_parameters_against_explicit_model(
            problem,
            shape,
            response0,
            fid_shape0,
            fid_total0,
            start=NOMINAL_PARAMS,
        )
        if not ok:
            continue
        point = dd_point(
            params_var, CENTRAL_DPS_REGION, sweights=sw_variant
        )
        variant_dd.append(point["f_before_closure"])
        variant_purity.append(point["purity"])
        variant_fit.append(float(params_var["f_dps"]))

if variant_dd:
    variant_dd = np.asarray(variant_dd, float)
    variant_purity = np.asarray(variant_purity, float)
    variant_fit = np.asarray(variant_fit, float)
    dd_splot_syst = float(
        np.sqrt(
            np.mean(
                (variant_dd - DD_CENTRAL["f_before_closure"]) ** 2
            )
        )
    )
    purity_splot_syst = float(
        np.sqrt(
            np.mean(
                (variant_purity - DD_CENTRAL["purity"]) ** 2
            )
        )
    )
else:
    dd_splot_syst = 0.0
    purity_splot_syst = 0.0

dd_stat_sigma = math.sqrt(
    dd_data_sigma**2 + dd_model_sigma**2 + mix_only_sigma**2
)
dd_syst_sigma = math.sqrt(dd_region_syst**2 + dd_splot_syst**2)
dd_total_sigma = math.sqrt(dd_stat_sigma**2 + dd_syst_sigma**2)

purity_stat_sigma = math.sqrt(
    purity_data_sigma**2 + purity_model_sigma**2
)
purity_syst_sigma = math.sqrt(
    purity_region_syst**2 + purity_splot_syst**2
)
purity_total_sigma = math.sqrt(
    purity_stat_sigma**2 + purity_syst_sigma**2
)

# Paired fit-versus-data-driven compatibility, retaining shared fluctuations.
delta_central = float(DD_CENTRAL["f_before_closure"] - NOMINAL_F)
delta_data_values = dd_data_values - dd_data_fit_values
delta_model_values = dd_model_values - dd_model_fit_values
delta_data_summary = ensemble_uncertainty_summary(
    delta_data_values, delta_central, "paired difference: data statistics"
)
delta_model_summary = ensemble_uncertainty_summary(
    delta_model_values, delta_central, "paired difference: model statistics"
)
delta_data_sigma = delta_data_summary["rms_about_nominal"]
delta_model_sigma = delta_model_summary["rms_about_nominal"]
if len(variant_dd):
    central_delta = DD_CENTRAL["f_before_closure"] - NOMINAL_F
    delta_splot_syst = float(
        np.sqrt(
            np.mean(
                ((variant_dd - variant_fit) - central_delta) ** 2
            )
        )
    )
else:
    delta_splot_syst = 0.0

delta_total_sigma = math.sqrt(
    delta_data_sigma**2
    + delta_model_sigma**2
    + mix_only_sigma**2
    + dd_region_syst**2
    + delta_splot_syst**2
)
delta_compatibility_z = (
    float(delta_central / delta_total_sigma)
    if delta_total_sigma > 0.0
    else np.nan
)
delta_compatibility_pvalue = (
    float(math.erfc(abs(delta_compatibility_z) / math.sqrt(2.0)))
    if np.isfinite(delta_compatibility_z)
    else np.nan
)

ensemble_diagnostics = pd.DataFrame(ENSEMBLE_DIAGNOSTIC_ROWS)
ensemble_diagnostics.to_csv(
    OUTDIR / "data_driven_replica_convergence_FINAL.csv", index=False
)

dd_budget = pd.DataFrame([
    {
        "source": "sWeighted data statistics with paired refit",
        "class": "stat",
        "sigma": dd_data_sigma,
        "centered_sigma": dd_data_summary["centered_std"],
        "bias": dd_data_bias,
        "included_in_total": True,
        "replicas": len(dd_data_values),
        "sigma_mcse_fraction": dd_data_summary["sigma_mcse_fraction"],
        "odd_even_sigma_pull": dd_data_summary["odd_even_sigma_pull"],
    },
    {
        "source": "model/efficiency/stitching/composition with paired refit",
        "class": "stat",
        "sigma": dd_model_sigma,
        "centered_sigma": dd_model_summary["centered_std"],
        "bias": dd_model_bias,
        "included_in_total": True,
        "replicas": len(dd_model_values),
        "sigma_mcse_fraction": dd_model_summary["sigma_mcse_fraction"],
        "odd_even_sigma_pull": dd_model_summary["odd_even_sigma_pull"],
    },
    {
        "source": "finite mixed-event pair statistics",
        "class": "stat",
        "sigma": mix_only_sigma,
        "included_in_total": True,
    },
    {
        "source": "four predefined one-at-a-time boundary variations",
        "class": "syst",
        "sigma": dd_region_syst,
        "replicas": len(DD_REGION_VARIATIONS),
        "included_in_total": True,
    },
    {
        "source": "automatic-region plateau stability (audit only)",
        "class": "audit",
        "sigma": dd_region_plateau_audit,
        "replicas": len(plateau_values),
        "included_in_total": False,
    },
    {
        "source": "signal extraction variants",
        "class": "syst",
        "sigma": dd_splot_syst,
        "replicas": len(variant_dd),
        "included_in_total": True,
    },
])
dd_budget.to_csv(
    OUTDIR / "data_driven_uncertainty_budget_FINAL.csv", index=False
)


In [ ]:
# DATA-DRIVEN STATISTICAL UNCERTAINTY — PRESENTATION PLOT
# Cheap visualization only: reuse the already-computed physics ensembles.

_dd_vis_rng = np.random.default_rng(RANDOM_SEED + 4201)
_dd_vis_n = int(N_DD_VISUALIZATION_TOYS)
if _dd_vis_n < 1000:
    raise ValueError("N_DD_VISUALIZATION_TOYS must be at least 1000")
_dd_nominal_fdps = float(DD_CENTRAL["f_before_closure"])
_dd_data_center = float(np.mean(dd_data_values))
_dd_model_center = float(np.mean(dd_model_values))
_dd_data_fluctuations = np.asarray(dd_data_values, float) - _dd_data_center
_dd_model_fluctuations = np.asarray(dd_model_values, float) - _dd_model_center
_dd_delta_data = _dd_vis_rng.choice(
    _dd_data_fluctuations, size=_dd_vis_n, replace=True
)
_dd_delta_model = _dd_vis_rng.choice(
    _dd_model_fluctuations, size=_dd_vis_n, replace=True
)
# The existing finite-mixed model is the analytic zero-mean variance above.
_dd_delta_mixed = _dd_vis_rng.normal(0.0, mix_only_sigma, size=_dd_vis_n)
dd_stat_visualization_toys = (
    _dd_nominal_fdps + _dd_delta_data + _dd_delta_model + _dd_delta_mixed
)
dd_stat_visualization_rms = float(np.sqrt(np.mean(
    (dd_stat_visualization_toys - _dd_nominal_fdps) ** 2
)))
dd_stat_visualization_mcse = float(
    dd_stat_sigma / math.sqrt(2.0 * (_dd_vis_n - 1))
)
dd_stat_visualization_pull = float(
    (dd_stat_visualization_rms - dd_stat_sigma)
    / dd_stat_visualization_mcse
)
if abs(dd_stat_visualization_pull) > 5.0:
    raise RuntimeError(
        "DD statistical visualization RMS disagrees with the authoritative "
        f"sigma by {dd_stat_visualization_pull:.2f} finite-toy standard errors"
    )

plt.close("all")
fig, ax = plt.subplots(figsize=(10.8, 6.1))
fig.subplots_adjust(left=0.12, right=0.97, bottom=0.15, top=0.84)
_dd_vis_bins = max(40, min(80, int(round(math.sqrt(_dd_vis_n)))))
ax.hist(
    dd_stat_visualization_toys, bins=_dd_vis_bins, density=True,
    color="#3f7fba", alpha=0.72, edgecolor="white", linewidth=0.55,
    label="Pseudoexperiments",
)
_dd_x = np.linspace(
    _dd_nominal_fdps - 4.5 * dd_stat_sigma,
    _dd_nominal_fdps + 4.5 * dd_stat_sigma, 500,
)
_dd_gaussian = (
    np.exp(-0.5 * ((_dd_x - _dd_nominal_fdps) / dd_stat_sigma) ** 2)
    / (math.sqrt(2.0 * math.pi) * dd_stat_sigma)
)
ax.plot(
    _dd_x, _dd_gaussian, color="#b2182b", linewidth=2.0,
    label="Gaussian",
)
ax.axvspan(
    _dd_nominal_fdps - dd_stat_sigma,
    _dd_nominal_fdps + dd_stat_sigma,
    color="#f4a261", alpha=0.20, label=r"$\pm1\sigma_{\rm stat}$",
)
ax.axvline(
    _dd_nominal_fdps, color="black", linewidth=2.2,
    label="Nominal result",
)
ax.axvline(
    _dd_nominal_fdps - dd_stat_sigma, color="#e76f51",
    linewidth=1.3, linestyle="--",
)
ax.axvline(
    _dd_nominal_fdps + dd_stat_sigma, color="#e76f51",
    linewidth=1.3, linestyle="--",
)
ax.set_xlabel(f"Pseudoexperiment {LABEL_FDPS_DD}")
ax.set_ylabel("Probability density")
ax.grid(axis="y", alpha=0.16)
ax.text(
    0.98, 0.92,
    (rf"$f_{{\mathrm{{DPS}}}}^{{\mathrm{{DD}}}} = "
     rf"{_dd_nominal_fdps*100:.1f} \pm {dd_stat_sigma*100:.1f}\%\ \mathrm{{(stat.)}}$"),
    transform=ax.transAxes, ha="right", va="top", fontsize=21,
    bbox=dict(facecolor="white", alpha=0.90, edgecolor="0.75", pad=4.0),
)
ax.legend(loc="upper left", bbox_to_anchor=(0.015, 0.88), fontsize=18)
save_and_show(fig, "data_driven_statistical_uncertainty_toys_CMS_FINAL", OUTDIR)
print(
    "[DD stat plot] RMS / authoritative sigma = "
    f"{dd_stat_visualization_rms:.6f} / {dd_stat_sigma:.6f} "
    f"(finite-toy pull {dd_stat_visualization_pull:+.2f})"
)
print(
    "[DD stat plot] diagnosed data/model centering biases = "
    f"{dd_data_bias:+.6f} / {dd_model_bias:+.6f}"
)


In [ ]:
# DATA-DRIVEN REGION SYSTEMATIC — PRESENTATION PLOT

_dd_region_table_columns = [
    "variation", "is_nominal", "dy_min", "dphi_min", "dphi_max",
    "purity", "data_region_neff", "mixed_region_neff", "closure",
    "fDPS_DD", "delta_fDPS_from_nominal", "accepted",
    "rejection_reason",
]
_dd_nominal_table_row = {
    "variation": "Nominal",
    "is_nominal": True,
    "dy_min": float(CENTRAL_DPS_REGION["dy_min"]),
    "dphi_min": float(CENTRAL_DPS_REGION["dphi_min"]),
    "dphi_max": float(CENTRAL_DPS_REGION["dphi_max"]),
    "purity": float(DD_CENTRAL["purity"]),
    "data_region_neff": float(DD_CENTRAL["data_region_neff"]),
    "mixed_region_neff": float(DD_CENTRAL["mixed_region_neff"]),
    "closure": float(DD_CENTRAL["dps_mc_to_mixed_closure"]),
    "fDPS_DD": float(DD_CENTRAL["f_before_closure"]),
    "delta_fDPS_from_nominal": 0.0,
    "accepted": bool(DD_SELECTED_SCAN_ROW["accepted"]),
    "rejection_reason": str(DD_SELECTED_SCAN_ROW["rejection_reason"]),
}
_dd_variation_table = DD_REGION_VARIATIONS.copy()
_dd_variation_table.insert(1, "is_nominal", False)
DD_REGION_PRESENTATION_TABLE = pd.concat(
    [pd.DataFrame([_dd_nominal_table_row]), _dd_variation_table],
    ignore_index=True,
)[_dd_region_table_columns]
if len(DD_REGION_VARIATIONS) != 4:
    raise RuntimeError("Exactly four predefined DD-region variations are required")
dd_region_plot_rms = float(np.sqrt(np.mean(
    DD_REGION_VARIATIONS["delta_fDPS_from_nominal"].to_numpy(float) ** 2
)))
np.testing.assert_allclose(
    dd_region_plot_rms, dd_region_syst, rtol=0.0, atol=1.0e-15,
)
np.testing.assert_allclose(
    dd_total_sigma**2, dd_stat_sigma**2 + dd_syst_sigma**2,
    rtol=2.0e-14, atol=1.0e-15,
)
_dd_budget_syst_variance = float(np.sum(
    dd_budget.loc[
        (dd_budget["class"] == "syst")
        & dd_budget["included_in_total"].astype(bool), "sigma"
    ].to_numpy(float) ** 2
))
np.testing.assert_allclose(
    _dd_budget_syst_variance, dd_syst_sigma**2,
    rtol=2.0e-14, atol=1.0e-15,
)

_dd_region_x = np.arange(len(DD_REGION_PRESENTATION_TABLE), dtype=float)
_dd_region_y = DD_REGION_PRESENTATION_TABLE["fDPS_DD"].to_numpy(float)
_dd_region_nominal = float(DD_CENTRAL["f_before_closure"])
_dd_region_sigma_plot = float(dd_region_syst)
_dd_region_band_low = _dd_region_nominal - _dd_region_sigma_plot
_dd_region_band_high = _dd_region_nominal + _dd_region_sigma_plot
_dd_region_relative_percent = 100.0 * _dd_region_sigma_plot / _dd_region_nominal
_dd_total_syst_relative_percent = 100.0 * float(dd_syst_sigma) / _dd_region_nominal
_dd_region_band_half_width = 0.5 * (
    _dd_region_band_high - _dd_region_band_low
)
np.testing.assert_allclose(
    _dd_region_band_half_width,
    _dd_region_relative_percent * _dd_region_nominal / 100.0,
    rtol=0.0, atol=1.0e-15,
)
_dd_region_labels = DD_REGION_PRESENTATION_TABLE["variation"].tolist()
plt.close("all")
fig, ax = plt.subplots(figsize=(10.8, 6.1))
fig.subplots_adjust(
    left=0.12, right=0.97,
    bottom=0.23, top=0.84,
)
ax.axhspan(
    _dd_region_band_low,
    _dd_region_band_high,
    color="#f4a261", alpha=0.25,
    label=r"Nominal $\pm\sigma_{\mathrm{region}}^{\mathrm{DD}}$", zorder=1,
)
ax.axhline(
    _dd_region_nominal, color="black", linewidth=2.2,
    label=f"Nominal {LABEL_FDPS_DD}", zorder=2,
)
ax.scatter(
    _dd_region_x[1:], _dd_region_y[1:], s=72, color="#2a6fbb",
    edgecolor="white", linewidth=0.8,
    label="Variation regions", zorder=4,
)
ax.scatter(
    _dd_region_x[0], _dd_region_y[0], s=105, marker="D",
    color="black", edgecolor="white", linewidth=0.9,
    label="Nominal region", zorder=5,
)
ax.set_xlim(-0.55, len(_dd_region_x) - 0.45)
_dd_region_lo = min(np.min(_dd_region_y), _dd_region_band_low)
_dd_region_hi = max(np.max(_dd_region_y), _dd_region_band_high)
_dd_region_pad = max(0.015, 0.24 * (_dd_region_hi - _dd_region_lo))

#ax.set_ylim(_dd_region_lo - _dd_region_pad, _dd_region_hi + _dd_region_pad)
ax.set_ylim( 0.31, 0.42)

ax.set_xticks(_dd_region_x)
ax.set_xticklabels(_dd_region_labels, rotation=18, ha="right", fontsize=20)
ax.set_xlabel("Region variation")
ax.set_ylabel(LABEL_FDPS_DD, fontsize=20)
ax.grid(axis="y", alpha=0.18)
ax.text(
    0.98, 0.92,
    rf"$\sigma_{{\mathrm{{region}}}}^{{\mathrm{{DD}}}}"
    rf" = {_dd_region_relative_percent:.1f}\%$",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=24,
    bbox=dict(
        facecolor="white",
        alpha=0.92,
        edgecolor="0.65",
        boxstyle="round,pad=0.35"
    ),
)
ax.legend(loc="upper left", bbox_to_anchor=(0.01, 0.99), fontsize=18)
save_and_show(fig, "data_driven_region_systematic_CMS_FINAL", OUTDIR)
print("\nNominal and four predefined DD-region variations:")
display(DD_REGION_PRESENTATION_TABLE)
print(
    f"[DD region plot] {len(DD_REGION_VARIATIONS)} predefined variations; "
    f"RMS = {dd_region_plot_rms:.6f}"
)
print(f"[DD region semantics] nominal = {_dd_region_nominal:.17g}")
print(f"[DD region semantics] dd_region_syst = {dd_region_syst:.17g}")
print(f"[DD region semantics] dd_syst_sigma = {dd_syst_sigma:.17g}")
print(f"[DD region semantics] dd_total_sigma = {dd_total_sigma:.17g}")
print(
    "[DD region semantics] relative region / total systematic = "
    f"{_dd_region_relative_percent:.15g}% / "
    f"{_dd_total_syst_relative_percent:.15g}%"
)
if np.isclose(dd_region_syst, dd_syst_sigma, rtol=0.0, atol=1.0e-15):
    print(
        "[DD region bookkeeping] region and total systematic agree because "
        f"the additional approved sPlot systematic is {float(dd_splot_syst):.17g}."
    )
else:
    print(
        "[DD region bookkeeping] region and total systematic differ because "
        f"the additional approved sPlot systematic is {float(dd_splot_syst):.17g} "
        "and enters the systematic budget in quadrature."
    )
print(
    f"[DD region audit] plateau RMS = {dd_region_plateau_audit:.6f} "
    "(audit only; excluded from totals)"
)

print("\nMIXED-EVENT DATA-DRIVEN CROSS-CHECK (REGION FROZEN)")
print(f"  raw fDPS                  = {DD_CENTRAL['f_raw']:.4f}")
print(
    f"  post-fit simulation purity= {DD_CENTRAL['purity']:.4f} "
    f"± {purity_total_sigma:.4f}"
)
print(
    f"  data-driven fDPS          = {DD_CENTRAL['f_before_closure']:.4f} "
    f"± {dd_total_sigma:.4f}"
)
print(
    f"  statistical / systematic  = {dd_stat_sigma:.4f} / "
    f"{dd_syst_sigma:.4f}"
)
print(
    f"  successful data/model reps= {len(dd_data_values)} / "
    f"{len(dd_model_values)}"
)
print(
    f"  paired data-driven-fit diff= {delta_central:+.4f} "
    f"± {delta_total_sigma:.4f} "
    f"(z={delta_compatibility_z:+.2f}, p={delta_compatibility_pvalue:.3g})"
)
print(f"  data-bootstrap centering  = {dd_data_bias:+.4f}")
print(f"  model-bootstrap centering = {dd_model_bias:+.4f}")


In [ ]:
dd_weights, valid = data_fiducial_event_weights(problem, NOMINAL_PARAMS)
plot_phi_edges = np.linspace(0, DPHI_MAX, 17)
plot_y_edges = np.linspace(0, problem["spec"]["dy_edges"][-1], 19)
data_map = np.histogram2d(
    data_sw.loc[valid,"delta_phi"], data_sw.loc[valid,"delta_y"],
    bins=[plot_phi_edges, plot_y_edges], weights=dd_weights[valid],
)[0]

fig, ax = plt.subplots(figsize=(8.8, 6.6))
fig.subplots_adjust(left=0.13, right=0.86, bottom=0.15, top=0.82)
mesh = ax.pcolormesh(plot_phi_edges, plot_y_edges, data_map.T, shading="auto")
cbar = fig.colorbar(mesh, ax=ax, shrink=0.88)
cbar.set_label("Efficiency-corrected sWeighted yield", fontsize=12, labelpad=10)
cbar.ax.tick_params(labelsize=10)
ax.axvline(CENTRAL_DPS_REGION["dphi_min"], linestyle="--", linewidth=1.4, color="black")
ax.axvline(CENTRAL_DPS_REGION["dphi_max"], linestyle="--", linewidth=1.4, color="black")
ax.axhline(CENTRAL_DPS_REGION["dy_min"], linestyle="--", linewidth=1.4, color="black")
ax.set_xlabel(XLABEL_DPHI)
ax.set_ylabel(XLABEL_DY)
label_box = dict(facecolor="white", alpha=0.82, edgecolor="none", pad=0.18)

info_text = (
    f"{DD_REGION_MODE_LABEL}\n"
    + rf"$|\Delta y|\geq{CENTRAL_DPS_REGION['dy_min']:.2f},\quad"
    + rf"{CENTRAL_DPS_REGION['dphi_min']:.2f}\leq|\Delta\phi|\leq"
    + rf"{CENTRAL_DPS_REGION['dphi_max']:.2f}$" "\n"

    + rf"$f_{{\mathrm{{DPS}}}}^{{\mathrm{{DD}}}}="
    + rf"{DD_CENTRAL['f_before_closure']:.3f}\pm{dd_total_sigma:.3f}$" "\n"

    + rf"$P_{{\mathrm{{DPS}}}}="
    + rf"{DD_CENTRAL['purity']:.3f}\pm{purity_total_sigma:.3f}$" "\n"

    + rf"$\sigma_{{\mathrm{{stat}}}}={dd_stat_sigma:.3f},\quad"
    + rf"\sigma_{{\mathrm{{syst}}}}={dd_syst_sigma:.3f}$" "\n"

    + rf"$q_{{\mathrm{{DPS}}}}^{{\mathrm{{sim}}}}/"
    + rf"q_{{\mathrm{{mix}}}}="
    + rf"{DD_CENTRAL['dps_mc_to_mixed_closure']:.2f}$"
)

place_on_right = CENTRAL_DPS_REGION["dphi_max"] <= 0.55*DD_DPHI_MAX
ax.text(
    0.97 if place_on_right else 0.03, 0.84, info_text,
    transform=ax.transAxes, ha="right" if place_on_right else "left", va="top",
    fontsize=14.0, linespacing=1.22,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.90, edgecolor="0.35"),
    zorder=10,
)
save_and_show(fig, "data_driven_enriched_region_FINAL", OUTDIR)


### 12a. Intermediate data-driven distributions
¶
These figures expose the central ingredients of the mixed-event method before
the uncertainty ensembles. They show the data/mixed-event shape comparison,
the normalization-region definition, and the central decomposition
$H_{\mathrm{data}} = H_{\mathrm{DPS}} + H_{\mathrm{SPS}}$,
with the mixed DPS template normalized using the post-fit DPS purity of the
selected normalization region.

In [ ]:
# ============================================================
# DATA-DRIVEN INTERMEDIATE PLOTS I:
# data vs mixed-event DPS shape in the slices defining the 2D region
# SWAN / Matplotlib-safe version
# ============================================================

plt.close("all")


# ------------------------------------------------------------
# Local CMS label, independent of mplhep
# ------------------------------------------------------------



# ------------------------------------------------------------
# Central fiducial weights
# ------------------------------------------------------------

dd_data_weights, dd_data_fit_valid = (
    data_fiducial_event_weights(
        problem,
        NOMINAL_PARAMS,
    )
)

dd_data_weights = np.asarray(
    dd_data_weights,
    dtype=float,
)

dd_data_fit_valid = np.asarray(
    dd_data_fit_valid,
    dtype=bool,
)

dd_data_domain = (
    dd_data_fit_valid
    & np.asarray(
        dd_domain_mask(
            data_sw,
            problem,
        ),
        dtype=bool,
    )
)


dd_mix_weights = np.asarray(
    _mixed_weights_central(
        NOMINAL_PARAMS
    ),
    dtype=float,
)

dd_mix_domain = np.asarray(
    MIXED_DOMAIN_MASK,
    dtype=bool,
).copy()


# ------------------------------------------------------------
# Normalized weighted histogram
# ------------------------------------------------------------

def dd_normalized_weighted_hist(
    values,
    weights,
    edges,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    edges = np.asarray(
        edges,
        dtype=float,
    )

    if values.shape[0] != weights.shape[0]:
        raise ValueError(
            "Values and weights have different lengths: "
            f"{values.shape[0]} vs {weights.shape[0]}"
        )

    finite = (
        np.isfinite(values)
        & np.isfinite(weights)
    )

    values = values[finite]
    weights = weights[finite]

    h = np.histogram(
        values,
        bins=edges,
        weights=weights,
    )[0].astype(float)

    h2 = np.histogram(
        values,
        bins=edges,
        weights=weights**2,
    )[0].astype(float)

    total = float(
        np.sum(h)
    )

    total_var = float(
        np.sum(h2)
    )

    if (
        not np.isfinite(total)
        or total <= 0.0
    ):
        raise ValueError(
            "Non-positive histogram normalization"
        )

    p = h / total

    # Variance of a weighted histogram normalized
    # to unit area, including the common-normalization
    # covariance contribution.
    var = (
        h2 / total**2
        + h**2 * total_var / total**4
        - 2.0 * h * h2 / total**3
    )

    return (
        p,
        np.sqrt(
            np.clip(
                var,
                0.0,
                None,
            )
        ),
    )


# ------------------------------------------------------------
# Use exactly the nominal template-fit binning
# ------------------------------------------------------------

phi_edges_intermediate = np.asarray(
    problem["spec"]["dphi_edges"],
    dtype=float,
)

dy_edges_intermediate = np.asarray(
    problem["spec"]["dy_edges"],
    dtype=float,
)


# Explicit scalar region boundaries
dy_min_region = float(
    np.asarray(
        CENTRAL_DPS_REGION["dy_min"],
        dtype=float,
    ).reshape(-1)[0]
)

dphi_min_region = float(
    np.asarray(
        CENTRAL_DPS_REGION["dphi_min"],
        dtype=float,
    ).reshape(-1)[0]
)

dphi_max_region = float(
    np.asarray(
        CENTRAL_DPS_REGION["dphi_max"],
        dtype=float,
    ).reshape(-1)[0]
)


# ============================================================
# DELTA-PHI PROJECTION
#
# Require the Delta-y component of the 2D DPS-enriched region.
# ============================================================

data_delta_y = (
    data_sw["delta_y"]
    .to_numpy(dtype=float)
)

mix_delta_y = (
    dps_mixed["delta_y"]
    .to_numpy(dtype=float)
)


data_phi_mask = (
    dd_data_domain
    & np.isfinite(data_delta_y)
    & (
        data_delta_y
        >= dy_min_region
    )
)

mix_phi_mask = (
    dd_mix_domain
    & np.isfinite(mix_delta_y)
    & (
        mix_delta_y
        >= dy_min_region
    )
)


data_phi, data_phi_err = (
    dd_normalized_weighted_hist(
        data_sw.loc[
            data_phi_mask,
            "delta_phi",
        ].to_numpy(dtype=float),

        dd_data_weights[
            data_phi_mask
        ],

        phi_edges_intermediate,
    )
)


mix_phi, mix_phi_err = (
    dd_normalized_weighted_hist(
        dps_mixed.loc[
            mix_phi_mask,
            "delta_phi",
        ].to_numpy(dtype=float),

        dd_mix_weights[
            mix_phi_mask
        ],

        phi_edges_intermediate,
    )
)


# ============================================================
# DELTA-Y PROJECTION
#
# Require the Delta-phi component of the 2D DPS-enriched region.
# ============================================================

data_delta_phi = (
    data_sw["delta_phi"]
    .to_numpy(dtype=float)
)

mix_delta_phi = (
    dps_mixed["delta_phi"]
    .to_numpy(dtype=float)
)


data_dy_mask = (
    dd_data_domain
    & np.isfinite(data_delta_phi)
    & (
        data_delta_phi
        >= dphi_min_region
    )
    & (
        data_delta_phi
        <= dphi_max_region
    )
)


mix_dy_mask = (
    dd_mix_domain
    & np.isfinite(mix_delta_phi)
    & (
        mix_delta_phi
        >= dphi_min_region
    )
    & (
        mix_delta_phi
        <= dphi_max_region
    )
)


data_dy, data_dy_err = (
    dd_normalized_weighted_hist(
        data_sw.loc[
            data_dy_mask,
            "delta_y",
        ].to_numpy(dtype=float),

        dd_data_weights[
            data_dy_mask
        ],

        dy_edges_intermediate,
    )
)


mix_dy, mix_dy_err = (
    dd_normalized_weighted_hist(
        dps_mixed.loc[
            mix_dy_mask,
            "delta_y",
        ].to_numpy(dtype=float),

        dd_mix_weights[
            mix_dy_mask
        ],

        dy_edges_intermediate,
    )
)


# ============================================================
# CMS-STYLE FIGURE
#
# No constrained_layout.
# No mplhep CMS label.
# No bbox_inches="tight".
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13.2, 5.2),
)

fig.subplots_adjust(
    left=0.09,
    right=0.985,
    bottom=0.18,
    top=0.80,
    wspace=0.20,
)


# ============================================================
# LEFT: DELTA PHI
# ============================================================

phi_centers = 0.5 * (
    phi_edges_intermediate[:-1]
    + phi_edges_intermediate[1:]
)


axes[0].errorbar(
    phi_centers,
    data_phi,
    yerr=data_phi_err,
    fmt="o",
    color="black",
    markersize=4.6,
    capsize=2.5,
    linewidth=1.1,
    label="Data",
    zorder=4,
)


axes[0].stairs(
    mix_phi,
    phi_edges_intermediate,
    linewidth=1.9,
    color="tab:orange",
    label="Mixed-event DPS template",
    zorder=3,
)


axes[0].axvspan(
    dphi_min_region,
    dphi_max_region,
    alpha=0.12,
    color="tab:blue",
    label="DPS-enriched normalization window",
    zorder=1,
)


axes[0].set_xlabel(
    XLABEL_DPHI
)

axes[0].set_ylabel(
    YLABEL_NORMALIZED
)

axes[0].set_ylim(0.0, 0.6)

axes[0].text(
    0.5, 0.55,
    rf"Projection for $|\Delta y|\geq {dy_min_region:.2f}$",
    transform=axes[0].transAxes,
    ha="center",
    va="top",
    fontsize=20,
)


axes[0].legend(
    loc="upper right",
    fontsize=15.0,
)




# ============================================================
# RIGHT: DELTA Y
# ============================================================

dy_centers = 0.5 * (
    dy_edges_intermediate[:-1]
    + dy_edges_intermediate[1:]
)


axes[1].errorbar(
    dy_centers,
    data_dy,
    yerr=data_dy_err,
    fmt="o",
    color="black",
    markersize=4.6,
    capsize=2.5,
    linewidth=1.1,
    label="Data",
    zorder=4,
)


axes[1].stairs(
    mix_dy,
    dy_edges_intermediate,
    linewidth=1.9,
    color="tab:orange",
    label="Mixed-event DPS template",
    zorder=3,
)


axes[1].axvspan(
    dy_min_region,
    float(
        dy_edges_intermediate[-1]
    ),
    alpha=0.12,
    color="tab:blue",
    label="DPS-enriched normalization window",
    zorder=1,
)


axes[1].set_xlabel(
    XLABEL_DY
)

axes[1].set_ylabel(
    YLABEL_NORMALIZED
)

axes[1].text(
    0.50, 0.55,
    rf"Projection for "
    rf"${dphi_min_region:.2f}\leq|\Delta\phi|\leq{dphi_max_region:.2f}$",
    transform=axes[1].transAxes,
    ha="center",
    va="top",
    fontsize=20,
)

axes[1].legend(
    loc="upper right",
    fontsize=15.0,
)




# ============================================================
# SAVE
# ============================================================

stem = (
    "data_driven_intermediate_"
    "data_vs_mixed_conditional_CMS"
)


save_and_show(fig, stem, OUTDIR)


# ------------------------------------------------------------
# Small numerical audit
# ------------------------------------------------------------

print()
print(
    "Intermediate DD plots:"
)

print(
    f"  data |dy| >= {dy_min_region:.2f}: "
    f"{np.count_nonzero(data_phi_mask)} candidates"
)

print(
    f"  mixed |dy| >= {dy_min_region:.2f}: "
    f"{np.count_nonzero(mix_phi_mask)} candidates"
)

print(
    f"  data {dphi_min_region:.2f} <= |dphi| <= "
    f"{dphi_max_region:.2f}: "
    f"{np.count_nonzero(data_dy_mask)} candidates"
)

print(
    f"  mixed {dphi_min_region:.2f} <= |dphi| <= "
    f"{dphi_max_region:.2f}: "
    f"{np.count_nonzero(mix_dy_mask)} candidates"
)


In [ ]:
# ============================================================
# DATA-DRIVEN INTERMEDIATE PLOTS II:
# central DPS normalization and SPS = data - DPS decomposition
# shown as differential fiducial cross sections
# SWAN / Matplotlib-safe version
# ============================================================

plt.close("all")


# ------------------------------------------------------------
# Safe local CMS label
# ------------------------------------------------------------



# ------------------------------------------------------------
# Weighted histogram with sumw2
# ------------------------------------------------------------

def weighted_hist_sumw2(
    values,
    weights,
    edges,
):

    values = np.asarray(
        values,
        dtype=float,
    )

    weights = np.asarray(
        weights,
        dtype=float,
    )

    edges = np.asarray(
        edges,
        dtype=float,
    )

    if len(values) != len(weights):
        raise ValueError(
            f"values/weights length mismatch: "
            f"{len(values)} vs {len(weights)}"
        )

    finite = (
        np.isfinite(values)
        & np.isfinite(weights)
    )

    values = values[finite]
    weights = weights[finite]

    h = np.histogram(
        values,
        bins=edges,
        weights=weights,
    )[0].astype(float)

    h2 = np.histogram(
        values,
        bins=edges,
        weights=weights**2,
    )[0].astype(float)

    return h, h2


# ============================================================
# FIDUCIAL CROSS-SECTION NORMALIZATION
# ============================================================

_lumi_pb = float(
    np.asarray(
        LUMI_RUN2_PB_INV,
        dtype=float,
    ).reshape(-1)[0]
)

_branching_product = float(
    np.asarray(
        BRANCHING_PRODUCT,
        dtype=float,
    ).reshape(-1)[0]
)

FIDUCIAL_XSEC_NORM_PB = (
    1.0
    / (
        _lumi_pb
        * _branching_product
    )
)

print(
    "Fiducial cross-section normalization "
    "[pb / corrected event] =",
    FIDUCIAL_XSEC_NORM_PB,
)


# ============================================================
# DPS-ENRICHED NORMALIZATION REGION
# ============================================================

data_region_mask = (
    np.asarray(
        dd_data_domain,
        dtype=bool,
    )
    & np.asarray(
        region_mask(
            data_sw,
            CENTRAL_DPS_REGION,
        ),
        dtype=bool,
    )
)

mix_region_mask = (
    np.asarray(
        dd_mix_domain,
        dtype=bool,
    )
    & np.asarray(
        region_mask(
            dps_mixed,
            CENTRAL_DPS_REGION,
        ),
        dtype=bool,
    )
)


data_region_yield = float(
    np.sum(
        np.asarray(
            dd_data_weights,
            dtype=float,
        )[data_region_mask]
    )
)

mixed_region_yield = float(
    np.sum(
        np.asarray(
            dd_mix_weights,
            dtype=float,
        )[mix_region_mask]
    )
)


if (
    data_region_yield <= 0.0
    or mixed_region_yield <= 0.0
):
    raise RuntimeError(
        "Non-positive normalization-region yield"
    )


# ------------------------------------------------------------
# Post-fit DPS purity correction
# ------------------------------------------------------------

dd_purity = float(
    np.asarray(
        DD_CENTRAL["purity"],
        dtype=float,
    ).reshape(-1)[0]
)

purity_scale_applied = float(
    np.asarray(
        dd_purity_scale(
            dd_purity
        ),
        dtype=float,
    ).reshape(-1)[0]
)


DD_MIXED_SCALE = float(
    purity_scale_applied
    * data_region_yield
    / mixed_region_yield
)


print(
    "Data-driven central mixed-template scale =",
    DD_MIXED_SCALE,
)

print(
    "Post-fit DPS purity                      =",
    dd_purity,
)

print(
    "Purity scale actually applied            =",
    purity_scale_applied,
)


# ============================================================
# BUILD CORRECTED-YIELD PROJECTIONS
# ============================================================

def build_dd_projection(
    axis,
    edges,
):

    edges = np.asarray(
        edges,
        dtype=float,
    )

    data_domain = np.asarray(
        dd_data_domain,
        dtype=bool,
    )

    mix_domain = np.asarray(
        dd_mix_domain,
        dtype=bool,
    )

    data_weights = np.asarray(
        dd_data_weights,
        dtype=float,
    )

    mix_weights = np.asarray(
        dd_mix_weights,
        dtype=float,
    )


    if axis == "dphi":

        data_values_all = (
            data_sw[
                "delta_phi"
            ].to_numpy(dtype=float)
        )

        mix_values_all = (
            dps_mixed[
                "delta_phi"
            ].to_numpy(dtype=float)
        )

    elif axis == "dy":

        data_values_all = (
            data_sw[
                "delta_y"
            ].to_numpy(dtype=float)
        )

        mix_values_all = (
            dps_mixed[
                "delta_y"
            ].to_numpy(dtype=float)
        )

    else:
        raise ValueError(axis)


    data_values = (
        data_values_all[
            data_domain
        ]
    )

    mix_values = (
        mix_values_all[
            mix_domain
        ]
    )


    h_data, v_data = (
        weighted_hist_sumw2(
            data_values,
            data_weights[
                data_domain
            ],
            edges,
        )
    )


    h_mix, v_mix = (
        weighted_hist_sumw2(
            mix_values,
            mix_weights[
                mix_domain
            ],
            edges,
        )
    )


    h_dps = (
        DD_MIXED_SCALE
        * h_mix
    )

    v_dps = (
        DD_MIXED_SCALE**2
        * v_mix
    )

    h_sps = (
        h_data
        - h_dps
    )


    return {
        "data":
            h_data,

        "data_var":
            v_data,

        "dps":
            h_dps,

        "dps_var":
            v_dps,

        "sps":
            h_sps,
    }


# ------------------------------------------------------------
# Use exactly the intermediate/template-fit binning
# ------------------------------------------------------------

phi_edges_intermediate = np.asarray(
    phi_edges_intermediate,
    dtype=float,
)

dy_edges_intermediate = np.asarray(
    dy_edges_intermediate,
    dtype=float,
)


dd_proj_phi = build_dd_projection(
    "dphi",
    phi_edges_intermediate,
)

dd_proj_dy = build_dd_projection(
    "dy",
    dy_edges_intermediate,
)


# ============================================================
# NORMALIZATION CROSS-CHECK
# ============================================================

sigma_fid_phi_pb = float(
    FIDUCIAL_XSEC_NORM_PB
    * np.sum(
        dd_proj_phi["data"]
    )
)

sigma_fid_dy_pb = float(
    FIDUCIAL_XSEC_NORM_PB
    * np.sum(
        dd_proj_dy["data"]
    )
)


print(
    f"Integrated fiducial cross section from |Delta phi| = "
    f"{sigma_fid_phi_pb:.3f} pb"
)

print(
    f"Integrated fiducial cross section from |Delta y|   = "
    f"{sigma_fid_dy_pb:.3f} pb"
)


# Cross-check that both projections cover the same fiducial domain
if not np.isclose(
    sigma_fid_phi_pb,
    sigma_fid_dy_pb,
    rtol=1e-10,
    atol=1e-8,
):
    raise RuntimeError(
        "Delta-phi and Delta-y projections do not integrate "
        "to the same fiducial cross section."
    )


# ============================================================
# SCALAR VALUES FOR PLOT ANNOTATION
# ============================================================

f_dd_central = float(
    np.asarray(
        DD_CENTRAL[
            "f_before_closure"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dd_sigma_total_plot = float(
    np.asarray(
        dd_total_sigma,
        dtype=float,
    ).reshape(-1)[0]
)

dy_min_plot = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dy_min"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dphi_min_plot = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dphi_min"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dphi_max_plot = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dphi_max"
        ],
        dtype=float,
    ).reshape(-1)[0]
)


# ============================================================
# CMS-STYLE DIFFERENTIAL FIDUCIAL CROSS-SECTION FIGURE
#
# IMPORTANT:
#   no constrained_layout
#   no mplhep cms_label
#   no bbox_inches="tight"
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15.0, 6.0),
)


fig.subplots_adjust(
    left=0.09,
    right=0.985,
    bottom=0.18,
    top=0.80,
    wspace=0.22,
)


_dd_projection_legends = []
_dd_projection_annotations = []


plot_definitions = (

    (
        axes[0],
        "dphi",
        phi_edges_intermediate,
        dd_proj_phi,
        XLABEL_DPHI,
        YLABEL_DPHI_XSEC,
    ),

    (
        axes[1],
        "dy",
        dy_edges_intermediate,
        dd_proj_dy,
        XLABEL_DY,
        YLABEL_DY_XSEC,
    ),
)


for (
    ax,
    observable,
    edges,
    result,
    xlabel,
    ylabel,
) in plot_definitions:

    edges = np.asarray(
        edges,
        dtype=float,
    )

    widths = np.diff(
        edges
    )

    centers = 0.5 * (
        edges[:-1]
        + edges[1:]
    )


    diff_xsec_norm = (
        FIDUCIAL_XSEC_NORM_PB
        / widths
    )


    data_diff_xsec = (
        np.asarray(
            result["data"],
            dtype=float,
        )
        * diff_xsec_norm
    )


    data_diff_xsec_err = (
        np.sqrt(
            np.clip(
                np.asarray(
                    result[
                        "data_var"
                    ],
                    dtype=float,
                ),
                0.0,
                None,
            )
        )
        * diff_xsec_norm
    )


    dps_diff_xsec = (
        np.asarray(
            result["dps"],
            dtype=float,
        )
        * diff_xsec_norm
    )


    sps_diff_xsec = (
        np.asarray(
            result["sps"],
            dtype=float,
        )
        * diff_xsec_norm
    )


    # Explicit pedagogical closure curve. By construction this is
    # SPS + DPS = (data - DPS) + DPS, so its central value coincides
    # with the data points; drawing it makes the decomposition visible.
    total_diff_xsec = sps_diff_xsec + dps_diff_xsec


    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------

    ax.errorbar(
        centers,
        data_diff_xsec,
        yerr=data_diff_xsec_err,
        fmt="o",
        color="black",
        markersize=4.6,
        capsize=2.5,
        linewidth=1.0,
        label="Data",
        zorder=4,
    )


    # --------------------------------------------------------
    # DPS
    # --------------------------------------------------------

    ax.stairs(
        dps_diff_xsec,
        edges,
        linewidth=2.0,
        color="tab:blue",
        label="Data-driven DPS",
        zorder=3,
    )


    # --------------------------------------------------------
    # SPS residual
    # --------------------------------------------------------

    ax.stairs(
        sps_diff_xsec,
        edges,
        linewidth=2.0,
        color="tab:orange",
        label=r"Data $-$ DPS",
        zorder=3,
    )


    # --------------------------------------------------------
    # SPS + DPS closure curve (same central values as the data)
    # --------------------------------------------------------
    ax.stairs(
        total_diff_xsec,
        edges,
        linewidth=1.8,
        linestyle=":",
        color="0.25",
        label="SPS + DPS",
        zorder=2,
    )


    ax.axhline(
        0.0,
        color="0.35",
        linewidth=0.8,
        zorder=1,
    )


    ax.set_xlabel(
        xlabel
    )

    ax.set_ylabel(
        ylabel
    )


    common_fiducial_xsec_ylim = {
        "dphi": (0.0, 600.0),
        "dy": (0.0, 700.0),
    }

    ax.set_ylim(
        *common_fiducial_xsec_ylim[observable]
    )


    _dd_projection_legends.append(ax.legend(
        loc="upper left",
        bbox_to_anchor=(0.02, 0.98),
        fontsize=18.0,
    ))


# ============================================================
# CMS LABEL
# ============================================================

# ============================================================
# DATA-DRIVEN RESULT ANNOTATION
# ============================================================

info_text = (
    rf"$f_{{\mathrm{{DPS}}}}^{{\mathrm{{DD}}}}="
    rf"{100.0*f_dd_central:.1f}"
    rf"\pm{100.0*dd_sigma_total_plot:.1f}\%$"
    "\n"
    rf"$|\Delta y|\geq {dy_min_plot:.2f}$"
    "\n"
    rf"${dphi_min_plot/np.pi:.2f}\pi"
    rf"\leq|\Delta\phi|\leq"
    rf"{dphi_max_plot/np.pi:.2f}\pi$"
)

for ax in axes:
    _dd_projection_annotations.append(ax.text(
        0.95,
        0.92,
        info_text,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=20,
        bbox=dict(
            facecolor="white",
            alpha=0.90,
            edgecolor="none",
            pad=2.5,
        ),
        zorder=10,
    ))


fig.canvas.draw()
_dd_projection_renderer = fig.canvas.get_renderer()
_dd_projection_overlap_audits = []
for _panel_index, (_legend, _annotation) in enumerate(zip(
    _dd_projection_legends, _dd_projection_annotations,
)):
    _intersection_px2 = float(_intersection_area(
        _legend.get_window_extent(_dd_projection_renderer),
        _annotation.get_window_extent(_dd_projection_renderer),
    ))
    np.testing.assert_allclose(_intersection_px2, 0.0, rtol=0.0, atol=1.0e-9)
    _dd_projection_overlap_audits.append({
        "name": f"DD intermediate panel {_panel_index}: legend/info",
        "panel_index": int(_panel_index),
        "intersection_area_px2": _intersection_px2,
        "status": "PASS",
    })
    print(
        f"[DD intermediate layout] panel {_panel_index} "
        f"legend/info intersection = {_intersection_px2:.6f} px^2"
    )
fig._layout_explicit_overlap_audits = _dd_projection_overlap_audits


# ============================================================
# SAVE
# ============================================================

stem = (
    "data_driven_intermediate_"
    "SPS_DPS_fiducial_xsec_projections_CMS"
)


save_and_show(fig, stem, OUTDIR)


print()
print(
    "[DD intermediate II] completed successfully"
)


In [ ]:
# ============================================================
# FULL-FIDUCIAL POST-FIT TEMPLATE PROJECTION
#
# Directly comparable to DATA-DRIVEN INTERMEDIATE PLOTS II.
#
# SAME:
#   - data events
#   - efficiency correction
#   - luminosity
#   - branching fractions
#   - Delta-phi binning
#   - full fiducial domain
#
# Therefore the DATA points in this plot MUST agree bin by bin
# with the Data points of the data-driven Delta-phi projection.
# ============================================================


problem_fullphi = FITS[
    NOMINAL_BINNING
]["total"]["problem"]

params_fullphi = FITS[
    NOMINAL_BINNING
]["total"]["values"].copy()

postfit_fullphi = central_mixture(
    problem_fullphi,
    params_fullphi,
)

f_dps_fullphi = float(
    params_fullphi["f_dps"]
)

f_cc_sps_fullphi = float(
    postfit_fullphi["f_cc_sps"]
)

f_cc_dps_fullphi = float(
    postfit_fullphi["f_cc_dps"]
)


# IMPORTANT:
# use exactly the same binning as the DD plot
phi_edges_fullphi = (
    np.asarray(
        phi_edges_intermediate,
        dtype=float,
    ).copy()
)

phi_widths_fullphi = np.diff(
    phi_edges_fullphi
)

phi_centers_fullphi = (
    0.5
    * (
        phi_edges_fullphi[:-1]
        + phi_edges_fullphi[1:]
    )
)


# ------------------------------------------------------------
# 1. DATA
#
# dd_data_weights = efficiency-corrected yields.
#
# Convert to pb with exactly the same normalization as PLOTS II.
# ------------------------------------------------------------

data_phi_full = (
    data_sw[
        "delta_phi"
    ].to_numpy(float)
)


h_data_full_yield = np.histogram(
    data_phi_full[
        dd_data_domain
    ],
    bins=phi_edges_fullphi,
    weights=dd_data_weights[
        dd_data_domain
    ],
)[0]


h_data_full_var_yield = np.histogram(
    data_phi_full[
        dd_data_domain
    ],
    bins=phi_edges_fullphi,
    weights=(
        dd_data_weights[
            dd_data_domain
        ]
        ** 2
    ),
)[0]


h_data_full_pb = (
    h_data_full_yield
    * FIDUCIAL_XSEC_NORM_PB
)

h_data_full_err_pb = (
    np.sqrt(
        np.clip(
            h_data_full_var_yield,
            0.0,
            None,
        )
    )
    * FIDUCIAL_XSEC_NORM_PB
)


data_full_diff = (
    h_data_full_pb
    / phi_widths_fullphi
)

data_full_diff_err = (
    h_data_full_err_pb
    / phi_widths_fullphi
)


sigma_data_fullphi = float(
    np.sum(
        h_data_full_pb
    )
)


# ------------------------------------------------------------
# 2. Full-fiducial MC component shapes
# ------------------------------------------------------------

component_hist_fullphi = {}


for component in COMPONENTS:

    h_component = np.zeros(
        len(phi_edges_fullphi) - 1,
        dtype=float,
    )

    total_component = 0.0


    for sample, frame in MC_SAMPLE_CACHE.items():

        if (
            SAMPLE_INFO[sample]["component"]
            != component
        ):
            continue


        _, mc_valid_fullphi, _ = assign_bins(
            frame,
            problem_fullphi["spec"],
        )


        selected_weight = (
            frame[
                "mc_weight_xsec_only"
            ].to_numpy(float)
            *
            CENTRAL_STITCH_SCALES.get(
                sample,
                1.0,
            )
            /
            (
                LUMI_RUN2_PB_INV
                * BRANCHING_PRODUCT
            )
        )


        component_eff = (
            evaluate_efficiency_product(
                frame,
                EFFICIENCY_SOURCE_FOR_COMPONENT[
                    component
                ],
            )
        )


        fid_weight = (
            selected_weight
            /
            np.maximum(
                component_eff,
                EFFICIENCY_FLOOR,
            )
        )


        phi_mc = (
            frame[
                "delta_phi"
            ].to_numpy(float)
        )

        dy_mc = (
            frame[
                "delta_y"
            ].to_numpy(float)
        )


        mc_domain = (
            mc_valid_fullphi
            & np.isfinite(phi_mc)
            & np.isfinite(dy_mc)
            & np.isfinite(fid_weight)
            & (
                phi_mc
                >= phi_edges_fullphi[0]
            )
            & (
                phi_mc
                <= phi_edges_fullphi[-1]
            )
        )


        h_component += np.histogram(
            phi_mc[mc_domain],
            bins=phi_edges_fullphi,
            weights=fid_weight[
                mc_domain
            ],
        )[0]


        total_component += float(
            np.sum(
                fid_weight[
                    mc_domain
                ]
            )
        )


    if total_component <= 0.0:
        raise RuntimeError(
            f"Non-positive normalization for {component}"
        )


    component_hist_fullphi[
        component
    ] = (
        h_component
        / total_component
    )


# ------------------------------------------------------------
# 3. Post-fit SPS and DPS shapes
# ------------------------------------------------------------

sps_shape_fullphi = (
    f_cc_sps_fullphi
    * component_hist_fullphi[
        "SPS-ccbar"
    ]
    +
    (1.0 - f_cc_sps_fullphi)
    * component_hist_fullphi[
        "SPS-bbbar"
    ]
)

dps_shape_fullphi = (
    f_cc_dps_fullphi
    * component_hist_fullphi[
        "DPS-ccbar"
    ]
    +
    (1.0 - f_cc_dps_fullphi)
    * component_hist_fullphi[
        "DPS-bbbar"
    ]
)


# Each shape must integrate to unity over the full domain.
print(
    "SPS full-domain shape integral =",
    np.sum(sps_shape_fullphi),
)

print(
    "DPS full-domain shape integral =",
    np.sum(dps_shape_fullphi),
)


# ------------------------------------------------------------
# 4. Scale by post-fit fractions and measured total cross section
# ------------------------------------------------------------

h_sps_full_pb = (
    sigma_data_fullphi
    * (1.0 - f_dps_fullphi)
    * sps_shape_fullphi
)

h_dps_full_pb = (
    sigma_data_fullphi
    * f_dps_fullphi
    * dps_shape_fullphi
)

h_model_full_pb = (
    h_sps_full_pb
    + h_dps_full_pb
)


sps_full_diff = (
    h_sps_full_pb
    / phi_widths_fullphi
)

dps_full_diff = (
    h_dps_full_pb
    / phi_widths_fullphi
)

model_full_diff = (
    h_model_full_pb
    / phi_widths_fullphi
)


purity_fullphi = safe_divide(
    h_dps_full_pb,
    h_model_full_pb,
    default=np.nan,
)


# ------------------------------------------------------------
# 5. EXPLICIT closure against the data-driven plot
#
# These arrays MUST be equal because they represent the same
# measured data differential cross section.
# ------------------------------------------------------------

dd_data_diff_reference = (
    dd_proj_phi["data"]
    * FIDUCIAL_XSEC_NORM_PB
    / phi_widths_fullphi
)


max_abs_difference = float(
    np.max(
        np.abs(
            data_full_diff
            - dd_data_diff_reference
        )
    )
)


print(
    "\n=== TEMPLATE vs DATA-DRIVEN DATA CROSS-CHECK ==="
)

print(
    f"Integrated template-data cross section = "
    f"{sigma_data_fullphi:.3f} pb"
)

print(
    f"Integrated DD-data cross section       = "
    f"{sigma_fid_phi_pb:.3f} pb"
)

print(
    "Maximum bin-by-bin difference [pb/rad] =",
    max_abs_difference,
)


if not np.allclose(
    data_full_diff,
    dd_data_diff_reference,
    rtol=1e-10,
    atol=1e-10,
):
    raise RuntimeError(
        "Template and DD data projections do not match "
        "despite using the same events, weights and binning."
    )

print(
    "PASS: template and DD Data points are identical "
    "bin by bin."
)


# ------------------------------------------------------------
# 6. Plot
# ------------------------------------------------------------

fig, (ax, axp) = plt.subplots(
    2,
    1,
    figsize=(9.0, 7.6),
    gridspec_kw={
        "height_ratios": [3.2, 1.0],
        "hspace": 0.06,
    },
    sharex=True,
)
fig.subplots_adjust(left=0.16, right=0.97, bottom=0.14, top=0.83, hspace=0.07)


ax.errorbar(
    phi_centers_fullphi,
    data_full_diff,
    yerr=data_full_diff_err,
    xerr=0.5 * phi_widths_fullphi,
    fmt="o",
    color="black",
    markersize=4.6,
    linewidth=1.0,
    capsize=2.5,
    label="Data",
    zorder=5,
)


ax.stairs(
    model_full_diff,
    phi_edges_fullphi,
    linewidth=2.0,
    label="Post-fit SPS + DPS",
)

ax.stairs(
    dps_full_diff,
    phi_edges_fullphi,
    linewidth=1.7,
    linestyle="--",
    label="Post-fit DPS",
)

ax.stairs(
    sps_full_diff,
    phi_edges_fullphi,
    linewidth=1.7,
    linestyle=":",
    label="Post-fit SPS",
)





ax.text(
    0.98,
    0.9,
    "Full fiducial region"
    + "\n"
    + rf"$f_{{\mathrm{{DPS}}}}^{{\mathrm{{post-fit}}}}="
      rf"{f_dps_fullphi:.3f}$",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=20,
)


ax.set_ylabel(
    YLABEL_DPHI_XSEC,
    fontsize=AXIS_LABEL_FONT_SIZE,
)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.50, 0.9),
    frameon=False,
    ncol=1,
    fontsize=15,
)


axp.stairs(
    purity_fullphi,
    phi_edges_fullphi,
    linewidth=1.8,
)

axp.set_ylabel(
    LABEL_PDPS,
    fontsize=20,
)

axp.set_xlabel(
    XLABEL_DPHI,
    fontsize=20,
)

axp.grid(
    axis="y",
    alpha=0.20,
)


save_and_show(
    fig,
    "templatefit_postfit_dphi_full_fiducial_differential",
    OUTDIR,
)


In [ ]:
# ============================================================
# DATA-DRIVEN INTERMEDIATE PLOTS III:
# 2D data, scaled DPS template, and SPS residual
# SWAN / Matplotlib-safe version
# ============================================================

plt.close("all")

from matplotlib.colors import TwoSlopeNorm


# ------------------------------------------------------------
# Safe local CMS label
# ------------------------------------------------------------



# ============================================================
# EXPLICIT SCALAR / ARRAY INPUTS
# ============================================================

phi_edges_2d = np.asarray(
    phi_edges_intermediate,
    dtype=float,
)

dy_edges_2d = np.asarray(
    dy_edges_intermediate,
    dtype=float,
)

data_domain_2d = np.asarray(
    dd_data_domain,
    dtype=bool,
)

mix_domain_2d = np.asarray(
    dd_mix_domain,
    dtype=bool,
)

data_weights_2d = np.asarray(
    dd_data_weights,
    dtype=float,
)

mix_weights_2d = np.asarray(
    dd_mix_weights,
    dtype=float,
)


dphi_min_2d = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dphi_min"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dphi_max_2d = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dphi_max"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dy_min_2d = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dy_min"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dd_mixed_scale_2d = float(
    np.asarray(
        DD_MIXED_SCALE,
        dtype=float,
    ).reshape(-1)[0]
)


# ============================================================
# DATA MAP
# ============================================================

data_phi_2d = (
    data_sw[
        "delta_phi"
    ].to_numpy(dtype=float)
)

data_y_2d = (
    data_sw[
        "delta_y"
    ].to_numpy(dtype=float)
)


data_valid_2d = (
    data_domain_2d
    & np.isfinite(
        data_phi_2d
    )
    & np.isfinite(
        data_y_2d
    )
    & np.isfinite(
        data_weights_2d
    )
)


data_map_intermediate = np.histogram2d(
    data_phi_2d[
        data_valid_2d
    ],
    data_y_2d[
        data_valid_2d
    ],
    bins=[
        phi_edges_2d,
        dy_edges_2d,
    ],
    weights=data_weights_2d[
        data_valid_2d
    ],
)[0].astype(float)


# ============================================================
# MIXED-EVENT MAP
# ============================================================

mix_phi_2d = (
    dps_mixed[
        "delta_phi"
    ].to_numpy(dtype=float)
)

mix_y_2d = (
    dps_mixed[
        "delta_y"
    ].to_numpy(dtype=float)
)


mix_valid_2d = (
    mix_domain_2d
    & np.isfinite(
        mix_phi_2d
    )
    & np.isfinite(
        mix_y_2d
    )
    & np.isfinite(
        mix_weights_2d
    )
)


mixed_map_intermediate = np.histogram2d(
    mix_phi_2d[
        mix_valid_2d
    ],
    mix_y_2d[
        mix_valid_2d
    ],
    bins=[
        phi_edges_2d,
        dy_edges_2d,
    ],
    weights=mix_weights_2d[
        mix_valid_2d
    ],
)[0].astype(float)


# ============================================================
# DPS AND SPS DECOMPOSITION
# ============================================================

dps_map_intermediate = (
    dd_mixed_scale_2d
    * mixed_map_intermediate
)

sps_map_intermediate = (
    data_map_intermediate
    - dps_map_intermediate
)


# ------------------------------------------------------------
# Numerical checks
# ------------------------------------------------------------

if not np.all(
    np.isfinite(
        data_map_intermediate
    )
):
    raise RuntimeError(
        "Non-finite values found in data 2D map"
    )

if not np.all(
    np.isfinite(
        dps_map_intermediate
    )
):
    raise RuntimeError(
        "Non-finite values found in DPS 2D map"
    )

if not np.all(
    np.isfinite(
        sps_map_intermediate
    )
):
    raise RuntimeError(
        "Non-finite values found in SPS 2D residual"
    )


print(
    "2D data integral        =",
    float(
        np.sum(
            data_map_intermediate
        )
    ),
)

print(
    "2D scaled DPS integral  =",
    float(
        np.sum(
            dps_map_intermediate
        )
    ),
)

print(
    "2D SPS residual integral=",
    float(
        np.sum(
            sps_map_intermediate
        )
    ),
)


# ============================================================
# COLOUR SCALE
# ============================================================

# Use a common scale for Data and DPS so that their
# absolute normalization can be visually compared.

positive_max = max(
    float(
        np.nanmax(
            data_map_intermediate
        )
    ),
    float(
        np.nanmax(
            dps_map_intermediate
        )
    ),
    1.0e-12,
)


# Signed colour scale for SPS residual.
sps_span = max(
    float(
        np.nanmax(
            np.abs(
                sps_map_intermediate
            )
        )
    ),
    1.0e-12,
)

sps_norm = TwoSlopeNorm(
    vmin=-sps_span,
    vcenter=0.0,
    vmax=sps_span,
)


# ============================================================
# FIGURE
#
# IMPORTANT:
#   no constrained_layout
#   no mplhep cms_label
#   no bbox_inches="tight"
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15.4, 5.8),
)


fig.subplots_adjust(
    left=0.07,
    right=0.97,
    bottom=0.18,
    top=0.73,
    wspace=0.20,
)


# ============================================================
# PANEL 1: DATA
# ============================================================

mesh_data = axes[0].pcolormesh(
    phi_edges_2d,
    dy_edges_2d,
    data_map_intermediate.T,
    shading="auto",
    vmin=0.0,
    vmax=positive_max,
)

cbar_data = fig.colorbar(
    mesh_data,
    ax=axes[0],
    shrink=0.90,
    pad=0.025,
)

cbar_data.set_label(
    " "
)
cbar_data.ax.tick_params(labelsize=9.5)

axes[0].set_title("")

axes[0].annotate(
    "Data",
    xy=(0.50, 1.0),
    xycoords="axes fraction",
    xytext=(0.0, 27.0),
    textcoords="offset points",
    ha="center",
    va="bottom",
    fontsize=15,
    annotation_clip=False,
)


# ============================================================
# PANEL 2: SCALED MIXED-EVENT DPS
# ============================================================

mesh_dps = axes[1].pcolormesh(
    phi_edges_2d,
    dy_edges_2d,
    dps_map_intermediate.T,
    shading="auto",
    vmin=0.0,
    vmax=positive_max,
)

cbar_dps = fig.colorbar(
    mesh_dps,
    ax=axes[1],
    shrink=0.90,
    pad=0.025,
)

cbar_dps.set_label(
    " "
)
cbar_dps.ax.tick_params(labelsize=9.5)

axes[1].set_title("")

axes[1].annotate(
    "Scaled mixed-event DPS",
    xy=(0.50, 1.0),
    xycoords="axes fraction",
    xytext=(0.0, 27.0),
    textcoords="offset points",
    ha="center",
    va="bottom",
    fontsize=15,
    annotation_clip=False,
)


# ============================================================
# PANEL 3: SPS RESIDUAL
# ============================================================

mesh_sps = axes[2].pcolormesh(
    phi_edges_2d,
    dy_edges_2d,
    sps_map_intermediate.T,
    shading="auto",
    cmap="coolwarm",
    norm=sps_norm,
)

cbar_sps = fig.colorbar(
    mesh_sps,
    ax=axes[2],
    shrink=0.90,
    pad=0.025,
)

cbar_sps.set_label(
    r" "
)
cbar_sps.ax.tick_params(labelsize=9.5)

axes[2].set_title("")

axes[2].annotate(
    r"SPS residual: data $-$ DPS",
    xy=(0.50, 1.0),
    xycoords="axes fraction",
    xytext=(0.0, 27.0),
    textcoords="offset points",
    ha="center",
    va="bottom",
    fontsize=15,
    annotation_clip=False,
)


# ============================================================
# DPS-ENRICHED REGION BOUNDARIES
# ============================================================

for ax in axes:

    ax.axvline(
        dphi_min_2d,
        linestyle="--",
        linewidth=1.2,
        color="black",
        zorder=5,
    )

    ax.axvline(
        dphi_max_2d,
        linestyle="--",
        linewidth=1.2,
        color="black",
        zorder=5,
    )

    ax.axhline(
        dy_min_2d,
        linestyle="--",
        linewidth=1.2,
        color="black",
        zorder=5,
    )

    ax.set_xlim(
        float(
            phi_edges_2d[0]
        ),
        float(
            phi_edges_2d[-1]
        ),
    )

    ax.set_ylim(
        float(
            dy_edges_2d[0]
        ),
        float(
            dy_edges_2d[-1]
        ),
    )

    ax.set_xlabel(
        XLABEL_DPHI
    )

    ax.set_ylabel(
        XLABEL_DY
    )


# ============================================================
# CMS LABEL
# ============================================================



# ------------------------------------------------------------
# Region annotation
# ------------------------------------------------------------

axes[2].text(
    0.97,
    0.96,
    (
        rf"DPS-enriched region"
        "\n"
        rf"$|\Delta y|\geq {dy_min_2d:.2f}$"
        "\n"
        rf"${dphi_min_2d/np.pi:.2f}\pi"
        rf"\leq|\Delta\phi|\leq"
        rf"{dphi_max_2d/np.pi:.2f}\pi$"
    ),
    transform=axes[2].transAxes,
    ha="right",
    va="top",
    fontsize=15,
    bbox=dict(
        facecolor="white",
        alpha=0.85,
        edgecolor="none",
        pad=2.0,
    ),
)


# ============================================================
# SAVE
# ============================================================

stem = (
    "data_driven_intermediate_"
    "2D_data_DPS_SPS_CMS"
)

save_and_show(fig, stem, OUTDIR)


print()
print(
    "[DD intermediate III] completed successfully"
)

## 13. Final method comparison
¶
This is the main convener-facing result plot. It places the 2D profile-likelihood determination and the mixed-event data-driven cross-check side by side. All annotations are kept inside the plotting area; detailed configuration and provenance remain in the audit tables.

In [ ]:
# ============================================================
# FINAL METHOD COMPARISON:
# 2D profile likelihood vs mixed-event data-driven cross-check
# SWAN / Matplotlib-safe version
# ============================================================

plt.close("all")


# ------------------------------------------------------------
# Safe local CMS label
# ------------------------------------------------------------



# ============================================================
# NOMINAL TEMPLATE-FIT RESULT
# ============================================================

profile = FITS[
    NOMINAL_BINNING
]["total"]

profile_stat = FITS[
    NOMINAL_BINNING
]["stat"]

profile_baseline = FITS[
    NOMINAL_BINNING
]["baseline"]


profile_report = (
    REPORTABILITY.loc[
        REPORTABILITY[
            "binning"
        ] == NOMINAL_BINNING
    ]
    .iloc[0]
)


# Explicit scalars
profile_central = float(
    np.asarray(
        profile[
            "values"
        ][
            "f_dps"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

profile_down = float(
    np.asarray(
        profile[
            "f_error_down"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

profile_up = float(
    np.asarray(
        profile[
            "f_error_up"
        ],
        dtype=float,
    ).reshape(-1)[0]
)


dd_central_final = float(
    np.asarray(
        DD_CENTRAL[
            "f_before_closure"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dd_sigma_final = float(
    np.asarray(
        dd_total_sigma,
        dtype=float,
    ).reshape(-1)[0]
)

paired_z_final = float(
    np.asarray(
        delta_compatibility_z,
        dtype=float,
    ).reshape(-1)[0]
)


dy_min_final = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dy_min"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dphi_min_final = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dphi_min"
        ],
        dtype=float,
    ).reshape(-1)[0]
)

dphi_max_final = float(
    np.asarray(
        CENTRAL_DPS_REGION[
            "dphi_max"
        ],
        dtype=float,
    ).reshape(-1)[0]
)


gof_pvalue_final = float(
    np.asarray(
        GOF[
            NOMINAL_BINNING
        ][
            "pvalue"
        ],
        dtype=float,
    ).reshape(-1)[0]
)


model_inadequacy_final = float(
    np.asarray(
        profile[
            "model_inadequacy_rel"
        ],
        dtype=float,
    ).reshape(-1)[0]
)


# ============================================================
# FINAL RESULTS TABLE
# ============================================================

final_rows = pd.DataFrame(
    [
        {
            "method":
                "2D profile likelihood",

            "fDPS":
                profile_central,

            "down":
                profile_down,

            "up":
                profile_up,

            "role":
                "nominal",

            "n_free_parameters":
                len(
                    PARAMETER_NAMES
                ),

            "binning":
                NOMINAL_BINNING,

            "actual_n_dphi_bins":
                ACTUAL_N_DPHI_BINS,

            "actual_n_dy_bins":
                ACTUAL_N_DY_BINS,

            "dphi_max":
                float(
                    np.asarray(
                        DPHI_MAX,
                        dtype=float,
                    ).reshape(-1)[0]
                ),

            "AN143_integrated_systematics":
                INCLUDE_AN143_INTEGRATED_SYSTEMATICS,

            "efficiency_response_systematics":
                bool(
                    PROBLEMS[
                        NOMINAL_BINNING
                    ][
                        "efficiency_response_systematic"
                    ][
                        "available"
                    ]
                ),

            "year_response_fdps_envelope":
                float(
                    EFFICIENCY_RESPONSE_SYSTEMATICS.get(
                        "year_fdps_envelope",
                        np.nan,
                    )
                ),

            "efficiency_binning_fdps_envelope":
                float(
                    EFFICIENCY_RESPONSE_SYSTEMATICS.get(
                        "binning_fdps_envelope",
                        np.nan,
                    )
                ),

            "model_inadequacy_rel":
                model_inadequacy_final,
        },

        {
            "method":
                "Mixed-event data-driven × post-fit purity",

            "fDPS":
                dd_central_final,

            "down":
                dd_sigma_final,

            "up":
                dd_sigma_final,

            "role":
                "cross-check",

            "n_free_parameters":
                len(
                    PARAMETER_NAMES
                ),

            "binning":
                NOMINAL_BINNING,

            "dphi_max":
                float(
                    np.asarray(
                        DPHI_MAX,
                        dtype=float,
                    ).reshape(-1)[0]
                ),

            "AN143_integrated_systematics":
                INCLUDE_AN143_INTEGRATED_SYSTEMATICS,

            # Dedicated year/binning covariance belongs
            # to the template-fit shape model and is not
            # copied into the DD estimator.
            "efficiency_response_systematics":
                False,

            "region_dy_min":
                dy_min_final,

            "region_dphi_min":
                dphi_min_final,

            "region_dphi_max":
                dphi_max_final,

            "paired_difference_z":
                paired_z_final,
        },
    ]
)


final_rows.to_csv(
    OUTDIR
    / "final_fdps_results_FINAL.csv",
    index=False,
)


print(
    final_rows.to_string(
        index=False
    )
)


# ============================================================
# FINAL COMPARISON FIGURE
#
# IMPORTANT:
#   no constrained_layout
#   no mplhep cms_label
#   no bbox_inches="tight"
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.2, 5.8)
)


fig.subplots_adjust(
    left=0.14,
    right=0.97,
    bottom=0.20,
    top=0.83,
)


# ------------------------------------------------------------
# Points and uncertainties
# ------------------------------------------------------------

x = np.asarray(
    [
        0.0,
        1.0,
    ],
    dtype=float,
)


y = np.asarray(
    [
        profile_central,
        dd_central_final,
    ],
    dtype=float,
)


err_down = np.asarray(
    [
        profile_down,
        dd_sigma_final,
    ],
    dtype=float,
)


err_up = np.asarray(
    [
        profile_up,
        dd_sigma_final,
    ],
    dtype=float,
)


err = np.vstack(
    [
        err_down,
        err_up,
    ]
)


ax.errorbar(
    x,
    y,
    yerr=err,
    fmt="o",
    color="black",
    ecolor="black",
    markersize=6.0,
    capsize=5,
    linewidth=1.8,
    zorder=4,
)


# ------------------------------------------------------------
# X-axis labels
# ------------------------------------------------------------

ax.set_xticks(
    x
)


ax.set_xticklabels(
    [
        (
            "Template fit\n"
            + r"$(2\mathrm{D}\ \mathrm{profile})$"
        ),

        (
            "Data-driven method\n"
            + r"$(\mathrm{DPS\ enriched\ region})$"
        ),
    ],
    fontsize=15,
)


# ------------------------------------------------------------
# Axes
# ------------------------------------------------------------

ax.set_ylabel(
    LABEL_FDPS
)


ax.set_xlim(
    -0.55,
    1.55,
)


upper = max(
    0.82,
    float(
        np.max(
            y + err_up
        )
    )
    * 1.75,
)


ax.set_ylim(
    0.0,
    min(
        1.0,
        upper,
    ),
)


ax.grid(
    axis="y",
    alpha=0.18,
)


# ============================================================
# CMS LABEL
# ============================================================



# ============================================================
# STATUS
# ============================================================

status = (
    "Reportable candidate"
    if bool(
        profile_report[
            "reportable"
        ]
    )
    else
    "Diagnostic only"
)


# ============================================================
# ANNOTATION
# ============================================================

text = (
    rf"$f_{{\mathrm{{DPS}}}}^{{\mathrm{{template}}}}="
    rf"{100.0*profile_central:.1f}"
    rf"^{{+{100.0*profile_up:.1f}}}"
    rf"_{{-{100.0*profile_down:.1f}}}\%"
    rf"\qquad"
    rf"f_{{\mathrm{{DPS}}}}^{{\mathrm{{DD}}}}="
    rf"{100.0*dd_central_final:.1f}"
    rf"\pm{100.0*dd_sigma_final:.1f}\%$"
    "\n"

    + "Goodness-of-fit: "
    + rf"$p={gof_pvalue_final:.2f}$"
    "\n"

    + rf"$|\Delta y|\geq {dy_min_final:.2f},\quad"
    + rf"{dphi_min_final:.2f}"
    + rf"\leq|\Delta\phi|\leq"
    + rf"{dphi_max_final:.2f}$"
)


ax.text(
    0.50,
    0.9,
    text,
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=18,
    linespacing=1.28,
    bbox=dict(
        facecolor="white",
        alpha=0.92,
        edgecolor="none",
        pad=3.0,
    ),
    zorder=10,
)


# ============================================================
# SAVE
# ============================================================

stem = (
    "final_fdps_profile_vs_"
    "datadriven_CMS_FINAL"
)

save_and_show(fig, stem, OUTDIR)


print()
print(
    "[final comparison] completed successfully"
)


## 14. Final audit summary

The final JSON audit records the complete Boolean configuration, fitted parameters, uncertainty-mode choices, GOF, template statistics, the integrated efficiency-response systematic covariance, and the differential inputs that remain pending. The year-response covariance is explicitly temporary until 2016APV/2016 efficiency maps are available.

In [ ]:
missing = UNCERTAINTY_INVENTORY.loc[
    UNCERTAINTY_INVENTORY["status"].isin(
        ["MISSING_INPUT", "PENDING_INDEPENDENT_SAMPLE"]
    ),
    "source",
].tolist()
nominal_reportable = bool(
    REPORTABILITY.loc[
        REPORTABILITY["binning"]==NOMINAL_BINNING, "reportable"
    ].iloc[0]
)
pred_bias = POSTFIT_PREDICTION[NOMINAL_BINNING]["max_model_bootstrap_bias_sigma"]
baseline_dof = max(1, PROBLEMS[NOMINAL_BINNING]["H"].shape[0] - len(PARAMETER_NAMES))
baseline_gof_asymptotic = float(chi2_distribution.sf(
    profile_baseline["quadratic"], baseline_dof
))
projection_max_pulls = {
    axis: float(np.nanmax(np.abs(PROJECTION_RESULTS[(NOMINAL_BINNING, axis)]["marginal_pull"])))
    for axis in ("dphi", "dy")
}

if not nominal_reportable:
    STATUS = "DIAGNOSTIC_ONLY_PROFILE_GATE_FAILED"
elif missing:
    STATUS = "PROFILE_PASSES_BUT_REMAINING_DIFFERENTIAL_SYSTEMATICS_PENDING"
elif USE_PARTIAL_RUN2_EFFICIENCY_PROXY:
    STATUS = "PRELIMINARY_RUN2_WITH_2017_2018_EFFICIENCY_RESPONSE"
else:
    STATUS = "REPORTABLE_CANDIDATE_PENDING_ANALYSIS_REVIEW"

print("\nFINAL STATUS:", STATUS)
print("Configuration:", CONFIG_TAG)
print(f"Nominal fDPS: {profile_central:.4f} +{profile_up:.4f}/-{profile_down:.4f}")
print(f"Baseline fDPS (floor=0): {profile_baseline['values']['f_dps']:.4f}")
print(f"Empirical MC-shape floor: {100*profile['model_inadequacy_rel']:.1f}%")
if FREE_CC_FRACTIONS:
    print(
        f"f_cc(SPS): {profile['values']['f_cc_sps']:.4f} "
        f"+/- {profile['errors']['f_cc_sps']:.4f}"
    )
    print(
        f"f_cc(DPS): {profile['values']['f_cc_dps']:.4f} "
        f"+/- {profile['errors']['f_cc_dps']:.4f}"
    )
else:
    print(f"f_cc(SPS) fixed: {profile['values']['f_cc_sps']:.4f}")
    print(f"f_cc(DPS) fixed: {profile['values']['f_cc_dps']:.4f}")
print(f"Descriptive toy GOF p-value: {GOF[NOMINAL_BINNING]['pvalue']:.4g}")
print(f"Baseline asymptotic GOF p-value: {baseline_gof_asymptotic:.4g}")
print("Max projected marginal pulls:", projection_max_pulls)
print(f"Max adjusted 2D marginal pull: {LOCAL_PULL_MAPS[NOMINAL_BINNING]['max_abs_adjusted']:.3f}")
print(f"Minimum template N_eff: {PROBLEMS[NOMINAL_BINNING]['min_neff']:.3f}")
print(f"Prediction-bootstrap max centering: {pred_bias:.2f} sigma")
print("Data period:", DATA_PERIOD)
print(f"Full Run-2 luminosity: {LUMI_RUN2_FB_INV:.2f} fb^-1")
print("Efficiency-response years:", list(EFFICIENCY_YEARS))
print(
    "Efficiency-response luminosity:",
    f"{LUMI_EFFICIENCY_RESPONSE_FB_INV:.2f} fb^-1",
)
print(
    "Efficiency scope:",
    "PRELIMINARY 2017+2018 response; 2016 pending"
    if USE_PARTIAL_RUN2_EFFICIENCY_PROXY
    else "complete Run-2 response",
)
print("sPlot closure validated:", SPLOT_CLOSURE_VALIDATED)
print("AN143 integrated normalization enabled:", INCLUDE_AN143_INTEGRATED_SYSTEMATICS)
print(
    "Efficiency-response differential systematics included:",
    PROBLEMS[NOMINAL_BINNING]["efficiency_response_systematic"]["available"],
)
if EFFICIENCY_RESPONSE_SYSTEMATICS["available"]:
    print(
        "  year-response |delta fDPS| envelope:",
        f"{EFFICIENCY_RESPONSE_SYSTEMATICS['year_fdps_envelope']:.6f}",
    )
    print(
        "  efficiency-binning |delta fDPS| envelope:",
        f"{EFFICIENCY_RESPONSE_SYSTEMATICS['binning_fdps_envelope']:.6f}",
    )
print("Remaining differential inputs:", missing if missing else "none")
print("Automatic data-driven region:", CENTRAL_DPS_REGION)
print(f"Data-driven fDPS: {DD_CENTRAL['f_before_closure']:.4f} +/- {dd_total_sigma:.4f}")
print(f"Paired data-driven--fit compatibility: z={delta_compatibility_z:+.3f}")

audit_payload = {
    "status": STATUS,
    "configuration": ANALYSIS_CONFIGURATION,
    "nominal_reportable": nominal_reportable,
    "nominal_parameters": {k: float(v) for k, v in profile["values"].items()},
    "nominal_parameter_errors_hesse": {k: float(v) for k, v in profile["errors"].items()},
    "nominal_fDPS": float(profile_central),
    "nominal_fDPS_down": float(profile_down),
    "nominal_fDPS_up": float(profile_up),
    "baseline_no_floor_fDPS": float(profile_baseline["values"]["f_dps"]),
    "baseline_no_floor_fDPS_down": float(profile_baseline["f_error_down"]),
    "baseline_no_floor_fDPS_up": float(profile_baseline["f_error_up"]),
    "baseline_no_floor_quadratic": float(profile_baseline["quadratic"]),
    "baseline_no_floor_asymptotic_gof_pvalue": baseline_gof_asymptotic,
    "toy_gof_pvalue": float(GOF[NOMINAL_BINNING]["pvalue"]),
    "toy_gof_pvalue_mcse": float(GOF[NOMINAL_BINNING]["pvalue_mcse"]),
    "toy_gof_pvalue_ci95": [float(x) for x in GOF[NOMINAL_BINNING]["pvalue_ci95"]],
    "toy_gof_successful": int(len(GOF[NOMINAL_BINNING]["q_toys"])),
    "toy_gof_role": "descriptive under the adopted fixed covariance model",
    "model_inadequacy": {
        "enabled": bool(USE_MODEL_INADEQUACY),
        "relative_2D_floor": float(profile["model_inadequacy_rel"]),
        "source": MODEL_INADEQUACY_SOURCE,
        "shape_only_normalization_mode_removed": True,
        "calibrated_on_same_data": bool(MODEL_INADEQUACY_CALIBRATED_ON_SAME_DATA),
        "externally_validated": bool(MODEL_INADEQUACY_EXTERNALLY_VALIDATED),
        "reportability_veto": False,
        "analysis_policy": "fixed adopted nominal covariance component",
        "max_2D_marginal_pull_baseline": float(LOCAL_PULL_MAPS[NOMINAL_BINNING]["max_abs_baseline"]),
        "max_2D_marginal_pull_adjusted": float(LOCAL_PULL_MAPS[NOMINAL_BINNING]["max_abs_adjusted"]),
        "max_projected_marginal_pulls": projection_max_pulls,
    },
    "minimum_template_neff": float(PROBLEMS[NOMINAL_BINNING]["min_neff"]),
    "prediction_bootstrap_max_centering_sigma": float(pred_bias),
    "efficiency_response_systematics": {
        "enabled": bool(INCLUDE_EFFICIENCY_RESPONSE_SYSTEMATICS),
        "available": bool(EFFICIENCY_RESPONSE_SYSTEMATICS["available"]),
        "source_directory": str(EFFICIENCY_RESPONSE_SYSTEMATICS_DIR),
        "source_nominal_fDPS": float(
            EFFICIENCY_RESPONSE_SYSTEMATICS.get("source_nominal_fdps", np.nan)
        ),
        "year_response_temporary_until_2016": bool(
            EFFICIENCY_RESPONSE_SYSTEMATICS.get(
                "year_response_temporary_until_2016", True
            )
        ),
        "year_response_fdps_envelope": float(
            EFFICIENCY_RESPONSE_SYSTEMATICS.get("year_fdps_envelope", np.nan)
        ),
        "year_response_fdps_rms": float(
            EFFICIENCY_RESPONSE_SYSTEMATICS.get("year_fdps_rms", np.nan)
        ),
        "efficiency_binning_fdps_envelope": float(
            EFFICIENCY_RESPONSE_SYSTEMATICS.get("binning_fdps_envelope", np.nan)
        ),
        "efficiency_binning_fdps_rms": float(
            EFFICIENCY_RESPONSE_SYSTEMATICS.get("binning_fdps_rms", np.nan)
        ),
        "covariance_dimension": int(
            EFFICIENCY_RESPONSE_SYSTEMATICS["cov"].shape[0]
        ),
        "normalization_mode_removed": True,
        "propagated_in_template_fit": bool(
            PROBLEMS[NOMINAL_BINNING]["efficiency_response_systematic"]["available"]
        ),
        "propagated_in_data_driven": False,
    },
    "missing_differential_sources": missing,
    "efficiency_cross_factor_covariance_available": EFFICIENCY_CROSS_FACTOR_COVARIANCE_AVAILABLE,
    "splot_closure_validated": SPLOT_CLOSURE_VALIDATED,
    "data_period": DATA_PERIOD,
    "full_run2_luminosity_fb_inv": float(LUMI_RUN2_FB_INV),
    "efficiency_response_years": list(EFFICIENCY_YEARS),
    "efficiency_response_luminosity_fb_inv": float(LUMI_EFFICIENCY_RESPONSE_FB_INV),
    "partial_run2_efficiency_proxy": bool(USE_PARTIAL_RUN2_EFFICIENCY_PROXY),
    "use_2017_mc_as_run2_proxy": bool(USE_2017_MC_AS_RUN2_PROXY),
    "AN143_common_rel_available": float(AN143_COMMON_REL_AVAILABLE),
    "AN143_common_rel_effective": float(AN143_COMMON_REL_EFFECTIVE),
    "AN143_integrated_systematics_enabled": bool(INCLUDE_AN143_INTEGRATED_SYSTEMATICS),
    "data_driven": {
        "method": "mixed-event cross-check multiplied by post-fit simulation purity",
        "independent_of_template_fit": False,
        "automatic_region": bool(AUTO_DD_REGION),

        "region_predeclared": bool(not AUTO_DD_REGION),

        "region_frozen_after_nominal_fit": bool(AUTO_DD_REGION),

        "region": {
            k: float(v)
            for k, v in CENTRAL_DPS_REGION.items()
        },

        "fixed_region_definition": {
            k: float(v)
            for k, v in FIXED_DPS_REGION.items()
        },

        "selection_rule": (
            "maximum post-fit DPS purity after support and "
            "mixed-event/simulation closure gates"
            if AUTO_DD_REGION
            else
            "predeclared fixed region: |dy|>=1.8 and "
            "pi/3<=|dphi|<=2pi/3"
        ),
        "maximum_mixed_dps_closure_deviation": float(DD_MAX_MIXED_DPS_CLOSURE_DEVIATION),
        "purity_plateau_width": float(DD_PURITY_PLATEAU),
        "accepted_candidates": int(np.count_nonzero(DD_REGION_SCAN["accepted"])),
        "plateau_candidates": int(len(DD_REGION_PLATEAU)),
        "maximum_purity": (
            float(DD_MAX_PURITY)
            if AUTO_DD_REGION
            else None
        ),

        "selected_purity": float(
            DD_CENTRAL["purity"]
        ),        
        "selected_mixed_purity_audit": float(DD_CENTRAL["mixed_purity"]),
        "data_region_fraction": float(DD_CENTRAL["data_region_fraction"]),
        "mixed_region_fraction": float(DD_CENTRAL["mixed_region_fraction"]),
        "data_region_neff": float(DD_CENTRAL["data_region_neff"]),
        "mixed_region_neff": float(DD_CENTRAL["mixed_region_neff"]),
        "dps_mc_region_neff": float(DD_SELECTED_SCAN_ROW["dps_mc_region_neff"]),
        "sps_mc_region_neff": float(DD_SELECTED_SCAN_ROW["sps_region_neff"]),
        "dps_mc_to_mixed_closure": float(DD_CENTRAL["dps_mc_to_mixed_closure"]),
        "fDPS_raw": float(DD_CENTRAL["f_raw"]),
        "fDPS_before_closure": float(DD_CENTRAL["f_before_closure"]),
        "fDPS_closure_divided_audit": float(DD_CENTRAL["f_corrected"]),
        "fDPS_mixed_hybrid_audit": float(DD_CENTRAL["f_mixed_hybrid_audit"]),
        "fDPS_solved_mixture_audit": float(DD_CENTRAL["f_solved_audit"]),
        "fDPS": float(DD_CENTRAL["f_before_closure"]),
        "fDPS_total_sigma": float(dd_total_sigma),
        "fDPS_statistical_sigma": float(dd_stat_sigma),
        "fDPS_systematic_sigma": float(dd_syst_sigma),
        "finite_mixed_pair_sigma": float(mix_only_sigma),
        "paired_difference": float(delta_central),
        "paired_difference_sigma": float(delta_total_sigma),
        "paired_difference_z": float(delta_compatibility_z),
        "paired_difference_pvalue": float(delta_compatibility_pvalue),
        "successful_data_replicas": int(len(dd_data_values)),
        "successful_model_replicas": int(len(dd_model_values)),
        "requested_data_replicas": int(N_DD_DATA_REPLICAS),
        "requested_model_replicas": int(N_DD_MODEL_REPLICAS),
        "replica_convergence": ENSEMBLE_DIAGNOSTIC_ROWS,
    },
}
with open(OUTDIR/"FINAL_analysis_status.json", "w") as handle:
    json.dump(audit_payload, handle, indent=2)


## 15. Optional $\sigma_{\mathrm{eff}}$
¶
This derived quantity remains disabled by default and should only be evaluated after the nominal quality gates pass and the external cross-section covariance assumptions are reviewed.

In [ ]:
# ================================================================
# PRELIMINARY sigma_eff EXTRACTION
# ================================================================
#
# This diagnostic is intentionally allowed even when the complete
# Run-2 systematic/reportability contract is not yet closed.
#
# Formula:
#
#   sigma_eff =
#       sigma(J/psi) * sigma(D*)
#       --------------------------------
#       f_DPS * sigma(J/psi + D*)
#
# The asymmetric profiled fDPS interval is propagated exactly through
# the 1/fDPS dependence.
#
# The external cross-section statistical and systematic uncertainties
# are propagated separately in quadrature, assuming no correlations.
#
# IMPORTANT:
#   Until 2016 efficiencies + remaining validation/systematic inputs
#   are closed, this value must be labelled PRELIMINARY / DIAGNOSTIC.
# ================================================================

RUN_SIGMA_EFF = bool(
    int(os.environ.get("JPSIDSTAR_RUN_SIGMA_EFF", "1"))
)


# ----------------------------------------------------------------
# External cross-section inputs
# ----------------------------------------------------------------

SIGMA_ASSOC_TOT_PB = 568.58
SIGMA_ASSOC_TOT_STAT_PB = 6.75
SIGMA_ASSOC_TOT_SYST_PB = 157.80

SIGMA_JPSI_NB = 3.74
SIGMA_JPSI_STAT_NB = 0.02
SIGMA_JPSI_SYST_NB = 0.06

SIGMA_DSTAR_UB = 442.29
SIGMA_DSTAR_STAT_UB = 9.90
SIGMA_DSTAR_SYST_UB = 26.05


# ----------------------------------------------------------------
# sigma_eff
# ----------------------------------------------------------------

def sigma_eff_mb(
    f_dps,
    sigma_assoc_pb=SIGMA_ASSOC_TOT_PB,
    sigma_jpsi_nb=SIGMA_JPSI_NB,
    sigma_dstar_ub=SIGMA_DSTAR_UB,
):
    """
    Return sigma_eff in mb.

    Unit conversion:
        nb -> pb : x 1e3
        ub -> pb : x 1e6
        pb -> mb : / 1e9
    """

    f_dps = float(f_dps)

    if not np.isfinite(f_dps) or f_dps <= 0.0:
        raise ValueError(
            f"Invalid fDPS={f_dps}; sigma_eff requires fDPS > 0."
        )

    return float(
        (sigma_jpsi_nb * 1e3)
        * (sigma_dstar_ub * 1e6)
        / (f_dps * sigma_assoc_pb)
        / 1e9
    )


if RUN_SIGMA_EFF:

    # ------------------------------------------------------------
    # Retrieve the final profile result
    # ------------------------------------------------------------

    f_central = float(profile_central)
    f_down = float(profile_down)
    f_up = float(profile_up)

    if not (
        np.isfinite(f_central)
        and np.isfinite(f_down)
        and np.isfinite(f_up)
    ):
        raise RuntimeError(
            "Non-finite profiled fDPS result."
        )

    if f_central <= 0.0:
        raise RuntimeError(
            f"Cannot evaluate sigma_eff for fDPS={f_central}."
        )


    # ------------------------------------------------------------
    # Central value
    # ------------------------------------------------------------

    sigma_eff_central = sigma_eff_mb(f_central)


    # ------------------------------------------------------------
    # Asymmetric uncertainty from profiled fDPS
    #
    # sigma_eff ~ 1/fDPS:
    #
    #   larger fDPS -> smaller sigma_eff
    #   smaller fDPS -> larger sigma_eff
    # ------------------------------------------------------------

    f_high = min(
        1.0 - 1e-9,
        f_central + f_up,
    )

    f_low = max(
        1e-9,
        f_central - f_down,
    )

    sigma_eff_from_f_high = sigma_eff_mb(f_high)
    sigma_eff_from_f_low = sigma_eff_mb(f_low)

    sigma_eff_fdps_down = (
        sigma_eff_central
        - sigma_eff_from_f_high
    )

    sigma_eff_fdps_up = (
        sigma_eff_from_f_low
        - sigma_eff_central
    )


    # ------------------------------------------------------------
    # External cross-section statistical uncertainty
    #
    # Assuming the three external measurements are independent:
    #
    # (dsigma_eff / sigma_eff)^2 =
    #
    #   (dsigma_Jpsi / sigma_Jpsi)^2
    # + (dsigma_Dstar / sigma_Dstar)^2
    # + (dsigma_assoc / sigma_assoc)^2
    #
    # Signs do not matter for the variance.
    # ------------------------------------------------------------

    rel_external_stat = np.sqrt(
        (SIGMA_JPSI_STAT_NB / SIGMA_JPSI_NB) ** 2
        +
        (SIGMA_DSTAR_STAT_UB / SIGMA_DSTAR_UB) ** 2
        +
        (SIGMA_ASSOC_TOT_STAT_PB / SIGMA_ASSOC_TOT_PB) ** 2
    )

    sigma_eff_external_stat = (
        sigma_eff_central
        * rel_external_stat
    )


    # ------------------------------------------------------------
    # External cross-section systematic uncertainty
    # ------------------------------------------------------------

    rel_external_syst = np.sqrt(
        (SIGMA_JPSI_SYST_NB / SIGMA_JPSI_NB) ** 2
        +
        (SIGMA_DSTAR_SYST_UB / SIGMA_DSTAR_UB) ** 2
        +
        (SIGMA_ASSOC_TOT_SYST_PB / SIGMA_ASSOC_TOT_PB) ** 2
    )

    sigma_eff_external_syst = (
        sigma_eff_central
        * rel_external_syst
    )


    # ------------------------------------------------------------
    # Combined uncertainty
    #
    # The fDPS profile uncertainty is kept asymmetric.
    #
    # External stat/syst uncertainties are treated as symmetric
    # and independent from the fDPS profile because no joint
    # covariance is currently available.
    # ------------------------------------------------------------

    sigma_eff_total_down = np.sqrt(
        sigma_eff_fdps_down ** 2
        +
        sigma_eff_external_stat ** 2
        +
        sigma_eff_external_syst ** 2
    )

    sigma_eff_total_up = np.sqrt(
        sigma_eff_fdps_up ** 2
        +
        sigma_eff_external_stat ** 2
        +
        sigma_eff_external_syst ** 2
    )


    # ------------------------------------------------------------
    # DPS associated-production cross section used internally
    # ------------------------------------------------------------

    sigma_dps_assoc_pb = (
        f_central
        * SIGMA_ASSOC_TOT_PB
    )


    # ------------------------------------------------------------
    # Status label
    # ------------------------------------------------------------

    _nominal_reportable = bool(
        globals().get(
            "nominal_reportable",
            False,
        )
    )

    sigma_eff_status = "PRELIMINARY / DIAGNOSTIC"
    # 2016 efficiencies, sPlot-model variants and an independent
    # efficiency closure are still pending in this analysis version.


    # ------------------------------------------------------------
    # Machine-readable result for later plots/slides
    # ------------------------------------------------------------

    SIGMA_EFF_RESULT = {
        "status": sigma_eff_status,

        "fDPS": f_central,
        "fDPS_error_down": f_down,
        "fDPS_error_up": f_up,

        "sigma_DPS_assoc_pb": sigma_dps_assoc_pb,

        "sigma_eff_mb": sigma_eff_central,

        "sigma_eff_fdps_error_down_mb":
            sigma_eff_fdps_down,

        "sigma_eff_fdps_error_up_mb":
            sigma_eff_fdps_up,

        "sigma_eff_external_stat_mb":
            sigma_eff_external_stat,

        "sigma_eff_external_syst_mb":
            sigma_eff_external_syst,

        "sigma_eff_total_error_down_mb":
            sigma_eff_total_down,

        "sigma_eff_total_error_up_mb":
            sigma_eff_total_up,

        "external_independence_assumption": True,

        "efficiency_scope":
            "2017+2018 response used as partial Run-2 proxy; "
            "2016 pending",
    }


    # ------------------------------------------------------------
    # Human-readable output
    # ------------------------------------------------------------

    print()
    print("=" * 78)
    print("PRELIMINARY sigma_eff EXTRACTION")
    print("=" * 78)

    print(
        f"Status: {sigma_eff_status}"
    )

    print()

    print(
        "Input fDPS = "
        f"{f_central:.4f} "
        f"+{f_up:.4f}/-{f_down:.4f}"
    )

    print(
        "sigma_DPS(J/psi + D*) = "
        f"{sigma_dps_assoc_pb:.3f} pb"
    )

    print()

    print(
        "sigma_eff = "
        f"{sigma_eff_central:.4f} "
        f"+{sigma_eff_fdps_up:.4f}"
        f"/-{sigma_eff_fdps_down:.4f} mb "
        "(fDPS profile only)"
    )

    print(
        "external statistical uncertainty = "
        f"+/- {sigma_eff_external_stat:.4f} mb "
        f"({100.0 * rel_external_stat:.2f}%)"
    )

    print(
        "external systematic uncertainty  = "
        f"+/- {sigma_eff_external_syst:.4f} mb "
        f"({100.0 * rel_external_syst:.2f}%)"
    )

    print()

    print(
        "sigma_eff = "
        f"{sigma_eff_central:.4f} "
        f"+{sigma_eff_total_up:.4f}"
        f"/-{sigma_eff_total_down:.4f} mb "
        "(combined diagnostic uncertainty)"
    )

    print()
    print(
        "Combination assumes no covariance between the profiled fDPS "
        "and the external cross-section measurements."
    )

    if not _nominal_reportable:
        print(
            "WARNING: this sigma_eff value is PRELIMINARY. "
            "2016 efficiencies and the remaining closure/sPlot-model "
            "checks are still pending."
        )

    print("=" * 78)


else:

    SIGMA_EFF_RESULT = None

    print(
        "sigma_eff disabled "
        "(set JPSIDSTAR_RUN_SIGMA_EFF=1 to enable)."
    )


In [ ]:
# ================================================================
# COMPLETE sigma_eff UNCERTAINTY BUDGET + DOUBLE-COUNTING AUDIT
# ================================================================
# Read-only diagnostic:
# - does NOT modify FITS
# - does NOT modify fDPS
# - does NOT modify sigma_eff
# - does NOT add any new systematic
#
# Only three NON-OVERLAPPING aggregate terms are used in the final
# sigma_eff uncertainty:
#
#   1) total profiled fDPS uncertainty
#   2) external cross-section statistical subtotal
#   3) external cross-section systematic subtotal
#
# All other rows are decomposition / diagnostic information only.
# ================================================================

if SIGMA_EFF_RESULT is None:
    raise RuntimeError(
        "SIGMA_EFF_RESULT is unavailable; run the existing sigma_eff cell first."
    )


# ----------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------

def _budget_profile_row(label):
    matches = POSTFIT_UNCERTAINTY_TABLE.loc[
        POSTFIT_UNCERTAINTY_TABLE["contribution"].eq(label)
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one POSTFIT_UNCERTAINTY_TABLE row for {label!r}; "
            f"found {len(matches)}."
        )
    return matches.iloc[0]


_total_fit = FITS[NOMINAL_BINNING]["total"]
_f0 = float(_total_fit["values"]["f_dps"])
_sigma0 = float(sigma_eff_mb(_f0))


def _fdps_to_sigma_eff_impact(delta_down, delta_up):
    """
    Exact propagation through sigma_eff ~ 1/fDPS.

    delta_down: fDPS downward uncertainty
    delta_up:   fDPS upward uncertainty

    Returns:
        sigma_eff downward impact,
        sigma_eff upward impact
    """
    delta_down = float(delta_down)
    delta_up = float(delta_up)

    f_high = min(1.0 - 1.0e-12, _f0 + delta_up)
    f_low = max(1.0e-12, _f0 - delta_down)

    sigma_down = _sigma0 - float(sigma_eff_mb(f_high))
    sigma_up = float(sigma_eff_mb(f_low)) - _sigma0

    return float(sigma_down), float(sigma_up)


def _symmetric_sigma_eff_impact(relative_uncertainty):
    return float(_sigma0 * float(relative_uncertainty))


def _safe_float(value):
    try:
        value = float(value)
    except Exception:
        return np.nan
    return value if np.isfinite(value) else np.nan


# ----------------------------------------------------------------
# AN-21-143 associated-cross-section systematic decomposition
# ----------------------------------------------------------------

AN21143_SOURCES_PERCENT = {
    "Luminosity": 2.42,
    "Branching ratios": 2.05,
    "Signal-yield extraction": 1.00,
    "Tracking": 9.25,
    "MC sample statistics": 0.10,
    "Pile-up reweighting": 16.33,
    "Yield instability + pile-up dependence": 20.00,
    "Trigger efficiency": 3.40,
}

an21143_quadrature_percent = float(
    np.sqrt(
        np.sum(
            np.square(
                list(AN21143_SOURCES_PERCENT.values())
            )
        )
    )
)

assoc_current_syst_percent = float(
    100.0 * SIGMA_ASSOC_TOT_SYST_PB / SIGMA_ASSOC_TOT_PB
)

print(
    "AN-21-143 eight-source quadrature subtotal = "
    f"{an21143_quadrature_percent:.3f}%"
)
print(
    "Current associated-cross-section systematic input = "
    f"{assoc_current_syst_percent:.3f}%"
)
print(
    "Difference = "
    f"{an21143_quadrature_percent - assoc_current_syst_percent:+.3f} "
    "percentage points (reported only; neither input is modified)."
)


# ----------------------------------------------------------------
# Existing fDPS profile components
# ----------------------------------------------------------------

stat_row = _budget_profile_row(
    "Statistical profile (data + finite MC/response statistics)"
)

approved_row = _budget_profile_row(
    "Efficiency response systematics (all approved)"
)

shape_row = _budget_profile_row(
    "Adopted MC-shape component"
)

total_profile_row = _budget_profile_row(
    "Total profile"
)


# ----------------------------------------------------------------
# Budget container
# ----------------------------------------------------------------

_budget_rows = []


def _add_budget_row(
    source,
    origin,
    status,
    affects_fdps=False,
    affects_sigma_assoc=False,
    input_uncertainty="",
    impact_down=np.nan,
    impact_up=np.nan,
    included_in_total=False,
    an21143=False,
    note="",
):
    _budget_rows.append({
        "source": str(source),
        "origin": str(origin),
        "status": str(status),
        "affects_fDPS": bool(affects_fdps),
        "affects_sigma_assoc": bool(affects_sigma_assoc),
        "input_uncertainty": str(input_uncertainty),
        "delta_sigma_eff_down_mb": _safe_float(impact_down),
        "delta_sigma_eff_up_mb": _safe_float(impact_up),
        "included_in_total": bool(included_in_total),
        "AN-21-143": bool(an21143),
        "note": str(note),
    })


# ================================================================
# 1. fDPS uncertainty side
# ================================================================

# Statistical profile
stat_down_mb, stat_up_mb = _fdps_to_sigma_eff_impact(
    stat_row["down"],
    stat_row["up"],
)

_add_budget_row(
    source="Statistical profile",
    origin="fDPS profile likelihood",
    status="DECOMPOSITION_ONLY",
    affects_fdps=True,
    input_uncertainty=(
        f"-{float(stat_row['down']):.6f}/"
        f"+{float(stat_row['up']):.6f} fDPS"
    ),
    impact_down=stat_down_mb,
    impact_up=stat_up_mb,
    note=(
        "Aggregated data + finite MC/response statistical profile. "
        "Already contained in the total profiled fDPS uncertainty."
    ),
)

# Internal contents of the statistical profile.
# No independent scalar contribution exists for these pieces.
for source in [
    "Data statistics",
    "Finite MC/template statistics",
    "Efficiency-map statistics",
    "DPS pT stitching statistics",
    "SPS/DPS ccbar/bbbar composition statistics",
]:
    _add_budget_row(
        source=source,
        origin="joint finite-simulation/statistical response ensemble",
        status="INCLUDED_IN_STATISTICAL_PROFILE",
        affects_fdps=True,
        input_uncertainty="jointly covaried; no independent scalar",
        note=(
            "Part of the statistical profile. "
            "Not added separately to sigma_eff."
        ),
    )


# Approved differential efficiency-response aggregate
approved_down_mb, approved_up_mb = _fdps_to_sigma_eff_impact(
    approved_row["down"],
    approved_row["up"],
)

_add_budget_row(
    source="Approved differential profile increment",
    origin="fDPS normalized-shape covariance",
    status="DECOMPOSITION_ONLY",
    affects_fdps=True,
    input_uncertainty=(
        f"-{float(approved_row['down']):.6f}/"
        f"+{float(approved_row['up']):.6f} fDPS"
    ),
    impact_down=approved_down_mb,
    impact_up=approved_up_mb,
    note="Already included in the total profiled fDPS uncertainty.",
)


# Dedicated efficiency-response studies
eff_syst = (
    EFFICIENCY_RESPONSE_SYSTEMATICS
    if isinstance(EFFICIENCY_RESPONSE_SYSTEMATICS, dict)
    else {}
)

year_envelope = _safe_float(
    eff_syst.get("year_fdps_envelope", np.nan)
)

binning_envelope = _safe_float(
    eff_syst.get("binning_fdps_envelope", np.nan)
)

for source, envelope in [
    ("Efficiency year response", year_envelope),
    ("Efficiency-map binning", binning_envelope),
]:
    if np.isfinite(envelope):
        impact_down, impact_up = _fdps_to_sigma_eff_impact(
            envelope,
            envelope,
        )

        _add_budget_row(
            source=source,
            origin="dedicated efficiency-response study",
            status="DECOMPOSITION_ONLY",
            affects_fdps=True,
            input_uncertainty=f"+/-{envelope:.6f} fDPS",
            impact_down=impact_down,
            impact_up=impact_up,
            note=(
                "Shown separately for interpretation; already propagated "
                "through the approved differential covariance."
            ),
        )
    else:
        _add_budget_row(
            source=source,
            origin="dedicated efficiency-response study",
            status="NO_NUMERIC_VALUE",
            affects_fdps=True,
            input_uncertainty="not available",
            note="No numerical value invented.",
        )


# Signal-yield / sPlot differential variation
try:
    splot_available = bool(
        PROBLEMS[NOMINAL_BINNING]["splot_systematic"]["available"]
    )
except Exception:
    splot_available = False

_add_budget_row(
    source="Signal-yield / sPlot fit variants",
    origin="differential signal-extraction variations",
    status=(
        "INCLUDED_IN_APPROVED_DIFFERENTIAL_PROFILE"
        if splot_available
        else "PENDING_INPUT"
    ),
    affects_fdps=True,
    input_uncertainty=(
        "differential covariance; no independent scalar"
        if splot_available
        else "pending differential input"
    ),
    note=(
        "No independent scalar is added to sigma_eff. "
        "If available, it is already part of the profiled fDPS covariance."
    ),
)


# MC-template shape inadequacy
shape_down_mb, shape_up_mb = _fdps_to_sigma_eff_impact(
    shape_row["down"],
    shape_row["up"],
)

_add_budget_row(
    source="Adopted MC-template shape inadequacy component",
    origin="nominal fDPS model prescription",
    status="DECOMPOSITION_ONLY",
    affects_fdps=True,
    input_uncertainty=(
        f"{100.0 * float(MODEL_INADEQUACY_REL):.1f}% normalized-shape floor; "
        f"profile increment "
        f"-{float(shape_row['down']):.6f}/"
        f"+{float(shape_row['up']):.6f} fDPS"
    ),
    impact_down=shape_down_mb,
    impact_up=shape_up_mb,
    note=(
        "Already included in the total profiled fDPS uncertainty. "
        "Not added again."
    ),
)


# TOTAL fDPS uncertainty — THIS is what enters sigma_eff
fdps_total_down_mb, fdps_total_up_mb = _fdps_to_sigma_eff_impact(
    total_profile_row["down"],
    total_profile_row["up"],
)

_add_budget_row(
    source="Total profiled fDPS uncertainty",
    origin="nominal HIST profile likelihood",
    status="INCLUDED_AGGREGATE",
    affects_fdps=True,
    input_uncertainty=(
        f"-{float(total_profile_row['down']):.6f}/"
        f"+{float(total_profile_row['up']):.6f} fDPS"
    ),
    impact_down=fdps_total_down_mb,
    impact_up=fdps_total_up_mb,
    included_in_total=True,
    note=(
        "Used exactly once in sigma_eff. "
        "Its statistical, efficiency-response and MC-shape pieces "
        "are not added again."
    ),
)


# ================================================================
# 2. Associated J/psi + D* cross section
# ================================================================

assoc_stat_rel = float(
    SIGMA_ASSOC_TOT_STAT_PB / SIGMA_ASSOC_TOT_PB
)
assoc_stat_mb = _symmetric_sigma_eff_impact(
    assoc_stat_rel
)

_add_budget_row(
    source="Associated-cross-section statistical uncertainty",
    origin="sigma(J/psi+D*)",
    status="DECOMPOSITION_ONLY",
    affects_sigma_assoc=True,
    input_uncertainty=(
        f"{SIGMA_ASSOC_TOT_STAT_PB:.2f} pb "
        f"({100.0 * assoc_stat_rel:.3f}%)"
    ),
    impact_down=assoc_stat_mb,
    impact_up=assoc_stat_mb,
    note="Included through the external statistical subtotal.",
)


# Individual AN-21-143 systematics.
# IMPORTANT: decomposition only.
for source, percent in AN21143_SOURCES_PERCENT.items():

    rel = float(percent / 100.0)
    impact = _symmetric_sigma_eff_impact(rel)

    _add_budget_row(
        source=f"AN-21-143: {source}",
        origin="AN-21-143 associated-cross-section systematic",
        status="DECOMPOSITION_ONLY",
        affects_sigma_assoc=True,
        input_uncertainty=f"{percent:.2f}%",
        impact_down=impact,
        impact_up=impact,
        an21143=True,
        note=(
            "Component of the associated-cross-section systematic subtotal. "
            "Not added separately to the final sigma_eff variance."
        ),
    )


# AN quadrature cross-check
an_quad_mb = _symmetric_sigma_eff_impact(
    an21143_quadrature_percent / 100.0
)

_add_budget_row(
    source="AN-21-143 eight-source quadrature subtotal",
    origin="AN-21-143 cross-check",
    status="CROSSCHECK_ONLY",
    affects_sigma_assoc=True,
    input_uncertainty=f"{an21143_quadrature_percent:.3f}%",
    impact_down=an_quad_mb,
    impact_up=an_quad_mb,
    an21143=True,
    note=(
        "Cross-check only. The current sigma(J/psi+D*) systematic input "
        "is used in the nominal sigma_eff propagation."
    ),
)


# Current associated-cross-section systematic input
assoc_syst_rel = float(
    SIGMA_ASSOC_TOT_SYST_PB / SIGMA_ASSOC_TOT_PB
)
assoc_syst_mb = _symmetric_sigma_eff_impact(
    assoc_syst_rel
)

_add_budget_row(
    source="Associated-cross-section systematic subtotal (current input)",
    origin="sigma(J/psi+D*) current external input",
    status="INCLUDED_VIA_EXTERNAL_SYSTEMATIC_SUBTOTAL",
    affects_sigma_assoc=True,
    input_uncertainty=(
        f"{SIGMA_ASSOC_TOT_SYST_PB:.2f} pb "
        f"({100.0 * assoc_syst_rel:.3f}%)"
    ),
    impact_down=assoc_syst_mb,
    impact_up=assoc_syst_mb,
    an21143=True,
    note=(
        "Used once inside the external systematic subtotal. "
        "The eight AN components are NOT added again."
    ),
)


# ================================================================
# 3. Inclusive J/psi and D* cross sections
# ================================================================

jpsi_stat_rel = float(
    SIGMA_JPSI_STAT_NB / SIGMA_JPSI_NB
)
jpsi_syst_rel = float(
    SIGMA_JPSI_SYST_NB / SIGMA_JPSI_NB
)

dstar_stat_rel = float(
    SIGMA_DSTAR_STAT_UB / SIGMA_DSTAR_UB
)
dstar_syst_rel = float(
    SIGMA_DSTAR_SYST_UB / SIGMA_DSTAR_UB
)


for source, origin, value, unit, rel, category in [
    (
        "J/psi statistical",
        "inclusive J/psi cross section",
        SIGMA_JPSI_STAT_NB,
        "nb",
        jpsi_stat_rel,
        "external statistical subtotal",
    ),
    (
        "J/psi systematic",
        "inclusive J/psi cross section",
        SIGMA_JPSI_SYST_NB,
        "nb",
        jpsi_syst_rel,
        "external systematic subtotal",
    ),
    (
        "D* statistical",
        "inclusive D* cross section",
        SIGMA_DSTAR_STAT_UB,
        "ub",
        dstar_stat_rel,
        "external statistical subtotal",
    ),
    (
        "D* systematic",
        "inclusive D* cross section",
        SIGMA_DSTAR_SYST_UB,
        "ub",
        dstar_syst_rel,
        "external systematic subtotal",
    ),
]:

    impact = _symmetric_sigma_eff_impact(rel)

    _add_budget_row(
        source=source,
        origin=origin,
        status="DECOMPOSITION_ONLY",
        input_uncertainty=(
            f"{float(value):.4g} {unit} "
            f"({100.0 * rel:.3f}%)"
        ),
        impact_down=impact,
        impact_up=impact,
        note=f"Included through the {category}.",
    )


# ================================================================
# 4. External aggregate uncertainties
# ================================================================

external_stat_rel = float(
    np.sqrt(
        assoc_stat_rel**2
        + jpsi_stat_rel**2
        + dstar_stat_rel**2
    )
)

external_stat_mb = _symmetric_sigma_eff_impact(
    external_stat_rel
)

_add_budget_row(
    source="External statistical subtotal",
    origin="associated + inclusive cross sections",
    status="INCLUDED_AGGREGATE",
    affects_sigma_assoc=True,
    input_uncertainty=f"{100.0 * external_stat_rel:.3f}%",
    impact_down=external_stat_mb,
    impact_up=external_stat_mb,
    included_in_total=True,
    note=(
        "Associated, J/psi and D* statistical components "
        "are represented once by this subtotal."
    ),
)


external_syst_rel = float(
    np.sqrt(
        assoc_syst_rel**2
        + jpsi_syst_rel**2
        + dstar_syst_rel**2
    )
)

external_syst_mb = _symmetric_sigma_eff_impact(
    external_syst_rel
)

_add_budget_row(
    source="External systematic subtotal",
    origin="associated + inclusive cross sections",
    status="INCLUDED_AGGREGATE",
    affects_sigma_assoc=True,
    input_uncertainty=f"{100.0 * external_syst_rel:.3f}%",
    impact_down=external_syst_mb,
    impact_up=external_syst_mb,
    included_in_total=True,
    note=(
        "Current associated systematic + J/psi systematic + D* systematic. "
        "AN components are not re-added."
    ),
)


# ================================================================
# 5. Reconstruct final sigma_eff uncertainty
# ================================================================

sigma_eff_total_down = float(
    np.sqrt(
        fdps_total_down_mb**2
        + external_stat_mb**2
        + external_syst_mb**2
    )
)

sigma_eff_total_up = float(
    np.sqrt(
        fdps_total_up_mb**2
        + external_stat_mb**2
        + external_syst_mb**2
    )
)


# Regression against already-existing nominal result.
regression = {
    "central": (
        _sigma0,
        float(SIGMA_EFF_RESULT["sigma_eff_mb"]),
    ),
    "fdps_down": (
        fdps_total_down_mb,
        float(SIGMA_EFF_RESULT["sigma_eff_fdps_error_down_mb"]),
    ),
    "fdps_up": (
        fdps_total_up_mb,
        float(SIGMA_EFF_RESULT["sigma_eff_fdps_error_up_mb"]),
    ),
    "external_stat": (
        external_stat_mb,
        float(SIGMA_EFF_RESULT["sigma_eff_external_stat_mb"]),
    ),
    "external_syst": (
        external_syst_mb,
        float(SIGMA_EFF_RESULT["sigma_eff_external_syst_mb"]),
    ),
    "total_down": (
        sigma_eff_total_down,
        float(SIGMA_EFF_RESULT["sigma_eff_total_error_down_mb"]),
    ),
    "total_up": (
        sigma_eff_total_up,
        float(SIGMA_EFF_RESULT["sigma_eff_total_error_up_mb"]),
    ),
}

for name, (reconstructed, stored) in regression.items():
    if not np.isclose(
        reconstructed,
        stored,
        rtol=1.0e-10,
        atol=1.0e-10,
    ):
        raise AssertionError(
            f"sigma_eff regression failed for {name}: "
            f"reconstructed={reconstructed:.12g}, "
            f"stored={stored:.12g}"
        )


_add_budget_row(
    source="Total sigma_eff uncertainty",
    origin="final diagnostic result",
    status="RESULT",
    affects_fdps=True,
    affects_sigma_assoc=True,
    input_uncertainty="non-overlapping aggregate terms only",
    impact_down=sigma_eff_total_down,
    impact_up=sigma_eff_total_up,
    note="Result row; not an additional uncertainty source.",
)


# ================================================================
# 6. Final budget table
# ================================================================

SIGMA_EFF_UNCERTAINTY_BUDGET = pd.DataFrame(
    _budget_rows
)

SIGMA_EFF_UNCERTAINTY_BUDGET[
    "relative_sigma_eff_down_percent"
] = (
    100.0
    * SIGMA_EFF_UNCERTAINTY_BUDGET[
        "delta_sigma_eff_down_mb"
    ]
    / _sigma0
)

SIGMA_EFF_UNCERTAINTY_BUDGET[
    "relative_sigma_eff_up_percent"
] = (
    100.0
    * SIGMA_EFF_UNCERTAINTY_BUDGET[
        "delta_sigma_eff_up_mb"
    ]
    / _sigma0
)


# "Equivalent" share is informative for decomposition rows.
# It is NOT additive for nested/decomposed sources.
SIGMA_EFF_UNCERTAINTY_BUDGET[
    "equivalent_variance_share_down_percent"
] = (
    100.0
    * SIGMA_EFF_UNCERTAINTY_BUDGET[
        "delta_sigma_eff_down_mb"
    ]**2
    / sigma_eff_total_down**2
)

SIGMA_EFF_UNCERTAINTY_BUDGET[
    "equivalent_variance_share_up_percent"
] = (
    100.0
    * SIGMA_EFF_UNCERTAINTY_BUDGET[
        "delta_sigma_eff_up_mb"
    ]**2
    / sigma_eff_total_up**2
)

SIGMA_EFF_UNCERTAINTY_BUDGET[
    "equivalent_variance_share_percent"
] = 0.5 * (
    SIGMA_EFF_UNCERTAINTY_BUDGET[
        "equivalent_variance_share_down_percent"
    ]
    +
    SIGMA_EFF_UNCERTAINTY_BUDGET[
        "equivalent_variance_share_up_percent"
    ]
)


# For the actual final variance, ONLY included aggregate rows count.
SIGMA_EFF_UNCERTAINTY_BUDGET[
    "used_variance_share_down_percent"
] = np.where(
    SIGMA_EFF_UNCERTAINTY_BUDGET["included_in_total"],
    SIGMA_EFF_UNCERTAINTY_BUDGET[
        "equivalent_variance_share_down_percent"
    ],
    np.nan,
)

SIGMA_EFF_UNCERTAINTY_BUDGET[
    "used_variance_share_up_percent"
] = np.where(
    SIGMA_EFF_UNCERTAINTY_BUDGET["included_in_total"],
    SIGMA_EFF_UNCERTAINTY_BUDGET[
        "equivalent_variance_share_up_percent"
    ],
    np.nan,
)


# Result row is not itself a source.
_result_mask = (
    SIGMA_EFF_UNCERTAINTY_BUDGET["source"]
    == "Total sigma_eff uncertainty"
)

SIGMA_EFF_UNCERTAINTY_BUDGET.loc[
    _result_mask,
    [
        "equivalent_variance_share_down_percent",
        "equivalent_variance_share_up_percent",
        "equivalent_variance_share_percent",
        "used_variance_share_down_percent",
        "used_variance_share_up_percent",
    ],
] = np.nan


# ================================================================
# 7. Robust double-counting audit
# ================================================================

included = SIGMA_EFF_UNCERTAINTY_BUDGET.loc[
    SIGMA_EFF_UNCERTAINTY_BUDGET["included_in_total"]
].copy()

expected_included = {
    "Total profiled fDPS uncertainty",
    "External statistical subtotal",
    "External systematic subtotal",
}

actual_included = set(
    included["source"].tolist()
)

check_exact_aggregates = (
    actual_included == expected_included
)

check_fdps_once = (
    (included["source"] == "Total profiled fDPS uncertainty").sum()
    == 1
)

check_external_stat_once = (
    (included["source"] == "External statistical subtotal").sum()
    == 1
)

check_external_syst_once = (
    (included["source"] == "External systematic subtotal").sum()
    == 1
)

# No individual AN component may be marked as an additive final term.
check_an_not_readded = not bool(
    SIGMA_EFF_UNCERTAINTY_BUDGET.loc[
        SIGMA_EFF_UNCERTAINTY_BUDGET["source"].str.startswith(
            "AN-21-143:"
        ),
        "included_in_total",
    ].any()
)

# Current associated systematic subtotal is represented through
# External systematic subtotal, not added independently.
check_assoc_syst_not_readded = not bool(
    SIGMA_EFF_UNCERTAINTY_BUDGET.loc[
        SIGMA_EFF_UNCERTAINTY_BUDGET["source"].eq(
            "Associated-cross-section systematic subtotal (current input)"
        ),
        "included_in_total",
    ].any()
)

# Pending inputs must never be additive.
pending_mask = (
    SIGMA_EFF_UNCERTAINTY_BUDGET["status"]
    .astype(str)
    .str.startswith("PENDING")
)

check_pending_not_added = not bool(
    SIGMA_EFF_UNCERTAINTY_BUDGET.loc[
        pending_mask,
        "included_in_total",
    ].any()
)

# The three actually-used variance pieces must close to 100%.
used_down_sum = float(
    included[
        "used_variance_share_down_percent"
    ].sum()
)

used_up_sum = float(
    included[
        "used_variance_share_up_percent"
    ].sum()
)

check_variance_closure = bool(
    np.isclose(
        used_down_sum,
        100.0,
        rtol=1.0e-10,
        atol=1.0e-8,
    )
    and
    np.isclose(
        used_up_sum,
        100.0,
        rtol=1.0e-10,
        atol=1.0e-8,
    )
)


def _audit_status(passed):
    if passed is None:
        return "REVIEW"
    return "PASS" if bool(passed) else "FAIL"


_audit_rows = [
    (
        "Exactly three non-overlapping aggregate terms enter sigma_eff",
        "DOUBLE COUNTING",
        check_exact_aggregates,
        (
            "Expected and found: total fDPS profile + external statistical "
            "subtotal + external systematic subtotal."
        ),
    ),
    (
        "Full fDPS profile is propagated exactly once",
        "DOUBLE COUNTING",
        check_fdps_once,
        (
            "Statistical, efficiency-response and MC-shape pieces "
            "are decomposition only."
        ),
    ),
    (
        "External statistical subtotal is propagated exactly once",
        "DOUBLE COUNTING",
        check_external_stat_once,
        (
            "Associated, J/psi and D* statistical components "
            "are not re-added."
        ),
    ),
    (
        "External systematic subtotal is propagated exactly once",
        "DOUBLE COUNTING",
        check_external_syst_once,
        (
            "Associated, J/psi and D* systematic components "
            "are not re-added."
        ),
    ),
    (
        "Individual AN-21-143 sources are not added on top of the subtotal",
        "DOUBLE COUNTING",
        check_an_not_readded,
        (
            "AN sources are displayed as decomposition only."
        ),
    ),
    (
        "Associated-cross-section systematic subtotal is not added twice",
        "DOUBLE COUNTING",
        check_assoc_syst_not_readded,
        (
            "It enters once through the external systematic aggregate."
        ),
    ),
    (
        "Pending inputs are not added numerically",
        "DOUBLE COUNTING",
        check_pending_not_added,
        "Pending rows are informational only.",
    ),
    (
        "Used variance shares close to 100%",
        "DOUBLE COUNTING",
        check_variance_closure,
        (
            f"down={used_down_sum:.6f}%, "
            f"up={used_up_sum:.6f}%."
        ),
    ),
    (
        "fDPS vs sigma(J/psi+D*) statistical correlation",
        "MISSING/UNKNOWN COVARIANCE",
        None,
        "Correlation currently assumed zero; review in final publication treatment.",
    ),
    (
        "Efficiency-related correlations between fDPS and cross-section inputs",
        "MISSING/UNKNOWN COVARIANCE",
        None,
        "Correlation currently assumed zero; this is not double counting.",
    ),
    (
        "Common luminosity/tracking/trigger/pileup correlations among external cross sections",
        "MISSING/UNKNOWN COVARIANCE",
        None,
        "Review if the external measurements share correlated inputs.",
    ),
]

SIGMA_EFF_DOUBLE_COUNTING_AUDIT = pd.DataFrame([
    {
        "audit_item": item,
        "classification": classification,
        "status": _audit_status(passed),
        "passed": passed if passed is not None else pd.NA,
        "evidence": evidence,
    }
    for item, classification, passed, evidence in _audit_rows
])


double_counting_failures = (
    SIGMA_EFF_DOUBLE_COUNTING_AUDIT[
        SIGMA_EFF_DOUBLE_COUNTING_AUDIT[
            "classification"
        ].eq("DOUBLE COUNTING")
    ]
)

if not bool(
    double_counting_failures["passed"]
    .astype(bool)
    .all()
):
    display(SIGMA_EFF_DOUBLE_COUNTING_AUDIT)
    raise AssertionError(
        "sigma_eff double-counting audit failed; "
        "see the displayed audit table for the exact failed item."
    )


# ================================================================
# 8. Slide-friendly table
# ================================================================

def _impact_text(row):
    down = row["delta_sigma_eff_down_mb"]
    up = row["delta_sigma_eff_up_mb"]

    if not (
        np.isfinite(down)
        and np.isfinite(up)
    ):
        return "included elsewhere / no scalar"

    if np.isclose(
        down,
        up,
        rtol=1.0e-10,
        atol=1.0e-10,
    ):
        return f"+/-{up:.4f} mb"

    return f"-{down:.4f}/+{up:.4f} mb"


def _share_text(row):
    share = row["equivalent_variance_share_percent"]

    if not np.isfinite(share):
        return "-"

    if row["included_in_total"]:
        return f"{share:.1f}%"

    return f"{share:.1f}% (decomp.)"


slide_sources = [
    # fDPS side
    "Statistical profile",
    "Efficiency-map statistics",
    "Efficiency year response",
    "Efficiency-map binning",
    "Signal-yield / sPlot fit variants",
    "Adopted MC-template shape inadequacy component",
    "Total profiled fDPS uncertainty",

    # associated cross section
    "Associated-cross-section statistical uncertainty",
    *[
        f"AN-21-143: {name}"
        for name in AN21143_SOURCES_PERCENT
    ],
    "Associated-cross-section systematic subtotal (current input)",

    # inclusive cross sections
    "J/psi statistical",
    "J/psi systematic",
    "D* statistical",
    "D* systematic",

    # aggregates / result
    "External statistical subtotal",
    "External systematic subtotal",
    "Total sigma_eff uncertainty",
]

slide_budget = (
    SIGMA_EFF_UNCERTAINTY_BUDGET
    .set_index("source")
    .loc[slide_sources]
    .reset_index()
)

SIGMA_EFF_UNCERTAINTY_SLIDE_TABLE = pd.DataFrame({
    "Source": slide_budget["source"],
    "Input uncertainty": slide_budget["input_uncertainty"],
    "Impact on sigma_eff": [
        _impact_text(row)
        for _, row in slide_budget.iterrows()
    ],
    "Share of final variance": [
        _share_text(row)
        for _, row in slide_budget.iterrows()
    ],
    "Used in final total?": [
        "YES" if value else "NO"
        for value in slide_budget["included_in_total"]
    ],
    "AN-21-143?": [
        "YES" if value else "NO"
        for value in slide_budget["AN-21-143"]
    ],
})


# ================================================================
# 9. Save + display
# ================================================================

SIGMA_EFF_UNCERTAINTY_BUDGET.to_csv(
    OUTDIR / "sigma_eff_uncertainty_budget.csv",
    index=False,
)

SIGMA_EFF_DOUBLE_COUNTING_AUDIT.to_csv(
    OUTDIR / "sigma_eff_double_counting_audit.csv",
    index=False,
)

SIGMA_EFF_UNCERTAINTY_SLIDE_TABLE.to_csv(
    OUTDIR / "sigma_eff_uncertainty_slide_table.csv",
    index=False,
)


print("\n=== COMPLETE sigma_eff UNCERTAINTY BUDGET ===")
display(SIGMA_EFF_UNCERTAINTY_BUDGET)

print("\n=== DOUBLE-COUNTING AUDIT ===")
display(SIGMA_EFF_DOUBLE_COUNTING_AUDIT)

print("\n=== SLIDE-FRIENDLY UNCERTAINTY TABLE ===")
display(SIGMA_EFF_UNCERTAINTY_SLIDE_TABLE)


print("\n=== FINAL CLOSURE ===")
print(
    f"sigma_eff = {_sigma0:.4f} "
    f"-{sigma_eff_total_down:.4f}"
    f"/+{sigma_eff_total_up:.4f} mb"
)

print(
    "Variance actually used in final result: "
    f"down={used_down_sum:.3f}%, "
    f"up={used_up_sum:.3f}%"
)

print(
    "Double-counting audit: PASS"
)

## 16. Recommended convener-facing figure sequence

The notebook deliberately produces more diagnostics than should appear in one talk. For the final presentation, the most compact physics narrative is:

1. **Data preparation:** `data_splot_before_after_CMS` and the sPlot-correlation audit.
2. **Detector response:** MC before/after efficiency-correction diagnostic.
3. **Pre-fit discrimination:** `prefit_data_SPS_DPS_2D_inputs_CMS` and/or `prefit_SPS_DPS_projection_separation_CMS`.
4. **Nominal template fit:** post-fit $|\Delta\phi|$, post-fit $|\Delta y|$, 2D local-pull map, profile-likelihood scan, and toy GOF.
5. **Template-fit validation:** injected-$f_{\mathrm{DPS}}$ closure plot.
6. **Data-driven construction:** data vs mixed events in the enriched region, then the SPS/DPS decomposition with the dotted SPS+DPS closure curve, followed by the 2D data/DPS/SPS maps.
7. **Final result:** the data-driven $f_{\mathrm{DPS}}$ result and the final template-fit vs data-driven comparison.

The pre-fit plot is particularly useful pedagogically because it shows the discrimination information **before** revealing the fitted fraction.

## 17. Final nominal audit and convener checklist

This final cell is deliberately HIST-only. It checks the numerical contracts of the nominal result, records fit/GOF/closure/data-driven summary values, and writes a machine-readable final audit. It does not compare alternative template estimators.

In [ ]:
# ================================================================
# MACHINE-READABLE FINAL NOMINAL HIST AUDIT
# ================================================================
profile = FITS[NOMINAL_BINNING]["total"]
profile_stat = FITS[NOMINAL_BINNING]["stat"]
problem = PROBLEMS[NOMINAL_BINNING]
gof = GOF[NOMINAL_BINNING]

closure_max_bias = (
    float(np.max(np.abs(TEMPLATE_CLOSURE_SUMMARY["bias"])))
    if len(TEMPLATE_CLOSURE_SUMMARY) else np.nan
)
closure_pull_mean = (
    float(np.mean(TEMPLATE_CLOSURE_TOYS["pull"]))
    if len(TEMPLATE_CLOSURE_TOYS) else np.nan
)
closure_pull_width = (
    float(np.std(TEMPLATE_CLOSURE_TOYS["pull"], ddof=1))
    if len(TEMPLATE_CLOSURE_TOYS) > 1 else np.nan
)
max_pull_dphi = float(np.nanmax(np.abs(
    PROJECTION_RESULTS[(NOMINAL_BINNING, "dphi")]["marginal_pull"]
)))
max_pull_dy = float(np.nanmax(np.abs(
    PROJECTION_RESULTS[(NOMINAL_BINNING, "dy")]["marginal_pull"]
)))
max_pull_2d = float(LOCAL_PULL_MAPS[NOMINAL_BINNING]["max_abs_adjusted"])

# Mandatory numerical audits.
audit_failures = []
central_model = central_mixture(problem, profile["values"])["shape"]
if not np.all(np.isfinite(central_model)) or not np.isclose(np.sum(central_model), 1.0, atol=1e-12):
    audit_failures.append("central HIST template is non-finite or not normalized")
if len(data_sw) != REFERENCE_SPLOT_ENTRIES:
    audit_failures.append("data/sWeight aligned entry count changed")

for label, cov in {
    "data_cov": problem["data_cov"],
    "fit_full_cov": profile["full_cov"],
    "fit_reduced_cov": profile["V_reduced"],
    "efficiency_systematic_cov": problem["efficiency_response_systematic"]["cov"],
}.items():
    cov = np.asarray(cov, float)
    if not np.all(np.isfinite(cov)):
        audit_failures.append(f"{label} contains NaN/inf")
        continue
    scale = max(float(np.max(np.abs(cov))), 1e-20)
    if not np.allclose(cov, cov.T, rtol=0.0, atol=1e-10 * scale):
        audit_failures.append(f"{label} is not symmetric")
    if float(np.min(np.linalg.eigvalsh(0.5 * (cov + cov.T)))) < -1e-8 * scale:
        audit_failures.append(f"{label} is not PSD")

for component in COMPONENTS:
    response = np.asarray(problem["central_states"][component]["response"], float)
    if np.any(~np.isfinite(response)) or float(np.sum(response)) <= 0.0:
        audit_failures.append(f"invalid nominal HIST response for {component}")

pending = [
    "signal-yield/sPlot fit variants",
    "independent efficiency closure",
    "2016APV/2016 efficiencies",
]

summary = {
    "template_estimator": "hist",
    "high_statistics_mode": bool(HIGH_STATISTICS_MODE),
    "fit_binning": NOMINAL_BINNING,
    "actual_fit_grid": f"{ACTUAL_N_DPHI_BINS}x{ACTUAL_N_DY_BINS}",
    "dphi_range": [0.0, float(DPHI_MAX)],
    "number_fit_parameters": len(PARAMETER_NAMES),
    "fDPS": float(profile["values"]["f_dps"]),
    "stat_down": float(profile_stat["f_error_down"]),
    "stat_up": float(profile_stat["f_error_up"]),
    "total_down": float(profile["f_error_down"]),
    "total_up": float(profile["f_error_up"]),
    "gof_toy_pvalue": float(gof["pvalue"]),
    "gof_pvalue_mcse": float(gof["pvalue_mcse"]),
    "gof_statistic_quadratic": float(profile["quadratic"]),
    "gof_toys_requested": int(gof["n_requested"]),
    "gof_toys_successful": int(len(gof["q_toys"])),
    "gof_toys_failed": int(gof["n_requested"] - len(gof["q_toys"])),
    "max_2d_pull": max_pull_2d,
    "max_projected_dphi_pull": max_pull_dphi,
    "max_projected_dy_pull": max_pull_dy,
    "minimum_template_N_eff": float(problem["min_neff"]),
    "closure_max_bias": closure_max_bias,
    "closure_pull_mean": closure_pull_mean,
    "closure_pull_width": closure_pull_width,
    "closure_status": (
        "PASS" if len(TEMPLATE_CLOSURE_SUMMARY)
        and bool(np.all(TEMPLATE_CLOSURE_SUMMARY["closure_status"] == "PASS"))
        else "WARNING" if len(TEMPLATE_CLOSURE_SUMMARY) else "NOT_RUN"
    ),
    "efficiency_source": "luminosity-weighted validated 2017+2018 response used as current Run-2 proxy",
    "systematics_included": [
        "finite MC/template", "efficiency-map statistical replicas",
        "year response", "efficiency binning", "adopted MC-shape floor",
    ],
    "pending_systematics": pending,
    "data_driven_fDPS": float(DD_CENTRAL["f_before_closure"]),
    "data_driven_total_sigma": float(dd_total_sigma),
    "template_vs_DD_compatibility_z": float(delta_compatibility_z),
    "sigma_eff": SIGMA_EFF_RESULT,
    "reportability_status": STATUS,
    "audit_failures": audit_failures,
}

summary_path = OUTDIR / "FINAL_NOMINAL_HIST_analysis_status.json"
with summary_path.open("w") as handle:
    json.dump(summary, handle, indent=2, allow_nan=False)

FINAL_AUDIT = pd.DataFrame([
    ("template estimator", "HIST (nominal)"),
    ("configuration", CONFIG_TAG),
    ("high-statistics mode", HIGH_STATISTICS_MODE),
    ("fit binning", f"{NOMINAL_BINNING} ({ACTUAL_N_DPHI_BINS}x{ACTUAL_N_DY_BINS})"),
    ("DeltaPhi range", f"0--{DPHI_MAX:.8f}"),
    ("number of fit parameters", len(PARAMETER_NAMES)),
    ("fDPS", summary["fDPS"]),
    ("stat interval", f"-{summary['stat_down']:.5f}/+{summary['stat_up']:.5f}"),
    ("total interval", f"-{summary['total_down']:.5f}/+{summary['total_up']:.5f}"),
    ("GOF p-value", summary["gof_toy_pvalue"]),
    ("GOF failed toys", summary["gof_toys_failed"]),
    ("closure max bias", closure_max_bias),
    ("closure pull mean", closure_pull_mean),
    ("closure pull width", closure_pull_width),
    ("max post-fit pull", max(max_pull_2d, max_pull_dphi, max_pull_dy)),
    ("minimum template N_eff", summary["minimum_template_N_eff"]),
    ("efficiency source", summary["efficiency_source"]),
    ("pending", "; ".join(pending)),
    ("data-driven fDPS", summary["data_driven_fDPS"]),
    ("template-vs-DD compatibility z", summary["template_vs_DD_compatibility_z"]),
    ("sigma_eff", SIGMA_EFF_RESULT),
    ("status", STATUS),
    ("audit", "PASS" if not audit_failures else "FAIL: " + "; ".join(audit_failures)),
], columns=["item", "value"])

display(FINAL_AUDIT)
FINAL_AUDIT.to_csv(OUTDIR / "FINAL_NOMINAL_HIST_AUDIT.csv", index=False)

print("\n" + "=" * 92)
print("FINAL NOMINAL HIST AUDIT")
print("=" * 92)
print(FINAL_AUDIT.to_string(index=False))
print("=" * 92)
if audit_failures:
    raise RuntimeError("Mandatory final audit failed: " + "; ".join(audit_failures))

In [ ]:
# ================================================================
# FINAL PRESENTATION-LAYOUT AUDIT AND CONTACT SHEET
# ================================================================
EXPECTED_PRESENTATION_STEMS = {
    "data_splot_before_after_CMS",
    "prefit_component_HIST_templates_2D_CMS",
    "mc_efficiency_before_after_dphi_CMS",
    "mc_efficiency_before_after_dy_CMS",
    "prefit_data_SPS_DPS_2D_inputs_CMS",
    "prefit_SPS_DPS_projection_separation_CMS",
    f"profile_likelihood_scan_{NOMINAL_BINNING}",
    f"gof_toys_{NOMINAL_BINNING}_CMS",
    "templatefit_closure_linearity_CMS",
    "templatefit_closure_pull_CMS",
    f"prediction_uncertainty_breakdown_{NOMINAL_BINNING}_dphi",
    f"prediction_uncertainty_breakdown_{NOMINAL_BINNING}_dy",
    f"profile_{NOMINAL_BINNING}_dphi_CMS_FINAL",
    f"profile_{NOMINAL_BINNING}_dy_CMS_FINAL",
    f"profile_{NOMINAL_BINNING}_2D_local_pull_FINAL",
    f"templatefit_postfit_dphi_after_dy_{DPS_CUT:.2f}_differential",
    "templatefit_postfit_dphi_full_fiducial_differential",
    "dps_mc_vs_mixed_SELECTED_before_efficiency",
    "dps_mc_vs_mixed_FIDUCIAL_after_efficiency",
    "data_driven_statistical_uncertainty_toys_CMS_FINAL",
    "data_driven_region_systematic_CMS_FINAL",
    "data_driven_enriched_region_FINAL",
    "data_driven_intermediate_data_vs_mixed_conditional_CMS",
    "data_driven_intermediate_SPS_DPS_fiducial_xsec_projections_CMS",
    "data_driven_intermediate_2D_data_DPS_SPS_CMS",
    "final_fdps_profile_vs_datadriven_CMS_FINAL",
}

_layout_rows = []
for _record in FIGURE_LAYOUT_AUDIT:
    _row = dict(_record)
    for _key in (
        "issues", "bbox_issues", "semantic_issues", "cms_issues",
        "panel_issues", "raster_issues", "explicit_overlap_issues",
    ):
        if _key in _row:
            _row[_key] = " | ".join(_row[_key])
    for _key in ("cms_panel_status", "explicit_overlap_audits"):
        if _key in _row:
            _row[_key] = json.dumps(_row[_key], sort_keys=True)
    _layout_rows.append(_row)
FIGURE_LAYOUT_AUDIT_TABLE = pd.DataFrame(_layout_rows)
if FIGURE_LAYOUT_AUDIT_TABLE.empty:
    raise RuntimeError("No presentation figures reached the final layout audit")

_audited_stems = set(FIGURE_LAYOUT_AUDIT_TABLE["stem"].astype(str))
_missing_layout_figures = sorted(EXPECTED_PRESENTATION_STEMS - _audited_stems)
_unexpected_layout_figures = sorted(_audited_stems - EXPECTED_PRESENTATION_STEMS)
if _missing_layout_figures or _unexpected_layout_figures:
    raise RuntimeError(
        "Presentation-figure completeness failure: missing="
        f"{_missing_layout_figures}; unexpected={_unexpected_layout_figures}"
    )
if FIGURE_LAYOUT_AUDIT_TABLE["stem"].duplicated().any():
    duplicates = FIGURE_LAYOUT_AUDIT_TABLE.loc[
        FIGURE_LAYOUT_AUDIT_TABLE["stem"].duplicated(keep=False), "stem"
    ].tolist()
    raise RuntimeError(f"Duplicate presentation audit rows: {duplicates}")

_contact_sheet_path = OUTDIR / "presentation_figure_contact_sheet.png"
create_presentation_contact_sheet(
    OUTDIR,
    FIGURE_LAYOUT_AUDIT,
    _contact_sheet_path,
)
FIGURE_LAYOUT_AUDIT_TABLE.to_csv(
    OUTDIR / "figure_layout_audit_FINAL.csv", index=False
)

_failed_layout = FIGURE_LAYOUT_AUDIT_TABLE.loc[
    FIGURE_LAYOUT_AUDIT_TABLE["overall_status"] != "PASS"
]
print("\nFINAL PRESENTATION LAYOUT AUDIT")
display(FIGURE_LAYOUT_AUDIT_TABLE[[
    "stem", "source_cell", "bbox_status", "semantic_status",
    "cms_spacing_status", "cms_panel_count", "cms_expected_panel_count",
    "panel_spacing_status", "explicit_overlap_status",
    "raster_whitespace_status", "overall_status",
]])
_expected_dense_panel_counts = {
    "mc_efficiency_before_after_dphi_CMS": 8,
    "mc_efficiency_before_after_dy_CMS": 8,
    "data_driven_intermediate_SPS_DPS_fiducial_xsec_projections_CMS": 2,
}
for _stem, _expected_count in _expected_dense_panel_counts.items():
    _match = FIGURE_LAYOUT_AUDIT_TABLE.loc[
        FIGURE_LAYOUT_AUDIT_TABLE["stem"] == _stem
    ]
    if len(_match) != 1:
        raise RuntimeError(f"Missing unique panel-count audit row for {_stem}")
    _observed_count = int(_match.iloc[0]["cms_panel_count"])
    if _observed_count != _expected_count:
        raise RuntimeError(
            f"{_stem}: expected {_expected_count} CMS-owning panels, "
            f"found {_observed_count}"
        )
    print(
        f"Per-panel CMS audit: {_stem} = {_observed_count}/"
        f"{_expected_count} PASS"
    )

print("Maximum raster outer margins:")
for _side in ("left", "right", "top", "bottom"):
    print(
        f"  {_side:>6s} = "
        f"{FIGURE_LAYOUT_AUDIT_TABLE[f'raster_margin_{_side}'].max():.4f}"
    )
print(
    "Largest normalized inter-panel gap =",
    max(
        float(FIGURE_LAYOUT_AUDIT_TABLE["horizontal_gap_ratio_max"].max()),
        float(FIGURE_LAYOUT_AUDIT_TABLE["vertical_gap_ratio_max"].max()),
    ),
)
print("Contact sheet =", _contact_sheet_path)
print("Strict mode =", LAYOUT_AUDIT_STRICT)

if len(_failed_layout) and LAYOUT_AUDIT_STRICT:
    raise RuntimeError(
        "Final layout audit contains failures:\n"
        + _failed_layout[["stem", "issues"]].to_string(index=False)
    )
if not LAYOUT_AUDIT_STRICT:
    print(
        "COLLECTION PASS COMPLETE:", len(_failed_layout),
        "figure(s) require geometry fixes before the strict run."
    )
else:
    print("FINAL STRICT LAYOUT AUDIT: PASS (zero failures)")
